<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/New_stablez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================
# CROSS SPORT MODEL CORE
# ==============================

import pandas as pd
import numpy as np
import requests
import os
from datetime import datetime

print("✅ Core Libraries Loaded")

✅ Core Libraries Loaded


In [2]:
# ==============================
# LIVE ODDS DATA PULL
# ==============================

import requests
API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

SPORT = "basketball_ncaab"
URL = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals"
}

response = requests.get(URL, params=params)

if response.status_code != 200:
    raise Exception(f"API Error: {response.text}")

games_raw = response.json()

print("✅ Odds Loaded")
print("Games Retrieved:", len(games_raw))

✅ Odds Loaded
Games Retrieved: 28


In [ ]:
# =====================================
# CONVERT RAW ODDS → CLEAN GAME TABLE
# =====================================

games_df = []

for game in games_raw:
    home = game.get("home_team")
    away = game.get("away_team")

    # Extract markets
    home_ml = None
    away_ml = None

    for book in game.get("bookmakers", []):
        for market in book.get("markets", []):
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    if outcome["name"] == home:
                        home_ml = outcome["price"]
                    if outcome["name"] == away:
                        away_ml = outcome["price"]

    games_df.append({
        "home_team": home,
        "away_team": away,
        "home_ml": home_ml,
        "away_ml": away_ml
    })

games_df = pd.DataFrame(games_df)

print("✅ Clean Table Created")
print(games_df.head())

✅ Clean Table Created
                     home_team             away_team  home_ml  away_ml
0          Grambling St Tigers  Florida A&M Rattlers     9.00     1.06
1  Western Carolina Catamounts       Furman Paladins      NaN      NaN
2           Creighton Bluejays     Providence Friars     2.60     1.48
3   West Virginia Mountaineers           BYU Cougars     1.11     6.00
4      Sam Houston St Bearkats     Missouri St Bears      NaN      NaN


In [ ]:
# ==========================================
# CONVERT MONEYLINE → IMPLIED PROBABILITY
# ==========================================

def ml_to_prob(ml):
    if ml is None:
        return None

    ml = float(ml)

    if ml < 0:
        return abs(ml) / (abs(ml) + 100)
    else:
        return 100 / (ml + 100)


games_df["home_implied_prob"] = games_df["home_ml"].apply(ml_to_prob)
games_df["away_implied_prob"] = games_df["away_ml"].apply(ml_to_prob)

print("✅ Implied Probabilities Added")
print(games_df.head())

✅ Implied Probabilities Added
                     home_team             away_team  home_ml  away_ml  \
0          Grambling St Tigers  Florida A&M Rattlers     9.00     1.06   
1  Western Carolina Catamounts       Furman Paladins      NaN      NaN   
2           Creighton Bluejays     Providence Friars     2.60     1.48   
3   West Virginia Mountaineers           BYU Cougars     1.11     6.00   
4      Sam Houston St Bearkats     Missouri St Bears      NaN      NaN   

   home_implied_prob  away_implied_prob  
0           0.917431           0.989511  
1                NaN                NaN  
2           0.974659           0.985416  
3           0.989022           0.943396  
4                NaN                NaN  


In [ ]:
# ==========================================
# FIX: CONVERT DECIMAL ODDS → IMPLIED PROB
# ==========================================

games_df["home_implied_prob"] = 1 / games_df["home_ml"]
games_df["away_implied_prob"] = 1 / games_df["away_ml"]

print("✅ Corrected Implied Probabilities")
print(games_df[[
    "home_team",
    "away_team",
    "home_implied_prob",
    "away_implied_prob"
]].head())

✅ Corrected Implied Probabilities
                     home_team             away_team  home_implied_prob  \
0          Grambling St Tigers  Florida A&M Rattlers           0.111111   
1  Western Carolina Catamounts       Furman Paladins                NaN   
2           Creighton Bluejays     Providence Friars           0.384615   
3   West Virginia Mountaineers           BYU Cougars           0.900901   
4      Sam Houston St Bearkats     Missouri St Bears                NaN   

   away_implied_prob  
0           0.943396  
1                NaN  
2           0.675676  
3           0.166667  
4                NaN  


In [ ]:
print(type(locals()))
print([name for name in globals() if isinstance(globals()[name], __import__("pandas").DataFrame)])

<class 'dict'>
['inv', 'games_df']


In [ ]:
odds_df["home_implied_prob"] = 1 / odds_df["home_ml"]
odds_df["away_implied_prob"] = 1 / odds_df["away_ml"]

print(odds_df.head())

NameError: name 'odds_df' is not defined

In [ ]:
import os

print(os.environ.get("ODDS_API_KEY"))

In [ ]:
import os

os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

print("API Key Set")

In [ ]:
print(os.environ.get("ODDS_API_KEY"))

In [ ]:
import requests
import pandas as pd

# 🔑 PUT YOUR KEY HERE
API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

SPORT = "basketball_ncaab"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals"
}

response = requests.get(url, params=params)

if response.status_code != 200:
    raise Exception(f"API Error: {response.text}")

data = response.json()

print("✅ API Connected")
print("Games Returned:", len(data))

In [ ]:
rows = []

for game in data:
    home = game["home_team"]
    away = game["away_team"]

    bookmakers = game.get("bookmakers", [])

    if not bookmakers:
        continue

    book = bookmakers[0]
    markets = book["markets"]

    h2h = next((m for m in markets if m["key"] == "h2h"), None)

    if h2h:
        outcomes = h2h["outcomes"]

        home_price = next(o["price"] for o in outcomes if o["name"] == home)
        away_price = next(o["price"] for o in outcomes if o["name"] == away)

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_ml": home_price,
            "away_ml": away_price
        })

odds_df = pd.DataFrame(rows)

print("✅ Clean Table Created")
print(odds_df.head())

In [ ]:
# ===============================
# CONVERT MONEYLINE → IMPLIED PROB
# ===============================

def ml_to_prob(ml):
    if ml > 0:
        return 100 / (ml + 100)
    else:
        return -ml / (-ml + 100)

odds_df["home_implied_prob"] = odds_df["home_ml"].apply(ml_to_prob)
odds_df["away_implied_prob"] = odds_df["away_ml"].apply(ml_to_prob)

print("✅ Implied Probabilities Added")
print(odds_df.head())

In [ ]:
# ==========================================
# REMOVE VIG (Normalize Probabilities)
# ==========================================

odds_df["total_implied"] = (
    odds_df["home_implied_prob"] + odds_df["away_implied_prob"]
)

odds_df["home_true_prob"] = (
    odds_df["home_implied_prob"] / odds_df["total_implied"]
)

odds_df["away_true_prob"] = (
    odds_df["away_implied_prob"] / odds_df["total_implied"]
)

print("✅ Vig Removed — True Probabilities Created")
print(odds_df[["home_team","home_true_prob","away_true_prob"]].head())

In [ ]:
# ==================================================
# SIMPLE MODEL PROBABILITY (INITIAL MODEL BASELINE)
# ==================================================

# Example model assumption:
# Home advantage = +0.03 probability boost
# Based on historical home advantage in basketball

HOME_ADVANTAGE = 0.03

def model_probability(row):

    home_prob = row["home_true_prob"]
    away_prob = row["away_true_prob"]

    # Basic model:
    # If home team stronger in market → slight boost
    model_home = home_prob + HOME_ADVANTAGE
    model_away = away_prob - HOME_ADVANTAGE

    # Normalize
    total = model_home + model_away

    return model_home / total


odds_df["home_model_prob"] = odds_df.apply(model_probability, axis=1)

print("✅ Model Probability Column Added")
print(odds_df[["home_team","home_model_prob"]].head())

In [ ]:
# ==========================================
# CALCULATE BETTING EDGE
# ==========================================

odds_df["edge"] = odds_df["home_model_prob"] - odds_df["home_true_prob"]

# Sort strongest edges first
odds_df_sorted = odds_df.sort_values(by="edge", ascending=False)

print("✅ Edge Calculated")
print(odds_df_sorted[[
    "home_team",
    "home_true_prob",
    "home_model_prob",
    "edge"
]].head(10))

In [ ]:
# ==================================================
# STRENGTH-BASED MODEL PROBABILITY (REAL VERSION)
# ==================================================

# Assign team strength using market implied probability as baseline
# Then introduce regression toward 0.5 (mean reversion)

def strength_model(row):

    market_prob = row["home_true_prob"]

    # Mean reversion factor
    regression_weight = 0.4

    # Pull probability toward 0.5 slightly
    adjusted = market_prob * (1 - regression_weight) + 0.5 * regression_weight

    return adjusted


odds_df["home_model_prob"] = odds_df.apply(strength_model, axis=1)

print("✅ Strength Model Applied")
print(odds_df[["home_team","home_model_prob"]].head())

In [ ]:
# ==================================================
# REAL TEAM STRENGTH MODEL (DATA DRIVEN)
# ==================================================

# ⚠️ This is a placeholder until we integrate historical data
# It creates a synthetic strength metric based on market deviation

odds_df["team_strength_score"] = (
    (1 - odds_df["home_true_prob"]) * 0.6 +
    (odds_df["home_model_prob"] - 0.5) * 0.4
)

# Convert strength into probability via sigmoid
def sigmoid(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))

odds_df["final_model_prob"] = odds_df["team_strength_score"].apply(sigmoid)

print("✅ Real Strength Layer Added")
print(odds_df[[
    "home_team",
    "final_model_prob"
]].head())

In [ ]:
import numpy as np

In [ ]:
odds_df["team_strength_score"] = (
    (1 - odds_df["home_true_prob"]) * 0.6 +
    (odds_df["home_model_prob"] - 0.5) * 0.4
)

def sigmoid(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))

odds_df["final_model_prob"] = odds_df["team_strength_score"].apply(sigmoid)

print("✅ Real Strength Layer Added")
print(odds_df[["home_team","final_model_prob"]].head())

In [ ]:
# ==================================================
# TEMPORARY BETTER BASELINE MODEL
# ==================================================

# Use market probability as baseline
# Then introduce controlled uncertainty

noise = np.random.normal(0, 0.02, size=len(odds_df))

odds_df["final_model_prob"] = (
    odds_df["home_true_prob"] * 0.7
    + 0.5 * 0.3
    + noise
)

# Clamp between 0 and 1
odds_df["final_model_prob"] = odds_df["final_model_prob"].clip(0.01, 0.99)

print("✅ Improved Temporary Model Built")
print(odds_df[["home_team","final_model_prob"]].head())

In [ ]:
# ==========================================
# PULL HISTORICAL TEAM STATS (FREE DATA)
# ==========================================

import requests

STATS_URL = "https://stats.ncaa.org/teams"

try:
    response = requests.get(STATS_URL, timeout=10)
    print("Status Code:", response.status_code)
    print("Raw Length:", len(response.text))
except Exception as e:
    print("Error pulling historical data:", e)

In [ ]:
# ==========================================
# TEST HISTORICAL SCORES FROM ODDS API
# ==========================================

import requests

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/scores"

params = {
    "apiKey": API_KEY,
    "daysFrom": 3  # pulls last 3 days of results
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

if response.status_code != 200:
    print("Error:", response.text)
else:
    data = response.json()
    print("✅ Games Returned:", len(data))
    print(data[:2])  # preview first 2 games

In [ ]:
# ==========================================
# CONVERT SCORES → HISTORICAL DATAFRAME
# ==========================================

historical_rows = []

for game in data:

    if not game.get("completed"):
        continue

    home = game["home_team"]
    away = game["away_team"]

    scores = game.get("scores", [])

    home_score = None
    away_score = None

    for s in scores:
        if s["name"] == home:
            home_score = int(s["score"])
        if s["name"] == away:
            away_score = int(s["score"])

    if home_score is None or away_score is None:
        continue

    historical_rows.append({
        "home_team": home,
        "away_team": away,
        "home_score": home_score,
        "away_score": away_score,
        "point_diff": home_score - away_score
    })

historical_df = pd.DataFrame(historical_rows)

print("✅ Historical Dataset Built")
print(historical_df.head())
print("Total Games Stored:", len(historical_df))

In [ ]:
# ==========================================
# BUILD TEAM PERFORMANCE METRICS
# ==========================================

# Create combined dataset for both home & away perspectives

home_stats = historical_df[["home_team","home_score","away_score"]].copy()
home_stats.rename(columns={
    "home_team":"team",
    "home_score":"points_scored",
    "away_score":"points_allowed"
}, inplace=True)

away_stats = historical_df[["away_team","away_score","home_score"]].copy()
away_stats.rename(columns={
    "away_team":"team",
    "away_score":"points_scored",
    "home_score":"points_allowed"
}, inplace=True)

all_stats = pd.concat([home_stats, away_stats], ignore_index=True)

team_metrics = all_stats.groupby("team").agg({
    "points_scored":"mean",
    "points_allowed":"mean"
}).rename(columns={
    "points_scored":"off_rating",
    "points_allowed":"def_rating"
})

team_metrics["net_rating"] = team_metrics["off_rating"] - team_metrics["def_rating"]

print("✅ Team Metrics Built")
print(team_metrics.head())

In [ ]:
# ==========================================
# MERGE TEAM STRENGTH INTO LIVE GAMES
# ==========================================

# Reset index so team name becomes a column
team_metrics_reset = team_metrics.reset_index()

# Merge home team stats
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="home_team",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"home_off_rating",
    "def_rating":"home_def_rating",
    "net_rating":"home_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

# Merge away team stats
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="away_team",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"away_off_rating",
    "def_rating":"away_def_rating",
    "net_rating":"away_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

print("✅ Team Strength Merged Into Live Games")
print(odds_df.head())

In [ ]:
# ==========================================
# NORMALIZE TEAM NAMES (CRITICAL FIX)
# ==========================================

def normalize_name(name):
    if pd.isna(name):
        return name

    name = name.lower()
    name = name.replace("state", "st")
    name = name.replace("university", "")
    name = name.replace("college", "")
    name = name.replace("  ", " ")
    return name.strip()

# Apply normalization to historical data
historical_df["home_team_norm"] = historical_df["home_team"].apply(normalize_name)
historical_df["away_team_norm"] = historical_df["away_team"].apply(normalize_name)

# Rebuild team metrics using normalized names
home_stats = historical_df[["home_team_norm","home_score","away_score"]].rename(columns={"home_team_norm":"team"})
away_stats = historical_df[["away_team_norm","away_score","home_score"]].rename(columns={"away_team_norm":"team"})

all_stats = pd.concat([home_stats, away_stats], ignore_index=True)

team_metrics = all_stats.groupby("team").agg({
    "home_score":"mean",
    "away_score":"mean"
}).rename(columns={
    "home_score":"off_rating",
    "away_score":"def_rating"
})

team_metrics["net_rating"] = team_metrics["off_rating"] - team_metrics["def_rating"]

print("✅ Normalized Team Metrics Rebuilt")
print(team_metrics.head())

In [ ]:
# ==========================================
# NORMALIZE LIVE TEAM NAMES
# ==========================================

odds_df["home_team_norm"] = odds_df["home_team"].apply(normalize_name)
odds_df["away_team_norm"] = odds_df["away_team"].apply(normalize_name)

team_metrics_reset = team_metrics.reset_index()

In [ ]:
# ==========================================
# MERGE USING NORMALIZED TEAM NAMES
# ==========================================

# Merge home stats
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="home_team_norm",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"home_off_rating",
    "def_rating":"home_def_rating",
    "net_rating":"home_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

# Merge away stats
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="away_team_norm",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"away_off_rating",
    "def_rating":"away_def_rating",
    "net_rating":"away_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

print("✅ Clean Merge Completed")
print(odds_df.head())

In [ ]:
!pip install thefuzz

In [ ]:
from thefuzz import process

# ==========================================
# FUZZY MATCH LIVE TEAMS TO HISTORICAL TEAMS
# ==========================================

team_list = team_metrics.index.tolist()

def fuzzy_match_team(name):
    if pd.isna(name):
        return None

    match, score = process.extractOne(name, team_list)

    # Only accept strong matches
    if score > 75:
        return match
    else:
        return None

odds_df["home_team_matched"] = odds_df["home_team_norm"].apply(fuzzy_match_team)
odds_df["away_team_matched"] = odds_df["away_team_norm"].apply(fuzzy_match_team)

print("✅ Fuzzy Matching Complete")
print(odds_df[["home_team","home_team_matched"]].head())

In [ ]:
team_metrics_reset = team_metrics.reset_index()

# Merge home
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="home_team_matched",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"home_off_rating",
    "def_rating":"home_def_rating",
    "net_rating":"home_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

# Merge away
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="away_team_matched",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"away_off_rating",
    "def_rating":"away_def_rating",
    "net_rating":"away_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

print("✅ Fuzzy Merge Completed")
print(odds_df.head())

In [ ]:
# ==========================================================
# FULL TEAM MATCH FIX + RE-MERGE (PRODUCTION VERSION)
# ==========================================================

from thefuzz import process

# ---------- 1. NORMALIZE AGAIN (SAFETY) ----------
def normalize_name(name):
    if pd.isna(name):
        return name
    name = name.lower()
    name = name.replace("state", "st")
    name = name.replace("university", "")
    name = name.replace("college", "")
    name = name.replace("  ", " ")
    return name.strip()

odds_df["home_team_norm"] = odds_df["home_team"].apply(normalize_name)
odds_df["away_team_norm"] = odds_df["away_team"].apply(normalize_name)

# ---------- 2. FUZZY MATCH ----------
team_list = team_metrics.index.tolist()

def fuzzy_match_team(name):
    if pd.isna(name):
        return None
    match, score = process.extractOne(name, team_list)
    if score > 75:
        return match
    return None

odds_df["home_team_matched"] = odds_df["home_team_norm"].apply(fuzzy_match_team)
odds_df["away_team_matched"] = odds_df["away_team_norm"].apply(fuzzy_match_team)

# ---------- 3. MANUAL OVERRIDE DICTIONARY ----------
manual_match = {
    # Add fixes here if needed later
    "binghamton bearcats": "binghamton bearcats",
    "albany great danes": "albany great danes",
    "rhode island rams": "rhode island rams",
    "new hampshire wildcats": "new hampshire wildcats"
}

odds_df["home_team_matched"] = odds_df.apply(
    lambda row: manual_match.get(row["home_team_norm"], row["home_team_matched"]),
    axis=1
)

odds_df["away_team_matched"] = odds_df.apply(
    lambda row: manual_match.get(row["away_team_norm"], row["away_team_matched"]),
    axis=1
)

# ---------- 4. DROP OLD RATING COLUMNS ----------
rating_cols = [
    "home_off_rating","home_def_rating","home_net_rating",
    "away_off_rating","away_def_rating","away_net_rating"
]

for col in rating_cols:
    if col in odds_df.columns:
        odds_df.drop(columns=[col], inplace=True)

# ---------- 5. RE-MERGE TEAM METRICS ----------
team_metrics_reset = team_metrics.reset_index()

# Merge home
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="home_team_matched",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"home_off_rating",
    "def_rating":"home_def_rating",
    "net_rating":"home_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

# Merge away
odds_df = odds_df.merge(
    team_metrics_reset,
    left_on="away_team_matched",
    right_on="team",
    how="left"
).rename(columns={
    "off_rating":"away_off_rating",
    "def_rating":"away_def_rating",
    "net_rating":"away_net_rating"
})

odds_df.drop(columns=["team"], inplace=True)

# ---------- 6. FINAL CLEAN CHECK ----------
print("✅ FULL MATCH + RE-MERGE COMPLETE")
print(odds_df.head())

print("\nNaN Counts in Rating Columns:")
print(odds_df[rating_cols].isna().sum())

In [ ]:
# ==================================================
# FILL MISSING RATINGS WITH LEAGUE AVERAGES
# ==================================================

# Compute league averages from available ratings
league_home_off_avg = odds_df["home_off_rating"].mean()
league_home_def_avg = odds_df["home_def_rating"].mean()
league_home_net_avg = odds_df["home_net_rating"].mean()

league_away_off_avg = odds_df["away_off_rating"].mean()
league_away_def_avg = odds_df["away_def_rating"].mean()
league_away_net_avg = odds_df["away_net_rating"].mean()

# Fill NaNs
odds_df["home_off_rating"].fillna(league_home_off_avg, inplace=True)
odds_df["home_def_rating"].fillna(league_home_def_avg, inplace=True)
odds_df["home_net_rating"].fillna(league_home_net_avg, inplace=True)

odds_df["away_off_rating"].fillna(league_away_off_avg, inplace=True)
odds_df["away_def_rating"].fillna(league_away_def_avg, inplace=True)
odds_df["away_net_rating"].fillna(league_away_net_avg, inplace=True)

print("✅ Missing Ratings Filled With League Averages")
print("Remaining NaNs:")
print(odds_df[[
    "home_off_rating","away_off_rating"
]].isna().sum())

In [ ]:
# FUTURE-PROOF VERSION
odds_df["home_off_rating"] = odds_df["home_off_rating"].fillna(league_home_off_avg)
odds_df["home_def_rating"] = odds_df["home_def_rating"].fillna(league_home_def_avg)
odds_df["home_net_rating"] = odds_df["home_net_rating"].fillna(league_home_net_avg)

odds_df["away_off_rating"] = odds_df["away_off_rating"].fillna(league_away_off_avg)
odds_df["away_def_rating"] = odds_df["away_def_rating"].fillna(league_away_def_avg)
odds_df["away_net_rating"] = odds_df["away_net_rating"].fillna(league_away_net_avg)

In [ ]:
import numpy as np

# League average points per game (replace with actual league avg if you have it)
league_avg_points = 75

# Calculate expected points for each team
odds_df["proj_home_pts"] = league_avg_points + \
    (odds_df["home_off_rating"] - league_avg_points) - \
    (odds_df["away_def_rating"] - league_avg_points)

odds_df["proj_away_pts"] = league_avg_points + \
    (odds_df["away_off_rating"] - league_avg_points) - \
    (odds_df["home_def_rating"] - league_avg_points)

# Expected margin
odds_df["proj_margin"] = odds_df["proj_home_pts"] - odds_df["proj_away_pts"]

# Convert margin into win probability via logistic (simple scaling factor)
def margin_to_prob(margin, scale=10):
    """Convert expected margin to home win probability"""
    return 1 / (1 + np.exp(-margin / scale))

odds_df["home_proj_prob"] = odds_df["proj_margin"].apply(lambda x: margin_to_prob(x))

# Away probability
odds_df["away_proj_prob"] = 1 - odds_df["home_proj_prob"]

print("✅ Projected Points and Win Probabilities Added")
odds_df[[
    "home_team", "away_team", "proj_home_pts", "proj_away_pts",
    "proj_margin", "home_proj_prob", "away_proj_prob"
]].head()

In [ ]:
# ==================================================
# CALCULATE MARKET EDGE
# ==================================================

# Use home win market probability
odds_df["market_home_prob"] = odds_df["home_implied_prob"] / (
    odds_df["home_implied_prob"] + odds_df["away_implied_prob"]
)

# Edge calculation
odds_df["home_edge"] = odds_df["home_proj_prob"] - odds_df["market_home_prob"]

# Sort by strongest positive edge
edges = odds_df.sort_values("home_edge", ascending=False)

print("✅ Edge Calculated")
print(edges[[
    "home_team",
    "away_team",
    "home_proj_prob",
    "market_home_prob",
    "home_edge"
]].head(15))

In [ ]:
# ==================================================
# FILTER FOR ACTIONABLE +EV BETS
# ==================================================

MIN_EDGE = 0.05  # 5% edge threshold

good_bets = edges[edges["home_edge"] > MIN_EDGE].copy()

# Kelly Fraction (simple proportional model)
# Kelly = Edge / Odds Variance Approx
good_bets["kelly_fraction"] = good_bets["home_edge"] / good_bets["market_home_prob"]

# Cap Kelly at 0.5 (risk control)
good_bets["kelly_fraction"] = good_bets["kelly_fraction"].clip(upper=0.5)

print("✅ +EV Bets Identified")
print(good_bets[[
    "home_team",
    "away_team",
    "home_proj_prob",
    "market_home_prob",
    "home_edge",
    "kelly_fraction"
]])

In [ ]:
# ==================================================
# MONTE CARLO SIMULATION FOR PROBABILITY REFINEMENT
# ==================================================

import numpy as np

SIMS = 5000

def simulate_win_prob(home_mean, away_mean):
    home_wins = 0
    for _ in range(SIMS):
        home_score = np.random.normal(home_mean, 10)
        away_score = np.random.normal(away_mean, 10)

        if home_score > away_score:
            home_wins += 1

    return home_wins / SIMS


# Apply simulation to projection table
sim_probs = []

for _, row in projections.iterrows():
    prob = simulate_win_prob(row["proj_home_pts"], row["proj_away_pts"])
    sim_probs.append(prob)

projections["simulated_win_prob"] = sim_probs

print("✅ Monte Carlo Layer Added")
print(projections.head())

In [ ]:
print([name for name in globals() if "proj" in name.lower()])

In [ ]:
import numpy as np

# Make a copy of your current live odds + metrics table
projections = odds_df.copy()

# Projected points based on simple adjusted net rating
# We'll use home_net_rating and away_net_rating to estimate points
# Scale factor is arbitrary for demonstration; can be tuned
scale = 0.7  # adjust for typical game scores

projections["proj_home_pts"] = 75 + projections["home_net_rating"] * scale
projections["proj_away_pts"] = 75 + projections["away_net_rating"] * scale

# Projected margin
projections["proj_margin"] = projections["proj_home_pts"] - projections["proj_away_pts"]

# Convert projected points into simple win probabilities
def win_prob(home_pts, away_pts):
    # sigmoid function to map margin to probability between 0 and 1
    return 1 / (1 + np.exp(-0.1 * (home_pts - away_pts)))

projections["home_proj_prob"] = projections.apply(
    lambda row: win_prob(row["proj_home_pts"], row["proj_away_pts"]), axis=1
)
projections["away_proj_prob"] = 1 - projections["home_proj_prob"]

# Verify
print("✅ Projections rebuilt")
print(projections[["home_team","away_team","proj_home_pts","proj_away_pts","proj_margin","home_proj_prob","away_proj_prob"]].head())

In [ ]:
# Calculate edges vs market probabilities
projections["home_edge"] = projections["home_proj_prob"] - projections["home_true_prob"]

# Kelly fraction: simple half-Kelly sizing
# formula: kelly = edge / odds_converted - 1, but simplified here as a fraction of your bankroll
projections["kelly_fraction"] = 0.5 * (projections["home_edge"] / (projections["home_ml"] - 1))

# Cap Kelly fraction at 0.5 (half Kelly max)
projections["kelly_fraction"] = projections["kelly_fraction"].clip(upper=0.5)

# Filter +EV bets (home edge > 0)
plus_ev = projections[projections["home_edge"] > 0].sort_values(by="home_edge", ascending=False)

# Show top +EV bets
print("✅ +EV Bets Identified")
print(plus_ev[["home_team","away_team","home_proj_prob","home_true_prob","home_edge","kelly_fraction"]].head(15))

In [ ]:
# ==============================
# TRUE BETTING ENGINE LAYER
# ==============================

# Expected Value Calculation
# EV = (win_prob * (odds - 1)) - (1 - win_prob)

projections["ev_percent"] = (
    projections["home_proj_prob"] * (projections["home_ml"] - 1)
    - (1 - projections["home_proj_prob"])
)

# Sort by strongest EV
ranked_bets = projections.sort_values(by="ev_percent", ascending=False)

print("✅ TOP RANKED BETS BY EXPECTED VALUE")
print(
    ranked_bets[
        [
            "home_team",
            "away_team",
            "home_proj_prob",
            "home_true_prob",
            "home_edge",
            "ev_percent",
            "kelly_fraction",
        ]
    ].head(20)
)

In [ ]:
# ==============================
# PROFESSIONAL EV ENGINE
# ==============================

def moneyline_to_decimal(ml):
    """Convert American moneyline to decimal odds"""
    if ml > 0:
        return 1 + (ml / 100)
    else:
        return 1 + (100 / abs(ml))


# Convert implied probability into fair odds
projections["decimal_odds"] = projections["home_ml"].apply(moneyline_to_decimal)

# Correct Expected Value Formula
projections["ev_percent"] = (
    projections["home_proj_prob"] * (projections["decimal_odds"] - 1)
    - (1 - projections["home_proj_prob"])
) * 100


# Rank properly
ranked_bets = projections.sort_values(by="ev_percent", ascending=False)

print("🔥 UPDATED +EV RANKING")
print(
    ranked_bets[
        [
            "home_team",
            "away_team",
            "home_proj_prob",
            "home_true_prob",
            "home_edge",
            "ev_percent",
            "kelly_fraction",
        ]
    ].head(20)
)

In [ ]:
# ============================================
# PROFESSIONAL MARKET NORMALIZATION (VIG REMOVAL)
# ============================================

def remove_vig(home_prob, away_prob):
    total = home_prob + away_prob
    return home_prob / total, away_prob / total


# Normalize probabilities
projections["home_market_adj"], projections["away_market_adj"] = zip(
    *projections.apply(
        lambda row: remove_vig(
            row["home_implied_prob"],
            row["away_implied_prob"],
        ),
        axis=1,
    )
)

print("✅ Market Probabilities De-Vigged")

In [ ]:
# ============================================
# TRUE EDGE CALCULATION
# ============================================

projections["true_edge"] = (
    projections["home_proj_prob"] - projections["home_market_adj"]
)

print("🔥 TRUE EDGE REBUILT")
print(projections.sort_values("true_edge", ascending=False).head(20))

In [ ]:
# ==========================================
# PROBABILITY CALIBRATION LAYER
# ==========================================

import numpy as np
from sklearn.isotonic import IsotonicRegression

# Make sure your dataframe is called: odds_df
# and it contains: home_proj_prob + home_market_prob

# ---- Safety Check ----
if "home_proj_prob" not in odds_df.columns:
    raise Exception("home_proj_prob column missing")

# ---- Isotonic Calibration ----
iso = IsotonicRegression(out_of_bounds="clip")

raw_probs = odds_df["home_proj_prob"].values

# Fit isotonic on itself as proxy calibration
calibrated_probs = iso.fit_transform(raw_probs, raw_probs)

odds_df["home_calibrated_prob"] = calibrated_probs

# ---- Recalculate True Edge Using Calibrated Prob ----
odds_df["true_edge"] = (
    odds_df["home_calibrated_prob"] - odds_df["home_market_prob"]
)

# ---- Recalculate EV % Properly ----
odds_df["ev_percent"] = (
    odds_df["true_edge"] * 100
)

print("✅ Calibration Layer Applied")
print(odds_df[[ "home_team", "home_proj_prob", "home_calibrated_prob", "true_edge"]].head())

In [ ]:
# ==========================================
# CALIBRATION + EDGE REBUILD (AUTO DETECT)
# ==========================================

import numpy as np
from sklearn.isotonic import IsotonicRegression

# Check required column
if "home_proj_prob" not in odds_df.columns:
    raise Exception("home_proj_prob missing from dataframe")

# ---- Find Market Probability Column Automatically ----
market_prob_col = None

for col in odds_df.columns:
    if "market" in col.lower() and "home" in col.lower():
        market_prob_col = col
        break

if market_prob_col is None:
    raise Exception("No home_market_prob column found. Check column names.")

print("Using market column:", market_prob_col)

# ---- Isotonic Calibration ----
iso = IsotonicRegression(out_of_bounds="clip")

raw_probs = odds_df["home_proj_prob"].values
calibrated_probs = iso.fit_transform(raw_probs, raw_probs)

odds_df["home_calibrated_prob"] = calibrated_probs

# ---- True Edge ----
odds_df["true_edge"] = (
    odds_df["home_calibrated_prob"] - odds_df[market_prob_col]
)

# ---- EV ----
odds_df["ev_percent"] = odds_df["true_edge"] * 100

print("✅ Calibration + Edge Rebuilt")
print(odds_df[["home_team","home_calibrated_prob","true_edge"]].head())

In [ ]:
# ===== DEFINE BET TYPE =====
def get_bet_type(row):
    if row["home_calibrated_prob"] > row["market_home_prob"]:
        return "Home ML"
    else:
        return "Away ML"

odds_df["bet_type"] = odds_df.apply(get_bet_type, axis=1)

print("✅ Bet Type Column Added")
print(odds_df[["home_team","away_team","bet_type"]].head())

In [ ]:
# ==========================================
# KELLY FRACTION CALCULATION
# ==========================================
# Standard fractional Kelly for single bet
# kelly_fraction = edge / (odds - 1)

# Detect odds column automatically
odds_col = None
for col in odds_df.columns:
    if "ml" in col.lower() and "home" in col.lower():
        odds_col = col
        break

if odds_col is None:
    raise Exception("No home moneyline column found. Check column names.")

print("Using odds column:", odds_col)

# ---- Calculate Kelly ----
odds_df["kelly_fraction"] = odds_df["true_edge"] / (odds_df[odds_col] - 1)

# Cap Kelly between 0 and 0.5 (half-Kelly max) for safety
odds_df["kelly_fraction"] = odds_df["kelly_fraction"].clip(lower=0, upper=0.5)

# ---- Display top +EV bets ----
top_ev = odds_df.sort_values("true_edge", ascending=False).head(10)
print(top_ev[["home_team","away_team","home_calibrated_prob","true_edge","kelly_fraction"]])

In [ ]:
# ==========================================
# DISCORD-READY +EV BET SNIPPET
# ==========================================
top_ev_discord = odds_df.sort_values("true_edge", ascending=False).head(10)

discord_snippet = "\n".join([
    f"💰 **{row.home_team} vs {row.away_team}**\n"
    f" - Edge: {row.true_edge:.3f}\n"
    f" - Kelly: {row.kelly_fraction:.3f}u\n"
    f" - Calibrated Prob: {row.home_calibrated_prob:.3f}\n"
    f" - Odds: {row.home_ml}\n"
    for _, row in top_ev_discord.iterrows()
])

print(discord_snippet)

In [ ]:
# ===== IMPROVED BET TYPE ENGINE =====

def get_bet_type(row):

    edge_threshold = 0.02  # minimum edge required

    if row["true_edge"] < edge_threshold:
        return "No Bet"

    # If probability is higher than market → we bet that side
    if row["home_calibrated_prob"] > row["market_home_prob"]:
        return "Home ML"
    else:
        return "Away ML"


odds_df["bet_type"] = odds_df.apply(get_bet_type, axis=1)

# Remove No Bets
odds_df = odds_df[odds_df["bet_type"] != "No Bet"]

print("✅ Filtered + Strong Bet Types Only")
print(odds_df[["home_team","away_team","bet_type","true_edge"]].head())

In [ ]:
# ===== RANK + DISCORD EXPORT =====

# Sort by strongest edge first
strong_bets = odds_df.sort_values(by="true_edge", ascending=False)

# Format for Discord
discord_snippet = ""
for _, row in strong_bets.iterrows():
    discord_snippet += f"💰 **{row['home_team']} vs {row['away_team']}**\n"
    discord_snippet += f" - Bet Type: {row['bet_type']}\n"
    discord_snippet += f" - Edge: {row['true_edge']:.3f}\n"
    discord_snippet += f" - Calibrated Prob: {row['home_calibrated_prob']:.3f}\n"
    if row['bet_type'] == "Home ML":
        odds = row["home_ml"]
    else:
        odds = row["away_ml"]
    discord_snippet += f" - Odds: {odds:.2f}\n\n"

# Print Discord-ready snippet
print("✅ Discord-ready bet snippet:\n")
print(discord_snippet)

In [ ]:
# ===== DISCORD SNIPPET WITH AMERICAN ODDS =====

def decimal_to_american(decimal_odds):
    if decimal_odds >= 2:
        return f"+{int((decimal_odds-1)*100)}"
    else:
        return f"-{int(100/(decimal_odds-1))}"

# Assume we have a 'book' column with the sportsbook name
discord_snippet = ""
for _, row in strong_bets.iterrows():
    # Determine American odds for the bet type
    if row['bet_type'] == "Home ML":
        odds_american = decimal_to_american(row["home_ml"])
    else:
        odds_american = decimal_to_american(row["away_ml"])

    # If 'book' column exists, include it
    book = row['book'] if 'book' in row.columns else "Unknown Book"

    discord_snippet += f"💰 **{row['home_team']} vs {row['away_team']}**\n"
    discord_snippet += f" - Bet Type: {row['bet_type']}\n"
    discord_snippet += f" - Edge: {row['true_edge']:.3f}\n"
    discord_snippet += f" - Calibrated Prob: {row['home_calibrated_prob']:.3f}\n"
    discord_snippet += f" - Odds: {odds_american} ({book})\n\n"

print("✅ Discord-ready snippet with American odds:\n")
print(discord_snippet)

In [ ]:
# ===== DISCORD SNIPPET WITH AMERICAN ODDS + BOOK =====

def decimal_to_american(decimal_odds):
    if decimal_odds >= 2:
        return f"+{int((decimal_odds - 1) * 100)}"
    else:
        return f"-{int(100 / (decimal_odds - 1))}"

discord_snippet = ""

for _, row in strong_bets.iterrows():

    # Determine odds source
    if row["bet_type"] == "Home ML":
        decimal_odds = row["home_ml"]
    else:
        decimal_odds = row["away_ml"]

    odds_american = decimal_to_american(decimal_odds)

    # Safe book column handling
    if "book" in strong_bets.columns:
        book = row["book"]
    else:
        book = "Unknown Book"

    discord_snippet += f"💰 **{row['home_team']} vs {row['away_team']}**\n"
    discord_snippet += f" - Bet Type: {row['bet_type']}\n"
    discord_snippet += f" - Edge: {row['true_edge']:.3f}\n"
    discord_snippet += f" - Calibrated Prob: {row['home_calibrated_prob']:.3f}\n"
    discord_snippet += f" - Odds: {odds_american} ({book})\n\n"

print(discord_snippet)

In [ ]:
threshold = 0.03  # minimum meaningful edge

if edge > threshold:
    bet_type = "Home ML"

elif edge < -threshold:
    bet_type = "Away ML"

else:
    bet_type = "No Bet"

In [ ]:
threshold = 0.03  # minimum meaningful edge

def assign_bet_type(row):
    if row['true_edge'] > threshold:
        return "Home ML"
    elif row['true_edge'] < -threshold:
        return "Away ML"
    else:
        return "No Bet"

# Apply to your dataframe
odds_df['bet_type'] = odds_df.apply(assign_bet_type, axis=1)

In [ ]:
threshold = 0.03

conditions = [
    odds_df["true_edge"] > threshold,
    odds_df["true_edge"] < -threshold
]

choices = ["Home ML", "Away ML"]

odds_df["bet_type"] = np.select(conditions, choices, default="No Bet")

In [ ]:
odds_df = odds_df.copy()

In [ ]:
threshold = 0.03

conditions = [
    odds_df["true_edge"] > threshold,
    odds_df["true_edge"] < -threshold
]

choices = ["Home ML", "Away ML"]

odds_df["bet_type"] = np.select(conditions, choices, default="No Bet")

In [ ]:
df = df.copy()

In [ ]:
# Make a safe copy to avoid SettingWithCopyWarning
odds_df = odds_df.copy()

# Set the threshold for meaningful edge
threshold = 0.03

# Assign bet types based on true_edge
conditions = [
    odds_df["true_edge"] > threshold,   # positive edge → Home ML
    odds_df["true_edge"] < -threshold   # negative edge → Away ML
]
choices = ["Home ML", "Away ML"]

# Apply the conditions
odds_df["bet_type"] = np.select(conditions, choices, default="No Bet")

# Check result
odds_df[["home_team", "away_team", "true_edge", "bet_type"]].head(10)

In [ ]:
# Make safe copy
odds_df = odds_df.copy()

# ===== Convert Decimal → American Odds =====
def decimal_to_american(decimal_odds):
    if decimal_odds >= 2:
        return f"+{int((decimal_odds - 1) * 100)}"
    else:
        return f"-{int(100 / (decimal_odds - 1))}"

# Choose which odds column to use
def get_odds(row):
    if row["bet_type"] == "Home ML":
        return row["home_ml"]
    elif row["bet_type"] == "Away ML":
        return row["away_ml"]
    else:
        return None

# Assign selected decimal odds
odds_df["selected_decimal_odds"] = odds_df.apply(get_odds, axis=1)

# Convert to American
odds_df["american_odds"] = odds_df["selected_decimal_odds"].apply(
    lambda x: decimal_to_american(x) if x is not None else None
)

# View results
print(odds_df[[
    "home_team",
    "away_team",
    "bet_type",
    "true_edge",
    "american_odds"
]])

In [ ]:
# Make safe copy
odds_df = odds_df.copy()

# ===== Decimal → American Conversion (Safe Version) =====
def decimal_to_american(decimal_odds):
    if decimal_odds is None or pd.isna(decimal_odds):
        return None

    if decimal_odds >= 2:
        return f"+{int((decimal_odds - 1) * 100)}"
    else:
        return f"-{int(100 / (decimal_odds - 1))}"

# Select correct odds
def get_odds(row):
    if row["bet_type"] == "Home ML":
        return row["home_ml"]
    elif row["bet_type"] == "Away ML":
        return row["away_ml"]
    else:
        return None

# Apply selection
odds_df["selected_decimal_odds"] = odds_df.apply(get_odds, axis=1)

# Convert safely
odds_df["american_odds"] = odds_df["selected_decimal_odds"].apply(decimal_to_american)

# View clean result
print(odds_df[[
    "home_team",
    "away_team",
    "bet_type",
    "true_edge",
    "american_odds"
]])

In [ ]:
# ===== FILTER TO CLEAN +EV BETS =====

# Remove No Bets
clean_bets = odds_df[odds_df["bet_type"] != "No Bet"].copy()

# Remove extreme juice traps (avoid -1000 type bets)
clean_bets = clean_bets[
    (clean_bets["true_edge"] >= 0.05) &  # minimum real edge
    (~clean_bets["american_odds"].astype(str).str.contains("None"))
]

# Sort strongest edge first
clean_bets = clean_bets.sort_values(by="true_edge", ascending=False)

# Show final betting card
print(clean_bets[[
    "home_team",
    "away_team",
    "bet_type",
    "true_edge",
    "american_odds"
]])

In [ ]:
# ===== PROFESSIONAL BET FILTER =====

clean_bets = odds_df[odds_df["bet_type"] != "No Bet"].copy()

# Minimum edge requirement (increase to remove noise)
min_edge = 0.08

# Remove extreme juice (-800 or worse)
def safe_juice_filter(odds):
    if odds is None:
        return False
    if odds.startswith("-"):
        return int(odds) > -800  # remove -800 and worse
    return True

clean_bets = clean_bets[
    (clean_bets["true_edge"] >= min_edge) &
    (clean_bets["american_odds"].apply(safe_juice_filter))
]

# Sort strongest plays first
clean_bets = clean_bets.sort_values(by="true_edge", ascending=False)

print(clean_bets[[
    "home_team",
    "away_team",
    "bet_type",
    "true_edge",
    "american_odds"
]])

In [ ]:
# ===== AUTO UNIT SIZING BASED ON EDGE =====

def assign_units(edge):
    if edge >= 0.35:
        return 1.0
    elif edge >= 0.20:
        return 0.5
    elif edge >= 0.10:
        return 0.25
    else:
        return 0

clean_bets["units"] = clean_bets["true_edge"].apply(assign_units)

# Remove plays with 0 units
clean_bets = clean_bets[clean_bets["units"] > 0]

# Sort by strongest edge
clean_bets = clean_bets.sort_values(by="true_edge", ascending=False)

print(clean_bets[[
    "home_team",
    "away_team",
    "bet_type",
    "true_edge",
    "american_odds",
    "units"
]])

In [ ]:
print(odds_df[["true_edge"]].describe())

In [ ]:
# Make safe copy
odds_df = odds_df.copy()

threshold = 0.03

# ---- Compute Away Edge Properly ----
odds_df["away_edge"] = odds_df["away_proj_prob"] - odds_df["away_implied_prob"]

# ---- Classify Using BOTH Edges ----
def classify(row):
    if row["true_edge"] > threshold and row["true_edge"] >= row["away_edge"]:
        return "Home ML"
    elif row["away_edge"] > threshold and row["away_edge"] > row["true_edge"]:
        return "Away ML"
    else:
        return "No Bet"

odds_df["bet_type"] = odds_df.apply(classify, axis=1)

# Show results
print(odds_df[["home_team","away_team","true_edge","away_edge","bet_type"]])

In [ ]:
# Make safe copy
odds_df = odds_df.copy()

# ---- Normalize Market Prob (Remove Juice Properly) ----
odds_df["market_total_prob"] = (
    odds_df["home_implied_prob"] + odds_df["away_implied_prob"]
)

odds_df["home_market_norm"] = odds_df["home_implied_prob"] / odds_df["market_total_prob"]
odds_df["away_market_norm"] = odds_df["away_implied_prob"] / odds_df["market_total_prob"]

# ---- Compute True Edges Properly ----
odds_df["home_edge"] = odds_df["home_proj_prob"] - odds_df["home_market_norm"]
odds_df["away_edge"] = odds_df["away_proj_prob"] - odds_df["away_market_norm"]

# ---- Classify ----
threshold = 0.03

def classify(row):
    if row["home_edge"] > threshold and row["home_edge"] >= row["away_edge"]:
        return "Home ML"
    elif row["away_edge"] > threshold and row["away_edge"] > row["home_edge"]:
        return "Away ML"
    else:
        return "No Bet"

odds_df["bet_type"] = odds_df.apply(classify, axis=1)

print(odds_df[[
    "home_team",
    "away_team",
    "home_edge",
    "away_edge",
    "bet_type"
]])

In [ ]:
print(odds_df[["home_proj_prob","away_proj_prob"]].head())

In [ ]:
import numpy as np

# ===== SIMULATION-BASED WIN PROBABILITY ENGINE =====

def simulate_win_prob(home_pts, away_pts, sims=10000, std_dev=10):
    """
    Monte Carlo simulation to estimate win probability.
    Uses scoring distributions instead of forced probabilities.
    """
    home_wins = 0

    for _ in range(sims):
        home_sim = np.random.normal(home_pts, std_dev)
        away_sim = np.random.normal(away_pts, std_dev)

        if home_sim > away_sim:
            home_wins += 1

    return home_wins / sims


# ===== APPLY TO YOUR PROJECTED POINTS =====

odds_df = odds_df.copy()

# Simulate probabilities from projected points
odds_df["home_sim_prob"] = odds_df.apply(
    lambda row: simulate_win_prob(
        row["proj_home_pts"],
        row["proj_away_pts"]
    ),
    axis=1
)

odds_df["away_sim_prob"] = 1 - odds_df["home_sim_prob"]

print(odds_df[[
    "home_team",
    "away_team",
    "proj_home_pts",
    "proj_away_pts",
    "home_sim_prob",
    "away_sim_prob"
]])

In [ ]:
# ===== NORMALIZE MARKET PROB (REMOVE JUICE) =====

odds_df["market_total_prob"] = (
    odds_df["home_implied_prob"] + odds_df["away_implied_prob"]
)

odds_df["home_market_norm"] = (
    odds_df["home_implied_prob"] / odds_df["market_total_prob"]
)

odds_df["away_market_norm"] = (
    odds_df["away_implied_prob"] / odds_df["market_total_prob"]
)

# ===== CALCULATE REAL EDGE =====

odds_df["home_edge"] = odds_df["home_sim_prob"] - odds_df["home_market_norm"]
odds_df["away_edge"] = odds_df["away_sim_prob"] - odds_df["away_market_norm"]

# ===== CLASSIFY BETS =====

threshold = 0.03

def classify(row):
    if row["home_edge"] > threshold and row["home_edge"] >= row["away_edge"]:
        return "Home ML"
    elif row["away_edge"] > threshold and row["away_edge"] > row["home_edge"]:
        return "Away ML"
    else:
        return "No Bet"

odds_df["bet_type"] = odds_df.apply(classify, axis=1)

print(odds_df[[
    "home_team",
    "away_team",
    "home_edge",
    "away_edge",
    "bet_type"
]])

In [ ]:
# ===== RE-CALCULATE EDGES PROPERLY =====

threshold = 0.03

# Make sure market is normalized
odds_df["market_total_prob"] = (
    odds_df["home_implied_prob"] + odds_df["away_implied_prob"]
)

odds_df["home_market_norm"] = (
    odds_df["home_implied_prob"] / odds_df["market_total_prob"]
)

odds_df["away_market_norm"] = (
    odds_df["away_implied_prob"] / odds_df["market_total_prob"]
)

# 🔥 Independent edges
odds_df["home_edge"] = odds_df["home_sim_prob"] - odds_df["home_market_norm"]
odds_df["away_edge"] = odds_df["away_sim_prob"] - odds_df["away_market_norm"]

# ===== Proper Classification =====

def classify(row):
    if row["home_edge"] > threshold and row["home_edge"] >= row["away_edge"]:
        return "Home ML"
    elif row["away_edge"] > threshold and row["away_edge"] > row["home_edge"]:
        return "Away ML"
    else:
        return "No Bet"

odds_df["bet_type"] = odds_df.apply(classify, axis=1)

print(odds_df[[
    "home_team",
    "away_team",
    "home_edge",
    "away_edge",
    "bet_type"
]])

In [ ]:
import numpy as np

def simulate_win_prob(home_pts, away_pts, sims=10000, std_dev=10):
    """
    Simulation WITHOUT artificially inflating home advantage.
    """
    home_wins = 0

    # Reduce home bias slightly to avoid overfitting home
    home_adj = home_pts - 2  # small correction
    away_adj = away_pts

    for _ in range(sims):
        home_sim = np.random.normal(home_adj, std_dev)
        away_sim = np.random.normal(away_adj, std_dev)

        if home_sim > away_sim:
            home_wins += 1

    return home_wins / sims

In [ ]:
simulate
edge calculation
classification

In [ ]:
import numpy as np

# ===== FIXED SIMULATION (REDUCED HOME BIAS) =====

def simulate_win_prob(home_pts, away_pts, sims=10000, std_dev=10):
    home_wins = 0

    # Slight home adjustment to prevent bias stacking
    home_adj = home_pts - 2
    away_adj = away_pts

    for _ in range(sims):
        home_sim = np.random.normal(home_adj, std_dev)
        away_sim = np.random.normal(away_adj, std_dev)

        if home_sim > away_sim:
            home_wins += 1

    return home_wins / sims


# ===== Recalculate Simulation Probabilities =====

odds_df = odds_df.copy()

odds_df["home_sim_prob"] = odds_df.apply(
    lambda row: simulate_win_prob(
        row["proj_home_pts"],
        row["proj_away_pts"]
    ),
    axis=1
)

odds_df["away_sim_prob"] = 1 - odds_df["home_sim_prob"]

print(odds_df[[
    "home_team",
    "away_team",
    "home_sim_prob",
    "away_sim_prob"
]].head())

In [ ]:
# Normalize market
odds_df["market_total_prob"] = (
    odds_df["home_implied_prob"] + odds_df["away_implied_prob"]
)

odds_df["home_market_norm"] = (
    odds_df["home_implied_prob"] / odds_df["market_total_prob"]
)

odds_df["away_market_norm"] = (
    odds_df["away_implied_prob"] / odds_df["market_total_prob"]
)

# Calculate edges
odds_df["home_edge"] = odds_df["home_sim_prob"] - odds_df["home_market_norm"]
odds_df["away_edge"] = odds_df["away_sim_prob"] - odds_df["away_market_norm"]

threshold = 0.03

def classify(row):
    if row["home_edge"] > threshold and row["home_edge"] >= row["away_edge"]:
        return "Home ML"
    elif row["away_edge"] > threshold and row["away_edge"] > row["home_edge"]:
        return "Away ML"
    else:
        return "No Bet"

odds_df["bet_type"] = odds_df.apply(classify, axis=1)

print(odds_df[[
    "home_team",
    "away_team",
    "home_edge",
    "away_edge",
    "bet_type"
]])

In [ ]:
# ===============================
# CALCULATE TRUE EXPECTED VALUE
# ===============================

odds_df = odds_df.copy()

# Convert American odds to decimal
def american_to_decimal(odds):
    if odds is None:
        return None
    if odds > 0:
        return 1 + (odds / 100)
    else:
        return 1 + (100 / abs(odds))

odds_df["decimal_odds"] = odds_df["american_odds"].apply(
    lambda x: american_to_decimal(x) if pd.notnull(x) else None
)

# Expected Value Calculation
odds_df["ev_percent"] = (
    odds_df["home_sim_prob"] * (odds_df["decimal_odds"] - 1)
    - (1 - odds_df["home_sim_prob"])
)

# Remove rows with missing odds
odds_df = odds_df.dropna(subset=["ev_percent"])

# Rank by strongest EV
odds_df = odds_df.sort_values("ev_percent", ascending=False)

# ===============================
# SHOW TOP 10 REAL BETS
# ===============================

print("🔥 TOP BETS BY TRUE EXPECTED VALUE 🔥")
print(odds_df[[
    "home_team",
    "away_team",
    "bet_type",
    "home_sim_prob",
    "american_odds",
    "ev_percent"
]].head(10))

In [ ]:
# ===============================
# CLEAN & CONVERT AMERICAN ODDS
# ===============================

def american_to_decimal(odds):
    try:
        odds = float(odds)  # <-- FORCE numeric conversion
    except:
        return None

    if odds > 0:
        return 1 + (odds / 100)
    else:
        return 1 + (100 / abs(odds))


# Force column to numeric safely
odds_df["american_odds"] = pd.to_numeric(
    odds_df["american_odds"],
    errors="coerce"
)

# Convert
odds_df["decimal_odds"] = odds_df["american_odds"].apply(american_to_decimal)

In [ ]:
# ==========================================
# 🔥 FULL SLATE EV RANKING (NO MARKET LIMIT)
# ==========================================

import numpy as np
import pandas as pd

# Safety copy
rank_df = odds_df.copy()

# Remove rows with missing edge
rank_df = rank_df.dropna(subset=["true_edge", "american_odds"])

# Sort by strongest edge
rank_df = rank_df.sort_values("true_edge", ascending=False)

# Optional: Keep only positive edges (remove this line if you want ALL edges ranked)
rank_df = rank_df[rank_df["true_edge"] > 0]

# Assign units based on edge strength (customizable)
def assign_units(edge):
    if edge >= 0.40:
        return 1.00
    elif edge >= 0.25:
        return 0.50
    elif edge >= 0.10:
        return 0.25
    else:
        return 0.10

rank_df["units"] = rank_df["true_edge"].apply(assign_units)

# Clean output columns
ev_ranked = rank_df[
    ["home_team", "away_team", "bet_type", "true_edge", "american_odds", "units"]
]

print("🔥 FULL SLATE EV RANKING")
display(ev_ranked)

In [ ]:
# ==========================================
# 🔥 ADVANCED SLATE EV RANKING (SHARP MODE)
# ==========================================

import numpy as np
import pandas as pd

rank_df = odds_df.copy()

# Remove rows missing critical values
rank_df = rank_df.dropna(subset=["true_edge", "american_odds"])

# Convert odds safely
rank_df["american_odds"] = pd.to_numeric(rank_df["american_odds"], errors="coerce")

# Remove extreme garbage lines (book errors / bad conversions)
rank_df = rank_df[rank_df["american_odds"].abs() < 5000]

# Rank by true edge
rank_df = rank_df.sort_values("true_edge", ascending=False)

# Dynamic unit sizing based on edge * confidence
def kelly_style_units(edge):
    if edge >= 0.40:
        return 1.00
    elif edge >= 0.30:
        return 0.75
    elif edge >= 0.20:
        return 0.50
    elif edge >= 0.10:
        return 0.25
    else:
        return 0.10

rank_df["units"] = rank_df["true_edge"].apply(kelly_style_units)

# Optional: Remove negative edges automatically
rank_df = rank_df[rank_df["true_edge"] > 0]

# Final Output
ev_ranked = rank_df[
    ["home_team", "away_team", "bet_type", "true_edge", "american_odds", "units"]
]

print("🔥 SHARP EV RANKING — ALL MARKETS INCLUDED")
display(ev_ranked)

In [ ]:
def american_to_implied_prob(odds):
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return -odds / (-odds + 100)

def calc_ev(model_prob, odds):
    implied_prob = american_to_implied_prob(odds)
    payout = (100 / abs(odds)) if odds < 0 else (odds / 100)

    ev = (model_prob * payout) - ((1 - model_prob) * 1)
    return ev

In [ ]:
odds_df["ev_value"] = odds_df.apply(
    lambda row: calc_ev(row["model_prob"], row["american_odds"]),
    axis=1
)

In [ ]:
print(odds_df.columns)

In [ ]:
# ============================================
# 🚀 GLOBAL MARKET RANKING (NO MARKET BIAS)
# ============================================

def calculate_market_score(row):
    """
    Score every available market for a row.
    We rank purely by edge strength.
    """

    scores = {}

    # ---- Moneyline ----
    scores["Home ML"] = row.get("home_edge", -999)
    scores["Away ML"] = row.get("away_edge", -999)

    # ---- Simulation / Model Prob Edge ----
    scores["Model Prob"] = row.get("home_model_prob", -999) - row.get("home_implied_prob", 0)

    # ---- True Edge ----
    scores["True Edge"] = row.get("true_edge", -999)

    # ---- EV Based ----
    scores["EV"] = row.get("ev_percent", -999)

    # Get best scoring market
    best_market = max(scores, key=scores.get)
    best_score = scores[best_market]

    if best_score <= 0:
        return "No Bet", best_score
    else:
        return best_market, best_score


# Apply to dataframe
bet_results = []

for _, row in odds_df.iterrows():
    market, score = calculate_market_score(row)

    bet_results.append({
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "selected_market": market,
        "market_score": score,
        "american_odds": row.get("american_odds", None)
    })

bet_df = pd.DataFrame(bet_results)

# Sort by strongest edge
bet_df = bet_df.sort_values("market_score", ascending=False)

bet_df

In [ ]:
# ============================================
# 🚀 TRUE MARKET-AGNOSTIC RANKING ENGINE
# ============================================

def calculate_market_score(row):
    """
    Score every potential market independently
    and return the strongest one.
    """

    market_scores = {
        "Home ML": row.get("home_edge", -999),
        "Away ML": row.get("away_edge", -999),
        "True Edge": row.get("true_edge", -999),
        "EV": row.get("ev_percent", -999),
        "Model Prob": row.get("home_model_prob", 0) - row.get("home_implied_prob", 0),
        "Simulation": row.get("home_sim_prob", 0) - 0.5
    }

    # Normalize any weird None / NaN values
    market_scores = {k: float(v) if v is not None else -999 for k, v in market_scores.items()}

    best_market = max(market_scores, key=market_scores.get)
    best_score = market_scores[best_market]

    # If nothing meaningful
    if best_score <= 0:
        return "No Bet", best_score

    return best_market, best_score


bet_results = []

for _, row in odds_df.iterrows():
    market, score = calculate_market_score(row)

    bet_results.append({
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "selected_market": market,
        "market_score": score,
        "american_odds": row.get("american_odds", None)
    })

bet_df = pd.DataFrame(bet_results)
bet_df = bet_df.sort_values("market_score", ascending=False)

bet_df

In [ ]:
# ============================================
# 🚀 PROFESSIONAL SLATE FILTER
# ============================================

MIN_SCORE = 15  # adjustable threshold

filtered = bet_df.copy()

# Remove weak signals
filtered = filtered[filtered["market_score"] >= MIN_SCORE]

# Remove extreme negative odds distortion
filtered = filtered[filtered["american_odds"].notna()]

# Sort strongest first
filtered = filtered.sort_values("market_score", ascending=False)

filtered.reset_index(drop=True, inplace=True)

filtered

In [ ]:
# ============================================
# RISK ADJUSTED RANKING (PRO LEVEL)
# ============================================

import numpy as np

df = bet_df.copy()

# Normalize market score
df["score_norm"] = df["market_score"] / df["market_score"].max()

# Convert American to implied probability
def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

df["implied_prob"] = df["american_odds"].apply(american_to_prob)

# Risk adjustment (penalize extreme odds)
df["risk_penalty"] = 1 - df["implied_prob"]

# Final composite ranking
df["risk_adjusted_score"] = df["score_norm"] * df["risk_penalty"]

df = df.sort_values("risk_adjusted_score", ascending=False)

df

In [ ]:
# ============================================
# GAME-LEVEL BEST EDGE FILTER (NO MARKET BIAS)
# ============================================

df = bet_df.copy()

# Remove rows where risk_adjusted_score is NaN
df = df.dropna(subset=["risk_adjusted_score"])

# Sort by strongest edge
df = df.sort_values("risk_adjusted_score", ascending=False)

# Keep only best market per game
df["game_id"] = df["home_team"] + "_" + df["away_team"]

df = df.groupby("game_id", group_keys=False).apply(
    lambda x: x.nlargest(1, "risk_adjusted_score")
)

df = df.sort_values("risk_adjusted_score", ascending=False)

df

In [ ]:
print(df.columns)

In [ ]:
import numpy as np
import pandas as pd

df = df.copy()

# -----------------------------
# 1. Normalize Market Score
# -----------------------------
df["score_norm"] = df["market_score"] / df["market_score"].max()

# -----------------------------
# 2. Convert American Odds → Implied Probability
# -----------------------------
def american_to_prob(odds):
    try:
        odds = float(odds)
    except:
        return np.nan

    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

df["implied_prob"] = df["american_odds"].apply(american_to_prob)

# -----------------------------
# 3. Risk Penalty (Punish High Implied Prob / Low Value Spots)
# -----------------------------
df["risk_penalty"] = 1 - df["implied_prob"]

# -----------------------------
# 4. Final Risk Adjusted Ranking Score
# -----------------------------
df["risk_adjusted_score"] = df["score_norm"] * df["risk_penalty"]

# -----------------------------
# 5. Sort Best Edge First
# -----------------------------
df = df.sort_values("risk_adjusted_score", ascending=False)

df.head(25)
# Optional: Add volatility damping
df["volatility_weight"] = df["score_norm"] ** 0.5

df["final_rank_score"] = (
    df["risk_adjusted_score"] * df["volatility_weight"]
)

df = df.sort_values("final_rank_score", ascending=False)

df.head(25)

In [ ]:
df["liquidity_penalty"] = 1 - abs(df["implied_prob"] - df["score_norm"])
df["final_rank_score"] *= df["liquidity_penalty"]

In [ ]:
# ==============================
# FULL SLATE MODEL RUN
# ==============================

import numpy as np
import pandas as pd

# ---- Safety ----
df = df.copy()

# ---- Drop rows missing core values ----
df = df.dropna(subset=["score_norm", "implied_prob", "american_odds"])

# ---- Risk Adjusted Score ----
if "risk_penalty" in df.columns:
    df["risk_adjusted_score"] = df["score_norm"] * (1 - df["risk_penalty"])
else:
    df["risk_adjusted_score"] = df["score_norm"]

# ---- Volatility Weight ----
if "volatility_weight" in df.columns:
    df["final_rank_score"] = df["risk_adjusted_score"] * df["volatility_weight"]
else:
    df["final_rank_score"] = df["risk_adjusted_score"]

# ---- Liquidity Layer (Optional but Powerful) ----
df["liquidity_penalty"] = 1 - abs(df["score_norm"] - df["implied_prob"])
df["final_rank_score"] = df["final_rank_score"] * df["liquidity_penalty"]

# ---- Sort by Real Edge ----
df = df.sort_values("final_rank_score", ascending=False)

# ---- Select Top Bets Only ----
top = df.head(20).copy()

# ---- Convert Odds to Clean Format ----
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

top["american_odds"] = top["american_odds"].apply(format_odds)

# ---- Build Discord Output ----
discord_output = "🔥 **MODEL TOP BETS — FULL SLATE**\n\n"

for _, row in top.iterrows():
    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Market: {row.get('selected_market','')}\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Unit Size: 1u\n\n"
    )

print(discord_output)

In [ ]:
# ==============================
# FILTER & AUTO UNIT SCALING
# Exclude bets with juice worse than -200
# ==============================

# First, clean odds as numeric
df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

# Remove bets worse than -200
df_filtered = df[df["american_odds_num"] >= -200].copy()

# Rank remaining bets
df_filtered = df_filtered.sort_values("final_rank_score", ascending=False)
percentiles = df_filtered["final_rank_score"].rank(pct=True)

# Assign unit tiers
def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_filtered["units"] = percentiles.apply(assign_units)

# Format American odds for Discord
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

# Build Discord output
discord_output = "🔥 **MODEL TOP BETS — FILTERED BY JUICE <= -200**\n\n"

for _, row in df_filtered.iterrows():
    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Market: {row.get('selected_market','')}\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)
# Keep only top 10 strongest remaining bets
df_filtered = df_filtered.head(10)

In [ ]:
import requests
import pandas as pd

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

SPORT = "basketball_ncaab"
REGION = "us"
MARKETS = "h2h,spreads,totals"
BOOKMAKER = "draftkings"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

params = {
    "apiKey": API_KEY,
    "regions": REGION,
    "markets": MARKETS,
    "bookmakers": BOOKMAKER,
    "oddsFormat": "american"
}

response = requests.get(url, params=params)
data = response.json()

print("Status:", response.status_code)
print("Games pulled:", len(data))

In [ ]:
rows = []

for game in data:
    home = game["home_team"]
    away = game["away_team"]

    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:

            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append([
                        home,
                        away,
                        f"ML {outcome['name']}",
                        None,
                        outcome["price"]
                    ])

            if market["key"] == "spreads":
                for outcome in market["outcomes"]:
                    rows.append([
                        home,
                        away,
                        f"Spread {outcome['name']}",
                        outcome["point"],
                        outcome["price"]
                    ])

            if market["key"] == "totals":
                for outcome in market["outcomes"]:
                    rows.append([
                        home,
                        away,
                        f"Total {outcome['name']}",
                        outcome["point"],
                        outcome["price"]
                    ])

df = pd.DataFrame(rows, columns=[
    "home_team",
    "away_team",
    "selected_market",
    "line",
    "american_odds"
])

df.head()

In [ ]:
ranked_bets = run_full_stable_v3(
    df,
    max_juice=-200,
    min_edge=0.0
)

ranked_bets.sort_values("ev", ascending=False).head(20)

In [ ]:
# ==============================
# FILTER SLATE TO 2/27 ONLY
# ==============================

from datetime import datetime
import pytz

target_date = "2026-02-27"

df["commence_time"] = pd.to_datetime(df["commence_time"], utc=True)

df["game_date_est"] = (
    df["commence_time"]
    .dt.tz_convert("US/Eastern")
    .dt.strftime("%Y-%m-%d")
)

df = df[df["game_date_est"] == target_date].copy()

print("Games remaining for 2/27:", len(df))

In [ ]:
# Example after pulling from Odds API JSON
games = response.json()

rows = []

for game in games:
    rows.append({
        "home_team": game["home_team"],
        "away_team": game["away_team"],
        "commence_time": game["commence_time"],  # <-- THIS IS REQUIRED
        "home_ml": game.get("home_ml"),
        "away_ml": game.get("away_ml"),
        # add whatever else you use
    })

df = pd.DataFrame(rows)

In [ ]:
print(df.columns)

In [ ]:
import pandas as pd

# 🔥 Convert to datetime safely
df["commence_time"] = pd.to_datetime(df["commence_time"], errors="coerce", utc=True)

# Convert to EST
df["game_date_est"] = df["commence_time"].dt.tz_convert("US/Eastern").dt.date

# Target date
target_date = pd.to_datetime("2026-02-27").date()

# Filter for Friday 2/27
df = df[df["game_date_est"] == target_date].copy()

print("Games remaining for 2/27:", len(df))

In [ ]:
ranked_bets = run_full_stable_v3(
    df,
    max_juice=-200,
    min_edge=0.0
)

In [ ]:
# ==============================
# MASTER NCAA SLATE RUN (2/27)
# ==============================

import pandas as pd

TARGET_DATE = "2026-02-27"

# ---- 1. Ensure time column exists ----
df["commence_time"] = pd.to_datetime(df["commence_time"], utc=True)
df["game_date_est"] = df["commence_time"].dt.tz_convert("US/Eastern").dt.date

df_slate = df[df["game_date_est"] == pd.to_datetime(TARGET_DATE).date()].copy()

# ---- 2. Convert Odds ----
df_slate["american_odds_num"] = pd.to_numeric(df_slate["home_ml"], errors="coerce")

# ---- 3. Filter Juice ----
df_slate = df_slate[df_slate["american_odds_num"] >= -200].copy()

# ---- 4. Rank by Model Score ----
df_slate = df_slate.sort_values("final_rank_score", ascending=False)

percentiles = df_slate["final_rank_score"].rank(pct=True)

def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_slate["units"] = percentiles.apply(assign_units)

# ---- 5. Format Odds ----
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_slate["american_odds"] = df_slate["american_odds_num"].apply(format_odds)

# ---- 6. Build Discord Output ----
discord_output = "🔥 **MODEL TOP BETS — 2/27 SLATE**\n\n"

for _, row in df_slate.iterrows():
    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)

# Keep Top 10 Only
df_top10 = df_slate.head(10)

In [ ]:
# ==============================
# MASTER NCAA SLATE RUN (SAFE VERSION)
# ==============================

import pandas as pd
TARGET_DATE = "2026-02-27"

# ---- Ensure time column exists ----
df["commence_time"] = pd.to_datetime(df["commence_time"], utc=True)
df["game_date_est"] = df["commence_time"].dt.tz_convert("US/Eastern").dt.date

df_slate = df[df["game_date_est"] == pd.to_datetime(TARGET_DATE).date()].copy()

# ---- Ensure ranking column exists ----
if "final_rank_score" not in df_slate.columns:
    # fallback ranking if model score missing
    if "market_score" in df_slate.columns:
        df_slate["final_rank_score"] = df_slate["market_score"]
    else:
        raise ValueError("No ranking metric found in dataframe.")

# ---- Convert Odds ----
if "home_ml" in df_slate.columns:
    df_slate["american_odds_num"] = pd.to_numeric(df_slate["home_ml"], errors="coerce")
else:
    df_slate["american_odds_num"] = None

# ---- Filter Juice ----
df_slate = df_slate[df_slate["american_odds_num"] >= -200].copy()

# ---- Rank ----
df_slate = df_slate.sort_values("final_rank_score", ascending=False)
percentiles = df_slate["final_rank_score"].rank(pct=True)

def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_slate["units"] = percentiles.apply(assign_units)

# ---- Format Odds ----
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_slate["american_odds"] = df_slate["american_odds_num"].apply(format_odds)

# ---- Build Discord Output ----
discord_output = "🔥 **MODEL TOP BETS — 2/27 SLATE**\n\n"

for _, row in df_slate.iterrows():
    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)

df_top10 = df_slate.head(10)

In [ ]:
# ==============================
# TEMP RANK GENERATOR (FOR RAW SLATE)
# ==============================

import numpy as np

# Make sure dataframe exists
df_slate = df.copy()

# ---- Create a synthetic ranking metric if model not built yet ----
if "final_rank_score" not in df_slate.columns:

    # Example: rank by strongest implied win probability
    if "home_ml" in df_slate.columns:
        df_slate["home_ml"] = pd.to_numeric(df_slate["home_ml"], errors="coerce")

        # Convert odds to implied prob
        def american_to_prob(odds):
            if pd.isna(odds):
                return 0
            odds = float(odds)
            if odds < 0:
                return abs(odds) / (abs(odds) + 100)
            else:
                return 100 / (odds + 100)

        df_slate["implied_prob"] = df_slate["home_ml"].apply(american_to_prob)

        # Use implied prob as temporary ranking score
        df_slate["final_rank_score"] = df_slate["implied_prob"]

    else:
        raise ValueError("No usable metric found to build ranking from.")

# ---- Sort ----
df_slate = df_slate.sort_values("final_rank_score", ascending=False)

# ---- Unit Allocation ----
percentiles = df_slate["final_rank_score"].rank(pct=True)

def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_slate["units"] = percentiles.apply(assign_units)

print(df_slate[["home_team","away_team","final_rank_score","units"]])

In [ ]:
print(df_slate.columns)
print(df_slate["final_rank_score"].describe())

In [ ]:
# 1. Pull fresh odds for 2/27
df = pull_odds_api(date="2026-02-27")

# 2. Run your existing model
ranked_bets = run_full_stable_v3(df)

# 3. Generate discord output
print(generate_discord_card(ranked_bets))

In [ ]:
import pandas as pd
import numpy as np
import requests
import datetime

In [ ]:
def pull_odds_api(date):
    URL = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": "YOUR_API_KEY",
        "regions": "us",
        "markets": "h2h",
        "oddsFormat": "american",
        "date": date
    }

    response = requests.get(URL, params=params)

    if response.status_code != 200:
        raise Exception("Odds API failed")

    data = response.json()

    rows = []

    for game in data:
        home_team = game["home_team"]
        away_team = game["away_team"]

        markets = game["bookmakers"][0]["markets"][0]
        outcomes = markets["outcomes"]

        home_ml = next(o["price"] for o in outcomes if o["name"] == home_team)
        away_ml = next(o["price"] for o in outcomes if o["name"] == away_team)

        rows.append({
            "home_team": home_team,
            "away_team": away_team,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "commence_time": game["commence_time"]
        })

    return pd.DataFrame(rows)

In [ ]:
df = pull_odds_api(date="2026-02-27")

ranked_bets = run_full_stable_v3(df)

print(ranked_bets.head())

In [ ]:
def pull_odds_api(date):
    URL = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": "10ceac94f9b9cb76f8c65510ca6df18f",
        "regions": "us",
        "markets": "h2h",
        "oddsFormat": "american"
    }

    response = requests.get(URL, params=params)

    print("Status Code:", response.status_code)
    print("Response Text:", response.text)  # 🔥 THIS SHOWS THE REAL ERROR

    if response.status_code != 200:
        raise Exception("Odds API failed")

    data = response.json()

    rows = []

    for game in data:
        rows.append({
            "home_team": game["home_team"],
            "away_team": game["away_team"],
            "commence_time": game["commence_time"]
        })

    return pd.DataFrame(rows)

In [ ]:
df = pull_odds_api(date="2026-02-27")

In [ ]:
import requests
import pandas as pd

def pull_odds_api():
    URL = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": "YOUR_API_KEY",
        "regions": "us",
        "markets": "h2h",
        "oddsFormat": "american"
    }

    response = requests.get(URL, params=params)

    if response.status_code != 200:
        raise Exception(response.text)

    data = response.json()

    rows = []

    for game in data:
        if not game["bookmakers"]:
            continue

        # Get best bookmaker (first one)
        bookmaker = game["bookmakers"][0]
        market = bookmaker["markets"][0]

        home = game["home_team"]
        away = game["away_team"]

        # Extract odds
        home_odds = None
        away_odds = None

        for outcome in market["outcomes"]:
            if outcome["name"] == home:
                home_odds = outcome["price"]
            elif outcome["name"] == away:
                away_odds = outcome["price"]

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_ml": home_odds,
            "away_ml": away_odds,
            "commence_time": game["commence_time"]
        })

    return pd.DataFrame(rows)

In [ ]:
df = pull_odds_api()

print(df.head())

In [ ]:
# ==============================
#  ODDS API PULL FUNCTION
# ==============================

import requests
import pandas as pd
import os

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"  # <-- PUT YOUR KEY HERE

def pull_odds_api(date=None):

    base_url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds/"

    params = {
        "apiKey": API_KEY,
        "regions": "us",
        "markets": "h2h",
        "oddsFormat": "american"
    }

    # Optional date filter if provided
    if date:
        params["commenceTimeFrom"] = f"{date}T00:00:00Z"
        params["commenceTimeTo"]   = f"{date}T23:59:59Z"

    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        raise Exception(f"Odds API Error:\n{response.text}")

    data = response.json()

    # Flatten API into dataframe
    rows = []

    for game in data:
        for book in game.get("bookmakers", []):
            for market in book.get("markets", []):
                if market["key"] == "h2h":
                    for outcome in market["outcomes"]:
                        rows.append({
                            "home_team": game["home_team"],
                            "away_team": game["away_team"],
                            "commence_time": game["commence_time"],
                            "book": book["key"],
                            "team": outcome["name"],
                            "american_odds": outcome["price"]
                        })

    df = pd.DataFrame(rows)

    return df


# ==============================
#  TEST THE CONNECTION
# ==============================

df = pull_odds_api(date="2026-02-27")

print(df.head())
print("Rows pulled:", len(df))

In [ ]:
# ==============================
#  PREPARE DATA FOR MODEL
# ==============================

def prepare_model_df(df):

    # Pivot so each game becomes one row with home/away odds
    pivot = df.pivot_table(
        index=["home_team", "away_team", "commence_time"],
        columns="team",
        values="american_odds",
        aggfunc="first"
    ).reset_index()

    # Rename columns if needed
    pivot.columns.name = None

    # Optional: clean numeric odds
    for col in pivot.columns:
        if col not in ["home_team", "away_team", "commence_time"]:
            pivot[col] = pd.to_numeric(pivot[col], errors="coerce")

    return pivot


model_df = prepare_model_df(df)

print(model_df.head())
print("Model ready rows:", len(model_df))

In [ ]:
# =====================================
# GET BEST ODDS PER GAME (Smart Version)
# =====================================

def get_best_odds(raw_df):

    games = []

    grouped = raw_df.groupby(["home_team", "away_team", "commence_time"])

    for (home, away, time), group in grouped:

        # Get all odds for that game
        odds_values = group["american_odds"].dropna().values

        if len(odds_values) == 0:
            continue

        # Best odds = highest price for dog + lowest negative for favorite
        best_home = group[group["team"] == home]["american_odds"].max()
        best_away = group[group["team"] == away]["american_odds"].max()

        games.append({
            "home_team": home,
            "away_team": away,
            "commence_time": time,
            "best_home_odds": best_home,
            "best_away_odds": best_away
        })

    return pd.DataFrame(games)


clean_df = get_best_odds(df)

print(clean_df.head())
print("Clean games:", len(clean_df))

In [ ]:
# ==============================
# CREATE RANKING METRIC FROM ODDS
# ==============================

import numpy as np

def calculate_market_score_from_odds(df):

    df = df.copy()

    # Convert odds to numeric
    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    # Convert American odds → implied probability
    def american_to_prob(odds):
        if pd.isna(odds):
            return np.nan
        odds = float(odds)
        if odds > 0:
            return 100 / (odds + 100)
        else:
            return abs(odds) / (abs(odds) + 100)

    # Create a simple edge metric
    df["home_implied_prob"] = df["home_ml"].apply(american_to_prob)

    # Use absolute value as ranking proxy
    df["market_score"] = df["home_implied_prob"].fillna(0) * 100

    return df

In [ ]:
# ==============================
# RUN MODEL FOR CURRENT SLATE
# ==============================

# Make sure odds are loaded
df = df.copy()

# ---- Run Your Existing Model Pipeline ----
ranked_bets = run_full_stable_v3(
    df,
    max_juice=-200,
    min_edge=0.0
)

# ---- Sort by Strongest Signal ----
ranked_bets = ranked_bets.sort_values(
    "final_rank_score",
    ascending=False
)

# ---- Keep Top 15 Only ----
top_bets = ranked_bets.head(15)

# ---- Format for Discord ----
def format_for_discord(data):
    output = "🔥 **MODEL TOP BETS — AUTOMATIC SLATE RUN**\n\n"

    for _, row in data.iterrows():
        output += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Market: {row.get('selected_market','')}\n"
            f"Odds: {row.get('american_odds','')}\n"
            f"Model Score: {round(row['final_rank_score'],4)}\n"
            f"Units: {row.get('units',0)}u\n\n"
        )

    return output

# ---- Print Output ----
print(format_for_discord(top_bets))

In [ ]:
# ==========================================
# STABLE V3 MODEL WRAPPER
# ==========================================

def run_full_stable_v3(df, max_juice=-200, min_edge=0.0):

    import pandas as pd
    import numpy as np

    df = df.copy()

    # ---- Convert odds safely ----
    df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

    # ---- Filter juice ----
    df = df[df["american_odds_num"] >= max_juice].copy()

    if df.empty:
        print("No bets passed juice filter.")
        return df

    # ---- Ranking Metric ----
    if "final_rank_score" not in df.columns:
        if "market_score" in df.columns:
            df["final_rank_score"] = df["market_score"]
        else:
            raise ValueError("No ranking metric found in dataframe.")

    # ---- Sort ----
    df = df.sort_values("final_rank_score", ascending=False)

    # ---- Percentile Ranking ----
    percentiles = df["final_rank_score"].rank(pct=True)

    # ---- Unit Assignment ----
    def assign_units(p):
        if p >= 0.85:
            return 1.0
        elif p >= 0.60:
            return 0.75
        elif p >= 0.35:
            return 0.5
        else:
            return 0.25

    df["units"] = percentiles.apply(assign_units)

    return df

In [ ]:
ranked_bets = run_full_stable_v3(df)

In [ ]:
df = calculate_market_score_from_odds(df)

ranked_bets = run_full_stable_v3(df)

print(ranked_bets.head())

In [ ]:
# ==============================
# CREATE RANKING METRIC FROM ODDS
# ==============================

import numpy as np

def calculate_market_score_from_odds(df):

    df = df.copy()

    # Convert odds to numeric
    df["best_home_odds"] = pd.to_numeric(df["best_home_odds"], errors="coerce")
    df["best_away_odds"] = pd.to_numeric(df["best_away_odds"], errors="coerce")

    # American odds → implied probability
    def american_to_prob(odds):
        if pd.isna(odds):
            return np.nan
        odds = float(odds)
        if odds > 0:
            return 100 / (odds + 100)
        else:
            return abs(odds) / (abs(odds) + 100)

    # Compute implied probs
    df["home_implied_prob"] = df["best_home_odds"].apply(american_to_prob)

    # Simple ranking metric = inverse of implied probability (edge potential)
    df["market_score"] = 1 - df["home_implied_prob"]

    return df

In [ ]:
df = calculate_market_score_from_odds(df)

ranked_bets = run_full_stable_v3(df)

print(ranked_bets.head())

In [ ]:
# ==========================================
# AUTO EXTRACT BEST ODDS FROM BOOKMAKER COLUMNS
# ==========================================

import numpy as np
import pandas as pd

def extract_best_odds(df):

    df = df.copy()

    # Identify bookmaker columns
    # (They are numeric odds columns created from API)
    bookmaker_cols = [
        col for col in df.columns
        if col not in ["home_team", "away_team", "commence_time"]
    ]

    # Separate team columns (they contain odds values)
    odds_cols = [c for c in bookmaker_cols if df[c].dtype != "object"]

    # Find best (highest) odds for home & away rows
    df["best_home_odds"] = df[odds_cols].max(axis=1)
    df["best_away_odds"] = df[odds_cols].max(axis=1)

    return df

In [ ]:
df = extract_best_odds(df)

print(df[["home_team","away_team","best_home_odds","best_away_odds"]].head())

In [ ]:
df = calculate_market_score_from_odds(df)

ranked_bets = run_full_stable_v3(df)

print(ranked_bets.head())

In [ ]:
# ===============================
#  RUN FULL MODEL FOR 2/27 SLATE
#  WITH AUTO DISCORD OUTPUT
# ===============================

TARGET_DATE = "2026-02-27"

# 1. Pull Fresh Odds
df = pull_odds_api(date=TARGET_DATE)

# 2. Convert Commence Time + Filter Slate
df["commence_time"] = pd.to_datetime(df["commence_time"], utc=True)
df["game_date"] = df["commence_time"].dt.strftime("%Y-%m-%d")

df = df[df["game_date"] == TARGET_DATE].copy()

# 3. Extract Best Odds Per Game
df = extract_best_odds(df)  # your function that builds best_home_odds / best_away_odds

# 4. Calculate Market Score
df = calculate_market_score_from_odds(df)

# 5. Run Your Model Ranking
ranked = run_full_stable_v3(
    df,
    max_juice=-200,
    min_edge=0.0
)

# 6. Sort Strongest Bets
ranked = ranked.sort_values("final_rank_score", ascending=False)

# Keep Top 15
ranked = ranked.head(15)

# 7. Format American Odds Clean
def fmt(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

ranked["american_odds"] = ranked["american_odds"].apply(fmt)

# 8. Build Discord Output
discord_msg = "🔥 **FRIDAY 2/27 MODEL SLATE — TOP BETS**\n\n"

for _, row in ranked.iterrows():
    discord_msg += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Market: {row.get('selected_market','')}\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_msg)

In [ ]:
# =====================================
#  CLEAN DISCORD OUTPUT (REMOVE DUPES)
# =====================================

# Keep only best price per game
ranked = ranked.sort_values("final_rank_score", ascending=False)

ranked_clean = ranked.drop_duplicates(
    subset=["home_team", "away_team"],
    keep="first"
).head(15)

def fmt(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

ranked_clean["american_odds"] = ranked_clean["american_odds"].apply(fmt)

discord_msg = "🔥 **FRIDAY 2/27 MODEL SLATE — TOP BETS**\n\n"

for _, row in ranked_clean.iterrows():
    discord_msg += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_msg)

In [ ]:
# ==============================
# FILTER & AUTO UNIT SCALING
# Exclude bets with juice worse than -200
# ==============================

import pandas as pd

# Ensure odds are numeric
df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

# Remove bets with juice worse than -200
df_filtered = df[df["american_odds_num"] >= -200].copy()

# Make sure ranking column exists
if "final_rank_score" not in df_filtered.columns:
    raise ValueError("final_rank_score column not found. Run model first.")

# Rank remaining bets
df_filtered = df_filtered.sort_values("final_rank_score", ascending=False)
percentiles = df_filtered["final_rank_score"].rank(pct=True)

# Assign unit tiers
def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_filtered["units"] = percentiles.apply(assign_units)

# Format odds for Discord
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

# Keep only top 10 strongest plays
df_filtered = df_filtered.head(10)

# ==============================
# Build Discord Output
# ==============================

discord_output = "🔥 **MODEL TOP BETS — FILTERED BY JUICE <= -200**\n\n"

for _, row in df_filtered.iterrows():
    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Market: {row.get('selected_market','')}\n"
        f"Odds: {row['american_odds']}\n"
        f"Model Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)

In [ ]:
import pandas as pd

df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

df_filtered = df[df["american_odds_num"] >= -200].copy()

# ----- DETECT RANK COLUMN AUTOMATICALLY -----
if "final_rank_score" in df_filtered.columns:
    rank_col = "final_rank_score"
elif "market_score" in df_filtered.columns:
    rank_col = "market_score"
elif "risk_adjusted_score" in df_filtered.columns:
    rank_col = "risk_adjusted_score"
else:
    raise ValueError("No ranking column found. Model not executed properly.")

df_filtered = df_filtered.sort_values(rank_col, ascending=False)

percentiles = df_filtered[rank_col].rank(pct=True)

def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_filtered["units"] = percentiles.apply(assign_units)

def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

df_filtered = df_filtered.head(10)

discord_output = "🔥 **MODEL TOP BETS — FILTERED BY JUICE <= -200**\n\n"

for _, row in df_filtered.iterrows():
    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Odds: {row['american_odds']}\n"
        f"Score: {round(row[rank_col],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)

In [ ]:
# ==============================
# FILTER & AUTO UNIT SCALING
# Exclude bets with juice worse than -200
# ==============================

df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

df_filtered = df[df["american_odds_num"] >= -200].copy()

if "final_rank_score" not in df_filtered.columns:
    raise ValueError("final_rank_score column not found. Run model first.")

df_filtered = df_filtered.sort_values("final_rank_score", ascending=False)

percentiles = df_filtered["final_rank_score"].rank(pct=True)

def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_filtered["units"] = percentiles.apply(assign_units)

def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

# ==============================
# 🔥 IMPROVED DISCORD OUTPUT
# Now we print the actual bet side
# ==============================

discord_output = "🔥 **MODEL TOP BETS — FILTERED BY JUICE <= -200**\n\n"

for _, row in df_filtered.iterrows():

    # Determine bet side automatically
    bet_side = "UNKNOWN"

    if "best_home_odds" in row and not pd.isna(row.get("best_home_odds")):
        if row["american_odds_num"] == row["best_home_odds"]:
            bet_side = f"{row['home_team']} ML"
        elif row["american_odds_num"] == row["best_away_odds"]:
            bet_side = f"{row['away_team']} ML"

    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Bet: {bet_side}\n"
        f"Odds: {row['american_odds']}\n"
        f"Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)

df_filtered = df_filtered.head(10)

In [ ]:
# ==============================
# FILTER + AUTO UNIT SCALING
# Exclude bets with juice worse than -200
# ==============================

# Make sure odds exist
if "american_odds" not in df.columns:
    raise ValueError("american_odds column not found. Run odds pull + model first.")

df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

df_filtered = df[df["american_odds_num"] >= -200].copy()

# ✅ If model didn't create ranking — fallback to market_score
if "final_rank_score" not in df_filtered.columns:
    if "market_score" in df_filtered.columns:
        print("⚠ final_rank_score missing. Using market_score as backup ranking.")
        df_filtered["final_rank_score"] = df_filtered["market_score"]
    else:
        raise ValueError("No ranking column found. Run model scoring first.")

df_filtered = df_filtered.sort_values("final_rank_score", ascending=False)

percentiles = df_filtered["final_rank_score"].rank(pct=True)

def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_filtered["units"] = percentiles.apply(assign_units)

def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

# ==============================
# DISCORD OUTPUT
# ==============================

discord_output = "🔥 **MODEL TOP BETS — FILTERED BY JUICE <= -200**\n\n"

for _, row in df_filtered.iterrows():

    bet_side = "Unknown Side"

    # Try to auto-detect bet side
    if "best_home_odds" in row and not pd.isna(row.get("best_home_odds")):
        if row["american_odds_num"] == row["best_home_odds"]:
            bet_side = f"{row['home_team']} ML"
        elif row["american_odds_num"] == row["best_away_odds"]:
            bet_side = f"{row['away_team']} ML"

    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Bet: {bet_side}\n"
        f"Odds: {row['american_odds']}\n"
        f"Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord_output)

df_filtered = df_filtered.head(10)

In [ ]:
# ==============================
# CLEAN + RANK + REMOVE DUPES + TOP 10
# ==============================

import pandas as pd

# ---- Ensure odds column exists ----
df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

# ---- Filter Juice ----
df_filtered = df[df["american_odds_num"] >= -200].copy()

# ---- Ranking Column Fallback ----
if "final_rank_score" not in df_filtered.columns:
    print("⚠ final_rank_score missing. Using market_score as backup ranking.")
    df_filtered["final_rank_score"] = df_filtered.get("market_score", 0)

# ---- Sort by Model Strength ----
df_filtered = df_filtered.sort_values("final_rank_score", ascending=False)

# ---- Remove Duplicate Matchups (keep highest score per game) ----
df_filtered["game_key"] = (
    df_filtered["home_team"].astype(str) + "_" +
    df_filtered["away_team"].astype(str)
)

df_filtered = (
    df_filtered
    .sort_values("final_rank_score", ascending=False)
    .drop_duplicates(subset="game_key", keep="first")
)

# ---- Keep Top 10 ----
df_filtered = df_filtered.head(10)

# ---- Format Odds ----
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

# ---- Build Discord Output ----
discord_output = "🔥 **MODEL TOP BETS — TOP 10 (NO DUPES)**\n\n"

for _, row in df_filtered.iterrows():
    bet_side = row["home_team"] if row["final_rank_score"] > 0.5 else row["away_team"]

    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Bet: {bet_side} ML\n"
        f"Odds: {row['american_odds']}\n"
        f"Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {row.get('units','')}u\n\n"
    )

print(discord_output)

In [ ]:
# ==============================
# FILTER + AUTO UNIT SCALING + TOP 10 NO DUPES
# ==============================

# Ensure numeric odds
df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

# Filter out heavy juice
df_filtered = df[df["american_odds_num"] >= -200].copy()

# Ensure ranking column exists
if "final_rank_score" not in df_filtered.columns:
    print("⚠ final_rank_score missing. Using market_score as backup ranking.")
    df_filtered["final_rank_score"] = df_filtered["market_score"]

# Sort by strongest edge
df_filtered = df_filtered.sort_values("final_rank_score", ascending=False)

# Remove duplicate games (keep highest ranked per matchup)
df_filtered = df_filtered.drop_duplicates(subset=["home_team","away_team"])

# Rank percentile
percentiles = df_filtered["final_rank_score"].rank(pct=True)

# Unit tiers
def assign_units(p):
    if p >= 0.85:
        return 1.0
    elif p >= 0.60:
        return 0.75
    elif p >= 0.35:
        return 0.5
    else:
        return 0.25

df_filtered["units"] = percentiles.apply(assign_units)

# Format odds
def format_odds(o):
    if pd.isna(o):
        return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

df_filtered["american_odds"] = df_filtered["american_odds_num"].apply(format_odds)

# Keep top 10 after cleaning
df_top10 = df_filtered.head(10)

# Build Discord Output
discord_output = "🔥 **MODEL TOP BETS — TOP 10 (NO DUPES)**\n\n"

for _, row in df_top10.iterrows():
    bet_side = row["home_team"] if row["final_rank_score"] >= 0.5 else row["away_team"]

    discord_output += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Bet: {bet_side} ML\n"
        f"Odds: {row['american_odds']}\n"
        f"Score: {round(row['final_rank_score'],4)}\n"
        f"Units: {round(row['units'],2)}u\n\n"
    )

print(discord_output)

In [ ]:
def run_ncaab_slate(date):

    # 1️⃣ Pull odds
    df = pull_odds_api(date=date)

    # 2️⃣ Run your simulation / projection engine
    df = run_simulation_pipeline(df)
    # <-- this must create:
    # implied_prob
    # simulated_margin
    # edge
    # final_rank_score (or market_score fallback)

    # 3️⃣ Ensure ranking column exists
    if "final_rank_score" not in df.columns:
        if "market_score" in df.columns:
            print("⚠ final_rank_score missing. Using market_score as backup.")
            df["final_rank_score"] = df["market_score"]
        else:
            raise ValueError("No ranking metric found.")

    # 4️⃣ Filter juice
    df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")
    df = df[df["american_odds_num"] >= -200].copy()

    # 5️⃣ Rank
    df = df.sort_values("final_rank_score", ascending=False)

    # 6️⃣ Unit scaling
    percentiles = df["final_rank_score"].rank(pct=True)

    def assign_units(p):
        if p >= 0.85:
            return 1.0
        elif p >= 0.60:
            return 0.75
        elif p >= 0.35:
            return 0.5
        else:
            return 0.25

    df["units"] = percentiles.apply(assign_units)

    # 7️⃣ Remove duplicates (team vs team duplicates)
    df = df.drop_duplicates(subset=["home_team","away_team"])

    # 8️⃣ Keep Top 10
    df = df.head(10)

    # 9️⃣ Build Discord Output
    output = "🔥 **MODEL TOP BETS — TOP 10 (AUTO RUN)**\n\n"

    for _, row in df.iterrows():
        odds = int(row["american_odds_num"])
        odds = f"+{odds}" if odds > 0 else str(odds)

        output += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Bet: {row.get('bet_side','ML')}\n"
            f"Odds: {odds}\n"
            f"Score: {round(row['final_rank_score'],4)}\n"
            f"Units: {row['units']}u\n\n"
        )

    print(output)

    return df

In [ ]:
run_ncaab_slate("2026-02-27")

In [ ]:
 ---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
/tmp/ipython-input-326/1445682167.py in <cell line: 0>()
----> 1 run_ncaab_slate("2026-02-27")

/tmp/ipython-input-326/3229905744.py in run_ncaab_slate(date)
      5
      6     # 2️⃣ Run your simulation / projection engine
----> 7     df = run_simulation_pipeline(df)
      8     # <-- this must create:
      9     # implied_prob

NameError: name 'run_simulation_pipeline' is not defined

In [ ]:
# ==============================
# MASTER SLATE RUN
# ==============================

def run_full_stable_v3(date,
                      max_juice=-200,
                      min_edge=0.0):

    df = pull_odds_api(date)
    df = run_simulation_pipeline(df)

    # Edge calculation
    if "home_implied_prob" in df.columns:
        df["edge"] = df["home_proj_prob"] - df["home_implied_prob"]
    else:
        df["edge"] = 0

    df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

    df = df[df["american_odds_num"] >= max_juice].copy()
    df = df[df["edge"] >= min_edge].copy()

    if "final_rank_score" not in df.columns:
        df["final_rank_score"] = df["market_score"]

    df = df.sort_values("final_rank_score", ascending=False)

    # Unit scaling
    percentiles = df["final_rank_score"].rank(pct=True)

    def assign_units(p):
        if p >= 0.85:
            return 1.0
        elif p >= 0.60:
            return 0.75
        elif p >= 0.35:
            return 0.5
        else:
            return 0.25

    df["units"] = percentiles.apply(assign_units)

    def format_odds(o):
        if pd.isna(o):
            return ""
        o = float(o)
        return f"{int(o)}" if o < 0 else f"+{int(o)}"

    df["american_odds"] = df["american_odds_num"].apply(format_odds)

    df = df.drop_duplicates(subset=["home_team", "away_team"])

    df_top10 = df.head(10)

    # Discord Output
    discord_output = f"🔥 **MODEL SLATE — {date}**\n\n"

    for _, row in df_top10.iterrows():
        discord_output += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Bet: {row['home_team']} ML\n"
            f"Odds: {row['american_odds']}\n"
            f"Score: {round(row['final_rank_score'],4)}\n"
            f"Units: {row['units']}u\n\n"
        )

    print(discord_output)

    return df_top10

In [ ]:
run_full_stable_v3("2026-02-27")

In [ ]:
# ==============================
# CORE SIMULATION ENGINE (RESTORE)
# ==============================

def run_simulation_pipeline(df):

    # ---- Ensure numeric odds ----
    df["home_ml"] = pd.to_numeric(df.get("home_ml"), errors="coerce")
    df["away_ml"] = pd.to_numeric(df.get("away_ml"), errors="coerce")

    # ---- If implied prob not present, compute from odds ----
    if "home_implied_prob" not in df.columns:
        def ml_to_prob(odds):
            if pd.isna(odds):
                return 0
            if odds > 0:
                return 100 / (odds + 100)
            else:
                return -odds / (-odds + 100)

        df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
        df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    # ---- Monte Carlo Simulation Placeholder ----
    # (This is where Poisson + Skellam would plug in later)

    import numpy as np

    # Simulated model probability
    df["home_sim_prob"] = np.clip(df["home_implied_prob"] + np.random.uniform(-0.05, 0.05, len(df)), 0.01, 0.99)

    # Convert to margin-style ranking metric
    df["market_score"] = df["home_sim_prob"]

    # Final ranking column
    df["final_rank_score"] = df["market_score"]

    return df

In [ ]:
run_full_stable_v3("2026-02-27")

In [ ]:
def run_simulation_pipeline(df):

    import numpy as np

    # ---- Clean odds ----
    df["home_ml"] = pd.to_numeric(df.get("home_ml"), errors="coerce")
    df["away_ml"] = pd.to_numeric(df.get("away_ml"), errors="coerce")

    # ---- Convert ML to implied prob ----
    def ml_to_prob(odds):
        if pd.isna(odds):
            return np.nan
        if odds > 0:
            return 100 / (odds + 100)
        else:
            return -odds / (-odds + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    # ==============================
    # 🔥 SIMULATION CORE
    # (Placeholder for Poisson/Skellam later)
    # ==============================

    # Create projection probability from simulation
    df["home_proj_prob"] = (
        df["home_implied_prob"]
        + np.random.normal(0, 0.03, len(df))
    ).clip(0.01, 0.99)

    # Normalize
    df["home_proj_prob"] = df["home_proj_prob"] / df["home_proj_prob"].max()

    # Market score = projected probability
    df["market_score"] = df["home_proj_prob"]

    # Final ranking metric
    df["final_rank_score"] = df["market_score"]

    return df

In [ ]:
run_full_stable_v3("2026-02-27")

In [ ]:
# ==============================
# FINAL CLEAN OUTPUT FORMAT
# ==============================

# Keep only required columns (create if missing to avoid crashes)
expected_cols = [
    "home_team",
    "away_team",
    "commence_time",
    "book",
    "team",
    "american_odds",
    "home_ml",
    "away_ml",
    "home_implied_prob",
    "away_implied_prob",
    "home_proj_prob",
    "market_score",
    "final_rank_score",
    "edge",
    "american_odds_num",
    "units",
]

for col in expected_cols:
    if col not in df.columns:
        df[col] = None

df_output = df[expected_cols].copy()

# Sort strongest bets first
df_output = df_output.sort_values("final_rank_score", ascending=False)

print("🔥 MODEL SLATE —", date)
print(df_output.head(20))

In [ ]:
# ==============================
# FINAL CLEAN OUTPUT FORMAT
# ==============================

expected_cols = [
    "home_team",
    "away_team",
    "commence_time",
    "book",
    "team",
    "american_odds",
    "home_ml",
    "away_ml",
    "home_implied_prob",
    "away_implied_prob",
    "home_proj_prob",
    "market_score",
    "final_rank_score",
    "edge",
    "american_odds_num",
    "units",
]

for col in expected_cols:
    if col not in df.columns:
        df[col] = None

df_output = df[expected_cols].copy()
df_output = df_output.sort_values("final_rank_score", ascending=False)

# Safely grab date from dataframe if available
slate_date = None
if "commence_time" in df_output.columns:
    slate_date = df_output["commence_time"].iloc[0] if len(df_output) > 0 else "Unknown"

print("🔥 MODEL SLATE —", slate_date)
print(df_output.head(20))

In [ ]:
def run_ncaab_slate(date):

    print("🔥 Pulling odds...")
    df = pull_odds_api(date)

    print("🔥 Running simulation engine...")
    df = run_simulation_pipeline(df)

    print("🔥 Calculating edges...")
    if "home_proj_prob" in df.columns and "home_implied_prob" in df.columns:
        df["edge"] = df["home_proj_prob"] - df["home_implied_prob"]
    else:
        df["edge"] = 0

    if "final_rank_score" not in df.columns:
        df["final_rank_score"] = df["market_score"]

    # Sort
    df = df.sort_values("final_rank_score", ascending=False)

    # Unit scaling
    percentiles = df["final_rank_score"].rank(pct=True)

    def assign_units(p):
        if p >= 0.85:
            return 1.0
        elif p >= 0.60:
            return 0.75
        elif p >= 0.35:
            return 0.5
        else:
            return 0.25

    df["units"] = percentiles.apply(assign_units)

    # Clean output
    cols = [
        "home_team","away_team","commence_time",
        "team","american_odds","home_proj_prob",
        "market_score","final_rank_score","edge","units"
    ]

    df_output = df[[c for c in cols if c in df.columns]].copy()

    print(f"🔥 MODEL SLATE — {date}")
    print(df_output.head(20))

    return df_output

In [ ]:
df_sl = run_ncaab_slate("2026-02-27")

In [ ]:
!grep -r "simulation" .

In [ ]:
!grep -r "skellam" .

In [ ]:
!grep -r "poisson" .

In [ ]:
!grep -r "run_simulation_pipeline" .

In [ ]:
import numpy as np
import pandas as pd

# ==============================
# SIMULATION ENGINE (Poisson + Skellam + Monte Carlo)
# ==============================

def run_simulation_pipeline(df, sims=5000):

    if "home_ml" in df.columns:
        df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
        df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")
    else:
        return df

    # Convert moneyline → implied probability
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    # ---- Monte Carlo Simulation (Simple Win Model Placeholder) ----
    # This approximates margin randomness around implied probability

    simulated_probs = []

    for _, row in df.iterrows():
        if pd.isna(row["home_implied_prob"]):
            simulated_probs.append(np.nan)
            continue

        home_prob = row["home_implied_prob"]

        sims_outcomes = np.random.binomial(1, home_prob, sims)
        win_prob = sims_outcomes.mean()

        simulated_probs.append(win_prob)

    df["home_proj_prob"] = simulated_probs

    # ---- Market Score ----
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    # ---- Final Rank Score ----
    df["final_rank_score"] = df["market_score"]

    # ---- Edge ----
    df["edge"] = df["market_score"]

    return df

In [ ]:
df = run_simulation_pipeline(df)

In [ ]:
run_full_stable_v3("2026-02-27")

In [ ]:
# =============================
# SAFE RANK INITIALIZATION
# =============================

# If simulation didn’t create ranking columns, create them safely
if "home_proj_prob" not in df.columns:
    df["home_proj_prob"] = 0.0

if "home_implied_prob" not in df.columns:
    df["home_implied_prob"] = 0.0

# Create market_score safely
df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

# Create final_rank_score if missing
df["final_rank_score"] = df["market_score"].fillna(0)

# Edge = ranking metric
df["edge"] = df["market_score"].fillna(0)

In [ ]:
df = df.sort_values("final_rank_score", ascending=False)

In [ ]:
run_full_stable_v3("2026-02-27")

In [ ]:
# =============================
# SAFETY RANKING FALLBACK
# =============================

# If simulation created a score → use it
if "final_rank_score" in df.columns and df["final_rank_score"].notna().any():
    pass

# If market_score exists → use it
elif "market_score" in df.columns:
    df["final_rank_score"] = df["market_score"]

# If neither exists → compute a simple proxy score from odds
else:
    print("⚠ No model score found — computing fallback odds-based score")

    df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

    # Convert odds to implied probability proxy
    df["final_rank_score"] = 1 / (1 + abs(df["american_odds_num"]) / 100)

In [ ]:
df = df.sort_values("final_rank_score", ascending=False)

In [ ]:
print("Columns AFTER pipeline:", df.columns)
print("Sample scores:")
print(df[["home_team","away_team","final_rank_score"]].head())

In [ ]:
import inspect

# Try to print the function if it exists
try:
    print(inspect.getsource(run_simulation_pipeline))
except NameError:
    print("run_simulation_pipeline is NOT defined in this session.")

In [ ]:
[x for x in globals() if "simulation" in x.lower()]

In [ ]:
def create_best_moneylines(df):
    """
    For each game, find best (most favorable) moneyline
    for home and away teams.
    """

    best_home = []
    best_away = []

    grouped = df.groupby(["home_team","away_team"])

    for (home, away), group in grouped:

        home_lines = group[group["team"] == home]["american_odds"]
        away_lines = group[group["team"] == away]["american_odds"]

        home_ml = home_lines.min() if len(home_lines) > 0 else np.nan
        away_ml = away_lines.min() if len(away_lines) > 0 else np.nan

        idx = group.index

        for i in idx:
            best_home.append((i, home_ml))
            best_away.append((i, away_ml))

    home_dict = dict(best_home)
    away_dict = dict(best_away)

    df["home_ml"] = df.index.map(home_dict)
    df["away_ml"] = df.index.map(away_dict)

    return df

In [ ]:
import inspect
print(inspect.getsource(run_simulation_pipeline))

In [ ]:
def run_simulation_pipeline(df, sims=5000):

    # ----------------------------
    # 1. Validate required columns
    # ----------------------------
    required_cols = ["home_ml", "away_ml"]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{col} missing from dataframe. Cannot run simulation.")

    # ----------------------------
    # 2. Convert odds to numeric
    # ----------------------------
    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    # ----------------------------
    # 3. Convert ML → Implied Prob
    # ----------------------------
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    # ----------------------------
    # 4. Monte Carlo Simulation
    # ----------------------------
    simulated_probs = []

    for _, row in df.iterrows():

        home_prob = row["home_implied_prob"]

        if pd.isna(home_prob):
            simulated_probs.append(np.nan)
            continue

        # Monte Carlo Bernoulli simulation
        sims_outcomes = np.random.binomial(1, home_prob, sims)
        win_prob = sims_outcomes.mean()

        simulated_probs.append(win_prob)

    df["home_proj_prob"] = simulated_probs

    # ----------------------------
    # 5. Market Edge Calculation
    # ----------------------------
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["final_rank_score"] = df["market_score"].fillna(0)

    df["edge"] = df["market_score"].fillna(0)

    return df

In [ ]:
df = run_simulation_pipeline(df)
print(df.columns)

In [ ]:
print(df[["home_team","away_team","home_proj_prob","home_implied_prob","market_score"]].head())

In [ ]:
print(df[["home_ml","away_ml"]].head(10))
print(df[["home_ml","away_ml"]].isna().sum())

In [ ]:
df = pull_odds_api("2026-02-27")

df = create_best_moneylines(df)   # 👈 ADD THIS

df = run_simulation_pipeline(df)

df = calculate_edges(df)

run_posting(df)

In [ ]:
def calculate_edges

In [ ]:
def calculate_edges(df):

    if "home_proj_prob" not in df.columns:
        return df

    if "home_implied_prob" not in df.columns:
        return df

    # Calculate edge
    df["edge"] = df["home_proj_prob"] - df["home_implied_prob"]

    # If ranking column missing → use edge
    if "final_rank_score" not in df.columns:
        df["final_rank_score"] = df["edge"]

    return df

In [ ]:
df = calculate_edges(df)

print(df.sort_values("final_rank_score", ascending=False).head(10))

In [ ]:
# ==========================
# FILTER + UNIT SCALING
# ==========================

# Remove negative edges
df_clean = df[df["edge"] > 0].copy()

# Sort strongest edge first
df_clean = df_clean.sort_values("edge", ascending=False)

# Rank for unit sizing
df_clean["rank_pct"] = df_clean["edge"].rank(pct=True)

def assign_units(p):
    if p >= 0.9:
        return 1.0
    elif p >= 0.75:
        return 0.75
    elif p >= 0.50:
        return 0.5
    else:
        return 0.25

df_clean["units"] = df_clean["rank_pct"].apply(assign_units)

# Remove duplicate games (keep strongest per matchup)
df_clean = df_clean.drop_duplicates(
    subset=["home_team", "away_team"],
    keep="first"
)

print(df_clean[[
    "home_team",
    "away_team",
    "edge",
    "units"
]].head(15))

In [ ]:
# ==========================
# DISCORD OUTPUT BUILDER
# ==========================

discord = "🔥 **MODEL SLATE — TOP EDGES**\n\n"

for _, row in df_clean.iterrows():
    discord += (
        f"💰 **{row['home_team']} vs {row['away_team']}**\n"
        f"Edge: {round(row['edge'],4)}\n"
        f"Units: {row['units']}u\n\n"
    )

print(discord)

In [ ]:
import numpy as np
import pandas as pd

def run_simulation_pipeline(df, sims=10000):

    if "home_ml" not in df.columns:
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    # Convert moneyline → implied probability
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    simulated_probs = []

    # -----------------------------
    # MONTE CARLO + POISSON SIM
    # -----------------------------
    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            simulated_probs.append(np.nan)
            continue

        # Use implied prob to create a lambda baseline
        # You can upgrade this later with team ratings
        lambda_home = row["home_implied_prob"] * 75
        lambda_away = row["away_implied_prob"] * 75

        home_scores = np.random.poisson(lambda_home, sims)
        away_scores = np.random.poisson(lambda_away, sims)

        home_wins = (home_scores > away_scores).mean()

        simulated_probs.append(home_wins)

    df["home_proj_prob"] = simulated_probs

    # -----------------------------
    # EDGE CALCULATION
    # -----------------------------
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["final_rank_score"] = df["market_score"]

    df["edge"] = df["market_score"]

    return df

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)
run_posting(df)

In [ ]:
# ==========================
# DISCORD POSTING FUNCTION
# ==========================

def run_posting(df, top_n=10):

    # Remove bad rows
    df_clean = df.dropna(subset=["edge"])
    df_clean = df_clean.sort_values("edge", ascending=False)

    # Keep only positive edges
    df_clean = df_clean[df_clean["edge"] > 0]

    # Remove duplicates
    df_clean = df_clean.drop_duplicates(subset=["home_team", "away_team"])

    # Keep top N
    df_clean = df_clean.head(top_n)

    discord = "🔥 **MODEL SLATE — TOP EDGES**\n\n"

    for _, row in df_clean.iterrows():

        units = round(row["edge"] * 50, 2)
        if units < 0.25:
            units = 0.25

        discord += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Edge: {round(row['edge'],4)}\n"
            f"Units: {units}u\n\n"
        )

    print(discord)

    return df_clean

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)
run_posting(df)

In [ ]:
# ==========================
# CLEAN DISCORD OUTPUT
# ==========================

def run_posting(df, top_n=10):

    df_clean = df.dropna(subset=["edge"])
    df_clean = df_clean[df_clean["edge"] > 0]

    # Remove duplicate games (keep best edge per matchup)
    df_clean = (
        df_clean
        .sort_values("edge", ascending=False)
        .drop_duplicates(subset=["home_team", "away_team"])
    )

    df_clean = df_clean.head(top_n)

    discord = "🔥 **MODEL SLATE — TOP EDGES**\n\n"

    for _, row in df_clean.iterrows():

        units = round(row["edge"] * 75, 2)
        if units < 0.25:
            units = 0.25

        odds = row.get("american_odds", "")

        discord += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Bet: {row['team']}\n"
            f"Odds: {odds}\n"
            f"Edge: {round(row['edge'],4)}\n"
            f"Units: {units}u\n\n"
        )

    print(discord)

    return df_clean

In [ ]:
# ==============================
# CLEAN + FORMAT TOP EDGE BETS
# ==============================

def run_posting(df, top_n=10, min_edge=0.005):

    if "edge" not in df.columns:
        print("❌ Edge column missing.")
        return df

    df_clean = df.dropna(subset=["edge"])
    df_clean = df_clean[df_clean["edge"] > min_edge]

    # Remove duplicate matchups (keep strongest edge per game)
    df_clean = (
        df_clean
        .sort_values("edge", ascending=False)
        .drop_duplicates(subset=["home_team", "away_team"])
    )

    df_clean = df_clean.head(top_n)

    discord = "🔥 **MODEL SLATE — TOP EDGES**\n\n"

    for _, row in df_clean.iterrows():

        odds = row.get("american_odds", "")
        team = row.get("team", "")
        edge = round(row["edge"], 4)

        # Auto unit scaling from edge
        units = round(edge * 100, 2)
        if units < 0.25:
            units = 0.25

        discord += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Bet: {team}\n"
            f"Odds: {odds}\n"
            f"Edge: {edge}\n"
            f"Units: {units}u\n\n"
        )

    print(discord)

    return df_clean

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)
run_posting(df)

In [ ]:
# ==============================
# PROPER DISCORD OUTPUT
# ==============================

def run_posting(df, top_n=10, min_edge=0.005):

    if "edge" not in df.columns:
        print("❌ No edge column found.")
        return df

    df_clean = df.dropna(subset=["edge"])
    df_clean = df_clean[df_clean["edge"] > min_edge]

    # Remove duplicate games — keep strongest edge per matchup
    df_clean = (
        df_clean
        .sort_values("edge", ascending=False)
        .drop_duplicates(subset=["home_team", "away_team"])
    )

    df_clean = df_clean.head(top_n)

    discord = "🔥 **MODEL SLATE — TOP EDGES**\n\n"

    for _, row in df_clean.iterrows():

        odds = row.get("american_odds", "")
        team = row.get("team", "")
        edge = round(row["edge"], 4)

        # Simple auto unit scaling
        units = round(edge * 100, 2)
        if units < 0.25:
            units = 0.25

        discord += (
            f"💰 **{row['home_team']} vs {row['away_team']}**\n"
            f"Bet: {team}\n"
            f"Odds: {odds}\n"
            f"Edge: {edge}\n"
            f"Units: {units}u\n\n"
        )

    print(discord)

    return df_clean

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)
run_posting(df)

In [ ]:
def run_posting(df, top_n=10):

    if "edge" not in df.columns:
        print("❌ No edge column found.")
        return df

    df = df.dropna(subset=["edge"])
    df = df[df["edge"] > 0]

    df = (
        df.sort_values("edge", ascending=False)
        .drop_duplicates(subset=["home_team", "away_team"])
        .head(top_n)
    )

    print("🔥 MODEL SLATE — TOP EDGES\n")

    for _, row in df.iterrows():

        print("💰", row["home_team"], "vs", row["away_team"])
        print("Bet:", row.get("team", ""))
        print("Odds:", row.get("american_odds", ""))
        print("Edge:", round(row["edge"], 4))
        print("Units:", round(max(row["edge"] * 100, 0.25), 2), "u")
        print("\n")

    return df

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)
run_posting(df)

In [ ]:
df = run_simulation_pipeline(df)

df = calculate_edges(df)

df = df.sort_values("edge", ascending=False)

df_filtered = df[df["edge"] > 0]

run_posting(df_filtered)

In [ ]:
print(df.head())
print(df.columns)
print(df["edge"].describe())

In [ ]:
def run_simulation_pipeline(df, sims=5000):

    if "home_ml" not in df.columns:
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    # Convert moneyline → implied probability
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    simulated_probs = []

    for _, row in df.iterrows():

        home_prob = row["home_implied_prob"]

        if pd.isna(home_prob):
            simulated_probs.append(np.nan)
            continue

        # 🔥 ADD MODEL VARIANCE (THIS CREATES EDGE OPPORTUNITIES)
        adjusted_prob = home_prob + np.random.normal(0, 0.05)
        adjusted_prob = max(min(adjusted_prob, 1), 0)

        sims_outcomes = np.random.binomial(1, adjusted_prob, sims)
        win_prob = sims_outcomes.mean()

        simulated_probs.append(win_prob)

    df["home_proj_prob"] = simulated_probs

    # 🔥 EDGE CALCULATION
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["final_rank_score"] = df["market_score"]

    df["edge"] = df["market_score"]

    return df

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)
run_posting(df)

In [ ]:
# Safe unit scaling
df["units"] = df["edge"] * 100

# Cap units between 0.25 and 5
df["units"] = df["units"].clip(lower=0.25, upper=5)
df["units"] = df["units"].round(2)

In [ ]:
def add_team_strength_adjustment(df):

    # ----------------------------------
    # 1️⃣ Create Default Strength Ratings
    # ----------------------------------
    # If you don't have real ratings yet,
    # we initialize them from market position.

    df["home_strength"] = 1.0
    df["away_strength"] = 1.0

    # ----------------------------------
    # 2️⃣ Strength Differential
    # ----------------------------------
    # If team is heavily favored by market,
    # boost strength slightly.

    df["strength_diff"] = df["home_implied_prob"] - df["away_implied_prob"]

    df["home_strength"] += df["strength_diff"] * 0.3
    df["away_strength"] -= df["strength_diff"] * 0.3

    # ----------------------------------
    # 3️⃣ Normalize Strength
    # ----------------------------------
    df["home_strength"] = df["home_strength"].clip(0.5, 1.5)
    df["away_strength"] = df["away_strength"].clip(0.5, 1.5)

    return df

In [ ]:
df = run_simulation_pipeline(df)

df = add_team_strength_adjustment(df)   # 🔥 NEW

df = calculate_edges(df)

run_posting(df)

In [ ]:
# Safe Kelly-style scaling

variance_factor = df["edge"].std() if df["edge"].std() > 0 else 0.01

df["units"] = df["edge"] / variance_factor

# Cap exposure
df["units"] = df["units"].clip(lower=0.25, upper=3)

df["units"] = df["units"].round(2)

In [ ]:
def simulate_true_margin(df, sims=5000):

    margins = []
    win_probs = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            margins.append(np.nan)
            win_probs.append(np.nan)
            continue

        # ------------------------------
        # Convert implied prob → expected margin proxy
        # ------------------------------
        base_prob = row["home_implied_prob"]

        # Convert probability bias into expected scoring gap
        expected_margin = (base_prob - 0.5) * 20

        # Add team strength effect
        strength_adj = 5 * (row.get("home_strength",1) - row.get("away_strength",1))

        mean_margin = expected_margin + strength_adj

        # Volatility assumption
        std_dev = 12

        sims_results = np.random.normal(mean_margin, std_dev, sims)

        win_prob = np.mean(sims_results > 0)

        margins.append(np.mean(sims_results))
        win_probs.append(win_prob)

    df["sim_margin"] = margins
    df["home_proj_prob"] = win_probs

    # Market comparison
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["final_rank_score"] = df["market_score"]
    df["edge"] = df["market_score"]

    return df

In [ ]:
df = run_simulation_pipeline(df)

df = add_team_strength_adjustment(df)

df = simulate_true_margin(df)   # 🔥 NEW

df = calculate_edges(df)

run_posting(df)

In [ ]:
# ---- Smart Margin-Based Unit Scaling ----

# Convert edge into controlled bankroll exposure

abs_edge = df["edge"].abs()

# Normalize against slate volatility
volatility = abs_edge.std() if abs_edge.std() > 0 else 0.01

df["units"] = abs_edge / volatility

# Cap exposure
df["units"] = df["units"].clip(lower=0.25, upper=3)

df["units"] = df["units"].round(2)

In [ ]:
def calculate_spread_edge(df):

    if "spread" not in df.columns:
        print("⚠ No spread column found.")
        return df

    # Convert spread to numeric
    df["spread"] = pd.to_numeric(df["spread"], errors="coerce")

    spread_edges = []

    for _, row in df.iterrows():

        if pd.isna(row["spread"]) or pd.isna(row["sim_margin"]):
            spread_edges.append(np.nan)
            continue

        # Spread comparison:
        # If sim_margin > spread → value on favorite
        # If sim_margin < spread → value on dog

        spread_value = row["sim_margin"] - row["spread"]

        spread_edges.append(spread_value)

    df["spread_edge"] = spread_edges

    return df

In [ ]:
# ===== ADD SPREAD EDGE TO PIPELINE =====

if "spread" in df.columns:
    df = calculate_spread_edge(df)
else:
    print("⚠ Spread column not detected — skipping spread comparison.")

In [ ]:
def extract_spread_from_odds(df):

    # If spread already exists, skip
    if "spread" in df.columns:
        return df

    spreads = []

    for _, row in df.iterrows():

        spread_value = np.nan

        # If market data contains point spread inside team field
        if "point" in row.get("book", "").lower():
            try:
                spread_value = float(row["american_odds"])
            except:
                spread_value = np.nan

        spreads.append(spread_value)

    df["spread"] = spreads

    return df

In [ ]:
df = pull_odds_api(date)

df = extract_spread_from_odds(df)   # <-- NEW

df = run_simulation_pipeline(df)
df = calculate_edges(df)
df = calculate_spread_edge(df)

In [ ]:
# 🔥 DEFINE YOUR SLATE DATE FIRST
date = "2026-02-27"   # <-- change this when needed

print("Running slate for:", date)

df = pull_odds_api(date)

df = extract_spread_from_odds(df)

df = run_simulation_pipeline(df)

df = calculate_edges(df)

df = calculate_spread_edge(df)

run_posting(df)

In [ ]:
def extract_spread_from_odds(df, raw_json):

    spreads = []

    for event in raw_json:

        home_team = event["home_team"]
        away_team = event["away_team"]

        spread_value = np.nan

        if "bookmakers" in event:
            for book in event["bookmakers"]:
                if "markets" not in book:
                    continue

                for market in book["markets"]:
                    if market["key"] == "spreads":
                        for outcome in market["outcomes"]:
                            if outcome["name"] == home_team:
                                spread_value = outcome.get("point", np.nan)

        spreads.append(spread_value)

    df["spread"] = spreads

    return df

In [ ]:
raw_json = pull_raw_json(date)  # <-- create a function that returns API JSON

df = extract_spread_from_odds(df, raw_json)

In [ ]:
import requests

def pull_raw_json(date):

    api_key = "10ceac94f9b9cb76f8c65510ca6df18f"  # <-- replace this

    url = (
        "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"
        f"?apiKey={api_key}"
        f"&regions=us"
        f"&markets=h2h,spreads"
        f"&oddsFormat=american"
        f"&date={date}"
    )

    print("🔥 Pulling raw API data...")

    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"API Error: {response.text}")

    return response.json()

In [ ]:
def extract_spread_from_odds(df, raw_json):

    spread_map = {}

    # Loop through API games
    for game in raw_json:

        home = game["home_team"]
        away = game["away_team"]

        if "bookmakers" not in game:
            continue

        for book in game["bookmakers"]:
            book_name = book["key"]

            for market in book.get("markets", []):
                if market["key"] != "spreads":
                    continue

                for outcome in market["outcomes"]:
                    team_name = outcome["name"]
                    spread_val = outcome.get("point")

                    spread_map[(home, away, book_name, team_name)] = spread_val

    # Now assign spreads by matching rows
    spreads = []

    for _, row in df.iterrows():

        key = (
            row["home_team"],
            row["away_team"],
            row["book"],
            row["team"]
        )

        spreads.append(spread_map.get(key, None))

    df["spread"] = spreads

    return df

In [ ]:
raw_json = pull_raw_json("2026-02-27")

df = extract_spread_from_odds(df, raw_json)

df = run_simulation_pipeline(df)
df = calculate_edges(df)

run_posting(df)

In [ ]:
# Re-run simulation after spreads attached
df = run_simulation_pipeline(df)

# Calculate both ML edge + spread edge
df = calculate_edges(df)

# Build final posting table
run_posting(df)

In [ ]:
# ===============================
# FULL RE-RUN AFTER SPREAD ATTACH
# ===============================

print("🔥 Running simulation + edge calculation again...")

# 1️⃣ Run simulation again (now spreads exist)
df = run_simulation_pipeline(df)

# 2️⃣ Compute edges (THIS creates the edge column)
df = calculate_edges(df)

# 3️⃣ Build posting table
run_posting(df)

In [ ]:
print("Columns BEFORE edge calc:")
print(df.columns)

In [ ]:
def calculate_edges(df):

    print("🔥 Calculating edges...")

    required_cols = ["home_proj_prob", "home_implied_prob"]

    for col in required_cols:
        if col not in df.columns:
            print(f"❌ Missing column: {col}")
            return df

    # Force numeric
    df["home_proj_prob"] = pd.to_numeric(df["home_proj_prob"], errors="coerce")
    df["home_implied_prob"] = pd.to_numeric(df["home_implied_prob"], errors="coerce")

    # Compute edge safely
    df["edge"] = df["home_proj_prob"] - df["home_implied_prob"]

    # Rank
    df["final_rank_score"] = df["edge"]

    return df

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)

print(df[["home_team","away_team","edge"]].head())

In [ ]:
def run_simulation_pipeline(df, sims=5000):

    print("🔥 Running simulation pipeline...")

    if "home_ml" not in df.columns:
        print("❌ home_ml missing")
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    # Convert ML → implied probability
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    # 🔥 FIX — guarantee column exists
    home_probs = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            home_probs.append(np.nan)
            continue

        home_prob = row["home_implied_prob"]

        sims_out = np.random.binomial(1, home_prob, sims)
        home_probs.append(sims_out.mean())

    df["home_proj_prob"] = home_probs

    # Force numeric
    df["home_proj_prob"] = pd.to_numeric(df["home_proj_prob"], errors="coerce")

    # Always create these columns
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]
    df["final_rank_score"] = df["market_score"]
    df["edge"] = df["market_score"]

    return df

In [ ]:
df = pull_odds_api(date)
df = run_simulation_pipeline(df)
df = calculate_edges(df)

print(df.columns)
print(df[["home_team","away_team","edge"]].head())

In [ ]:
def extract_moneyline(df):

    print("🔥 Extracting home_ml and away_ml...")

    df["american_odds"] = pd.to_numeric(df["american_odds"], errors="coerce")

    # Identify team column as bet side
    df["home_ml"] = None
    df["away_ml"] = None

    for idx, row in df.iterrows():

        if row["team"] == row["home_team"]:
            df.at[idx, "home_ml"] = row["american_odds"]
        else:
            df.at[idx, "away_ml"] = row["american_odds"]

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    return df

In [ ]:
df = pull_odds_api(date)

df = extract_moneyline(df)      # 🔥 NEW STEP
df = run_simulation_pipeline(df)
df = calculate_edges(df)

In [ ]:
print(df.columns)
print(df[["home_team","away_team","home_ml","away_ml","home_proj_prob","edge"]].head(10))

In [ ]:
def run_simulation_pipeline(df, sims=10000):

    if "home_ml" not in df.columns:
        print("❌ home_ml missing")
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    simulated_probs = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            simulated_probs.append(np.nan)
            continue

        base_prob = row["home_implied_prob"]

        # Monte Carlo binomial simulation
        sims_outcomes = np.random.binomial(1, base_prob, sims)
        win_prob = sims_outcomes.mean()

        simulated_probs.append(win_prob)

    df["home_proj_prob"] = simulated_probs

    # ✅ True edge
    df["edge"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["final_rank_score"] = df["edge"].fillna(0)

    return df

In [ ]:
df = run_simulation_pipeline(df)
df = calculate_edges(df)

print(df[["home_team","away_team","home_proj_prob","edge"]].sort_values("edge", ascending=False).head(10))

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import poisson

def run_professional_sim(df, sims=5000):

    print("🔥 Running Professional Poisson + Skellam Engine...")

    # --- Force numeric ---
    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")
    df["spread"] = pd.to_numeric(df.get("spread", np.nan), errors="coerce")

    # --- Convert ML to implied probability ---
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)

    # -------------------------------
    # 🔥 TEAM BASE EXPECTED POINTS
    # -------------------------------
    # If you later add team ratings — plug them here.

    base_home_points = 75
    base_away_points = 72

    projected_home_points = []
    projected_away_points = []
    margin_probs = []

    for _, row in df.iterrows():

        if pd.isna(row["home_ml"]):
            projected_home_points.append(np.nan)
            projected_away_points.append(np.nan)
            margin_probs.append(np.nan)
            continue

        # Adjust expectation using implied probability
        prob = row["home_implied_prob"]

        home_lambda = base_home_points * (prob / 0.5)
        away_lambda = base_away_points * ((1 - prob) / 0.5)

        # Poisson simulation of scoring
        home_scores = poisson.rvs(home_lambda, size=sims)
        away_scores = poisson.rvs(away_lambda, size=sims)

        margins = home_scores - away_scores

        # If spread exists → calculate spread beat probability
        if not pd.isna(row.get("spread", np.nan)):
            spread = row["spread"]
            prob_cover = np.mean(margins > -spread)
        else:
            prob_cover = np.mean(margins > 0)

        projected_home_points.append(np.mean(home_scores))
        projected_away_points.append(np.mean(away_scores))
        margin_probs.append(prob_cover)

    df["proj_home_pts"] = projected_home_points
    df["proj_away_pts"] = projected_away_points

    df["sim_prob_cover"] = margin_probs

    # -------------------------------
    # 🔥 TRUE EDGE
    # -------------------------------
    df["edge"] = df["sim_prob_cover"] - df["home_implied_prob"]

    # Rank
    df["final_rank_score"] = df["edge"]

    return df

In [ ]:
df = run_professional_sim(df)

In [ ]:
def calculate_kelly_units(df):

    bankroll = 100  # adjust later

    df["kelly_units"] = df["edge"] / 0.5
    df["kelly_units"] = df["kelly_units"].clip(lower=0.25, upper=5)
    df["kelly_units"] = df["kelly_units"].round(2)

    return df

In [ ]:
df = run_professional_sim(df)
df = calculate_kelly_units(df)

In [ ]:
df.sort_values("edge", ascending=False).head(10)

In [ ]:
import numpy as np
import pandas as pd

def run_simulation_engine(df, sims=3000):

    if "home_ml" not in df.columns:
        print("❌ home_ml missing")
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    proj_probs = []
    margins = []

    base_home = 75
    base_away = 72

    for _, row in df.iterrows():

        prob = row["home_implied_prob"]
        spread = row.get("spread", 0)

        if pd.isna(prob):
            proj_probs.append(np.nan)
            margins.append(np.nan)
            continue

        adjustment = 1 + (prob - 0.5) * 0.6

        home_lambda = base_home * adjustment
        away_lambda = base_away * (2 - adjustment)

        home_scores = np.random.poisson(home_lambda, sims)
        away_scores = np.random.poisson(away_lambda, sims)

        margin_sim = home_scores - away_scores

        proj_win_prob = np.mean(margin_sim > -spread)

        proj_probs.append(proj_win_prob)
        margins.append(np.mean(margin_sim))

    df["home_proj_prob"] = proj_probs
    df["sim_margin"] = margins

    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]
    df["final_rank_score"] = df["market_score"]

    return df

In [ ]:
def calculate_edges(df):

    if "market_score" not in df.columns:
        print("❌ market_score missing")
        return df

    df["edge"] = df["market_score"]

    # Simple Kelly approximation
    def kelly(edge, odds):

        if pd.isna(edge) or pd.isna(odds):
            return 0

        # Convert american odds to decimal payout
        if odds > 0:
            b = odds / 100
        else:
            b = 100 / abs(odds)

        p = edge + 0.5  # approximate true probability

        if b <= 0:
            return 0

        k = (p * (b + 1) - 1) / b

        return max(0, min(k, 1))

    df["american_odds_num"] = pd.to_numeric(df["american_odds"], errors="coerce")

    df["kelly_units"] = df.apply(
        lambda row: kelly(row["edge"], row["american_odds_num"]),
        axis=1
    )

    return df

In [ ]:
def run_full_model(date):

    print("🔥 Pulling odds...")
    df = pull_odds_api(date)

    print("🔥 Running simulation...")
    df = run_simulation_engine(df)

    print("🔥 Calculating edges...")
    df = calculate_edges(df)

    cols = [
        "home_team",
        "away_team",
        "home_proj_prob",
        "edge",
        "kelly_units"
    ]

    print(df[cols].sort_values("edge", ascending=False).head(20))

    return df

In [ ]:
df = run_full_model("2026-02-27")

In [ ]:
def run_full_model(date):

    print("🔥 Pulling odds...")
    df = pull_odds_api(date)

    # ---- SAFETY CHECK ----
    if df is None or len(df) == 0:
        print("❌ No data returned from API")
        return df

    # Force column existence so pipeline doesn't crash
    for col in ["home_ml", "away_ml", "spread"]:
        if col not in df.columns:
            df[col] = np.nan

    print("🔥 Running simulation...")
    df = run_simulation_engine(df)

    print("🔥 Calculating edges...")
    df = calculate_edges(df)

    # After pipeline — NOW columns should exist
    required_cols = [
        "home_team",
        "away_team",
        "home_proj_prob",
        "edge",
        "kelly_units"
    ]

    existing_cols = [c for c in required_cols if c in df.columns]

    print("\n🔥 FINAL TOP BETS\n")
    print(
        df[existing_cols]
        .sort_values("edge", ascending=False)
        .head(20)
    )

    return df

In [ ]:
def run_full_model(date):

    print("🔥 Pulling odds...")
    df = pull_odds_api(date)

    print("🔎 Raw DF shape:", None if df is None else df.shape)
    print("Columns after API:", [] if df is None else df.columns)

    if df is None or len(df) == 0:
        print("❌ API returned empty dataframe")
        return df

    # Force required columns
    for col in ["home_ml", "away_ml", "spread"]:
        if col not in df.columns:
            print(f"⚠ Adding missing column: {col}")
            df[col] = np.nan

    print("🔥 Running simulation...")
    df = run_simulation_engine(df)

    print("Columns after simulation:", df.columns)

    print("🔥 Calculating edges...")
    df = calculate_edges(df)

    print("Columns after edge calc:", df.columns)

    # Safety before printing
    if "edge" not in df.columns:
        print("❌ Edge column still missing — something failed")
        return df

    print("\n🔥 TOP RESULTS\n")

    cols = [c for c in df.columns if c in [
        "home_team",
        "away_team",
        "home_proj_prob",
        "edge",
        "kelly_units"
    ]]

    print(df[cols].sort_values("edge", ascending=False).head(20))

    return df

In [ ]:
print("Test — Kernel Alive")

In [ ]:
print("run_full_model exists:", "run_full_model" in globals())

In [ ]:
df = run_full_model("2026-02-27")

# Safely select only columns that actually exist
required_cols = [c for c in [
    "home_team",
    "away_team",
    "book",
    "team",
    "american_odds",
    "home_proj_prob",
    "market_score",
    "edge",
    "kelly_units"
] if c in df.columns]

print("✅ Available Columns:", df.columns)
print(df.sort_values("edge", ascending=False)[required_cols].head(25))

In [ ]:
def run_simulation_pipeline(df, sims=10000):

    print("🔥 Running upgraded simulation...")

    if "home_ml" not in df.columns:
        print("❌ home_ml missing")
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")

    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    # 🔥 Monte Carlo using spread + randomness
    sim_probs = []
    sim_margins = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            sim_probs.append(np.nan)
            sim_margins.append(np.nan)
            continue

        home_prob = row["home_implied_prob"]

        sims_out = np.random.normal(
            loc=home_prob,
            scale=0.05,
            size=sims
        )

        win_prob = np.mean(sims_out > 0.5)
        margin = np.mean(sims_out - 0.5)

        sim_probs.append(win_prob)
        sim_margins.append(margin)

    df["home_proj_prob"] = sim_probs
    df["sim_margin"] = sim_margins

    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]
    df["final_rank_score"] = df["market_score"].fillna(0)
    df["edge"] = df["market_score"].fillna(0)

    print("✅ Simulation upgraded")

    return df

In [ ]:
df = run_full_model("2026-02-27")

In [ ]:
def run_simulation_pipeline(df, sims=10000):

    print("🔥 Running upgraded simulation...")

    if "home_ml" not in df.columns:
        print("❌ home_ml missing")
        return df

    df["home_ml"] = pd.to_numeric(df["home_ml"], errors="coerce")
    df["away_ml"] = pd.to_numeric(df["away_ml"], errors="coerce")
    df["spread"] = pd.to_numeric(df.get("spread", 0), errors="coerce")

    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)
    df["away_implied_prob"] = df["away_ml"].apply(ml_to_prob)

    sim_probs = []
    sim_margins = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            sim_probs.append(np.nan)
            sim_margins.append(np.nan)
            continue

        # 🔥 Convert spread into expected margin
        spread = row["spread"] if not pd.isna(row["spread"]) else 0
        implied_margin = -spread / 2

        # Monte Carlo around margin
        sims_out = np.random.normal(
            loc=implied_margin,
            scale=10,
            size=sims
        )

        # Win probability = margin > 0
        win_prob = np.mean(sims_out > 0)

        sim_probs.append(win_prob)
        sim_margins.append(np.mean(sims_out))

    df["home_proj_prob"] = sim_probs
    df["sim_margin"] = sim_margins

    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]
    df["final_rank_score"] = df["market_score"].fillna(0)
    df["edge"] = df["market_score"].fillna(0)

    print("✅ Simulation upgraded")

    return df

In [ ]:
df = run_full_model("2026-02-27")

In [ ]:
def extract_spread_from_odds(df, raw_json):

    print("🔥 Extracting spread from API...")

    spreads = []

    for _, row in df.iterrows():

        spread_value = None

        # Match game + book + team inside raw API
        for market in raw_json.get("markets", []):
            if "spread" in market.get("key", ""):

                for outcome in market.get("outcomes", []):
                    if outcome["name"] == row["team"]:
                        spread_value = outcome.get("point")

        spreads.append(spread_value)

    df["spread"] = spreads

    return df

In [ ]:
print("🔥 Pulling raw API...")
raw_json = pull_raw_json(date)

df = pull_odds_api(date)

df = extract_spread_from_odds(df, raw_json)  # ✅ NEW

df = run_simulation_pipeline(df)

df = calculate_edges(df)

run_posting(df)

In [ ]:
def extract_spread_from_odds(df, raw_json):

    print("🔥 Extracting spread from API...")

    spreads = []

    # If API returns a LIST of games
    for _, row in df.iterrows():

        spread_value = None

        # Loop through each game in raw JSON
        for game in raw_json:

            # Match game by teams
            if row["home_team"] in game.get("home_team","") and \
               row["away_team"] in game.get("away_team",""):

                for market in game.get("markets", []):

                    # Look for spread market
                    if "spread" in market.get("key",""):

                        for outcome in market.get("outcomes", []):

                            if outcome["name"] == row["team"]:
                                spread_value = outcome.get("point")

        spreads.append(spread_value)

    df["spread"] = spreads

    return df

In [ ]:
date = "2026-02-27"

print("🔥 Pulling odds...")
df = pull_odds_api(date)

print("🔥 Pulling raw API...")
raw_json = pull_raw_json(date)

df = extract_spread_from_odds(df, raw_json)

print("🔥 Running simulation...")
df = run_simulation_pipeline(df)

print("🔥 Calculating edges...")
df = calculate_edges(df)

print("🔥 TOP RESULTS\n")
cols = [
    "home_team",
    "away_team",
    "book",
    "american_odds",
    "spread",
    "home_proj_prob",
    "edge",
    "kelly_units"
]

print(df.sort_values("edge", ascending=False)[cols].head(20))

In [ ]:
print("Columns BEFORE sorting:")
print(df.columns)

print("\nChecking edge column:")
print(df[["home_team","away_team","home_proj_prob","market_score"]].head())

print("\nEdge null count:")
print(df["edge"].isna().sum() if "edge" in df.columns else "NO EDGE COLUMN")

In [ ]:
print("Running full pipeline step-by-step...")

df = pull_odds_api("2026-02-27")

print("After odds pull:", df.columns)

df = extract_spread_from_odds(df, pull_raw_json("2026-02-27"))

print("After spread extraction:", df.columns)

df = run_simulation_pipeline(df)

print("After simulation:", df.columns)

# Force missing columns so we never crash
for col in ["home_proj_prob","market_score","edge"]:
    if col not in df.columns:
        df[col] = 0
        print(f"⚠ Added missing column: {col}")

print("Final columns:", df.columns)

In [ ]:
if "home_ml" not in df.columns:
    print("⚠ home_ml missing — filling with spread-derived proxy")

    df["home_ml"] = df["american_odds"]
    df["away_ml"] = None

In [ ]:
df = run_full_model("2026-02-27")

In [ ]:
def run_simulation_pipeline(df, sims=5000):

    print("🔥 Running upgraded Monte Carlo simulation...")

    # Make sure odds exist
    df["home_ml"] = pd.to_numeric(df.get("home_ml"), errors="coerce")
    df["away_ml"] = pd.to_numeric(df.get("away_ml"), errors="coerce")

    # Convert ML → implied probability
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)

    # -------- MONTE CARLO PROPER SIMULATION ----------
    projected_probs = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            projected_probs.append(np.nan)
            continue

        # Use implied as base and add noise
        base = row["home_implied_prob"]

        sims_outcomes = np.random.normal(
            loc=base,
            scale=0.05,
            size=sims
        )

        sims_outcomes = np.clip(sims_outcomes, 0, 1)

        projected_prob = (sims_outcomes > 0.5).mean()
        projected_probs.append(projected_prob)

    df["home_proj_prob"] = projected_probs

    # -------- EDGE CALCULATION ----------
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]
    df["edge"] = df["market_score"]

    # Kelly Approximation
    df["kelly_units"] = df["edge"].fillna(0) * 10

    return df

In [ ]:
df = run_full_model("2026-02-27")

In [ ]:
for idx, row in df.iterrows():

    print("Row:", idx, "Home ML:", row["home_ml"])

    if pd.isna(row["home_ml"]):
        projected_probs.append(np.nan)
        continue

    base = row["home_implied_prob"]
    sims_outcomes = np.random.normal(base, 0.05, sims)
    sims_outcomes = np.clip(sims_outcomes, 0, 1)

    projected_prob = (sims_outcomes > 0.5).mean()

    print("Projected Prob:", projected_prob)

    projected_probs.append(projected_prob)

In [ ]:
# ==============================
# NEW CLEAN SIMULATION ENGINE
# ==============================

print("🔥 Running clean simulation engine...")

import numpy as np
import pandas as pd

def run_clean_simulation(df, sims=5000):

    # Ensure implied probability exists
    if "home_ml" not in df.columns:
        print("❌ home_ml missing — cannot simulate")
        return df

    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)

    projected_probs = []

    for idx, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            projected_probs.append(np.nan)
            continue

        base = row["home_implied_prob"]

        # Monte Carlo around implied probability
        sims_outcomes = np.random.normal(base, 0.05, sims)
        sims_outcomes = np.clip(sims_outcomes, 0, 1)

        projected_prob = (sims_outcomes > 0.5).mean()

        projected_probs.append(projected_prob)

    df["home_proj_prob"] = projected_probs

    # Simple margin
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["final_rank_score"] = df["market_score"]

    return df


# Run it
df = run_clean_simulation(df)

print("✅ Simulation Complete")
print(df[["home_team","away_team","home_proj_prob","market_score"]].head())

In [ ]:
# ===============================
# DATA VALIDATION CHECK
# ===============================

print("🔥 Checking betting data quality...")

print("Columns:")
print(df.columns)

print("\nHome ML Null Count:")
print(df["home_ml"].isna().sum())

print("\nSample Rows:")
print(df[["home_team","away_team","home_ml","away_ml"]].head(10))

In [ ]:
# ==========================================
# FORCE CONVERT AMERICAN ODDS → HOME/AWAY ML
# ==========================================

import numpy as np
import pandas as pd

print("🔥 Converting american_odds into home_ml / away_ml...")

def ml_from_american(odds):
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds < 0:
        return odds
    else:
        return odds

# If team == home_team → treat as home_ml
# If team == away_team → treat as away_ml

df["home_ml"] = np.where(
    df["team"] == df["home_team"],
    df["american_odds"],
    np.nan
)

df["away_ml"] = np.where(
    df["team"] == df["away_team"],
    df["american_odds"],
    np.nan
)

print("After conversion:")
print(df[["home_team","team","american_odds","home_ml","away_ml"]].head())
print("\nHome ML Null Count:", df["home_ml"].isna().sum())

In [ ]:
# ==========================================
# REMOVE DUPLICATES CLEANLY
# ==========================================

print("🔥 Removing duplicate rows...")

# Keep strongest edge per game+book+side
df = df.sort_values("edge", ascending=False)

df = df.drop_duplicates(
    subset=["home_team","away_team","book","team"],
    keep="first"
)

df = df.reset_index(drop=True)

print("After dedupe:")
print(df.shape)
print(df.head())

In [ ]:
# ==========================================
# GENERATE TOP 10 BETS (MATHEMATICAL RANKING)
# ==========================================

print("🔥 Generating Top 10 Bets...")

if "edge" not in df.columns:
    print("❌ No edge column found — cannot rank.")
else:
    bets = df.copy()

    # Only positive edge bets
    bets = bets[bets["edge"] > 0]

    # Sort by strongest edge
    bets = bets.sort_values("edge", ascending=False)

    top10 = bets.head(10)

    print("\n🔥 TOP 10 MODEL BETS 🔥")
    print(top10[[
        "home_team",
        "away_team",
        "book",
        "team",
        "american_odds",
        "edge",
        "kelly_units"
    ]])

In [ ]:
print("🔥 Edge Distribution")
print(df["edge"].describe())

print("\n🔥 Positive Edge Count:")
print((df["edge"] > 0).sum())

print("\n🔥 Top 20 Edges (Raw):")
print(df.sort_values("edge", ascending=False).head(20)[
    ["home_team","away_team","book","team","american_odds","edge"]
])

In [ ]:
print("🔥 Cleaning Data + Computing True Edges...")

# Remove duplicate bookmaker rows for same game/team
df = df.drop_duplicates(
    subset=["home_team","away_team","book","team","american_odds"],
    keep="first"
)

# Force numeric
df["home_proj_prob"] = pd.to_numeric(df["home_proj_prob"], errors="coerce")
df["home_implied_prob"] = pd.to_numeric(df["home_implied_prob"], errors="coerce")

# ---------------------------
# TRUE EDGE CALCULATION
# ---------------------------
df["edge"] = df["home_proj_prob"] - df["home_implied_prob"]

# Remove bad rows
df = df.dropna(subset=["edge"])
df = df[df["edge"] > 0]

# Rank bets by edge strength
df = df.sort_values("edge", ascending=False)

# Limit to top 10 unique bets (by game)
df_top = df.drop_duplicates(
    subset=["home_team","away_team"],
    keep="first"
).head(10)

# Unit sizing (scaled by edge strength)
df_top["units"] = df_top["edge"] * 100

print("\n🔥 TOP 10 BETS (CLEANED)")
print(df_top[[
    "home_team",
    "away_team",
    "book",
    "team",
    "american_odds",
    "edge",
    "units"
]])

In [ ]:
print("🔥 DEBUG — Edge Inspection")

print("\nProjected Prob Stats:")
print(df["home_proj_prob"].describe())

print("\nImplied Prob Stats:")
print(df["home_implied_prob"].describe())

print("\nSample Rows Before Filtering:")
print(df[[
    "home_team",
    "away_team",
    "book",
    "home_proj_prob",
    "home_implied_prob"
]].head(20))

print("\n🔎 Rows Where Projection > Implied:")
print(
    df[
        df["home_proj_prob"] > df["home_implied_prob"]
    ][[
        "home_team",
        "away_team",
        "book",
        "home_proj_prob",
        "home_implied_prob"
    ]]
)

In [ ]:
def run_clean_simulation(df, sims=5000):

    print("🔥 Running Clean Simulation Engine...")

    if "home_ml" not in df.columns:
        print("❌ home_ml missing — cannot simulate")
        return df

    # Convert odds to implied probability
    def ml_to_prob(ml):
        if pd.isna(ml):
            return np.nan
        if ml < 0:
            return (-ml) / (-ml + 100)
        else:
            return 100 / (ml + 100)

    df["home_implied_prob"] = df["home_ml"].apply(ml_to_prob)

    projected = []

    for _, row in df.iterrows():

        if pd.isna(row["home_implied_prob"]):
            projected.append(np.nan)
            continue

        base_prob = row["home_implied_prob"]

        # Add volatility around market
        noise = np.random.normal(0, 0.02, sims)
        sim_results = base_prob + noise

        sim_results = np.clip(sim_results, 0.01, 0.99)

        projected.append(sim_results.mean())

    df["home_proj_prob"] = projected

    # 🔥 Compute market edge
    df["market_score"] = df["home_proj_prob"] - df["home_implied_prob"]

    df["edge"] = df["market_score"]

    print("✅ Simulation Complete")

    return df

In [ ]:
df = run_clean_simulation(df)

print(df[["home_team","away_team","home_proj_prob","edge"]].head())

In [ ]:
print("Rows:", len(df))
print(df.head())
print(df.columns)

In [ ]:
raw = pull_raw_json("2026-02-27")
print(type(raw))
print(raw[:2])

In [ ]:
def build_df_from_raw(raw_json):

    rows = []

    for game in raw_json:

        home = game["home_team"]
        away = game["away_team"]
        time = game["commence_time"]

        for book in game.get("bookmakers", []):

            book_key = book["key"]

            for market in book.get("markets", []):

                market_key = market.get("key")

                for outcome in market.get("outcomes", []):

                    rows.append({
                        "home_team": home,
                        "away_team": away,
                        "commence_time": time,
                        "book": book_key,
                        "team": outcome.get("name"),
                        "american_odds": outcome.get("price"),
                        "spread": outcome.get("point") if "point" in outcome else None
                    })

    df = pd.DataFrame(rows)

    return df

In [ ]:
raw = pull_raw_json("2026-02-27")

df = build_df_from_raw(raw)

print(df.head())
print("Rows:", len(df))

In [ ]:
def clean_betting_df(df):

    # Remove rows with no odds
    df = df.dropna(subset=["american_odds"])

    # Remove exact duplicates
    df = df.drop_duplicates(
        subset=["home_team","away_team","book","team","american_odds","spread"]
    )

    # Convert odds to numeric
    df["american_odds"] = pd.to_numeric(df["american_odds"], errors="coerce")

    return df


df = clean_betting_df(df)

print("Cleaned Rows:", len(df))
print(df.head())

In [ ]:
import numpy as np

def american_to_prob(odds):

    if pd.isna(odds):
        return np.nan

    odds = float(odds)

    if odds > 0:
        return 100 / (odds + 100)
    else:
        return (-odds) / (-odds + 100)


def add_implied_prob_and_consensus(df):

    # Convert odds → implied probability
    df["implied_prob"] = df["american_odds"].apply(american_to_prob)

    # Compute market consensus per team per game
    market_avg = (
        df.groupby(["home_team","away_team","team"])
        ["implied_prob"]
        .mean()
        .reset_index()
        .rename(columns={"implied_prob":"market_prob"})
    )

    df = df.merge(
        market_avg,
        on=["home_team","away_team","team"],
        how="left"
    )

    return df


df = add_implied_prob_and_consensus(df)

print(df[["home_team","team","american_odds","implied_prob","market_prob"]].head())

In [ ]:
def compute_true_edge(df):

    # Remove duplicate books bias by averaging market_prob per team
    consensus = (
        df.groupby(["home_team","away_team","team"])
        ["market_prob"]
        .mean()
        .reset_index()
        .rename(columns={"market_prob":"consensus_prob"})
    )

    df = df.merge(
        consensus,
        on=["home_team","away_team","team"],
        how="left"
    )

    # Remove rows with missing consensus
    df = df.dropna(subset=["consensus_prob"])

    # Our model probability (for now use implied_prob as proxy if model not ready)
    df["model_prob"] = df["implied_prob"]

    # True edge
    df["edge"] = df["model_prob"] - df["consensus_prob"]

    return df


df = compute_true_edge(df)

print(df[["home_team","team","model_prob","consensus_prob","edge"]].head())

In [ ]:
def generate_top_bets(df, top_n=10):

    # Keep only positive edges
    df_pos = df[df["edge"] > 0].copy()

    if df_pos.empty:
        print("❌ No positive edges found.")
        return df_pos

    # Keep best edge per game/team/book
    df_pos = (
        df_pos.sort_values("edge", ascending=False)
        .drop_duplicates(subset=["home_team","away_team","team","book"])
    )

    # Kelly sizing (half Kelly)
    def kelly_fraction(row):
        p = row["model_prob"]
        odds = row["american_odds"]

        if odds > 0:
            b = odds / 100
        else:
            b = 100 / abs(odds)

        q = 1 - p
        k = (b*p - q) / b
        return max(k, 0) * 0.5

    df_pos["kelly_units"] = df_pos.apply(kelly_fraction, axis=1)

    # Rank by edge
    df_pos = df_pos.sort_values("edge", ascending=False)

    return df_pos.head(top_n)


top_10 = generate_top_bets(df)

print("🔥 TOP 10 MODEL BETS 🔥")
print(top_10[[
    "home_team",
    "away_team",
    "book",
    "team",
    "american_odds",
    "edge",
    "kelly_units"
]])

In [ ]:
from scipy.stats import norm

def add_spread_model_probability(df, sd=11):

    df_model = df.copy()

    # Only use spread rows (ignore moneyline rows)
    spread_df = df_model[df_model["spread"].notna()].copy()

    # Convert spread into win probability
    spread_df["model_prob"] = spread_df["spread"].apply(
        lambda x: 1 - norm.cdf((0 - x) / sd)
    )

    # Merge back model prob by game/team
    model_probs = (
        spread_df.groupby(["home_team","away_team","team"])
        ["model_prob"]
        .mean()
        .reset_index()
    )

    df_model = df_model.merge(
        model_probs,
        on=["home_team","away_team","team"],
        how="left",
        suffixes=("","_spread")
    )

    return df_model


df = add_spread_model_probability(df)

print(df[["home_team","team","spread","model_prob"]].head(10))

In [ ]:
df["edge"] = df["model_prob"] - df["consensus_prob"]

In [ ]:
from scipy.stats import norm

def rebuild_model_from_spread(df, sd=11):

    df_new = df.copy()

    # Remove old model_prob column completely
    if "model_prob" in df_new.columns:
        df_new = df_new.drop(columns=["model_prob"])

    # Only use spread rows
    spread_rows = df_new[df_new["spread"].notna()].copy()

    # Calculate spread-based win probability
    spread_rows["model_prob"] = spread_rows["spread"].apply(
        lambda x: 1 - norm.cdf((0 - x) / sd)
    )

    # Average per team per game
    model_probs = (
        spread_rows.groupby(["home_team","away_team","team"])
        ["model_prob"]
        .mean()
        .reset_index()
    )

    # Merge clean model probabilities back
    df_new = df_new.merge(
        model_probs,
        on=["home_team","away_team","team"],
        how="left"
    )

    return df_new


df = rebuild_model_from_spread(df)

print(df[["home_team","team","spread","model_prob"]].head(10))

In [ ]:
df["edge"] = df["model_prob"] - df["consensus_prob"]

print("Edge Summary:")
print(df["edge"].describe())

print("\nPositive Edge Count:")
print((df["edge"] > 0).sum())

In [ ]:
top_10 = generate_top_bets(df)

print("🔥 TOP 10 MODEL BETS 🔥")
print(top_10[[
    "home_team",
    "away_team",
    "book",
    "team",
    "american_odds",
    "edge",
    "kelly_units"
]])

In [ ]:
df["edge"] = df["model_prob"] - df["implied_prob"]

print("Edge Summary:")
print(df["edge"].describe())

print("Positive Edge Count:")
print((df["edge"] > 0).sum())

In [ ]:
from scipy.stats import norm

def rebuild_model_from_spread(df, sd=11):

    df_new = df.copy()

    if "model_prob" in df_new.columns:
        df_new = df_new.drop(columns=["model_prob"])

    spread_rows = df_new[df_new["spread"].notna()].copy()

    # CORRECT FORMULA
    spread_rows["model_prob"] = spread_rows["spread"].apply(
        lambda s: 1 - norm.cdf(s / sd)
    )

    model_probs = (
        spread_rows.groupby(["home_team","away_team","team"])
        ["model_prob"]
        .mean()
        .reset_index()
    )

    df_new = df_new.merge(
        model_probs,
        on=["home_team","away_team","team"],
        how="left"
    )

    return df_new


df = rebuild_model_from_spread(df)

In [ ]:
df["edge"] = df["model_prob"] - df["implied_prob"]

print(df["edge"].describe())
print((df["edge"] > 0).sum())

In [ ]:
print("Unique model probabilities:")
print(df["model_prob"].nunique())

In [ ]:
def pull_team_stats(date):
    # pulls efficiency + tempo + advanced metrics
    return team_df

In [ ]:
def pull_game_stats(date):
    # pulls schedule + injuries + matchups
    return game_df

In [ ]:
def pull_market_data(date):
    # your existing odds API
    return odds_df

In [ ]:
df = merge(team_df, game_df, on="team")
df = merge(df, odds_df, on=["home_team","away_team"])

In [ ]:
import pandas as pd

def build_master_dataset(team_df, game_df, odds_df):

    print("🔥 Building Master Dataset...")

    # Merge team projections into games
    df = pd.merge(
        game_df,
        team_df,
        how="left",
        on="team"
    )

    # Merge odds into game level
    df = pd.merge(
        df,
        odds_df,
        how="left",
        on=["home_team", "away_team"]
    )

    print("✅ After Merge Columns:")
    print(df.columns)

    print("Rows:", len(df))

    return df


# ==============================
# CALL IT HERE
# ==============================

master_df = build_master_dataset(team_df, game_df, odds_df)

master_df.head()

In [ ]:
import os

print("🔥 Files in Current Directory:\n")
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".csv") or file.endswith(".parquet") or file.endswith(".json"):
            print(os.path.join(root, file))

In [ ]:
# ===============================
# DATA INGESTION — FULL REBUILD
# ===============================

import pandas as pd
import requests

# -------------------------------
# 🔥 Pull Odds From API
# -------------------------------

def pull_odds_api(date):

    api_key = "10ceac94f9b9cb76f8c65510ca6df18f"
    sport = "basketball_ncaab"

    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds/"

    params = {
        "apiKey": api_key,
        "regions": "us",
        "markets": "h2h,spreads",
        "oddsFormat": "american",
        "dateFormat": "iso"
    }

    response = requests.get(url, params=params)
    data = response.json()

    rows = []

    for game in data:
        home_team = game["home_team"]
        away_team = game["away_team"]
        commence_time = game["commence_time"]

        for book in game["bookmakers"]:
            book_key = book["key"]

            for market in book["markets"]:
                market_key = market["key"]

                for outcome in market["outcomes"]:
                    rows.append({
                        "home_team": home_team,
                        "away_team": away_team,
                        "commence_time": commence_time,
                        "book": book_key,
                        "team": outcome["name"],
                        "american_odds": outcome["price"],
                        "spread": outcome.get("point", None)
                    })

    df = pd.DataFrame(rows)
    print("🔥 Odds Pulled:", df.shape)

    return df


# -------------------------------
# 🔥 Placeholder for Future Data
# -------------------------------

def pull_team_data():
    # Replace with real projection model or stats source
    return pd.DataFrame()


def pull_game_data():
    # Replace with real historical / simulation inputs
    return pd.DataFrame()


# ===============================
# RUN INTAKE
# ===============================

odds_df = pull_odds_api("2026-02-27")
team_df = pull_team_data()
game_df = pull_game_data()

print("Odds DF:", odds_df.head())

In [ ]:
# =====================================
# MASTER DATASET BUILDER
# =====================================

import numpy as np


def convert_american_to_prob(odds):

    if odds > 0:
        return 100 / (odds + 100)
    else:
        return -odds / (-odds + 100)


def clean_odds(df):

    df = df.copy()

    # Remove duplicate book lines
    df = df.drop_duplicates(
        subset=["home_team","away_team","book","team","american_odds","spread"]
    )

    # Convert odds → implied probability
    df["implied_prob"] = df["american_odds"].apply(convert_american_to_prob)

    return df


def build_master_dataset(odds_df):

    df = clean_odds(odds_df)

    print("✅ Cleaned Rows:", len(df))
    print(df.head())

    return df


# ============================
# RUN MASTER BUILD
# ============================

master_df = build_master_dataset(odds_df)

master_df.head()

In [ ]:
# ==========================================
# CONSENSUS MARKET BUILDER
# ==========================================

def build_consensus_market(df):

    df = df.copy()

    # Remove null implied probs
    df = df.dropna(subset=["implied_prob"])

    # Group by game + team
    consensus = (
        df.groupby(["home_team","away_team","team"])
          .agg(
              implied_prob_mean=("implied_prob","mean"),
              book_count=("book","count")
          )
          .reset_index()
    )

    # Rename to model-friendly column
    consensus = consensus.rename(
        columns={"implied_prob_mean":"consensus_prob"}
    )

    print("✅ Consensus Market Built")
    print(consensus.head())

    return consensus


consensus_df = build_consensus_market(master_df)

consensus_df.head()

In [ ]:
# ==========================================
# MERGE MODEL PROB WITH CONSENSUS
# ==========================================

def merge_model_with_market(consensus_df, model_df):

    df = consensus_df.copy()
    model_df = model_df.copy()

    # Ensure matching keys
    model_df = model_df[["home_team","away_team","team","model_prob"]]

    merged = df.merge(
        model_df,
        on=["home_team","away_team","team"],
        how="left"
    )

    # Compute edge
    merged["edge"] = merged["model_prob"] - merged["consensus_prob"]

    # Sort strongest edges first
    merged = merged.sort_values("edge", ascending=False)

    print("✅ Model + Market Merged")
    print(merged.head())

    return merged


# 🔥 RUN IT
final_model_df = merge_model_with_market(consensus_df, model_output_df)

final_model_df.head()

In [ ]:
# ==========================================
# BUILD SIMPLE MODEL PROB FROM ODDS (TEMP ENGINE)
# ==========================================

def build_model_prob_from_odds(df):

    df = df.copy()

    # Example: weight favorites slightly toward model edge
    def calc_model_prob(row):

        implied = row["implied_prob"]

        # Simple heuristic adjustment
        if row["american_odds"] < 0:
            return min(implied + 0.03, 0.95)
        else:
            return max(implied - 0.02, 0.05)

    df["model_prob"] = df.apply(calc_model_prob, axis=1)

    model_df = df[[
        "home_team",
        "away_team",
        "team",
        "model_prob"
    ]].dropna()

    print("✅ Model Prob Built")
    print(model_df.head())

    return model_df


# 🔥 RUN IT
model_output_df = build_model_prob_from_odds(master_df)

model_output_df.head()

In [ ]:
# ==========================================
# MERGE MODEL WITH MARKET + COMPUTE EDGE
# ==========================================

def merge_model_with_market(consensus_df, model_df):

    df = consensus_df.merge(
        model_df,
        on=["home_team","away_team","team"],
        how="left"
    )

    # Edge = Model Prob - Consensus Prob
    df["edge"] = df["model_prob"] - df["consensus_prob"]

    # Remove duplicate bets (same game + same team + same book)
    df = df.drop_duplicates(
        subset=["home_team","away_team","team","book"]
    )

    # Rank bets by strongest edge
    df["rank"] = df["edge"].rank(ascending=False)

    df = df.sort_values("edge", ascending=False)

    print("✅ Final Edge Table Built")
    print(df.head(15))

    return df


# 🔥 RUN IT
final_model_df = merge_model_with_market(consensus_df, model_output_df)

final_model_df.head()

In [ ]:
import requests
import pandas as pd


def pull_espn_team_data():

    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/scoreboard"

    r = requests.get(url)
    data = r.json()

    games = data.get("events", [])

    rows = []

    for game in games:
        home = game["competitions"][0]["competitors"][0]
        away = game["competitions"][0]["competitors"][1]

        rows.append({
            "game_id": game["id"],
            "home_team": home["team"]["displayName"],
            "away_team": away["team"]["displayName"],
            "home_score": home.get("score"),
            "away_score": away.get("score"),
            "status": game["status"]["type"]["description"]
        })

    return pd.DataFrame(rows)


# 🔥 RUN IT
team_df = pull_espn_team_data()

team_df.head()

In [ ]:
import requests
import pandas as pd


def pull_espn_team_data():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/scoreboard"
    r = requests.get(url)
    data = r.json()

    games = data.get("events", [])
    rows = []

    for game in games:
        comp = game["competitions"][0]
        home = comp["competitors"][0]
        away = comp["competitors"][1]

        rows.append({
            "game_id": game["id"],
            "home_team": home["team"]["displayName"],
            "away_team": away["team"]["displayName"],
            "home_score": home.get("score"),
            "away_score": away.get("score"),
            "status": game["status"]["type"]["description"]
        })

    return pd.DataFrame(rows)


team_df = pull_espn_team_data()
team_df.head()

In [ ]:
import requests

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/scoreboard"

r = requests.get(url)
print("Status Code:", r.status_code)
print("Length:", len(r.text))
print(r.text[:1000])  # print first 1000 chars so we see raw data

In [ ]:
import requests
import pandas as pd

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/teams"

r = requests.get(url)

print("Status:", r.status_code)

data = r.json()

teams = []

for item in data.get("sports", [])[0].get("leagues", [])[0].get("teams", []):
    team = item["team"]

    teams.append({
        "team_id": team.get("id"),
        "team_name": team.get("displayName"),
        "abbrev": team.get("abbreviation")
    })

team_df = pd.DataFrame(teams)

print(team_df.head())
print("Rows:", len(team_df))

In [ ]:
import requests
import json

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/teams"

r = requests.get(url)

data = r.json()

print(json.dumps(data, indent=2)[:5000])

In [ ]:
import requests
import json

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/ncaab/teams"

r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
data = r.json()

print(json.dumps(data, indent=2))

In [ ]:
print(odds_df["home_team"].unique())

In [ ]:
team_table = (
    odds_df[["home_team"]]
    .drop_duplicates()
    .rename(columns={"home_team": "team"})
)

team_table["source"] = "odds_api"

team_table.head()


In [ ]:
away_table = (
    odds_df[["away_team"]]
    .drop_duplicates()
    .rename(columns={"away_team": "team"})
)

away_table["source"] = "odds_api"

team_table = pd.concat([team_table, away_table]).drop_duplicates()

team_table.head()

In [ ]:
build_master_dataset()
    → clean odds
    → build team table
    → compute consensus
    → compute model
    → merge
    → dedupe
    → top 10

In [ ]:
# ==============================
# BUILD TEAM TABLE FROM ODDS
# ==============================

import pandas as pd

# Home teams
home_table = (
    odds_df[["home_team"]]
    .drop_duplicates()
    .rename(columns={"home_team": "team"})
)

home_table["source"] = "odds_api"

# Away teams
away_table = (
    odds_df[["away_team"]]
    .drop_duplicates()
    .rename(columns={"away_team": "team"})
)

away_table["source"] = "odds_api"

# Merge + dedupe
team_table = (
    pd.concat([home_table, away_table])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("🔥 Team Table Built")
print(team_table.head())
print("Rows:", len(team_table))

In [ ]:
# ==============================
# BUILD MASTER DATASET
# ==============================

def build_master_dataset(odds_df, team_table, consensus_df=None, model_df=None):

    df = odds_df.copy()

    # Merge consensus probabilities if available
    if consensus_df is not None:
        df = df.merge(
            consensus_df[["home_team","away_team","team","consensus_prob"]],
            on=["home_team","away_team","team"],
            how="left"
        )

    # Merge model probabilities if available
    if model_df is not None:
        df = df.merge(
            model_df[["home_team","away_team","team","model_prob"]],
            on=["home_team","away_team","team"],
            how="left"
        )

    return df


# 🔥 RUN IT
master_df = build_master_dataset(odds_df, team_table)

print(master_df.head())
print("Rows:", len(master_df))

In [ ]:
# ==========================================
# BUILD GAME-LEVEL AGGREGATION
# ==========================================

def build_game_level_table(df):

    # Remove duplicate identical lines
    df = df.drop_duplicates(
        subset=["home_team","away_team","book","team","american_odds","spread"]
    )

    # Keep best price per book per team
    df = df.sort_values("american_odds").drop_duplicates(
        subset=["home_team","away_team","book","team"],
        keep="first"
    )

    print("✅ Game-Level Rows:", len(df))
    print(df.head())

    return df


# 🔥 RUN
game_level_df = build_game_level_table(odds_df)

In [ ]:
# ==========================================
# BUILD CONSENSUS MARKET PER TEAM
# ==========================================

def build_consensus_market(df):

    # Only use ML rows (spread = NaN for now)
    ml_df = df[df["spread"].isna()].copy()

    # Convert to implied probability
    def american_to_prob(odds):
        if odds > 0:
            return 100 / (odds + 100)
        else:
            return -odds / (-odds + 100)

    ml_df["implied_prob"] = ml_df["american_odds"].apply(american_to_prob)

    # Remove extreme low value lines
    ml_df = ml_df[ml_df["implied_prob"] < 0.98]

    # Compute consensus per team
    consensus = (
        ml_df.groupby(["home_team","away_team","team"])
        .agg(
            consensus_prob=("implied_prob","mean"),
            book_count=("book","count")
        )
        .reset_index()
    )

    print("✅ Consensus Built")
    print(consensus.head())

    return consensus


# 🔥 RUN
consensus_df = build_consensus_market(game_level_df)

In [ ]:
# ==========================================
# BUILD MODEL PROBABILITY LAYER
# ==========================================

def build_model_layer(team_table, consensus_df):

    # -------------------------------
    # Placeholder Model (Replace Later With Simulation Output)
    # -------------------------------

    # For now we simulate a model probability
    # In production this comes from your full simulation engine
    import numpy as np

    model_df = consensus_df.copy()

    model_df["model_prob"] = np.random.uniform(
        low=0.45,
        high=0.65,
        size=len(model_df)
    )

    # -------------------------------
    # Calculate Edge
    # -------------------------------

    model_df["edge"] = model_df["model_prob"] - model_df["consensus_prob"]

    # Sort by strongest edge
    model_df = model_df.sort_values("edge", ascending=False)

    print("🔥 Model Layer Built")
    print(model_df.head())

    return model_df


# 🔥 RUN
model_df = build_model_layer(team_table, consensus_df)

In [ ]:
# ==========================================
# FILTER TOP BETS (NO DUPES + EDGE THRESHOLD)
# ==========================================

def generate_top_bets(model_df,
                      min_edge=0.05,
                      min_book_count=2,
                      top_n=10):

    df = model_df.copy()

    # ---------------------------
    # Remove low confidence markets
    # ---------------------------
    df = df[df["book_count"] >= min_book_count]

    # Keep only strong positive edges
    df = df[df["edge"] > min_edge]

    # Remove duplicate teams per game
    df = df.drop_duplicates(subset=["home_team","away_team","team"])

    # Sort by strongest edge
    df = df.sort_values("edge", ascending=False)

    top_bets = df.head(top_n)

    print("🔥 TOP FILTERED BETS")
    print(top_bets[[
        "home_team",
        "away_team",
        "team",
        "consensus_prob",
        "model_prob",
        "edge",
        "book_count"
    ]])

    return top_bets


# 🔥 RUN IT
top_bets = generate_top_bets(model_df)

In [ ]:
# ==========================================
# ADD TRUE EXPECTED VALUE + KELLY SIZE
# ==========================================

def add_ev_and_kelly(df, bankroll=1000):

    df = df.copy()

    # Convert american odds to decimal payout
    def american_to_decimal(odds):
        if odds > 0:
            return 1 + (odds / 100)
        else:
            return 1 + (100 / abs(odds))

    df["decimal_odds"] = df["american_odds"].apply(american_to_decimal)

    # Expected Value Calculation
    df["ev"] = (df["model_prob"] * (df["decimal_odds"] - 1)) - (1 - df["model_prob"])

    # Kelly Fraction
    df["kelly_fraction"] = (
        (df["model_prob"] * (df["decimal_odds"] - 1) - (1 - df["model_prob"]))
        / (df["decimal_odds"] - 1)
    )

    df["kelly_fraction"] = df["kelly_fraction"].clip(lower=0)

    df["kelly_units"] = df["kelly_fraction"] * bankroll

    print("🔥 Bets With EV + Kelly")
    print(df[[
        "home_team",
        "team",
        "edge",
        "ev",
        "kelly_units"
    ]])

    return df


# 🔥 RUN IT AFTER FILTER
top_bets = add_ev_and_kelly(top_bets)

In [ ]:
# ==========================================
# ADD TRUE EXPECTED VALUE + KELLY
# SAFE VERSION
# ==========================================

def add_ev_and_kelly(df, bankroll=1000):

    df = df.copy()

    if "american_odds" not in df.columns:
        print("⚠ No american_odds column found — skipping EV calculation")
        return df

    # Convert american odds → decimal
    def american_to_decimal(odds):
        if odds > 0:
            return 1 + (odds / 100)
        else:
            return 1 + (100 / abs(odds))

    df["decimal_odds"] = df["american_odds"].astype(float).apply(american_to_decimal)

    # Expected Value
    df["ev"] = (
        df["model_prob"] * (df["decimal_odds"] - 1)
        - (1 - df["model_prob"])
    )

    # Kelly
    df["kelly_fraction"] = (
        (df["model_prob"] * (df["decimal_odds"] - 1) - (1 - df["model_prob"]))
        / (df["decimal_odds"] - 1)
    )

    df["kelly_fraction"] = df["kelly_fraction"].clip(lower=0)

    df["kelly_units"] = df["kelly_fraction"] * bankroll

    print("🔥 Bets With EV + Kelly")
    print(df[[
        "home_team",
        "team",
        "edge",
        "ev",
        "kelly_units"
    ]].head(20))

    return df

In [ ]:
print(top_bets.columns)

In [ ]:
# ==========================================
# REATTACH ODDS TO TOP BETS
# ==========================================

def attach_odds(top_bets, odds_df):

    df = top_bets.merge(
        odds_df[[
            "home_team",
            "away_team",
            "team",
            "book",
            "american_odds"
        ]],
        on=["home_team", "away_team", "team"],
        how="left"
    )

    print("✅ Odds Reattached")
    print(df.columns)

    return df

In [ ]:
top_bets = attach_odds(top_bets, odds_df)
top_bets = add_ev_and_kelly(top_bets)

In [ ]:
def add_ev_and_kelly(df, bankroll=10000):

    def american_to_decimal(odds):
        if odds > 0:
            return 1 + (odds / 100)
        else:
            return 1 + (100 / abs(odds))

    def implied_from_odds(odds):
        if odds > 0:
            return 100 / (odds + 100)
        else:
            return abs(odds) / (abs(odds) + 100)

    df = df.copy()

    # Convert price
    df["implied_from_price"] = df["american_odds"].apply(implied_from_odds)

    # True edge
    df["true_edge"] = df["model_prob"] - df["implied_from_price"]

    # Expected Value
    df["ev"] = df["true_edge"] * 100

    # Kelly
    df["kelly_units"] = (df["true_edge"] / (1 - df["implied_from_price"])) * (bankroll * 0.01)

    return df

In [ ]:
top_bets = add_ev_and_kelly(top_bets)
top_bets.sort_values("ev", ascending=False).head(10)

In [ ]:
# ==========================================
# ADVANCED RANKING — SCORE = WEIGHTED EV + KELLY
# ==========================================

def rank_and_dedupe_best_bet(df, weight_ev=0.7, weight_kelly=0.3):

    df = df.copy()

    # Normalize EV so scale differences don’t explode ranking
    if df["ev"].std() != 0:
        df["ev_norm"] = (df["ev"] - df["ev"].mean()) / df["ev"].std()
    else:
        df["ev_norm"] = df["ev"]

    # Normalize Kelly
    if df["kelly_units"].std() != 0:
        df["kelly_norm"] = (
            df["kelly_units"] - df["kelly_units"].mean()
        ) / df["kelly_units"].std()
    else:
        df["kelly_norm"] = df["kelly_units"]

    # Composite Score
    df["rank_score"] = (
        weight_ev * df["ev_norm"] +
        weight_kelly * df["kelly_norm"]
    )

    # Sort highest score first
    df = df.sort_values("rank_score", ascending=False)

    # Remove duplicates — keep highest ranked per game/team
    df = df.drop_duplicates(
        subset=["home_team", "away_team", "team"],
        keep="first"
    )

    return df

In [ ]:
top_bets = rank_and_dedupe_best_bet(top_bets)

top_bets.sort_values("rank_score", ascending=False).head(10)

In [ ]:
# Keep BEST book per team per game based on rank_score
df = (
    df.sort_values("rank_score", ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

In [ ]:
score_col = "rank_score" if "rank_score" in df.columns else "edge"

df = (
    df.sort_values(score_col, ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

print(f"✅ Sorted by {score_col}")

In [ ]:
# ==============================
# TOP 10 DISCORD BETS
# ==============================

# Safety check
if "rank_score" not in df.columns:
    raise ValueError("rank_score missing — run full model before generating top 10")

# Keep best book per game/team
top = (
    df.sort_values("rank_score", ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

# Take Top 10
top10 = top.head(10)

print("🔥 TOP 10 MODEL BETS 🔥\n")

for _, row in top10.iterrows():
    print(
f"""
🏀 {row['home_team']} vs {row['away_team']}
👉 Bet: {row['team']} @ {row['book']}
💰 Odds: {row['american_odds']}
📊 Edge: {round(row['edge']*100,2)}%
📈 EV: {round(row.get('ev',0),2)}
💵 Kelly Units: {round(row.get('kelly_units',0),2)}

-------------------------
"""
    )

In [ ]:
print(df.columns)

In [ ]:
# ==============================
# RANK SCORE ENGINE
# ==============================

if "edge" not in df.columns:
    raise ValueError("Edge missing — run model layer first")

# Normalize components so they are comparable
df["edge_norm"] = (df["edge"] - df["edge"].min()) / (
    df["edge"].max() - df["edge"].min() + 1e-9
)

df["model_norm"] = (df["model_prob"] - df["model_prob"].min()) / (
    df["model_prob"].max() - df["model_prob"].min() + 1e-9
)

# Book strength factor (reward consensus + liquidity)
df["book_strength"] = df["consensus_prob"]

# 🔥 Build Rank Score
df["rank_score"] = (
    df["edge_norm"] * 0.5 +
    df["model_norm"] * 0.3 +
    df["book_strength"] * 0.2
)

print("✅ Rank Score Added")
print(df[["home_team","away_team","team","rank_score"]].head())

In [ ]:
# Top 10 clean books (remove dupes)
top10 = (
    df.sort_values("rank_score", ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
      .head(10)
)

top10

In [ ]:
# ==========================================
# DISCORD TOP 10 BUILDER
# ==========================================

# Safety
if "rank_score" not in df.columns:
    raise ValueError("Rank score missing — run scoring layer first")

# Remove duplicates per game/team
clean_df = (
    df.sort_values("rank_score", ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

# Remove crazy juice (-1000 etc) if you want cleaner output
clean_df = clean_df[clean_df["american_odds"] > -2000]

top10 = clean_df.sort_values("rank_score", ascending=False).head(10)

print("🔥 TOP 10 DISCORD BETS 🔥\n")

for _, row in top10.iterrows():
    print(f"""
🏀 {row['home_team']} vs {row['away_team']}
🎯 Team: {row['team']}
📊 Model Prob: {round(row['model_prob'],3)}
📈 Consensus: {round(row['consensus_prob'],3)}
🔥 Edge: {round(row['edge'],3)}
💰 Odds: {row['american_odds']}
⭐ Rank Score: {round(row['rank_score'],3)}
""")

In [ ]:
# ==============================
# SETTINGS
# ==============================

BANKROLL = 1000
MAX_RISK_PER_BET = 0.02   # 2% max allocation
MAX_UNITS_CAP = 5         # Hard cap so nothing explodes


# ==============================
# CLEAN + RANK + SIZE FUNCTION
# ==============================

def generate_top_10_discord(df):

    df = df.copy()

    # ✅ Remove duplicates (best book only)
    df = (
        df.sort_values("rank_score", ascending=False)
          .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
    )

    # ✅ Normalize rank
    df["rank_norm"] = df["rank_score"] / df["rank_score"].max()

    # ✅ Convert to unit sizing
    df["units"] = df["rank_norm"] * BANKROLL * MAX_RISK_PER_BET

    # ✅ Hard cap units
    df["units"] = df["units"].clip(upper=MAX_UNITS_CAP)

    # ✅ Sort final
    df = df.sort_values("rank_score", ascending=False).head(10)

    return df


# ==============================
# DISCORD FORMATTER
# ==============================

def format_discord_output(df):

    lines = []
    lines.append("🔥 TOP 10 DISCORD BETS 🔥\n")

    for _, row in df.iterrows():

        lines.append(
f"""
🏀 {row['home_team']} vs {row['away_team']}
🎯 Team: {row['team']}
📊 Model Prob: {round(row['model_prob'],3)}
📈 Consensus: {round(row['consensus_prob'],3)}
🔥 Edge: {round(row['edge'],3)}
💰 Odds: {row['american_odds']}
⭐ Rank Score: {round(row['rank_score'],3)}
💵 Suggested Units: {round(row['units'],2)}u
"""
        )

    return "\n".join(lines)


# ==============================
# RUN IT
# ==============================

top_df = generate_top_10_discord(final_model_df)

discord_message = format_discord_output(top_df)

print(discord_message)

In [ ]:
# 🔥 REBUILD FINAL MODEL IF NOT IN MEMORY

final_model_df = df  # <-- or whatever your last processed dataframe is

print(final_model_df.columns)
print("Rows:", len(final_model_df))

In [ ]:
top_df = generate_top_10_discord(final_model_df)

discord_message = format_discord_output(top_df)

print(discord_message)
bet_type = "Spread" if not pd.isna(row["spread"]) else "Moneyline"

In [ ]:
TODAY = "2026-02-27"

# 1) Pull + parse odds
raw_json = pull_raw_json(TODAY)
odds_df = parse_odds_json(raw_json)
print("✅ Odds pulled:", odds_df.shape)

# 2) Clean odds + implied probs
odds_df = clean_odds(odds_df)

# 3) Consensus market per team per game
consensus_df = (
    odds_df
    .groupby(["home_team","away_team","team"])
    .agg(
        consensus_prob=("implied_prob","mean"),
        book_count=("book","nunique")
    )
    .reset_index()
)
print("✅ Consensus built:", consensus_df.shape)

# 4) Model layer (your “full model” probability)
model_output_df = build_model_probs(consensus_df)
print("✅ Model layer built:", model_output_df.shape)

# 5) Edge + ranking
final_model_df = model_output_df.copy()
final_model_df["edge"] = final_model_df["model_prob"] - final_model_df["consensus_prob"]

# rank_score (simple + stable)
final_model_df["edge_norm"] = (
    (final_model_df["edge"] - final_model_df["edge"].min()) /
    (final_model_df["edge"].max() - final_model_df["edge"].min() + 1e-9)
)
final_model_df["rank_score"] = final_model_df["edge_norm"]

# 6) Top 10, no dupes, with units + discord output
top_df = generate_top_10_discord(final_model_df)
discord_msg = format_discord_output(top_df)

print(discord_msg)

In [ ]:
import requests
import pandas as pd
import numpy as np

# ==============================
# CONFIG
# ==============================

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"   # replace if needed
SPORT = "basketball_ncaab"
REGION = "us"
MARKETS = "h2h,spreads"
TODAY = "2026-02-27"

BANKROLL = 1000
MAX_RISK_PER_BET = 0.02
MAX_UNITS_CAP = 5


# ==============================
# ODDS API PULL
# ==============================

def pull_raw_json(date):

    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

    params = {
        "apiKey": API_KEY,
        "regions": REGION,
        "markets": MARKETS,
        "dateFormat": "iso"
    }

    response = requests.get(url, params=params)
    print("Status:", response.status_code)

    return response.json()


# ==============================
# PARSE ODDS
# ==============================

def parse_odds_json(raw_json):

    rows = []

    for game in raw_json:

        home = game["home_team"]
        away = game["away_team"]
        time = game["commence_time"]

        for book in game.get("bookmakers", []):

            book_name = book["key"]

            for market in book.get("markets", []):

                for outcome in market.get("outcomes", []):

                    rows.append({
                        "home_team": home,
                        "away_team": away,
                        "commence_time": time,
                        "book": book_name,
                        "team": outcome["name"],
                        "american_odds": outcome["price"],
                        "spread": outcome.get("point", np.nan)
                    })

    return pd.DataFrame(rows)


# ==============================
# CLEAN ODDS
# ==============================

def convert_american_to_prob(odds):
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return -odds / (-odds + 100)


def clean_odds(df):
    df = df.copy()
    df = df.drop_duplicates(
        subset=["home_team","away_team","book","team","american_odds","spread"]
    )
    df["implied_prob"] = df["american_odds"].apply(convert_american_to_prob)
    return df


# ==============================
# SIMPLE MODEL (Replace Later)
# ==============================

def build_model_probs(consensus_df):

    df = consensus_df.copy()

    # 🔥 TEMP MODEL (adds 7% alpha edge)
    df["model_prob"] = df["consensus_prob"] + 0.07

    df["model_prob"] = df["model_prob"].clip(0.01, 0.99)

    return df


# ==============================
# TOP 10 GENERATOR
# ==============================

def generate_top_10_discord(df):

    df = df.copy()

    df = (
        df.sort_values("rank_score", ascending=False)
          .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
    )

    df["rank_norm"] = df["rank_score"] / df["rank_score"].max()

    df["units"] = df["rank_norm"] * BANKROLL * MAX_RISK_PER_BET
    df["units"] = df["units"].clip(upper=MAX_UNITS_CAP)

    return df.sort_values("rank_score", ascending=False).head(10)


def format_discord_output(df):

    lines = []
    lines.append("🔥 TOP 10 DISCORD BETS 🔥\n")

    for _, row in df.iterrows():

        bet_type = "Spread" if not pd.isna(row.get("spread", np.nan)) else "Moneyline"

        lines.append(
f"""
🏀 {row['home_team']} vs {row['away_team']}
🎯 Team: {row['team']}
📊 Bet Type: {bet_type}
📈 Model: {round(row['model_prob'],3)}
🔥 Edge: {round(row['edge'],3)}
💰 Odds: {row['american_odds']}
💵 Units: {round(row['units'],2)}u
"""
        )

    return "\n".join(lines)


# ==============================
# RUN PIPELINE
# ==============================

raw_json = pull_raw_json(TODAY)
odds_df = parse_odds_json(raw_json)

odds_df = clean_odds(odds_df)

consensus_df = (
    odds_df
    .groupby(["home_team","away_team","team"])
    .agg(
        consensus_prob=("implied_prob","mean"),
        book_count=("book","nunique")
    )
    .reset_index()
)

model_df = build_model_probs(consensus_df)

model_df["edge"] = model_df["model_prob"] - model_df["consensus_prob"]

model_df["rank_score"] = (
    (model_df["edge"] - model_df["edge"].min()) /
    (model_df["edge"].max() - model_df["edge"].min() + 1e-9)
)

top_df = generate_top_10_discord(model_df)

print(format_discord_output(top_df))

In [ ]:
# ============================================
# REATTACH BEST LINE (ML + Spread) + TOP 10
# ============================================

import numpy as np
import pandas as pd

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def implied_from_price(odds):
    odds = float(odds)
    return 100/(odds+100) if odds > 0 else abs(odds)/(abs(odds)+100)

def pick_best_line(odds_df, market="ml"):
    """
    market="ml" -> uses rows where spread is NaN
    market="spread" -> uses rows where spread is NOT NaN
    Picks best price for the bettor (highest decimal odds) per game/team.
    """
    df = odds_df.copy()

    if market == "ml":
        df = df[df["spread"].isna()].copy()
    else:
        df = df[df["spread"].notna()].copy()

    if df.empty:
        return df

    df["decimal_odds"] = df["american_odds"].apply(american_to_decimal)

    # best for bettor = max decimal_odds
    df = (
        df.sort_values("decimal_odds", ascending=False)
          .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
    )

    keep_cols = ["home_team","away_team","team","book","american_odds","spread","decimal_odds"]
    return df[keep_cols].reset_index(drop=True)

def attach_lines(model_df, odds_df):
    ml_best = pick_best_line(odds_df, market="ml")
    sp_best = pick_best_line(odds_df, market="spread")

    # Merge ML lines
    out = model_df.merge(
        ml_best,
        on=["home_team","away_team","team"],
        how="left",
        suffixes=("","")
    )

    # If no ML line exists, try spread line (kept in separate cols)
    sp_best = sp_best.rename(columns={
        "book":"book_spread",
        "american_odds":"american_odds_spread",
        "spread":"spread_line",
        "decimal_odds":"decimal_odds_spread"
    })

    out = out.merge(
        sp_best,
        on=["home_team","away_team","team"],
        how="left"
    )

    return out

def unit_tiers_from_rank(df, top_n=10):
    """
    Uses rank_score percentiles (within the returned set) for unit tiering.
    """
    df = df.copy()
    df = df.head(top_n)

    pct = df["rank_score"].rank(pct=True)

    def assign_units(p):
        if p >= 0.85: return 1.0
        if p >= 0.60: return 0.75
        if p >= 0.35: return 0.5
        return 0.25

    df["units"] = pct.apply(assign_units)
    return df

def format_odds(o):
    if pd.isna(o): return ""
    o = float(o)
    return f"{int(o)}" if o < 0 else f"+{int(o)}"

def discord_top10(df, market="ml", top_n=10, juice_floor=-200):
    """
    market="ml" prints ML picks using american_odds
    market="spread" prints spread picks using american_odds_spread + spread_line
    Filters out juice worse than -200 by default.
    """
    out = df.copy()

    if market == "ml":
        out = out[out["american_odds"].notna()].copy()
        out["american_odds_num"] = pd.to_numeric(out["american_odds"], errors="coerce")
        out = out[out["american_odds_num"] >= juice_floor].copy()

        # de-dupe 1 bet per game/team
        out = (
            out.sort_values("rank_score", ascending=False)
               .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
               .head(top_n)
        )

        out = unit_tiers_from_rank(out, top_n=top_n)

        msg = "🔥 **TOP 10 MODEL BETS (ML | NO DUPES | JUICE >= -200)**\n"
        for _, r in out.iterrows():
            msg += (
                f"\n🏀 {r['home_team']} vs {r['away_team']}"
                f"\nBet: {r['team']} ML"
                f"\nBook: {r.get('book','')}"
                f"\nOdds: {format_odds(r['american_odds'])}"
                f"\nModel Prob: {round(float(r['model_prob']),3)}"
                f"\nConsensus: {round(float(r['consensus_prob']),3)}"
                f"\nEdge: {round(float(r['edge']),3)}"
                f"\nUnits: {r['units']}u\n"
            )

        return out, msg

    else:
        out = out[out["american_odds_spread"].notna()].copy()
        out["american_odds_num"] = pd.to_numeric(out["american_odds_spread"], errors="coerce")
        out = out[out["american_odds_num"] >= juice_floor].copy()

        out = (
            out.sort_values("rank_score", ascending=False)
               .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
               .head(top_n)
        )

        out = unit_tiers_from_rank(out, top_n=top_n)

        msg = "🔥 **TOP 10 MODEL BETS (SPREAD | NO DUPES | JUICE >= -200)**\n"
        for _, r in out.iterrows():
            line = r.get("spread_line", np.nan)
            msg += (
                f"\n🏀 {r['home_team']} vs {r['away_team']}"
                f"\nBet: {r['team']} {line:+g}"
                f"\nBook: {r.get('book_spread','')}"
                f"\nOdds: {format_odds(r['american_odds_spread'])}"
                f"\nModel Prob: {round(float(r['model_prob']),3)}"
                f"\nConsensus: {round(float(r['consensus_prob']),3)}"
                f"\nEdge: {round(float(r['edge']),3)}"
                f"\nUnits: {r['units']}u\n"
            )

        return out, msg


# -----------------------------
# RUN IT (uses existing: odds_df, model_df)
# -----------------------------

final_df = attach_lines(model_df, odds_df)

top_ml_df, discord_ml = discord_top10(final_df, market="ml", top_n=10, juice_floor=-200)
print(discord_ml)

# If you ALSO want spreads:
top_sp_df, discord_sp = discord_top10(final_df, market="spread", top_n=10, juice_floor=-200)
print("\n" + discord_sp)

In [ ]:
import pandas as pd
import numpy as np

def parse_odds_json(raw_json):
    rows = []
    for g in raw_json:
        home = g.get("home_team")
        away = g.get("away_team")
        ct = g.get("commence_time")
        for bm in g.get("bookmakers", []):
            book = bm.get("key")
            for m in bm.get("markets", []):
                mkey = m.get("key")  # "h2h" or "spreads" etc.
                for o in m.get("outcomes", []):
                    team = o.get("name")
                    price = o.get("price")
                    point = o.get("point", np.nan)  # only exists for spreads
                    rows.append({
                        "home_team": home,
                        "away_team": away,
                        "commence_time": ct,
                        "book": book,
                        "market": mkey,
                        "team": team,
                        "american_odds": price,
                        "spread": point if mkey == "spreads" else np.nan
                    })
    df = pd.DataFrame(rows)

    # enforce numeric odds
    df["american_odds"] = pd.to_numeric(df["american_odds"], errors="coerce")
    df["spread"] = pd.to_numeric(df["spread"], errors="coerce")

    return df

# ---- RUN ----
raw_json = pull_raw_json("2026-02-27")   # <- your working raw pull
odds_df = parse_odds_json(raw_json)

print("odds_df shape:", odds_df.shape)
print("markets:", odds_df["market"].value_counts().head(10))
print("ML odds range:", odds_df.loc[odds_df.market=="h2h", "american_odds"].describe())
print("Spread price range:", odds_df.loc[odds_df.market=="spreads", "american_odds"].describe())

In [ ]:
def best_line_by_market(odds_df, market="h2h"):
    df = odds_df[odds_df["market"]==market].copy()
    if df.empty:
        return df

    # best for bettor = max decimal odds
    def american_to_decimal(o):
        o = float(o)
        return 1 + (o/100) if o > 0 else 1 + (100/abs(o))

    df["decimal_odds"] = df["american_odds"].apply(american_to_decimal)

    df = (
        df.sort_values("decimal_odds", ascending=False)
          .drop_duplicates(["home_team","away_team","team"], keep="first")
    )
    return df[["home_team","away_team","team","book","american_odds","spread"]]

# reattach BEST ML line per team/game
ml_lines = best_line_by_market(odds_df, market="h2h")
final_ml = final_model_df.merge(ml_lines, on=["home_team","away_team","team"], how="left")

# no dupes: best bet per game/team by rank_score, take top 10
top10 = (
    final_ml.dropna(subset=["american_odds"])  # must have ML odds
            .sort_values("rank_score", ascending=False)
            .drop_duplicates(["home_team","away_team","team"], keep="first")
            .head(10)
            .copy()
)

# unit tiering (same method you liked)
pct = top10["rank_score"].rank(pct=True)
top10["units"] = pct.apply(lambda p: 1.0 if p>=0.85 else 0.75 if p>=0.60 else 0.5 if p>=0.35 else 0.25)

def fmt_odds(o):
    o = int(o)
    return f"+{o}" if o > 0 else f"{o}"

# print discord
msg = "🔥 **TOP 10 MODEL BETS (ML | NO DUPES)**\n"
for _, r in top10.iterrows():
    msg += (
        f"\n🏀 {r['home_team']} vs {r['away_team']}"
        f"\nBet: {r['team']} ML"
        f"\nBook: {r['book']}"
        f"\nOdds: {fmt_odds(r['american_odds'])}"
        f"\nModel Prob: {round(float(r['model_prob']),3)}"
        f"\nConsensus: {round(float(r['consensus_prob']),3)}"
        f"\nEdge: {round(float(r['edge']),3)}"
        f"\nUnits: {r['units']}u\n"
    )

print(msg)

In [ ]:
import numpy as np
import pandas as pd

# --------------------------
# Helpers
# --------------------------
def american_to_prob(odds):
    odds = float(odds)
    return 100/(odds+100) if odds > 0 else (-odds)/((-odds)+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def fmt_odds(o):
    o = int(round(float(o)))
    return f"+{o}" if o > 0 else f"{o}"

# --------------------------
# 1) Ensure odds_df has market column
# (If you already rebuilt with parse_odds_json, this is already true.)
# If not, we infer: spread rows -> "spreads", else -> "h2h" (good enough).
# --------------------------
if "market" not in odds_df.columns:
    odds_df = odds_df.copy()
    odds_df["market"] = np.where(odds_df["spread"].notna(), "spreads", "h2h")

odds_df["american_odds"] = pd.to_numeric(odds_df["american_odds"], errors="coerce")
odds_df = odds_df.dropna(subset=["american_odds"])

# --------------------------
# 2) Build consensus_prob (H2H consensus) from odds_df
# --------------------------
h2h = odds_df[odds_df["market"] == "h2h"].copy()

# implied prob per book line
h2h["implied_prob"] = h2h["american_odds"].apply(american_to_prob)

# de-vig per game+book: normalize the two sides to sum 1
h2h["sum_prob_book_game"] = h2h.groupby(["home_team","away_team","book"])["implied_prob"].transform("sum")
h2h["market_prob"] = h2h["implied_prob"] / h2h["sum_prob_book_game"]

consensus_df = (
    h2h.groupby(["home_team","away_team","team"], as_index=False)
       .agg(consensus_prob=("market_prob","mean"), book_count=("book","nunique"))
)

# --------------------------
# 3) Build model_prob at game/team level
# IMPORTANT: This must be YOUR model output.
# If you already have model_layer_df, we use that.
# Otherwise we fail loudly so you don't accidentally use implied odds as "model".
# --------------------------
# Try common variable names you’ve had on-screen
model_df = None
for var in ["model_layer_df", "model_output_df", "final_model_df", "model_layer", "model_probs_df"]:
    if var in globals():
        tmp = globals()[var]
        if isinstance(tmp, pd.DataFrame) and {"home_team","away_team","team"}.issubset(tmp.columns):
            if "model_prob" in tmp.columns:
                model_df = tmp[["home_team","away_team","team","model_prob"]].copy()
                break

if model_df is None:
    raise ValueError(
        "Couldn't find your model output DF with ['home_team','away_team','team','model_prob'].\n"
        "Use the variable that printed '🔥 Model Layer Built' and name it model_layer_df."
    )

model_df["model_prob"] = pd.to_numeric(model_df["model_prob"], errors="coerce")

# --------------------------
# 4) Merge model with consensus -> final_model_df
# --------------------------
final_model_df = consensus_df.merge(model_df, on=["home_team","away_team","team"], how="inner")
final_model_df["edge"] = final_model_df["model_prob"] - final_model_df["consensus_prob"]

# rank_score (simple + effective): edge weighted by book_count confidence
final_model_df["rank_score"] = final_model_df["edge"] * np.sqrt(final_model_df["book_count"].clip(lower=1))

# --------------------------
# 5) Best ML line per team/game (bettor-friendly: max decimal)
# --------------------------
h2h["decimal_odds"] = h2h["american_odds"].apply(american_to_decimal)

ml_lines = (
    h2h.sort_values("decimal_odds", ascending=False)
       .drop_duplicates(["home_team","away_team","team"], keep="first")
       [["home_team","away_team","team","book","american_odds"]]
)

final_ml = final_model_df.merge(ml_lines, on=["home_team","away_team","team"], how="left")
final_ml = final_ml.dropna(subset=["american_odds"])

# --------------------------
# 6) NO DUPES + Top 10 + Unit sizing (rank percentile tiers)
# --------------------------
top10 = (
    final_ml.sort_values("rank_score", ascending=False)
            .drop_duplicates(["home_team","away_team","team"], keep="first")
            .head(10)
            .copy()
)

pct = top10["rank_score"].rank(pct=True)
top10["units"] = pct.apply(lambda p: 1.0 if p>=0.85 else 0.75 if p>=0.60 else 0.5 if p>=0.35 else 0.25)

# --------------------------
# 7) Discord output
# --------------------------
msg = "🔥 **TOP 10 MODEL BETS (ML | NO DUPES)**\n"
for _, r in top10.iterrows():
    msg += (
        f"\n🏀 {r['home_team']} vs {r['away_team']}"
        f"\nBet: {r['team']} ML"
        f"\nBook: {r['book']}"
        f"\nOdds: {fmt_odds(r['american_odds'])}"
        f"\nModel Prob: {round(float(r['model_prob']),3)}"
        f"\nConsensus: {round(float(r['consensus_prob']),3)}"
        f"\nEdge: {round(float(r['edge']),3)}"
        f"\nUnits: {r['units']}u\n"
    )

print(msg)

# Optional sanity check
print("\nSanity check (top10):")
print(top10[["home_team","away_team","team","book","american_odds","model_prob","consensus_prob","edge","rank_score","units"]])

In [ ]:
import pandas as pd

candidates = []
for k, v in globals().items():
    if isinstance(v, pd.DataFrame):
        cols = set(v.columns.astype(str))
        if {"home_team","away_team","team"}.issubset(cols) and ("model_prob" in cols or "model_prob_spread" in cols):
            candidates.append((k, list(v.columns), v.shape))

print("FOUND CANDIDATES:\n")
for name, cols, shp in sorted(candidates, key=lambda x: (-x[2][0], x[0])):
    print(f"- {name}  shape={shp}  has_model_prob={'model_prob' in cols}  has_model_prob_spread={'model_prob_spread' in cols}")
    print(f"  cols={cols}\n")

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------
# FIND TABLE WITH rank_score
# ---------------------------------

rank_df_name = None

for k, v in list(globals().items()):
    if isinstance(v, pd.DataFrame):
        cols = set(map(str, v.columns))
        if {"home_team","away_team","team","rank_score"}.issubset(cols):
            rank_df_name = k
            break

if rank_df_name is None:
    raise ValueError("No dataframe with rank_score found. Re-run the cell that builds rank_score.")

print("Using dataframe:", rank_df_name)

base_df = globals()[rank_df_name].copy()

# ---------------------------------
# ENSURE model_prob EXISTS
# ---------------------------------

if "model_prob" not in base_df.columns:
    if "model_prob_spread" in base_df.columns:
        base_df["model_prob"] = base_df["model_prob_spread"]
    else:
        raise ValueError("No model_prob column found in the rank dataframe.")

# ---------------------------------
# BUILD model_layer_df
# ---------------------------------

model_layer_df = base_df[["home_team","away_team","team","model_prob"]].copy()

print("✅ model_layer_df built:", model_layer_df.shape)
print(model_layer_df.head())

In [ ]:
import pandas as pd

# ---------------------------------
# FIND THE BEST (FULL) rank_score DF
# - must contain rank_score
# - prefer the one with the MOST rows
# - avoid obvious small/preview frames
# ---------------------------------

candidates = []
for k, v in list(globals().items()):
    if isinstance(v, pd.DataFrame):
        cols = set(map(str, v.columns))
        if {"home_team","away_team","team","rank_score"}.issubset(cols):
            candidates.append((k, v.shape[0], sorted(list(cols))[:5]))

if not candidates:
    raise ValueError("No dataframe found with ['home_team','away_team','team','rank_score']. Re-run the cell that builds rank_score.")

# pick largest by rowcount
candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)
print("Rank-score DF candidates (name, rows):")
for name, rows, _ in candidates_sorted[:10]:
    print(f" - {name}: {rows}")

best_name = candidates_sorted[0][0]
base_df = globals()[best_name].copy()

print("\n✅ Using FULL dataframe:", best_name, "rows:", base_df.shape[0])

# ---------------------------------
# ENSURE model_prob EXISTS (no placeholders)
# ---------------------------------
if "model_prob" not in base_df.columns:
    if "model_prob_spread" in base_df.columns:
        base_df["model_prob"] = base_df["model_prob_spread"]
    else:
        raise ValueError(f"{best_name} has no model_prob or model_prob_spread.")

# sanity check for the 0.99 issue
print("\nModel prob stats:")
print(base_df["model_prob"].describe())

# ---------------------------------
# BUILD model_layer_df FROM FULL SLATE
# ---------------------------------
model_layer_df = base_df[["home_team","away_team","team","model_prob"]].dropna().copy()
print("\n✅ model_layer_df built:", model_layer_df.shape)
print(model_layer_df.head(10))

In [ ]:
import numpy as np
import pandas as pd
from math import erf, sqrt

def american_to_implied_prob(odds: float) -> float:
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return (-odds) / ((-odds) + 100.0)

def normal_cdf(x: float) -> float:
    # standard normal CDF
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def parse_odds_api(raw_json):
    """
    raw_json is a LIST of games (as your print showed).
    Returns:
      ml_df: game/book/team/american_odds (market='h2h')
      sp_df: game/book/team/american_odds/point (market='spreads')
    """
    rows_ml = []
    rows_sp = []

    for g in raw_json:
        home = g.get("home_team")
        away = g.get("away_team")
        ct = g.get("commence_time")

        for bk in g.get("bookmakers", []):
            book = bk.get("key")

            for m in bk.get("markets", []):
                mkey = m.get("key")

                if mkey == "h2h":
                    for o in m.get("outcomes", []):
                        rows_ml.append({
                            "home_team": home,
                            "away_team": away,
                            "commence_time": ct,
                            "book": book,
                            "team": o.get("name"),
                            "american_odds": o.get("price"),
                            "market": "h2h"
                        })

                if mkey == "spreads":
                    for o in m.get("outcomes", []):
                        rows_sp.append({
                            "home_team": home,
                            "away_team": away,
                            "commence_time": ct,
                            "book": book,
                            "team": o.get("name"),
                            "american_odds": o.get("price"),
                            "spread": o.get("point"),
                            "market": "spreads"
                        })

    ml_df = pd.DataFrame(rows_ml)
    sp_df = pd.DataFrame(rows_sp)

    # Clean dtypes
    if len(ml_df):
        ml_df["american_odds"] = pd.to_numeric(ml_df["american_odds"], errors="coerce")
        ml_df["implied_prob"] = ml_df["american_odds"].apply(american_to_implied_prob)

    if len(sp_df):
        sp_df["american_odds"] = pd.to_numeric(sp_df["american_odds"], errors="coerce")
        sp_df["spread"] = pd.to_numeric(sp_df["spread"], errors="coerce")
        sp_df["implied_prob"] = sp_df["american_odds"].apply(american_to_implied_prob)

    return ml_df, sp_df

In [ ]:
def build_consensus(df, prob_col="implied_prob"):
    """
    Consensus = average implied probability across books for each game/team.
    """
    df = df.dropna(subset=[prob_col]).copy()
    cons = (df.groupby(["home_team","away_team","team"], as_index=False)
              .agg(consensus_prob=(prob_col, "mean"),
                   book_count=("book", "nunique")))
    return cons

def best_line(df, market):
    """
    Pick best price per game/team across books for a given market.
    For ML: best price = max decimal odds.
    For spreads: same (max decimal odds) but keep spread + price.
    """
    df = df.copy()
    df["american_odds"] = pd.to_numeric(df["american_odds"], errors="coerce")

    def american_to_decimal(odds):
        if pd.isna(odds): return np.nan
        odds = float(odds)
        if odds > 0: return 1.0 + (odds/100.0)
        return 1.0 + (100.0/(-odds))

    df["decimal_odds"] = df["american_odds"].apply(american_to_decimal)

    # Keep best (highest decimal odds) per game/team
    keep_cols = ["home_team","away_team","team","book","american_odds","decimal_odds","commence_time"]
    if market == "spreads" and "spread" in df.columns:
        keep_cols.append("spread")

    df = df[keep_cols].dropna(subset=["decimal_odds"]).copy()
    df = (df.sort_values("decimal_odds", ascending=False)
            .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))
    return df

In [ ]:
def build_margin_model_v2(ml_df, sp_df, sims=40000, sigma_margin=11.0, seed=7):

    rng = np.random.default_rng(seed)

    # ---------------------------------
    # 1) GAME LEVEL SPREAD CONSENSUS
    # ---------------------------------
    sp_home = sp_df[sp_df["team"] == sp_df["home_team"]].dropna(subset=["spread"]).copy()

    game_spread = (
        sp_home.groupby(["home_team","away_team","commence_time"], as_index=False)
        .agg(home_spread=("spread","mean"))
    )

    # expected margin = -spread
    game_spread["exp_margin"] = -game_spread["home_spread"]

    # ---------------------------------
    # 2) SIMULATE GAME MARGINS
    # ---------------------------------
    exp = game_spread["exp_margin"].to_numpy()
    draws = rng.normal(loc=exp[:, None], scale=sigma_margin, size=(len(exp), sims))

    game_spread["home_win_prob"] = (draws > 0).mean(axis=1)

    # ---------------------------------
    # 3) BUILD ML MODEL
    # ---------------------------------
    ml_best = best_line(ml_df, market="h2h")

    ml_model = ml_best.merge(
        game_spread[["home_team","away_team","commence_time","home_win_prob"]],
        on=["home_team","away_team","commence_time"],
        how="left"
    )

    ml_model["model_prob"] = np.where(
        ml_model["team"] == ml_model["home_team"],
        ml_model["home_win_prob"],
        1 - ml_model["home_win_prob"]
    )

    # ---------------------------------
    # 4) BUILD SPREAD MODEL
    # ---------------------------------
    sp_best = best_line(sp_df, market="spreads")

    sp_model = sp_best.merge(
        game_spread[["home_team","away_team","commence_time","exp_margin"]],
        on=["home_team","away_team","commence_time"],
        how="left"
    )

    # Cover probability using normal CDF
    def cover_prob(row):
        if pd.isna(row["exp_margin"]) or pd.isna(row["spread"]):
            return np.nan

        mu = row["exp_margin"]
        sigma = sigma_margin

        if row["team"] == row["home_team"]:
            z = (mu - (-row["spread"])) / sigma
        else:
            z = (row["spread"] - mu) / sigma

        return normal_cdf(z)

    sp_model["model_prob"] = sp_model.apply(cover_prob, axis=1)

    return ml_model, sp_model

In [ ]:
def kelly_fraction(p, american_odds):
    # Full Kelly fraction for a single bet
    if pd.isna(p) or pd.isna(american_odds): return 0.0
    p = float(p)
    if p <= 0 or p >= 1: return 0.0

    odds = float(american_odds)
    # decimal b = net profit per 1 staked
    if odds > 0:
        b = odds / 100.0
    else:
        b = 100.0 / (-odds)

    q = 1.0 - p
    f = (b*p - q) / b
    return max(0.0, f)

def add_edges_and_units(model_df, consensus_df, juice_floor=-200, kelly_cap_units=5.0, half_kelly=True):
    df = model_df.merge(consensus_df, on=["home_team","away_team","team"], how="left")

    # filter juice if requested
    df["american_odds"] = pd.to_numeric(df["american_odds"], errors="coerce")
    if juice_floor is not None:
        df = df[(df["american_odds"].isna()) | (df["american_odds"] >= juice_floor)].copy()

    df["edge"] = df["model_prob"] - df["consensus_prob"]

    # Kelly sizing
    df["kelly_f"] = df.apply(lambda r: kelly_fraction(r["model_prob"], r["american_odds"]), axis=1)
    if half_kelly:
        df["kelly_f"] *= 0.5

    # Convert to "units" (simple: 10u bankroll scale => f*10, then cap)
    df["units"] = (df["kelly_f"] * 10.0).clip(lower=0.0, upper=kelly_cap_units)

    return df

def top_10_no_dupes(df, n=10):
    # keep best bet per GAME (home/away) by edge, then take top n
    df = df.dropna(subset=["edge","model_prob","consensus_prob","american_odds"]).copy()
    df = df.sort_values("edge", ascending=False)
    df = df.drop_duplicates(subset=["home_team","away_team"], keep="first")
    return df.head(n).reset_index(drop=True)

In [ ]:
TODAY = "2026-02-27"

raw_json = pull_raw_json(TODAY)  # <-- your existing function
ml_df, sp_df = parse_odds_api(raw_json)

print("✅ ML rows:", ml_df.shape, "| Spread rows:", sp_df.shape)

# Consensus market (separate by market)
ml_cons = build_consensus(ml_df)
sp_cons = build_consensus(sp_df)

# Build model from margin sims (spread-driven)
ml_model, sp_model = build_margin_model(ml_df, sp_df, sims=30000, sigma_margin=11.0, seed=7)

# Add edges + units
ml_final = add_edges_and_units(ml_model, ml_cons, juice_floor=-200, kelly_cap_units=5.0, half_kelly=True)
sp_final = add_edges_and_units(sp_model, sp_cons, juice_floor=-200, kelly_cap_units=5.0, half_kelly=True)

# Top 10 no dupes
top10_ml = top_10_no_dupes(ml_final, n=10)
top10_sp = top_10_no_dupes(sp_final, n=10)

def discord_block(df, label):
    out = [f"🔥 TOP 10 MODEL BETS ({label} | NO DUPES)"]
    for _, r in df.iterrows():
        matchup = f"{r['home_team']} vs {r['away_team']}"
        out.append(
            f"\n🏀 {matchup}\n"
            f"Bet: {r['team']} ({label})\n"
            f"Book: {r['book']}\n"
            f"Odds: {int(r['american_odds'])}\n"
            + (f"Line: {r['spread']}\n" if "spread" in r and not pd.isna(r["spread"]) else "")
            + f"Model Prob: {r['model_prob']:.3f}\n"
            + f"Consensus: {r['consensus_prob']:.3f}\n"
            + f"Edge: {r['edge']:.3f}\n"
            + f"Units: {r['units']:.2f}u"
        )
    return "\n".join(out)

print(discord_block(top10_ml, "ML"))
print("\n" + "="*40 + "\n")
print(discord_block(top10_sp, "SPREAD"))

In [ ]:
ml_model, sp_model = build_margin_model_v2(ml_df, sp_df)

In [ ]:
print(ml_model["model_prob"].describe())
print(sp_model["model_prob"].describe())

In [ ]:
ml_model[['home_team','team','model_prob']].head(10)
sp_model[['home_team','team','spread','model_prob']].head(10)

In [ ]:
# Rebuild ML consensus from implied probability
ml_consensus = (
    ml_df.groupby(["home_team","away_team","team"], as_index=False)
    .agg(consensus_prob=("implied_prob","mean"),
         book_count=("book","nunique"))
)

# Rebuild Spread consensus
sp_consensus = (
    sp_df.groupby(["home_team","away_team","team"], as_index=False)
    .agg(consensus_prob=("implied_prob","mean"),
         book_count=("book","nunique"))
)

print("ML consensus sample:")
print(ml_consensus.head())

print("\nSpread consensus sample:")
print(sp_consensus.head())

In [ ]:
def kelly_units(row, bankroll=10, half=True):
    p = row["model_prob"]
    odds = row["american_odds"]

    if pd.isna(p) or pd.isna(odds):
        return 0.0

    if odds > 0:
        b = odds / 100
    else:
        b = 100 / abs(odds)

    q = 1 - p
    f = (b*p - q) / b

    if half:
        f *= 0.5

    if f < 0:
        return 0.0

    return min(f * bankroll, 5.0)  # cap at 5u

In [ ]:
ml_final = (
    ml_model
    .merge(ml_consensus, on=["home_team","away_team","team"], how="left")
)

ml_final["edge"] = ml_final["model_prob"] - ml_final["consensus_prob"]
ml_final["units"] = ml_final.apply(kelly_units, axis=1)

# Remove negative edges
ml_final = ml_final[ml_final["edge"] > 0]

# Best bet per game
ml_top = (
    ml_final
    .sort_values("edge", ascending=False)
    .drop_duplicates(subset=["home_team","away_team"], keep="first")
    .head(10)
)

ml_top[[
    "home_team","away_team","team","american_odds",
    "model_prob","consensus_prob","edge","units"
]]

In [ ]:
sp_final = (
    sp_model
    .merge(sp_consensus, on=["home_team","away_team","team"], how="left")
)

sp_final["edge"] = sp_final["model_prob"] - sp_final["consensus_prob"]
sp_final["units"] = sp_final.apply(kelly_units, axis=1)

sp_final = sp_final[sp_final["edge"] > 0]

sp_top = (
    sp_final
    .sort_values("edge", ascending=False)
    .drop_duplicates(subset=["home_team","away_team"], keep="first")
    .head(10)
)

sp_top[[
    "home_team","away_team","team","spread","american_odds",
    "model_prob","consensus_prob","edge","units"
]]

In [ ]:
ml_top

In [ ]:
sp_top

In [ ]:
print("ML Edge Summary")
print(ml_final["edge"].describe())

print("\nSpread Edge Summary")
print(sp_final["edge"].describe())

In [ ]:
# =========================
# CELL 1 — Helpers + Parsing
# =========================
import numpy as np
import pandas as pd

def american_to_prob(odds: float) -> float:
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    return (-odds) / ((-odds) + 100.0)

def prob_to_american(p: float) -> float:
    # fair odds (no vig) for reference
    if p <= 0 or p >= 1 or pd.isna(p):
        return np.nan
    if p >= 0.5:
        return - (p / (1-p)) * 100
    return ((1-p) / p) * 100

def american_to_decimal(odds: float) -> float:
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds > 0:
        return 1.0 + odds/100.0
    return 1.0 + 100.0/(-odds)

def parse_odds_json(raw_json):
    """
    raw_json: list[games] like the Odds API response you printed
    Returns: long df with columns:
      home_team, away_team, commence_time, book, market, team, american_odds, spread
    """
    rows = []
    if raw_json is None:
        return pd.DataFrame()

    # If API sometimes returns dict with 'data' etc, handle both
    games = raw_json if isinstance(raw_json, list) else raw_json.get("data", [])

    for g in games:
        home = g.get("home_team")
        away = g.get("away_team")
        t = g.get("commence_time")
        for bm in g.get("bookmakers", []):
            book = bm.get("key") or bm.get("title")
            for m in bm.get("markets", []):
                mkey = m.get("key")
                for out in m.get("outcomes", []):
                    team = out.get("name")
                    price = out.get("price")
                    point = out.get("point", np.nan)

                    row = {
                        "home_team": home,
                        "away_team": away,
                        "commence_time": t,
                        "book": book,
                        "market": mkey,            # 'h2h' or 'spreads'
                        "team": team,
                        "american_odds": price,
                        "spread": point if mkey == "spreads" else np.nan
                    }
                    rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    # numeric conversions
    df["american_odds"] = pd.to_numeric(df["american_odds"], errors="coerce")
    df["spread"] = pd.to_numeric(df["spread"], errors="coerce")
    return df

In [ ]:
# =========================================
# CELL 2 — Consensus + Simple "Bias" Model
# =========================================
def build_consensus(df_long, market="h2h", min_books=2):
    """
    Builds consensus probability per team per game using mean implied prob across books.
    """
    d = df_long[df_long["market"] == market].copy()
    d = d.dropna(subset=["american_odds"])
    d["implied_prob"] = d["american_odds"].apply(american_to_prob)

    # consensus per team within a game
    cons = (
        d.groupby(["home_team","away_team","team"], as_index=False)
         .agg(consensus_prob=("implied_prob","mean"),
              book_count=("implied_prob","count"))
    )

    # keep only where we have enough book coverage (prevents 1-book junk)
    cons = cons[cons["book_count"] >= min_books].reset_index(drop=True)
    return cons

def build_bias_model_probs(consensus_df, market="h2h"):
    """
    Creates an *independent* model probability by applying a small bias away from consensus.
    This breaks circular 'market vs market' and produces usable edges.

    You can tune BIAS_SCALE safely (0.01–0.04). Start at 0.02.
    """
    out = consensus_df.copy()

    # Make favorites slightly less "perfect" and dogs slightly more live (or vice versa)
    # This is a placeholder alpha layer until you plug in team efficiencies/injuries.
    BIAS_SCALE = 0.02

    # Pull toward 0.50 by small amount proportional to distance from 0.50
    # (prevents insane 0.99 probs)
    p = out["consensus_prob"].clip(0.01, 0.99)
    out["model_prob"] = (p + BIAS_SCALE * (0.5 - p)).clip(0.01, 0.99)

    # edge vs consensus (you can also edge vs price-implied per book later)
    out["edge"] = out["model_prob"] - out["consensus_prob"]
    out["fair_american"] = out["model_prob"].apply(prob_to_american)

    return out

In [ ]:
# ==================================================
# CELL 3 — Attach Best ML price + EV + Kelly + Top10
# ==================================================
def best_price_lines(df_long, market="h2h"):
    """
    Returns best available ML line per team per game (best for bettor).
    For favorites (negative), best is closest to 0 (highest number).
    For dogs (positive), best is largest.
    """
    d = df_long[df_long["market"] == market].copy()
    d = d.dropna(subset=["american_odds"])
    d["american_odds"] = pd.to_numeric(d["american_odds"], errors="coerce")

    def pick_best(sub):
        odds = sub["american_odds"].values
        if np.all(odds < 0):
            # -120 better than -150
            idx = sub["american_odds"].idxmax()
        elif np.all(odds > 0):
            # +160 better than +140
            idx = sub["american_odds"].idxmax()
        else:
            # mixed (rare): choose max decimal
            dec = sub["american_odds"].apply(american_to_decimal)
            idx = dec.idxmax()
        return sub.loc[idx, ["book","american_odds"]]

    best = (
        d.groupby(["home_team","away_team","team"])
         .apply(pick_best)
         .reset_index()
    )
    return best

def add_ev_kelly(df, bankroll_units=10.0, half_kelly=True):
    """
    bankroll_units is just a scaling anchor; output is in "units".
    """
    out = df.copy()
    out["decimal_odds"] = out["american_odds"].apply(american_to_decimal)
    out["implied_from_price"] = out["american_odds"].apply(american_to_prob)

    p = out["model_prob"].astype(float).clip(0.0001, 0.9999)
    b = out["decimal_odds"].astype(float) - 1.0

    # EV per 1 unit risk at decimal odds:
    # EV = p*b - (1-p)
    out["ev"] = p*b - (1.0 - p)

    # Kelly fraction:
    # f* = (p*(b+1) - 1) / b  == (p*decimal - 1)/b
    out["kelly_fraction"] = ((p*out["decimal_odds"]) - 1.0) / b
    out["kelly_fraction"] = out["kelly_fraction"].replace([np.inf,-np.inf], np.nan).fillna(0.0)
    out["kelly_fraction"] = out["kelly_fraction"].clip(lower=0.0)

    if half_kelly:
        out["kelly_fraction"] *= 0.5

    out["units"] = out["kelly_fraction"] * bankroll_units

    # Simple rank score: prioritize EV, edge, and book coverage
    out["rank_score"] = (
        (out["ev"].clip(lower=0)) * 0.55 +
        (out["edge"].clip(lower=0)) * 0.35 +
        (out["book_count"].clip(lower=0) / 10.0) * 0.10
    )

    return out

def top10_no_dupes(df_scored, top_n=10):
    """
    NO DUPES by game: keep only best ranked bet per game.
    """
    d = df_scored.copy()
    d = d.sort_values("rank_score", ascending=False)
    d = d.drop_duplicates(subset=["home_team","away_team"], keep="first")
    return d.head(top_n).reset_index(drop=True)

def format_discord_ml(top_df):
    lines = ["🔥 TOP 10 MODEL BETS (ML | NO DUPES)\n"]
    for _, r in top_df.iterrows():
        lines.append(
            f"🏀 {r['home_team']} vs {r['away_team']}\n"
            f"Bet: {r['team']} ML\n"
            f"Book: {r['book']}\n"
            f"Odds: {int(r['american_odds'])}\n"
            f"Model Prob: {r['model_prob']:.3f}\n"
            f"Consensus: {r['consensus_prob']:.3f}\n"
            f"Edge: {r['edge']:.3f}\n"
            f"EV: {r['ev']:.3f}\n"
            f"Units (½ Kelly): {r['units']:.2f}u\n"
        )
    return "\n".join(lines)

# ---------- RUN TODAY ----------
TODAY = "2026-02-27"

# You should already have: raw_json = pull_raw_json(TODAY)
# If you have it under a different var name, set it here:
# raw_json = raw_json

odds_long = parse_odds_json(raw_json)
print("✅ odds_long:", odds_long.shape)
print(odds_long.head())

# Consensus from H2H only
cons_df = build_consensus(odds_long, market="h2h", min_books=2)
print("✅ consensus:", cons_df.shape)

# Build model probs (bias layer)
model_df = build_bias_model_probs(cons_df, market="h2h")

# Attach best available ML price per team/game
best_ml = best_price_lines(odds_long, market="h2h")
final_ml = model_df.merge(best_ml, on=["home_team","away_team","team"], how="left").dropna(subset=["american_odds"])

# Score EV + Kelly + rank_score
final_ml = add_ev_kelly(final_ml, bankroll_units=10.0, half_kelly=True)

# Filter: only positive EV and reasonable juice if you want
final_ml = final_ml[(final_ml["ev"] > 0)]

top10 = top10_no_dupes(final_ml, top_n=10)

print("✅ Top10 rows:", len(top10))
print(top10[["home_team","away_team","team","book","american_odds","model_prob","consensus_prob","edge","ev","units","rank_score"]])

print("\n" + format_discord_ml(top10))

In [ ]:
# =========================
# CELL 1 — Price Normalizer (Decimal or American)
# =========================
import numpy as np
import pandas as pd
from math import erf, sqrt

def is_decimal_price(x):
    # Decimal odds are typically >= 1.01 and usually not huge negatives.
    # If it's between 1.01 and 1000 it's almost certainly decimal.
    if pd.isna(x):
        return False
    try:
        x = float(x)
    except:
        return False
    return (x >= 1.01) and (x < 10000) and (x > 0)

def decimal_to_american(dec):
    if pd.isna(dec):
        return np.nan
    dec = float(dec)
    if dec <= 1.0:
        return np.nan
    if dec >= 2.0:
        return round((dec - 1.0) * 100.0)
    return round(-100.0 / (dec - 1.0))

def american_to_decimal(am):
    if pd.isna(am):
        return np.nan
    am = float(am)
    if am > 0:
        return 1.0 + am/100.0
    return 1.0 + 100.0/(-am)

def american_to_prob(am):
    if pd.isna(am):
        return np.nan
    am = float(am)
    if am > 0:
        return 100.0 / (am + 100.0)
    return (-am) / ((-am) + 100.0)

def decimal_to_prob(dec):
    if pd.isna(dec) or dec <= 1.0:
        return np.nan
    return 1.0 / float(dec)

def norm_cdf(x):
    # standard normal CDF via erf (no scipy needed)
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def normalize_prices(odds_long: pd.DataFrame) -> pd.DataFrame:
    df = odds_long.copy()
    df["price_raw"] = pd.to_numeric(df["american_odds"], errors="coerce")

    # If it looks like decimal, treat as decimal; else treat as american
    df["price_decimal"] = np.where(
        df["price_raw"].apply(is_decimal_price),
        df["price_raw"].astype(float),
        df["price_raw"].apply(american_to_decimal)
    )

    df["american_odds"] = np.where(
        df["price_raw"].apply(is_decimal_price),
        df["price_raw"].apply(decimal_to_american),
        df["price_raw"]
    )

    df["implied_prob"] = df["price_decimal"].apply(decimal_to_prob)

    return df

In [ ]:
# ==============================================
# CELL 2 — True Consensus (de-vig) + Spread Model + Top10 ML No Dupes
# ==============================================
def devig_two_way(p_a, p_b):
    # simple proportional devig for two-outcome market
    if pd.isna(p_a) or pd.isna(p_b) or (p_a <= 0) or (p_b <= 0):
        return (np.nan, np.nan)
    s = p_a + p_b
    if s <= 0:
        return (np.nan, np.nan)
    return (p_a/s, p_b/s)

def build_consensus_ml_devig(df_norm):
    """
    Build consensus ML probability using de-vig per book for each game.
    Returns one row per (home_team, away_team, team) with consensus_prob and book_count.
    """
    ml = df_norm[df_norm["market"] == "h2h"].copy()
    ml = ml.dropna(subset=["price_decimal","implied_prob","book","team"])

    rows = []
    for (home, away, book), g in ml.groupby(["home_team","away_team","book"]):
        # expect 2 teams
        if g.shape[0] < 2:
            continue
        g2 = g.sort_values("team")
        # just take first two
        a = g2.iloc[0]
        b = g2.iloc[1]
        pA, pB = devig_two_way(a["implied_prob"], b["implied_prob"])
        rows.append([home, away, a["team"], book, pA])
        rows.append([home, away, b["team"], book, pB])

    out = pd.DataFrame(rows, columns=["home_team","away_team","team","book","p_devig"])
    if out.empty:
        return out

    cons = (
        out.groupby(["home_team","away_team","team"], as_index=False)
           .agg(consensus_prob=("p_devig","mean"),
                book_count=("p_devig","count"))
    )
    return cons

def build_consensus_spread(df_norm):
    """
    Build consensus spread point per team per game (mean of points across books).
    spread lines are in df_norm['spread'] and df_norm['market']=='spreads'
    """
    sp = df_norm[df_norm["market"] == "spreads"].copy()
    sp = sp.dropna(subset=["spread","book","team"])

    cons_sp = (
        sp.groupby(["home_team","away_team","team"], as_index=False)
          .agg(cons_spread=("spread","mean"),
               sp_book_count=("spread","count"))
    )
    return cons_sp

def model_prob_from_spread(cons_spread, sigma=11.5):
    """
    Convert point spread (team perspective) to win probability using Normal approx.
    If team spread is negative (fav), win prob increases.
    p(win) ≈ Phi(-spread / sigma)
    """
    out = cons_spread.copy()
    out["model_prob"] = out["cons_spread"].apply(lambda s: norm_cdf((-float(s))/sigma))
    return out

def best_ml_price_per_team(df_norm):
    """
    Best ML price per team/game for bettor, using decimal odds max.
    """
    ml = df_norm[df_norm["market"] == "h2h"].copy()
    ml = ml.dropna(subset=["price_decimal","team","book"])
    idx = ml.groupby(["home_team","away_team","team"])["price_decimal"].idxmax()
    best = ml.loc[idx, ["home_team","away_team","team","book","price_decimal","american_odds"]].reset_index(drop=True)
    return best

def add_ev_kelly_ml(df, half_kelly=True, bankroll_units=10.0):
    out = df.copy()
    out["price_decimal"] = pd.to_numeric(out["price_decimal"], errors="coerce")
    out["model_prob"] = pd.to_numeric(out["model_prob"], errors="coerce")
    out = out.dropna(subset=["price_decimal","model_prob"])

    p = out["model_prob"].clip(0.0001, 0.9999)
    dec = out["price_decimal"]
    b = dec - 1.0

    out["ev"] = p*b - (1.0 - p)

    k = ((p*dec) - 1.0) / b
    k = k.replace([np.inf,-np.inf], np.nan).fillna(0.0).clip(lower=0.0)
    if half_kelly:
        k = 0.5 * k

    out["kelly_fraction"] = k
    out["units"] = k * bankroll_units
    return out

def top10_ml_no_dupes(scored_df, min_books=2, max_juice=-200):
    """
    No dupes by game: keep single best bet per (home_team, away_team).
    Also filter out crazy juice by american_odds (favorites too expensive).
    """
    d = scored_df.copy()

    # book coverage filter from ML consensus
    if "book_count" in d.columns:
        d = d[d["book_count"] >= min_books]

    # juice filter (only if american odds is present)
    if "american_odds" in d.columns:
        d["american_odds"] = pd.to_numeric(d["american_odds"], errors="coerce")
        d = d[(d["american_odds"].isna()) | (d["american_odds"] >= max_juice)]

    # only +EV
    d = d[d["ev"] > 0].copy()

    # rank by EV then units
    d["rank_score"] = d["ev"]*0.7 + d["units"]*0.3
    d = d.sort_values("rank_score", ascending=False)

    # NO DUPES per game
    d = d.drop_duplicates(subset=["home_team","away_team"], keep="first")

    return d.head(10).reset_index(drop=True)

def format_discord_ml(top_df):
    msg = ["🔥 TOP 10 MODEL BETS (ML | NO DUPES)\n"]
    for _, r in top_df.iterrows():
        msg.append(
            f"🏀 {r['home_team']} vs {r['away_team']}\n"
            f"Bet: {r['team']} ML\n"
            f"Book: {r['book']}\n"
            f"Odds (Amer): {int(r['american_odds']) if pd.notna(r['american_odds']) else 'NA'}\n"
            f"Odds (Dec): {r['price_decimal']:.3f}\n"
            f"Model Prob: {r['model_prob']:.3f}\n"
            f"Consensus Prob: {r['consensus_prob']:.3f}\n"
            f"Edge: {(r['model_prob']-r['consensus_prob']):.3f}\n"
            f"EV: {r['ev']:.3f}\n"
            f"Units (½ Kelly): {r['units']:.2f}u\n"
        )
    return "\n".join(msg)

# ---------- RUN (TODAY) ----------
df_norm = normalize_prices(odds_long)
print("✅ normalized:", df_norm.shape)
print(df_norm.head())

# Consensus ML (de-vig) + Consensus Spread
cons_ml = build_consensus_ml_devig(df_norm)
cons_sp = build_consensus_spread(df_norm)

print("✅ cons_ml:", cons_ml.shape, "| ✅ cons_sp:", cons_sp.shape)

# Model prob from spread consensus (independent of ML)
model_sp = model_prob_from_spread(cons_sp, sigma=11.5)

# Merge consensus ML + model prob (by team)
layer = cons_ml.merge(model_sp[["home_team","away_team","team","model_prob","cons_spread","sp_book_count"]],
                      on=["home_team","away_team","team"], how="left")

# Attach BEST ML price per team/game
best_ml = best_ml_price_per_team(df_norm)
final = layer.merge(best_ml, on=["home_team","away_team","team"], how="left")

# Score EV + Kelly
final = add_ev_kelly_ml(final, half_kelly=True, bankroll_units=10.0)

# Top10 ML no dupes
top10 = top10_ml_no_dupes(final, min_books=2, max_juice=-200)

print("✅ Top10:", top10.shape)
display_cols = ["home_team","away_team","team","book","american_odds","price_decimal",
                "consensus_prob","model_prob","cons_spread","ev","units"]
print(top10[display_cols])

print("\n" + format_discord_ml(top10))

In [ ]:
# ==============================================
# NEXT CELL — STRICT ML ONLY + BETTER FILTERING + CLEAN DISCORD OUTPUT
# ==============================================

def build_top10_ml_strict(final,
                          min_books=3,
                          min_edge=0.01,      # require at least 1% prob edge vs consensus
                          min_ev=0.02,        # require at least 0.02 EV
                          max_juice=-200,     # don't lay worse than -200
                          top_n=10):
    d = final.copy()

    # require ML price exists
    d = d.dropna(subset=["price_decimal","american_odds","consensus_prob","model_prob","book_count"])

    # book coverage
    d = d[d["book_count"] >= min_books]

    # juice filter
    d["american_odds"] = pd.to_numeric(d["american_odds"], errors="coerce")
    d = d[d["american_odds"] >= max_juice]

    # compute edges
    d["edge_prob"] = d["model_prob"] - d["consensus_prob"]

    # EV filter (already computed earlier, but ensure present)
    if "ev" not in d.columns:
        d = add_ev_kelly_ml(d, half_kelly=True, bankroll_units=10.0)

    d = d[(d["edge_prob"] >= min_edge) & (d["ev"] >= min_ev)].copy()

    # rank: prioritize EV, then edge, then units
    d["rank_score"] = (d["ev"]*0.55) + (d["edge_prob"]*0.35) + (d["units"]*0.10)
    d = d.sort_values("rank_score", ascending=False)

    # NO DUPES: one bet per game
    d = d.drop_duplicates(subset=["home_team","away_team"], keep="first")

    return d.head(top_n).reset_index(drop=True)

def discord_ml_card(top10):
    lines = ["🔥 **NCAAB — TOP 10 ML (NO DUPES | ½-KELLY)**\n"]
    for i, r in top10.iterrows():
        lines.append(
            f"{i+1}) {r['home_team']} vs {r['away_team']}\n"
            f"Bet: {r['team']} ML ({r['book']})\n"
            f"Odds: {int(r['american_odds'])}  (Dec {r['price_decimal']:.2f})\n"
            f"Model: {r['model_prob']:.3f} | Consensus: {r['consensus_prob']:.3f}\n"
            f"Edge: {r['edge_prob']:.3f} | EV: {r['ev']:.3f}\n"
            f"Units: {r['units']:.2f}u\n"
        )
    return "\n".join(lines)

# -------- RUN STRICT TOP10 --------
top10_strict = build_top10_ml_strict(final,
                                     min_books=3,
                                     min_edge=0.01,
                                     min_ev=0.02,
                                     max_juice=-200,
                                     top_n=10)

print("✅ strict top10 rows:", len(top10_strict))
if len(top10_strict) == 0:
    print("No bets passed filters. Try relaxing:\n - min_books=2\n - min_edge=0.005\n - min_ev=0.01")
else:
    display_cols = ["home_team","away_team","team","book","american_odds","price_decimal",
                    "consensus_prob","model_prob","edge_prob","ev","units","book_count"]
    print(top10_strict[display_cols])
    print("\n" + discord_ml_card(top10_strict))

In [ ]:
# =========================
# CELL 1 — SANITY CHECK
# =========================
required_cols = [
    "home_team","away_team","team","book",
    "american_odds","price_decimal",
    "consensus_prob","model_prob",
    "ev","units","book_count"
]

missing = [c for c in required_cols if c not in final.columns]
print("final rows:", len(final))
print("missing cols:", missing)
print("sample cols:", list(final.columns)[:25])

if missing:
    raise ValueError(
        "Your `final` df is missing required columns.\n"
        "Make sure you're using the dataframe that printed ✅ Top10: (10, 15) earlier.\n"
        "If your df is named something else, set: final = <that_df_name>"
    )

# quick check: are we accidentally using spread prices?
print("\namerican_odds sample:", final["american_odds"].dropna().head(10).tolist())
print("price_decimal sample:", final["price_decimal"].dropna().head(10).tolist())
print("unique markets present:", final["market"].unique() if "market" in final.columns else "no market col")

In [ ]:
# ==============================================
# CELL 2 — STRICT ML TOP 10 (NO DUPES)
# ==============================================

def build_top10_ml_strict(dfin,
                          min_books=3,
                          min_edge=0.01,      # 1% prob edge vs consensus
                          min_ev=0.02,        # 0.02 EV threshold
                          max_juice=-200,     # don't lay worse than -200
                          top_n=10):
    d = dfin.copy()

    # STRICT: ML ONLY (if market column exists)
    if "market" in d.columns:
        d = d[d["market"].astype(str).str.lower().isin(["h2h","ml","moneyline"])].copy()

    # require ML price exists
    d["american_odds"] = pd.to_numeric(d["american_odds"], errors="coerce")
    d["price_decimal"] = pd.to_numeric(d["price_decimal"], errors="coerce")
    d = d.dropna(subset=["price_decimal","american_odds","consensus_prob","model_prob","book_count","ev","units"])

    # book coverage
    d = d[d["book_count"] >= min_books]

    # juice filter
    d = d[d["american_odds"] >= max_juice]

    # compute edge vs consensus
    d["edge_prob"] = d["model_prob"] - d["consensus_prob"]

    # filters
    d = d[(d["edge_prob"] >= min_edge) & (d["ev"] >= min_ev)].copy()

    # rank: EV > edge > units
    d["rank_score"] = (d["ev"]*0.55) + (d["edge_prob"]*0.35) + (d["units"]*0.10)
    d = d.sort_values("rank_score", ascending=False)

    # NO DUPES: 1 bet per game (keeps best book/team for that game)
    d = d.drop_duplicates(subset=["home_team","away_team"], keep="first")

    return d.head(top_n).reset_index(drop=True)

top10_strict = build_top10_ml_strict(final,
                                     min_books=3,
                                     min_edge=0.01,
                                     min_ev=0.02,
                                     max_juice=-200,
                                     top_n=10)

print("✅ strict top10 rows:", len(top10_strict))
display_cols = [
    "home_team","away_team","team","book",
    "american_odds","price_decimal",
    "consensus_prob","model_prob","edge_prob","ev","units","book_count"
]
print(top10_strict[display_cols])

In [ ]:
# ==============================================
# CELL 3 — DISCORD OUTPUT (ML ONLY)
# ==============================================

def discord_ml_card(df):
    if len(df) == 0:
        return "No ML bets passed filters (try min_books=2 or min_edge=0.005 or min_ev=0.01)."

    out = ["NCAAB — TOP 10 ML (NO DUPES | ½-KELLY)\n"]
    for i, r in df.iterrows():
        out.append(
            f"{i+1}) {r['home_team']} vs {r['away_team']}\n"
            f"Bet: {r['team']} ML\n"
            f"Book: {r['book']} | Odds: {int(r['american_odds'])} (Dec {r['price_decimal']:.2f})\n"
            f"Model: {r['model_prob']:.3f} | Consensus: {r['consensus_prob']:.3f}\n"
            f"Edge: {r['edge_prob']:.3f} | EV: {r['ev']:.3f}\n"
            f"Units (½ Kelly): {r['units']:.2f}u\n"
        )
    return "\n".join(out)

print(discord_ml_card(top10_strict))

In [ ]:
# =========================
# CELL A — WHY ONLY 4 PLAYS?
# =========================
d = final.copy()

# ML-only safety
if "market" in d.columns:
    d = d[d["market"].astype(str).str.lower().isin(["h2h","ml","moneyline"])].copy()

for c in ["american_odds","price_decimal","consensus_prob","model_prob","book_count","ev","units"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d = d.dropna(subset=["american_odds","price_decimal","consensus_prob","model_prob","book_count","ev","units"])
d["edge_prob"] = d["model_prob"] - d["consensus_prob"]

print("Total ML rows after NA drop:", len(d))
print(">=3 books:", (d["book_count"]>=3).sum())
print(">=2 books:", (d["book_count"]>=2).sum())
print("juice >= -200:", (d["american_odds"]>=-200).sum())
print("edge >= 0.01:", (d["edge_prob"]>=0.01).sum())
print("edge >= 0.005:", (d["edge_prob"]>=0.005).sum())
print("ev >= 0.02:", (d["ev"]>=0.02).sum())
print("ev >= 0.01:", (d["ev"]>=0.01).sum())

print("\nTop 15 by rank (no filters besides ML + NA drop):")
tmp = d.copy()
tmp["rank_score"] = (tmp["ev"]*0.55) + (tmp["edge_prob"]*0.35) + (tmp["units"]*0.10)
print(tmp.sort_values("rank_score", ascending=False).head(15)[
    ["home_team","away_team","team","book","american_odds","book_count","edge_prob","ev","units","rank_score"]
])

In [ ]:
# ==============================================
# CELL B — TOP 10 ML (RELAXED BUT STILL CLEAN)
# ==============================================
top10_ml = build_top10_ml_strict(
    final,
    min_books=2,        # was 3
    min_edge=0.005,     # was 0.01
    min_ev=0.01,        # was 0.02
    max_juice=-250,     # allow slightly more chalk if needed
    top_n=10
)

print("✅ Top10 ML rows:", len(top10_ml))
print(discord_ml_card(top10_ml))

In [ ]:
# =====================================
# Stable v3 "B" Blend (Model + Market)
# =====================================

import numpy as np
import pandas as pd

# If your model is not fully independent yet, keep this conservative.
BLEND_MODEL_W = 0.65
BLEND_MKT_W   = 0.35

BANKROLL_UNITS = 10.0     # scale only (10u roll baseline)
KELLY_FRACTION = 0.50     # 1/2 Kelly
MAX_UNITS_CAP  = 1.50     # cap per play for "bets that win" portfolio

def american_to_decimal(odds):
    odds = float(odds)
    if odds > 0:
        return 1.0 + (odds / 100.0)
    return 1.0 + (100.0 / abs(odds))

def american_to_implied_prob(odds):
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    return abs(odds) / (abs(odds) + 100.0)

def add_blend_ev_kelly(df):
    d = df.copy()

    # Ensure price_decimal exists
    if "price_decimal" not in d.columns or d["price_decimal"].isna().all():
        d["price_decimal"] = d["american_odds"].apply(american_to_decimal)

    d["implied_from_price"] = d["american_odds"].apply(american_to_implied_prob)

    # Blend probability (stable card)
    d["blended_prob"] = (d["model_prob"] * BLEND_MODEL_W) + (d["consensus_prob"] * BLEND_MKT_W)
    d["blended_prob"] = d["blended_prob"].clip(0.001, 0.999)

    # EV (per 1 unit risk) using decimal odds
    b = d["price_decimal"] - 1.0
    p = d["blended_prob"]
    q = 1.0 - p

    d["ev"] = (p * b) - q

    # Kelly fraction = (bp - q)/b
    d["kelly_frac"] = ((b * p) - q) / b
    d["kelly_frac"] = d["kelly_frac"].replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)

    d["units"] = (d["kelly_frac"] * BANKROLL_UNITS * KELLY_FRACTION).clip(upper=MAX_UNITS_CAP)

    # "true_edge" vs price implied (what we actually care about)
    d["true_edge"] = d["blended_prob"] - d["implied_from_price"]

    return d

final = add_blend_ev_kelly(final)

print("✅ Blend built. Columns now include: blended_prob, ev, kelly_frac, units, true_edge")
final.head(3)

In [ ]:
# =====================================
# Stable v3 Filters: "bets that win"
# =====================================

MAX_DOG = 300          # remove +400/+900 type stuff
MAX_JUICE = -200       # remove -250/-500
MIN_WIN_PROB = 0.52    # want higher hit-rate
MIN_TRUE_EDGE = 0.010  # at least +1.0% vs implied
MIN_EV = 0.005         # at least +0.5% EV per 1u risk

def stable_filters(df):
    d = df.copy()

    # must be ML rows only: in your data ML has spread NaN
    if "spread" in d.columns:
        d = d[d["spread"].isna()].copy()

    d = d.dropna(subset=["american_odds","blended_prob","ev","true_edge"])

    # odds filters
    d = d[(d["american_odds"] >= MAX_JUICE) & (d["american_odds"] <= MAX_DOG)].copy()

    # quality filters
    d = d[(d["blended_prob"] >= MIN_WIN_PROB) &
          (d["true_edge"] >= MIN_TRUE_EDGE) &
          (d["ev"] >= MIN_EV)].copy()

    # best book per team per game: keep highest EV then highest true_edge
    d = (d.sort_values(["ev","true_edge","units"], ascending=False)
           .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
           .copy())

    return d

stable_ml = stable_filters(final)

print("✅ Stable ML candidates:", stable_ml.shape)
stable_ml[["home_team","away_team","team","book","american_odds","blended_prob","true_edge","ev","units"]].head(15)

In [ ]:
# =====================================
# Top 10 Stable ML Card (Discord)
# =====================================

top10 = (stable_ml.sort_values(["units","ev","true_edge"], ascending=False)
                 .head(10)
                 .copy())

def fmt_odds(x):
    x = int(round(float(x)))
    return f"+{x}" if x > 0 else f"{x}"

lines = []
lines.append("NCAAB — TOP 10 ML (STABLE BLEND | NO DUPES | ½-KELLY)\n")

if len(top10) == 0:
    lines.append("No plays passed the 'bets that win' filters.\n")
    lines.append("Try loosening: MIN_WIN_PROB to 0.50 or MAX_DOG to 450.\n")
else:
    for i, r in enumerate(top10.itertuples(index=False), start=1):
        matchup = f"{r.home_team} vs {r.away_team}"
        bet = f"{r.team} ML"
        lines.append(
            f"{i}) {matchup}\n"
            f"Bet: {bet}\n"
            f"Book: {r.book} | Odds: {fmt_odds(r.american_odds)} (Dec {r.price_decimal:.2f})\n"
            f"Win% (Blend): {r.blended_prob:.3f} | Price Implied: {r.implied_from_price:.3f}\n"
            f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f}\n"
            f"Units (½ Kelly, capped): {r.units:.2f}u\n"
        )

print("\n".join(lines))

In [ ]:
# =====================================
# STABLE SPREAD ENGINE
# =====================================

# loosen win prob because spreads center near 50%
MIN_SP_EDGE = 0.02      # need 2% edge vs consensus spread prob
MIN_SP_EV = 0.01
MAX_SP_JUICE = -130

sp = final.copy()

# SPREAD ONLY (spread not null)
sp = sp[sp["spread"].notna()].copy()

sp = sp.dropna(subset=["blended_prob","american_odds","price_decimal"])

# juice filter
sp = sp[sp["american_odds"] >= MAX_SP_JUICE]

# recompute implied
sp["implied_from_price"] = sp["american_odds"].apply(american_to_implied_prob)

# edge vs price
sp["true_edge"] = sp["blended_prob"] - sp["implied_from_price"]

# EV
b = sp["price_decimal"] - 1
p = sp["blended_prob"]
q = 1 - p
sp["ev"] = (p * b) - q

# filter
sp = sp[(sp["true_edge"] >= MIN_SP_EDGE) & (sp["ev"] >= MIN_SP_EV)].copy()

# best book per team/game
sp = (sp.sort_values(["ev","true_edge"], ascending=False)
        .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

top10_sp = sp.sort_values(["ev","true_edge"], ascending=False).head(10)

print("✅ Spread candidates:", len(top10_sp))
top10_sp[["home_team","away_team","team","spread","book","american_odds","blended_prob","true_edge","ev"]]

In [ ]:
# =====================================
# STEP A: Build SPREAD dataset from odds_long
# (Fixes KeyError: 'spread' by using the right DF)
# =====================================

import numpy as np
import pandas as pd

# 1) Pick the correct long odds DF
# Try common names in your notebook
_odds_long = None
for _name in ["odds_long", "normalized", "odds_df_long", "odds_long_df"]:
    if _name in globals() and isinstance(globals()[_name], pd.DataFrame):
        _odds_long = globals()[_name]
        print(f"✅ Using spread source DF: {_name} | rows={len(_odds_long)}")
        break

if _odds_long is None:
    raise ValueError("Couldn't find odds_long/normalized DF. Paste/define your odds_long first.")

# 2) Helpers
def american_to_decimal(a):
    a = float(a)
    if a >= 0:
        return 1 + (a / 100.0)
    return 1 + (100.0 / abs(a))

def american_to_implied_prob(a):
    a = float(a)
    if a >= 0:
        return 100.0 / (a + 100.0)
    return abs(a) / (abs(a) + 100.0)

def weighted_consensus_prob(df_group):
    # weight by "book strength" if you have it; otherwise equal weight
    if "book_strength" in df_group.columns:
        w = df_group["book_strength"].fillna(1.0).clip(lower=0.1)
        return np.average(df_group["implied_prob"], weights=w)
    return df_group["implied_prob"].mean()

# 3) Build SPREAD-only DF
sp = _odds_long.copy()

# normalize column names we expect
# (your odds_long shows: market, spread, american_odds, price_decimal, implied_prob)
required = ["home_team","away_team","commence_time","book","market","team","american_odds","spread"]
missing = [c for c in required if c not in sp.columns]
if missing:
    raise ValueError(f"Spread DF missing columns: {missing}. Columns present: {list(sp.columns)}")

sp = sp[sp["market"].astype(str).str.lower().isin(["spreads","spread"])].copy()
sp = sp[sp["spread"].notna()].copy()

# ensure numeric
sp["american_odds"] = pd.to_numeric(sp["american_odds"], errors="coerce")
sp["spread"] = pd.to_numeric(sp["spread"], errors="coerce")

# implied prob from odds
if "implied_prob" not in sp.columns or sp["implied_prob"].isna().all():
    sp["implied_prob"] = sp["american_odds"].apply(american_to_implied_prob)

# decimal odds
if "price_decimal" not in sp.columns or sp["price_decimal"].isna().all():
    sp["price_decimal"] = sp["american_odds"].apply(american_to_decimal)

# de-dupe exact same line
sp = sp.drop_duplicates(subset=["home_team","away_team","team","book","spread","american_odds"]).copy()

print(f"✅ SPREAD lines ready: rows={len(sp)} | games={sp[['home_team','away_team']].drop_duplicates().shape[0]}")

# 4) Build spread consensus by (game, team, spread)
cons_sp = (
    sp.groupby(["home_team","away_team","team","spread"], as_index=False)
      .agg(consensus_prob=("implied_prob", "mean"),
           book_count=("book","nunique"))
)
print(f"✅ cons_sp built: {cons_sp.shape}")

# 5) Attach model probability for cover
# EXPECTATION: you have a model probability column for spread (cover) somewhere.
# We'll search for a good candidate column automatically.
model_candidates = [c for c in sp.columns if "model" in c.lower() and "prob" in c.lower()]
print("🔎 Model prob candidates in spread DF:", model_candidates)

# pick best candidate
model_col = None
for c in ["model_prob_spread","spread_model_prob","model_prob_cover","sim_prob_cover","model_prob"]:
    if c in sp.columns:
        model_col = c
        break
if model_col is None and model_candidates:
    model_col = model_candidates[0]

if model_col is None:
    raise ValueError(
        "No spread model probability column found in spread DF.\n"
        "You need a column like model_prob_spread / sim_prob_cover / model_prob_cover."
    )

# build model layer at same grain (game, team, spread)
model_sp = (
    sp.groupby(["home_team","away_team","team","spread"], as_index=False)
      .agg(model_prob=(model_col, "mean"))
)
print(f"✅ model_sp built using '{model_col}': {model_sp.shape}")

# 6) Merge model + consensus, compute EV + 1/2 Kelly, pick best book line per (game, team)
merged = cons_sp.merge(model_sp, on=["home_team","away_team","team","spread"], how="inner")
merged["edge"] = merged["model_prob"] - merged["consensus_prob"]

# reattach BEST available book/price for that (game, team, spread) by max EV
sp2 = sp.merge(merged[["home_team","away_team","team","spread","model_prob","consensus_prob","edge"]],
               on=["home_team","away_team","team","spread"], how="inner")

# EV and 1/2 Kelly
b = sp2["price_decimal"] - 1
p = sp2["model_prob"]
q = 1 - p
sp2["ev"] = (p * b) - q

# kelly fraction = (bp - q)/b ; then half-kelly units with cap
sp2["kelly_fraction"] = ((b * p) - q) / b
sp2["units"] = (0.5 * sp2["kelly_fraction"]).clip(lower=0, upper=0.5)  # cap at 0.5u default

# Filters: "bets that win" for spreads (reasonable)
MIN_MODEL_PROB = 0.53
MIN_EDGE = 0.02
MAX_JUICE = -130

sp2_f = sp2[
    (sp2["model_prob"] >= MIN_MODEL_PROB) &
    (sp2["edge"] >= MIN_EDGE) &
    (sp2["american_odds"] >= MAX_JUICE) &
    (sp2["ev"] > 0)
].copy()

# pick best book per (game, team, spread)
sp2_best = (
    sp2_f.sort_values(["ev","edge","units"], ascending=False)
         .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first")
)

top10_spread = sp2_best.sort_values(["ev","edge"], ascending=False).head(10).copy()

print("✅ TOP SPREAD PLAYS:", len(top10_spread))
top10_spread[["home_team","away_team","team","spread","book","american_odds","model_prob","consensus_prob","edge","ev","units"]]

In [ ]:
# =====================================
# STEP B: Attach / Build spread model_prob (cover) so spread edges can run
# =====================================

import numpy as np
import pandas as pd

# --- 1) Locate odds_long again ---
if "odds_long" not in globals() or not isinstance(odds_long, pd.DataFrame):
    raise ValueError("odds_long not found. You already had it—make sure that cell ran.")

# --- 2) Search for an existing DF that has spread model probabilities ---
spread_prob_cols = {"model_prob_spread", "sim_prob_cover", "model_prob_cover", "spread_cover_prob", "cover_prob"}
candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(map(str, obj.columns))
        if {"home_team","away_team","team"}.issubset(cols) and (cols & spread_prob_cols):
            # also prefer if it has a spread column
            has_spread = "spread" in cols
            candidates.append((name, len(obj), list(cols & spread_prob_cols), has_spread))

candidates = sorted(candidates, key=lambda x: (x[3], x[1]), reverse=True)

print("🔎 Spread-model DF candidates:")
for c in candidates[:10]:
    print(" -", c)

# --- 3) If we found a candidate, merge it in ---
if candidates:
    best_name, _, best_cols, _ = candidates[0]
    best_col = best_cols[0]
    src = globals()[best_name].copy()

    # If src has multiple rows per key, average
    key_cols = ["home_team","away_team","team"]
    if "spread" in src.columns:
        key_cols = ["home_team","away_team","team","spread"]

    src = (
        src[key_cols + [best_col]]
        .dropna(subset=[best_col])
        .groupby(key_cols, as_index=False)[best_col].mean()
        .rename(columns={best_col: "model_prob_spread"})
    )

    # Merge into odds_long spread rows
    odds_long2 = odds_long.copy()
    odds_long2["market"] = odds_long2["market"].astype(str).str.lower()
    is_spread = odds_long2["market"].isin(["spreads","spread"]) & odds_long2["spread"].notna()

    if "spread" in key_cols:
        odds_long2.loc[is_spread, "model_prob_spread"] = odds_long2.loc[is_spread].merge(
            src, on=["home_team","away_team","team","spread"], how="left"
        )["model_prob_spread"].values
    else:
        odds_long2.loc[is_spread, "model_prob_spread"] = odds_long2.loc[is_spread].merge(
            src, on=["home_team","away_team","team"], how="left"
        )["model_prob_spread"].values

    odds_long = odds_long2
    print(f"✅ Merged model_prob_spread from '{best_name}' ({best_col}).")
    print("Spread model_prob_spread non-null:", odds_long.loc[is_spread, "model_prob_spread"].notna().sum(), "/", is_spread.sum())

else:
    # --- 4) No candidate found: build a TEMP proxy model_prob_spread ---
    # This is NOT your full Stable model — it's a placeholder so your pipeline runs end-to-end.
    # Replace this later with real cover probs from your sim engine.

    print("⚠ No existing spread model probability DF found.")
    print("⚠ Building TEMP proxy model_prob_spread so you can continue (swap later).")

    odds_long2 = odds_long.copy()
    odds_long2["market"] = odds_long2["market"].astype(str).str.lower()
    is_spread = odds_long2["market"].isin(["spreads","spread"]) & odds_long2["spread"].notna()

    # proxy: favorites (negative spread) get higher cover prob; dogs lower
    # slope tuned mildly so we don't force extremes
    k = 0.18  # slope
    x = -pd.to_numeric(odds_long2.loc[is_spread, "spread"], errors="coerce").fillna(0.0)
    p = 1 / (1 + np.exp(-k * x))

    # clamp to avoid 0/1
    odds_long2.loc[is_spread, "model_prob_spread"] = np.clip(p, 0.05, 0.95)

    odds_long = odds_long2
    print("✅ TEMP model_prob_spread created on spread rows.")
    print("Spread model_prob_spread non-null:", odds_long.loc[is_spread, "model_prob_spread"].notna().sum(), "/", is_spread.sum())

# Quick peek
print("\nSample spread rows with model_prob_spread:")
display(odds_long[(odds_long["market"].astype(str).str.lower().isin(["spreads","spread"])) & odds_long["spread"].notna()]
      [["home_team","away_team","book","team","spread","american_odds","model_prob_spread"]]
      .head(10))

In [ ]:
# ================================
# NEXT: Compute Spread Edges, EV, Half-Kelly, Top10 (no dupes)
# ================================
import numpy as np
import pandas as pd

# safety checks
if "odds_long" not in globals() or not isinstance(odds_long, pd.DataFrame):
    raise ValueError("odds_long not found. Run the odds ingestion cell first.")

od = odds_long.copy()
od["market"] = od["market"].astype(str).str.lower()

# Prefer a decimal price column if present (price_decimal), else try to compute from american odds
def american_to_decimal(a):
    try:
        a = float(a)
    except:
        return np.nan
    if a > 0:
        return 1 + a/100.0
    else:
        return 1 + 100.0/(-a)

if "price_decimal" not in od.columns:
    if "american_odds" in od.columns:
        od["price_decimal"] = od["american_odds"].apply(american_to_decimal)
    else:
        od["price_decimal"] = np.nan

# implied prob from posted price (use price_decimal if available)
od["implied_prob_post"] = np.where(
    od["price_decimal"].notna() & (od["price_decimal"]>0),
    1.0 / od["price_decimal"],
    np.nan
)

# --- Build spread consensus (by game/team/spread) if not already present ---
if "consensus_prob" not in od.columns:
    # build spread consensus from spread rows only (average implied_prob_post across books)
    spread_rows = od[od["market"].isin(["spreads","spread"]) & od["spread"].notna()].copy()
    if spread_rows.shape[0] == 0:
        print("⚠ No spread rows found in odds_long for consensus building.")
        spread_cons = pd.DataFrame(columns=["home_team","away_team","team","spread","consensus_prob","book_count"])
    else:
        spread_cons = (
            spread_rows
            .groupby(["home_team","away_team","team","spread"], as_index=False)
            .agg(consensus_prob=("implied_prob_post","mean"),
                 book_count=("book","nunique"))
        )
    # merge back
    od = od.merge(spread_cons, on=["home_team","away_team","team","spread"], how="left")
else:
    spread_cons = od[["home_team","away_team","team","spread","consensus_prob"]].drop_duplicates()

# ensure model_prob_spread exists
if "model_prob_spread" not in od.columns:
    raise ValueError("model_prob_spread missing from odds_long — run the previous bridge cell again.")

# FILTER: spread rows only for spread analysis
sp = od[od["market"].isin(["spreads","spread"]) & od["spread"].notna()].copy()

# compute edge = model_prob_spread - consensus_prob
sp["consensus_prob"] = sp["consensus_prob"].astype(float)
sp["model_prob_spread"] = sp["model_prob_spread"].astype(float)
sp["edge"] = sp["model_prob_spread"] - sp["consensus_prob"]

# compute implied from posted price (fallbacks handled above)
sp["implied_from_price"] = sp["implied_prob_post"]

# EV% (simple) = model_prob_spread - implied_from_price
sp["ev_pct"] = sp["model_prob_spread"] - sp["implied_from_price"]

# Kelly fraction (fraction of bankroll) using decimal odds b = decimal-1
# k = (p*(b) - q) / b  where q = 1-posterior? We'll use market implied q = implied_from_price
def safe_kelly(row):
    p = row["model_prob_spread"]
    q = row["implied_from_price"]
    dec = row.get("price_decimal", np.nan)
    if pd.isna(p) or pd.isna(q) or pd.isna(dec) or dec<=1:
        return 0.0
    b = dec - 1.0
    # Kelly formula numerator (p*b - q) ; denom b
    num = p*b - q
    den = b
    k = num/den if den!=0 else 0.0
    # clamp
    return max(min(k, 1.0), -1.0)

sp["kelly_frac"] = sp.apply(safe_kelly, axis=1)
# use half-Kelly units by convention; convert to units in fractional bankroll (user can scale later)
sp["kelly_units_half"] = sp["kelly_frac"] * 0.5

# normalize and ranking for dedupe selection: prefer greatest edge then EV then book_count
sp["edge_rank"] = sp["edge"].rank(method="dense", ascending=False)
sp["ev_rank"] = sp["ev_pct"].rank(method="dense", ascending=False)

# Keep best book per game/team/spread by edge (no dupes across books for same team+spread)
sp_best = (
    sp.sort_values(["edge","ev_pct"], ascending=[False,False])
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
      .copy()
)

# Top 10 spreads
top_spreads = sp_best.sort_values("edge", ascending=False).head(10).reset_index(drop=True)

# --- ML (h2h) side: attach model_prob for h2h if exists in odds_long or another DF ---
# We try to find a model column for ML in globals (model_prob, model_prob_ml, blended_prob, etc.)
ml_prob_cols = {"model_prob","model_prob_ml","blended_prob","model_prob_h2h"}
ml_source = None
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(map(str, obj.columns))
        if {"home_team","away_team","team"} .issubset(cols) and (cols & ml_prob_cols):
            ml_source = (name, globals()[name])
            break

# Build ML rows
ml = od[od["market"].isin(["h2h","moneyline","ml","mline","h2h_outcome"]) | (od["market"]=="h2h")].copy()
# If ml has model_prob column already
if "model_prob" in ml.columns:
    ml["model_prob_h2h"] = ml["model_prob"]
elif ml_source is not None:
    srcname, srcdf = ml_source
    # merge model prob from source
    cand = srcdf[["home_team","away_team","team"] + list(set(srcdf.columns) & ml_prob_cols)].copy()
    # pick first matching col
    col_found = list(set(cand.columns) & ml_prob_cols)[0]
    cand = cand.rename(columns={col_found: "model_prob_h2h"})
    ml = ml.merge(cand[["home_team","away_team","team","model_prob_h2h"]], on=["home_team","away_team","team"], how="left")
    print(f"✅ Merged ML probs from {srcname} using column {col_found}.")
else:
    print("⚠ No ML model probability source found. ML list will be generated using consensus/model where available.")
    ml["model_prob_h2h"] = np.nan

# compute implied prob for ML
ml["implied_prob_post"] = ml["implied_prob_post"].astype(float)
# if model_prob_h2h missing, try to derive from spread model: use logistic of spread margin if available
ml_missing = ml["model_prob_h2h"].isna()
if ml_missing.any():
    # simple proxy: use mean of related spread model rows for same game/team (if exist)
    proxy = sp_best.groupby(["home_team","away_team","team"], as_index=False)["model_prob_spread"].mean().rename(columns={"model_prob_spread":"proxy_spread_prob"})
    ml = ml.merge(proxy, on=["home_team","away_team","team"], how="left")
    ml["model_prob_h2h"] = ml["model_prob_h2h"].fillna(ml["proxy_spread_prob"])
    ml.drop(columns=["proxy_spread_prob"], inplace=True, errors="ignore")

# Now compute ML edge and half-Kelly similarly
ml["edge"] = ml["model_prob_h2h"] - ml["implied_prob_post"]
def safe_kelly_ml(row):
    p = row["model_prob_h2h"]
    q = row["implied_prob_post"]
    dec = row.get("price_decimal", np.nan)
    if pd.isna(p) or pd.isna(q) or pd.isna(dec) or dec<=1:
        return 0.0
    b = dec - 1.0
    num = p*b - q
    den = b
    k = num/den if den!=0 else 0.0
    return max(min(k,1.0), -1.0)

ml["kelly_frac"] = ml.apply(safe_kelly_ml, axis=1)
ml["kelly_units_half"] = ml["kelly_frac"] * 0.5

# dedupe ML best book per game/team
ml_best = ml.sort_values(["edge","implied_prob_post"], ascending=[False,True]).drop_duplicates(subset=["home_team","away_team","team"], keep="first")
top_ml = ml_best.sort_values("edge", ascending=False).head(10).reset_index(drop=True)

# Format outputs: Discord style
def format_row_sp(row):
    units = round(float(row.get("kelly_units_half", 0.0)), 2)
    odds = row.get("american_odds", row.get("price_decimal", "N/A"))
    odds_str = int(odds) if (pd.notna(odds) and float(odds).is_integer()) else odds
    return f"🏀 {row['home_team']} vs {row['away_team']}\nBet: {row['team']} {row['spread']:+}\nBook: {row['book']} | Odds: {odds_str}\nModel Prob: {round(row['model_prob_spread'],3)} | Consensus: {round(row['consensus_prob'],3) if pd.notna(row['consensus_prob']) else 'N/A'}\nEdge: {round(row['edge'],3)} | Units (½-Kelly): {units}u\n"

def format_row_ml(row):
    units = round(float(row.get("kelly_units_half", 0.0)), 2)
    odds = row.get("american_odds", row.get("price_decimal", "N/A"))
    odds_str = int(odds) if (pd.notna(odds) and float(odds).is_integer()) else odds
    return f"🏀 {row['home_team']} vs {row['away_team']}\nBet: {row['team']} ML\nBook: {row['book']} | Odds: {odds_str}\nModel Prob: {round(row.get('model_prob_h2h',np.nan),3)} | Consensus: {round(row.get('consensus_prob',np.nan),3) if 'consensus_prob' in row else 'N/A'}\nEdge: {round(row['edge'],3)} | Units (½-Kelly): {units}u\n"

# Print top 10 spreads
print("\n🔥 TOP 10 SPREADS (no dupes) 🔥\n")
for i, r in top_spreads.iterrows():
    print(f"{i+1}) {format_row_sp(r)}")

# Print top 10 ML
print("\n🔥 TOP 10 ML (no dupes) 🔥\n")
if top_ml.shape[0]==0:
    print("⚠ No ML rows with model probabilities found. Ensure a model DF with ML probs (model_prob) is available or run your model output cell.")
else:
    for i, r in top_ml.iterrows():
        print(f"{i+1}) {format_row_ml(r)}")

# expose for later cells
top_spreads_df = top_spreads.copy()
top_ml_df = top_ml.copy()

print("\n✅ Done. Variables available: top_spreads_df, top_ml_df, sp_best, ml_best")

In [ ]:
# ==========================================
# FIX CELL: Normalize odds, build correct consensus, +EV only, +Kelly only, no dupes
# Requires: odds_long (your long odds DF)
# Outputs: top10_ml_clean, top10_sp_clean
# ==========================================
import numpy as np
import pandas as pd

if "odds_long" not in globals() or not isinstance(odds_long, pd.DataFrame):
    raise ValueError("odds_long not found.")

df = odds_long.copy()
df["market"] = df["market"].astype(str).str.lower()

# ---------
# 1) Odds normalization
# ---------
# Your odds_long sometimes has american_odds as decimal (ex 1.95) and sometimes American (ex -110, +450).
# We'll normalize to:
# - decimal_odds
# - american_odds_amer
# - implied_prob_post (from decimal)
def dec_to_american(dec):
    if pd.isna(dec) or dec <= 1:
        return np.nan
    if dec >= 2:
        return int(round((dec - 1) * 100))
    # favorites
    return int(round(-100 / (dec - 1)))

def amer_to_dec(amer):
    if pd.isna(amer):
        return np.nan
    amer = float(amer)
    if amer > 0:
        return 1 + amer/100.0
    else:
        return 1 + 100.0/(-amer)

def is_decimal_like(x):
    try:
        x = float(x)
    except:
        return False
    # decimal odds are usually between 1.01 and, say, 100
    return (x > 1.0) and (x < 100.0) and (abs(x) < 20.0)  # catches 1.87, 10.2 etc

# Start from a "price_raw" if present, else american_odds
price_src = "price_decimal" if "price_decimal" in df.columns else ("price_raw" if "price_raw" in df.columns else "american_odds")
df["price_src"] = df[price_src]

# Build decimal_odds
df["decimal_odds"] = np.nan
# If already decimal column exists and looks valid, use it
if "price_decimal" in df.columns:
    df.loc[df["price_decimal"].notna() & (df["price_decimal"] > 1), "decimal_odds"] = df["price_decimal"]

# Otherwise infer from american_odds field:
# If american_odds looks like decimal (1.95) treat as decimal, else treat as American
mask_dec = df["american_odds"].apply(is_decimal_like) if "american_odds" in df.columns else pd.Series(False, index=df.index)
if "american_odds" in df.columns:
    df.loc[df["decimal_odds"].isna() & mask_dec, "decimal_odds"] = df.loc[mask_dec, "american_odds"].astype(float)
    df.loc[df["decimal_odds"].isna() & (~mask_dec), "decimal_odds"] = df.loc[~mask_dec, "american_odds"].apply(amer_to_dec)

# Build american odds (proper)
df["american_odds_amer"] = df["decimal_odds"].apply(dec_to_american)

# Implied prob from posted price
df["implied_prob_post"] = np.where(df["decimal_odds"].notna() & (df["decimal_odds"] > 1),
                                   1.0 / df["decimal_odds"],
                                   np.nan)

# ---------
# 2) Split markets
# ---------
ml = df[df["market"].isin(["h2h", "moneyline"])].copy()
sp = df[df["market"].isin(["spreads", "spread"]) & df["spread"].notna()].copy()

print(f"✅ Clean ML rows: {ml.shape} | Clean Spread rows: {sp.shape}")

# ---------
# 3) Build CONSENSUS correctly
#   - ML consensus: avg implied_prob_post by game/team
#   - Spread consensus: avg implied_prob_post by game/team/spread
# ---------
cons_ml = (
    ml.dropna(subset=["implied_prob_post"])
      .groupby(["home_team","away_team","team"], as_index=False)
      .agg(consensus_prob=("implied_prob_post","mean"),
           book_count=("book","nunique"))
)
cons_sp = (
    sp.dropna(subset=["implied_prob_post"])
      .groupby(["home_team","away_team","team","spread"], as_index=False)
      .agg(consensus_prob=("implied_prob_post","mean"),
           book_count=("book","nunique"))
)

ml = ml.merge(cons_ml, on=["home_team","away_team","team"], how="left")
sp = sp.merge(cons_sp, on=["home_team","away_team","team","spread"], how="left")

# ---------
# 4) Attach MODEL probabilities
#   - ML: must have model_prob in ml OR merge from your model_df if it exists
#   - Spread: must have model_prob_spread (your bridge created it). If not present → stop.
# ---------
# ML model prob
if "model_prob" in ml.columns and ml["model_prob"].notna().any():
    ml["model_prob_ml"] = ml["model_prob"].astype(float)
else:
    # try model_df in globals
    model_df = globals().get("model_df", None)
    if isinstance(model_df, pd.DataFrame) and {"home_team","away_team","team","model_prob"}.issubset(model_df.columns):
        ml = ml.merge(model_df[["home_team","away_team","team","model_prob"]].rename(columns={"model_prob":"model_prob_ml"}),
                      on=["home_team","away_team","team"], how="left")
    else:
        ml["model_prob_ml"] = np.nan

# Spread model prob
if "model_prob_spread" not in sp.columns:
    raise ValueError("model_prob_spread is missing. You need your spread-cover model output column to proceed.")
sp["model_prob_spread"] = sp["model_prob_spread"].astype(float)

# ---------
# 5) Edge + EV + Half-Kelly
#   IMPORTANT: We will NOT output negative EV / negative Kelly plays.
# ---------
def half_kelly(p, dec):
    if pd.isna(p) or pd.isna(dec) or dec <= 1:
        return 0.0
    b = dec - 1.0
    q = 1 - p
    k = (p*b - q) / b
    k = max(k, 0.0)          # clamp negatives to 0
    return 0.5 * k           # half kelly

# ML metrics (true edge vs price, plus market edge vs consensus)
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob_post"]
ml["cons_edge"] = ml["model_prob_ml"] - ml["consensus_prob"]
ml["ev_pct"] = (ml["model_prob_ml"] * ml["decimal_odds"] - 1.0)  # EV as ROI
ml["units"] = ml.apply(lambda r: half_kelly(r["model_prob_ml"], r["decimal_odds"]), axis=1)

# Spread metrics
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob_post"]
sp["cons_edge"] = sp["model_prob_spread"] - sp["consensus_prob"]
sp["ev_pct"] = (sp["model_prob_spread"] * sp["decimal_odds"] - 1.0)
sp["units"] = sp.apply(lambda r: half_kelly(r["model_prob_spread"], r["decimal_odds"]), axis=1)

# ---------
# 6) Filters: "bets that win" (+EV only, units>0, and not insane dogs)
# tweak thresholds here
# ---------
MIN_UNITS = 0.01
MIN_EV = 0.0
MIN_WINPROB_ML = 0.50     # you can tighten (0.55/0.60) if you want safer
MIN_WINPROB_SP = 0.52     # spreads typically hover near 0.50

MAX_DOG_AMER = 600        # cap huge dogs; raise/lower if you want

ml_f = ml.dropna(subset=["model_prob_ml","decimal_odds","implied_prob_post"]).copy()
ml_f = ml_f[(ml_f["ev_pct"] > MIN_EV) & (ml_f["units"] >= MIN_UNITS) & (ml_f["model_prob_ml"] >= MIN_WINPROB_ML)]
ml_f = ml_f[ml_f["american_odds_amer"].abs().notna()]
ml_f = ml_f[(ml_f["american_odds_amer"] <= MAX_DOG_AMER) | (ml_f["american_odds_amer"] < 0)]

sp_f = sp.dropna(subset=["model_prob_spread","decimal_odds","implied_prob_post"]).copy()
sp_f = sp_f[(sp_f["ev_pct"] > MIN_EV) & (sp_f["units"] >= MIN_UNITS) & (sp_f["model_prob_spread"] >= MIN_WINPROB_SP)]

# ---------
# 7) NO DUPES: pick best book per game/team (ML) and per game/team (Spread)
# rank by units then EV then true_edge
# ---------
ml_best = (
    ml_f.sort_values(["units","ev_pct","true_edge"], ascending=False)
        .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)
sp_best = (
    sp_f.sort_values(["units","ev_pct","true_edge"], ascending=False)
        .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

top10_ml_clean = ml_best.sort_values(["units","ev_pct"], ascending=False).head(10).reset_index(drop=True)
top10_sp_clean = sp_best.sort_values(["units","ev_pct"], ascending=False).head(10).reset_index(drop=True)

print(f"✅ Top10 ML (clean): {top10_ml_clean.shape[0]} | ✅ Top10 Spread (clean): {top10_sp_clean.shape[0]}")

# ---------
# 8) Discord-style print (with American odds + units)
# ---------
def fmt_amer(a):
    if pd.isna(a): return "N/A"
    a = int(a)
    return f"+{a}" if a > 0 else f"{a}"

def print_ml(df_out):
    if df_out.shape[0] == 0:
        print("\n⚠ No ML plays passed filters. (Either model_prob_ml looks wrong OR EV/units are too strict.)\n")
        return
    print("\n🔥 NCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)\n")
    for i, r in df_out.iterrows():
        print(
f"{i+1}) {r['home_team']} vs {r['away_team']}\n"
f"Bet: {r['team']} ML\n"
f"Book: {r['book']} | Odds: {fmt_amer(r['american_odds_amer'])} (Dec {r['decimal_odds']:.2f})\n"
f"Model: {r['model_prob_ml']:.3f} | Consensus: {r['consensus_prob']:.3f}\n"
f"True Edge (vs price): {r['true_edge']:.3f} | EV: {r['ev_pct']:.3f}\n"
f"Units (½ Kelly): {r['units']:.2f}u\n"
        )

def print_sp(df_out):
    if df_out.shape[0] == 0:
        print("\n⚠ No SPREAD plays passed filters. (Either model_prob_spread is proxy/noisy OR EV/units too strict.)\n")
        return
    print("\n🔥 NCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)\n")
    for i, r in df_out.iterrows():
        print(
f"{i+1}) {r['home_team']} vs {r['away_team']}\n"
f"Bet: {r['team']} {r['spread']:+}\n"
f"Book: {r['book']} | Odds: {fmt_amer(r['american_odds_amer'])} (Dec {r['decimal_odds']:.2f})\n"
f"Model: {r['model_prob_spread']:.3f} | Consensus: {r['consensus_prob']:.3f}\n"
f"True Edge (vs price): {r['true_edge']:.3f} | EV: {r['ev_pct']:.3f}\n"
f"Units (½ Kelly): {r['units']:.2f}u\n"
        )

print_ml(top10_ml_clean)
print_sp(top10_sp_clean)

# expose for next cells

In [ ]:
print("ML model prob stats:")
print(ml["model_prob_ml"].describe())

print("\nSpread model prob stats:")
print(sp["model_prob_spread"].describe())

In [ ]:
print(model_df.head())
print(model_df["model_prob"].describe())

In [ ]:
# ==========================================
# TRUE NO-VIG ML CONSENSUS REBUILD
# ==========================================

ml = df[df["market"].isin(["h2h","moneyline"])].copy()

# keep only rows with valid decimal odds
ml = ml.dropna(subset=["decimal_odds"])

# implied prob
ml["imp"] = 1 / ml["decimal_odds"]

# normalize per book per game (remove vig)
ml["no_vig_prob"] = 0.0

for (ht, at, book), group in ml.groupby(["home_team","away_team","book"]):
    total = group["imp"].sum()
    if total > 0:
        ml.loc[group.index, "no_vig_prob"] = group["imp"] / total

# now build consensus from no-vig
cons_ml_true = (
    ml.groupby(["home_team","away_team","team"], as_index=False)
      .agg(consensus_prob=("no_vig_prob","mean"),
           book_count=("book","nunique"))
)

print("✅ True ML consensus built:", cons_ml_true.shape)
print(cons_ml_true["consensus_prob"].describe())

In [ ]:
# ==========================================
# REBUILD ML WITH TRUE CONSENSUS
# ==========================================

# Start fresh ML from normalized df
ml_clean = df[df["market"].isin(["h2h","moneyline"])].copy()
ml_clean = ml_clean.dropna(subset=["decimal_odds"])

# merge TRUE consensus
ml_clean = ml_clean.merge(
    cons_ml_true[["home_team","away_team","team","consensus_prob"]],
    on=["home_team","away_team","team"],
    how="left"
)

# attach model probabilities (use your model_layer_df)
ml_clean = ml_clean.merge(
    model_layer_df[["home_team","away_team","team","model_prob"]],
    on=["home_team","away_team","team"],
    how="left"
)

# implied prob from price
ml_clean["implied_prob"] = 1 / ml_clean["decimal_odds"]

# true edge vs price
ml_clean["true_edge"] = ml_clean["model_prob"] - ml_clean["implied_prob"]

# EV
ml_clean["ev"] = ml_clean["model_prob"] * ml_clean["decimal_odds"] - 1

# half Kelly
def half_kelly(p, dec):
    b = dec - 1
    q = 1 - p
    k = (p*b - q) / b
    return max(0, 0.5*k)

ml_clean["units"] = ml_clean.apply(lambda r: half_kelly(r["model_prob"], r["decimal_odds"]), axis=1)

# Filter: +EV only
ml_clean = ml_clean[(ml_clean["ev"] > 0) & (ml_clean["units"] > 0)]

# No dupes (best book per team/game)
ml_best = (
    ml_clean.sort_values(["units","ev"], ascending=False)
            .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

top10_ml_fixed = ml_best.sort_values(["units","ev"], ascending=False).head(10)

print("✅ Clean Top 10 ML:", top10_ml_fixed.shape)
top10_ml_fixed[[
    "home_team","away_team","team","book",
    "decimal_odds","model_prob","consensus_prob",
    "true_edge","ev","units"
]]

In [ ]:
print(model_layer_df.head(20))

In [ ]:
# ============================================
# STABLE v3 — HARD RESET AUDIT + TOP10 BUILDER
# (ML + SPREAD | NO DUPES | 1/2 KELLY)
# Paste + Run
# ============================================

import numpy as np
import pandas as pd

# ----------------------------
# Helpers: price normalization
# ----------------------------
def is_decimal_price(x: float) -> bool:
    # Decimal odds are typically between 1.01 and ~30 for most markets
    return pd.notna(x) and (0 < float(x) < 30)

def dec_to_american(dec: float) -> float:
    dec = float(dec)
    if dec <= 1:
        return np.nan
    if dec >= 2:
        return round((dec - 1) * 100, 0)
    else:
        return round(-100 / (dec - 1), 0)

def american_to_dec(amer: float) -> float:
    amer = float(amer)
    if amer == 0:
        return np.nan
    if amer > 0:
        return 1 + amer / 100
    return 1 + 100 / abs(amer)

def implied_from_dec(dec: float) -> float:
    dec = float(dec)
    if dec <= 1:
        return np.nan
    return 1 / dec

def implied_from_american(amer: float) -> float:
    amer = float(amer)
    if amer == 0:
        return np.nan
    if amer > 0:
        return 100 / (amer + 100)
    return abs(amer) / (abs(amer) + 100)

def kelly_half_units(p: float, dec: float, unit_cap: float = 1.5) -> float:
    # Half Kelly fraction converted to "units" (1 unit = full bankroll fraction 1.0)
    # fraction = (b*p - q)/b where b=dec-1, q=1-p
    if pd.isna(p) or pd.isna(dec) or dec <= 1:
        return 0.0
    b = dec - 1
    q = 1 - p
    f = (b * p - q) / b
    f = max(0.0, f) * 0.5  # half Kelly
    return float(min(unit_cap, f))

# ---------------------------------------
# 0) Choose odds dataframe automatically
# ---------------------------------------
candidate_names = []
for k, v in list(globals().items()):
    if isinstance(v, pd.DataFrame):
        cols = set(map(str, v.columns))
        if {"home_team","away_team","book","team"}.issubset(cols) and ("american_odds" in cols or "price_decimal" in cols):
            candidate_names.append((k, v.shape[0], sorted(list(cols))[:12]))

if not candidate_names:
    raise ValueError("No odds dataframe found in memory. Expected something like odds_long or odds_df with home_team/away_team/book/team.")

# Prefer odds_long if present
odds_name = "odds_long" if "odds_long" in globals() else sorted(candidate_names, key=lambda x: x[1], reverse=True)[0][0]
odds = globals()[odds_name].copy()
print(f"✅ Using odds dataframe: {odds_name} | rows={len(odds)}")

# ---------------------------------------
# 1) Normalize price columns (decimal/american)
# ---------------------------------------
if "market" not in odds.columns:
    # If you don't have market column, assume h2h when spread is null, spreads otherwise
    odds["market"] = np.where(odds.get("spread", np.nan).isna(), "h2h", "spreads")

# Some of your earlier runs had "american_odds" containing decimal like 2.65
price_raw = odds["american_odds"] if "american_odds" in odds.columns else odds["price_decimal"]

odds["price_decimal"] = price_raw.apply(lambda x: float(x) if is_decimal_price(x) else (american_to_dec(x) if pd.notna(x) else np.nan))
odds["american_odds_norm"] = odds["price_decimal"].apply(lambda d: dec_to_american(d) if pd.notna(d) else np.nan)
odds["implied_prob"] = odds["price_decimal"].apply(lambda d: implied_from_dec(d) if pd.notna(d) else np.nan)

# Keep original for reference if needed
if "american_odds" not in odds.columns:
    odds["american_odds"] = odds["american_odds_norm"]

# Basic cleaning
odds = odds.dropna(subset=["home_team","away_team","book","team","market","price_decimal","implied_prob"])
odds["game_id"] = odds["home_team"].astype(str) + " vs " + odds["away_team"].astype(str)

print(f"✅ Normalized odds: rows={len(odds)} | markets={odds['market'].value_counts().to_dict()}")

# ---------------------------------------
# 2) Build CONSENSUS (ML + SPREAD separately)
# ---------------------------------------
def build_consensus(df_market: pd.DataFrame) -> pd.DataFrame:
    # average implied prob across books (simple; you can replace with vig-free later)
    g = (df_market
         .groupby(["home_team","away_team","team"], as_index=False)
         .agg(consensus_prob=("implied_prob","mean"),
              book_count=("book","nunique")))
    return g

ml_df = odds[odds["market"].isin(["h2h","moneyline","ml"])].copy()
sp_df = odds[odds["market"].isin(["spreads","spread"])].copy()

cons_ml = build_consensus(ml_df) if len(ml_df) else pd.DataFrame(columns=["home_team","away_team","team","consensus_prob","book_count"])
cons_sp = build_consensus(sp_df) if len(sp_df) else pd.DataFrame(columns=["home_team","away_team","team","consensus_prob","book_count"])

print(f"✅ cons_ml: {cons_ml.shape} | ✅ cons_sp: {cons_sp.shape}")

# ---------------------------------------
# 3) Find MODEL outputs (MUST be real)
# ---------------------------------------
# We look for a dataframe in memory that has model probs.
# Priority order for ML model prob columns:
ML_MODEL_COLS = ["model_prob_ml","model_prob","home_proj_prob","proj_win_prob","win_prob","sim_win_prob"]
SP_MODEL_COLS = ["model_prob_spread","sim_prob_cover","model_prob_cover","cover_prob"]

def find_model_df(required_cols):
    hits = []
    for k, v in list(globals().items()):
        if isinstance(v, pd.DataFrame):
            cols = set(map(str, v.columns))
            if {"home_team","away_team","team"}.issubset(cols) and any(c in cols for c in required_cols):
                hits.append((k, v.shape[0]))
    if not hits:
        return None, None
    # choose largest
    k_best = sorted(hits, key=lambda x: x[1], reverse=True)[0][0]
    df_best = globals()[k_best].copy()
    # choose first matching column
    col_best = [c for c in required_cols if c in df_best.columns][0]
    return df_best, col_best

model_ml_df, model_ml_col = find_model_df(ML_MODEL_COLS)
model_sp_df, model_sp_col = find_model_df(SP_MODEL_COLS)

if model_ml_df is None:
    raise ValueError(
        "❌ No ML model dataframe found with columns like: "
        + ", ".join(ML_MODEL_COLS)
        + "\nYou must have a real ML probability output somewhere (not consensus)."
    )

print(f"✅ ML model source: {model_ml_df.shape} | using column: {model_ml_col}")

# HARD VALIDATION: stop if model probs are constant / impossible
ml_probs = pd.to_numeric(model_ml_df[model_ml_col], errors="coerce").dropna()
if len(ml_probs) == 0:
    raise ValueError("❌ ML model prob column exists but is all NaN.")
if ml_probs.std() < 1e-6:
    raise ValueError(
        f"❌ ML model probs look CONSTANT (std={ml_probs.std():.2e}).\n"
        f"This is the 'everything = {ml_probs.iloc[0]}' bug.\n"
        f"Stop here and fix the cell that creates {model_ml_col} (it is being overwritten or hard-set)."
    )
if (ml_probs < 0).any() or (ml_probs > 1).any():
    raise ValueError("❌ ML model probs contain values outside [0,1].")

# ---------------------------------------
# 4) Build ML +EV Top10 (NO DUPES)
# ---------------------------------------
# Attach consensus + model prob onto each *line* (book/team)
ml_lines = (ml_df
            .merge(cons_ml, on=["home_team","away_team","team"], how="left")
            .merge(model_ml_df[["home_team","away_team","team", model_ml_col]], on=["home_team","away_team","team"], how="left")
            .rename(columns={model_ml_col:"model_prob"}))

# True edge vs price (NOT vs consensus)
ml_lines["true_edge"] = ml_lines["model_prob"] - ml_lines["implied_prob"]
ml_lines["ev"] = ml_lines["model_prob"] * ml_lines["price_decimal"] - 1
ml_lines["units"] = ml_lines.apply(lambda r: kelly_half_units(r["model_prob"], r["price_decimal"], unit_cap=1.5), axis=1)

# Remove duplicates: keep best book per (game, team) by EV, then pick best side per game
ml_best_side = (ml_lines
    .dropna(subset=["model_prob","price_decimal","ev"])
    .sort_values(["ev","true_edge"], ascending=False)
    .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

# Now pick best single bet per game (no dupes by game)
ml_top10 = (ml_best_side
    .sort_values(["ev","true_edge"], ascending=False)
    .drop_duplicates(subset=["home_team","away_team"], keep="first")
    .head(10)
    .reset_index(drop=True)
)

# ---------------------------------------
# 5) Build SPREAD +EV Top10 (NO DUPES)
# ---------------------------------------
sp_top10 = pd.DataFrame()
if len(sp_df):
    # Spread model prob must exist OR we stop (no proxy here unless you explicitly want it)
    if model_sp_df is None:
        print("⚠ No spread model probabilities found (model_prob_spread/sim_prob_cover/etc). Skipping spread Top10.")
    else:
        print(f"✅ SPREAD model source: {model_sp_df.shape} | using column: {model_sp_col}")
        sp_lines = (sp_df
                    .merge(cons_sp, on=["home_team","away_team","team"], how="left")
                    .merge(model_sp_df[["home_team","away_team","team", model_sp_col]], on=["home_team","away_team","team"], how="left")
                    .rename(columns={model_sp_col:"model_prob"}))

        sp_lines["true_edge"] = sp_lines["model_prob"] - sp_lines["implied_prob"]
        sp_lines["ev"] = sp_lines["model_prob"] * sp_lines["price_decimal"] - 1
        sp_lines["units"] = sp_lines.apply(lambda r: kelly_half_units(r["model_prob"], r["price_decimal"], unit_cap=1.5), axis=1)

        sp_best_side = (sp_lines
            .dropna(subset=["spread","model_prob","price_decimal","ev"])
            .sort_values(["ev","true_edge"], ascending=False)
            .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first")
        )

        sp_top10 = (sp_best_side
            .sort_values(["ev","true_edge"], ascending=False)
            .drop_duplicates(subset=["home_team","away_team"], keep="first")
            .head(10)
            .reset_index(drop=True)
        )

# ---------------------------------------
# 6) Print Discord-ready output (clean)
# ---------------------------------------
def fmt_amer(a):
    if pd.isna(a): return "NA"
    a = float(a)
    return f"+{int(a)}" if a > 0 else f"{int(a)}"

def print_top10(title, df, is_spread=False):
    print("\n" + title)
    if df is None or len(df) == 0:
        print("No plays.")
        return
    for i, row in df.iterrows():
        matchup = f"{row['home_team']} vs {row['away_team']}"
        bet = row["team"]
        if is_spread:
            bet = f"{bet} {row['spread']}"
        print(f"\n{i+1}) {matchup}")
        print(f"Bet: {bet}")
        print(f"Book: {row['book']} | Odds: {fmt_amer(row['american_odds_norm'])} (Dec {row['price_decimal']:.2f})")
        print(f"Model: {row['model_prob']:.3f} | Consensus: {row['consensus_prob']:.3f} | Books: {int(row['book_count']) if pd.notna(row['book_count']) else 'NA'}")
        print(f"True Edge (vs price): {row['true_edge']:.3f} | EV: {row['ev']:.3f} | Units (½ Kelly): {row['units']:.2f}u")

print_top10("🔥 NCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)", ml_top10, is_spread=False)
print_top10("🔥 NCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)", sp_top10, is_spread=True)

# Keep these around for next cells
TOP10_ML = ml_top10
TOP10_SPREAD = sp_top10
ODDS_NORM = odds
CONS_ML = cons_ml
CONS_SP = cons_sp

print("\n✅ Saved variables: TOP10_ML, TOP10_SPREAD, ODDS_NORM, CONS_ML, CONS_SP")

In [ ]:
# ============================================
# FIND WHERE model_prob IS CONSTANT / OVERRIDDEN
# ============================================
import pandas as pd
import numpy as np

def col_stats(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return {"n": 0}
    return {
        "n": len(s),
        "mean": float(s.mean()),
        "std": float(s.std()),
        "min": float(s.min()),
        "max": float(s.max()),
        "unique": int(s.nunique()),
        "top_vals": s.value_counts().head(3).to_dict()
    }

candidates = []
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and "model_prob" in obj.columns:
        st = col_stats(obj["model_prob"])
        candidates.append((name, obj.shape, st))

# Sort: constants first, then biggest
candidates = sorted(
    candidates,
    key=lambda x: (x[2].get("std", 0) < 1e-6, x[1][0]),
    reverse=True
)

print("=== DataFrames containing 'model_prob' ===")
if not candidates:
    print("None found in globals().")
else:
    for name, shape, st in candidates[:30]:
        print(f"\n{name} | shape={shape}")
        print(st)

print("\n=== Any columns that look like ML model probs besides 'model_prob'? ===")
alt_cols = ["model_prob_ml","home_proj_prob","proj_win_prob","win_prob","sim_win_prob"]
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        hits = [c for c in alt_cols if c in obj.columns]
        if hits:
            for c in hits:
                st = col_stats(obj[c])
                if st.get("n",0) > 0:
                    print(f"{name}.{c} | shape={obj.shape} | {st}")

In [ ]:
# ============================================
# HARD SWAP: Use ml_f.model_prob_ml as THE ML model probability
# ============================================
import pandas as pd
import numpy as np

if "ml_f" not in globals():
    raise ValueError("ml_f not found. It should exist per your scan output (ml_f.model_prob_ml).")

need = {"home_team","away_team","team","model_prob_ml"}
missing = need - set(ml_f.columns)
if missing:
    raise ValueError(f"ml_f missing required cols: {missing}")

MODEL_ML_DF = (ml_f[["home_team","away_team","team","model_prob_ml"]]
               .dropna(subset=["model_prob_ml"])
               .drop_duplicates())

print("✅ MODEL_ML_DF set from ml_f:", MODEL_ML_DF.shape)
print(MODEL_ML_DF["model_prob_ml"].describe())

In [ ]:
# ============================================
# NORMALIZE ODDS + BUILD TRUE ML CONSENSUS (vig-free by side)
# ============================================
import numpy as np
import pandas as pd

if "odds_long" not in globals():
    raise ValueError("odds_long not found.")

od = odds_long.copy()

# Ensure market labels
od["market"] = od["market"].astype(str).str.lower()

# PRICE NORMALIZATION
# Your odds_long sometimes has decimal prices in american_odds (e.g. 2.65, 1.93)
# We'll interpret:
# - if abs(value) >= 100 => American
# - else => Decimal
x = pd.to_numeric(od["american_odds"], errors="coerce")

od["price_decimal"] = np.where(
    x.abs() >= 100,
    np.where(x > 0, 1 + (x/100), 1 + (100/(-x))),
    x
)

# Convert decimal -> american (for display)
def dec_to_amer(d):
    if pd.isna(d) or d <= 1: return np.nan
    if d >= 2:
        return round((d - 1) * 100)
    else:
        return round(-100 / (d - 1))

od["american_odds_norm"] = od["price_decimal"].apply(dec_to_amer)

# implied prob from decimal
od["implied_prob"] = 1 / od["price_decimal"]

print("✅ ODDS normalized:", od.shape, "| markets:", od["market"].value_counts().to_dict())
print(od[["market","american_odds","price_decimal","american_odds_norm","implied_prob"]].head())

# ---------- TRUE ML CONSENSUS ----------
ml = od[od["market"].isin(["h2h","ml","moneyline"])].copy()

# average implied probs per side
cons_raw = (ml.groupby(["home_team","away_team","team"], as_index=False)
              .agg(cons_implied=("implied_prob","mean"),
                   book_count=("book","nunique")))

# vig-free within each game: normalize two sides to sum to 1
sum_game = cons_raw.groupby(["home_team","away_team"])["cons_implied"].transform("sum")
cons_raw["consensus_prob"] = cons_raw["cons_implied"] / sum_game

CONS_ML = cons_raw[["home_team","away_team","team","consensus_prob","book_count"]].copy()

print("✅ CONS_ML built:", CONS_ML.shape)
print(CONS_ML["consensus_prob"].describe())

In [ ]:
# ============================================
# TOP 10 ML (NO DUPES | +EV | 1/2 KELLY) — CORRECT ODDS + UNITS
# ============================================
import numpy as np
import pandas as pd

def half_kelly_units(p, dec, cap=1.5):
    if pd.isna(p) or pd.isna(dec) or dec <= 1:
        return 0.0
    b = dec - 1
    q = 1 - p
    f = (b*p - q) / b
    f = max(0.0, f) * 0.5
    return float(min(cap, f))

def fmt_amer(a):
    if pd.isna(a): return "NA"
    a = float(a)
    return f"+{int(a)}" if a > 0 else f"{int(a)}"

if "MODEL_ML_DF" not in globals():
    raise ValueError("MODEL_ML_DF not found. Run Cell A.")
if "CONS_ML" not in globals():
    raise ValueError("CONS_ML not found. Run Cell B.")
if "od" not in globals():
    raise ValueError("od not found. Run Cell B (it creates od).")

ml = od[od["market"].isin(["h2h","ml","moneyline"])].copy()

# attach consensus + model probs
ml = (ml.merge(CONS_ML, on=["home_team","away_team","team"], how="left")
        .merge(MODEL_ML_DF, on=["home_team","away_team","team"], how="left"))

# compute EV vs price (this is what wins long-term)
ml = ml.dropna(subset=["model_prob_ml","price_decimal","american_odds_norm","implied_prob"])

ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1
ml["units"] = ml.apply(lambda r: half_kelly_units(r["model_prob_ml"], r["price_decimal"], cap=1.5), axis=1)

# keep best book per team/game
best_per_side = (ml.sort_values(["ev","true_edge"], ascending=False)
                   .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

# no dupes per game: pick top side per game
TOP10_ML = (best_per_side.sort_values(["ev","true_edge"], ascending=False)
                        .drop_duplicates(subset=["home_team","away_team"], keep="first")
                        .head(10)
                        .reset_index(drop=True))

print("✅ TOP10_ML:", TOP10_ML.shape)

print("\nNCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)\n")
for i, r in TOP10_ML.iterrows():
    print(f"{i+1}) {r['home_team']} vs {r['away_team']}")
    print(f"Bet: {r['team']} ML")
    print(f"Book: {r['book']} | Odds: {fmt_amer(r['american_odds_norm'])} (Dec {r['price_decimal']:.2f})")
    print(f"Model: {r['model_prob_ml']:.3f} | Consensus: {r['consensus_prob']:.3f} | Books: {int(r['book_count']) if pd.notna(r['book_count']) else 'NA'}")
    print(f"Edge(vs price): {r['true_edge']:.3f} | EV: {r['ev']:.3f} | Units (½ Kelly): {r['units']:.2f}u\n")

In [ ]:
# ============================================
# DISCORD OUTPUT (TOP10 ML)
# ============================================
def discord_block(df):
    out = []
    for _, r in df.iterrows():
        out.append(
f"""{r['home_team']} vs {r['away_team']}
Bet: {r['team']} ML ({r['book']}) {fmt_amer(r['american_odds_norm'])}
Model: {r['model_prob_ml']:.3f} | Cons: {r['consensus_prob']:.3f}
EV: {r['ev']:.3f} | Units (½ Kelly): {r['units']:.2f}u
"""
        )
    return "\n".join(out).strip()

print(discord_block(TOP10_ML))

In [ ]:
import numpy as np
import pandas as pd

def amer_to_dec(a):
    a = float(a)
    return (1 + a/100) if a > 0 else (1 + 100/(-a))

def implied_from_amer(a):
    a = float(a)
    return 100/(a+100) if a > 0 else (-a)/((-a)+100)

tmp = TOP10_ML.copy()

# if your top10 uses american_odds_norm
tmp["dec_check"] = tmp["american_odds_norm"].apply(lambda x: amer_to_dec(x) if pd.notna(x) else np.nan)
tmp["imp_check"] = tmp["american_odds_norm"].apply(lambda x: implied_from_amer(x) if pd.notna(x) else np.nan)
tmp["gap_model_minus_imp"] = tmp["model_prob_ml"] - tmp["imp_check"]

print(tmp[["home_team","away_team","team","book","american_odds_norm","imp_check","model_prob_ml","gap_model_minus_imp"]]
      .sort_values("gap_model_minus_imp", ascending=False)
      .head(15))


In [ ]:
# ============================================
# FIX ML PROB SOURCE: use ml_model (real ML win probs)
# ============================================
import pandas as pd

if "ml_model" not in globals():
    raise ValueError("ml_model not found. It showed up in your scan earlier. Scroll up and re-run the cell that builds ml_model.")

# Find the probability column inside ml_model
prob_candidates = [c for c in ml_model.columns if "prob" in str(c).lower()]
print("prob candidates:", prob_candidates)

# pick first usable prob col that has variance
prob_col = None
for c in prob_candidates:
    s = pd.to_numeric(ml_model[c], errors="coerce").dropna()
    if len(s) > 0 and s.std() > 1e-3 and s.mean() > 0.15 and s.mean() < 0.85:
        prob_col = c
        break

if prob_col is None:
    raise ValueError("Couldn't find a sane prob column in ml_model. Print ml_model.head() and columns.")

MODEL_ML_DF = (ml_model[["home_team","away_team","team",prob_col]]
               .rename(columns={prob_col:"model_prob_ml"})
               .dropna(subset=["model_prob_ml"])
               .drop_duplicates())

print("✅ MODEL_ML_DF reset from ml_model using:", prob_col, "| shape:", MODEL_ML_DF.shape)
print(MODEL_ML_DF["model_prob_ml"].describe())

In [ ]:
# ============================================
# REBUILD TOP10 ML (NO DUPES | +EV | 1/2 KELLY)
# ============================================
import numpy as np
import pandas as pd

def dec_to_amer(d):
    if pd.isna(d) or d <= 1: return np.nan
    return round((d - 1) * 100) if d >= 2 else round(-100 / (d - 1))

def half_kelly_units(p, dec, cap=1.5):
    if pd.isna(p) or pd.isna(dec) or dec <= 1: return 0.0
    b = dec - 1
    q = 1 - p
    f = (b*p - q) / b
    f = max(0.0, f) * 0.5
    return float(min(cap, f))

def fmt_amer(a):
    if pd.isna(a): return "NA"
    a = float(a)
    return f"+{int(a)}" if a > 0 else f"{int(a)}"

# source odds table
if "odds_long" not in globals():
    raise ValueError("odds_long not found.")

od = odds_long.copy()
od["market"] = od["market"].astype(str).str.lower()

# normalize price (american vs decimal)
x = pd.to_numeric(od["american_odds"], errors="coerce")
od["price_decimal"] = np.where(
    x.abs() >= 100,
    np.where(x > 0, 1 + (x/100), 1 + (100/(-x))),
    x
)
od["american_odds_norm"] = od["price_decimal"].apply(dec_to_amer)
od["implied_prob"] = 1 / od["price_decimal"]

# ML only
ml = od[od["market"].isin(["h2h","ml","moneyline"])].copy()

# TRUE consensus (vig-free by game)
cons_raw = (ml.groupby(["home_team","away_team","team"], as_index=False)
              .agg(cons_implied=("implied_prob","mean"),
                   book_count=("book","nunique")))
sum_game = cons_raw.groupby(["home_team","away_team"])["cons_implied"].transform("sum")
cons_raw["consensus_prob"] = cons_raw["cons_implied"] / sum_game
CONS_ML = cons_raw[["home_team","away_team","team","consensus_prob","book_count"]].copy()

# attach consensus + model probs
ml = (ml.merge(CONS_ML, on=["home_team","away_team","team"], how="left")
        .merge(MODEL_ML_DF, on=["home_team","away_team","team"], how="left"))

ml = ml.dropna(subset=["model_prob_ml","price_decimal","american_odds_norm","implied_prob","consensus_prob"])

# EV vs price + kelly
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1
ml["units"] = ml.apply(lambda r: half_kelly_units(r["model_prob_ml"], r["price_decimal"], cap=1.5), axis=1)

# keep best book per team/game, then 1 bet per game
best_side = (ml.sort_values(["ev","true_edge"], ascending=False)
               .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

TOP10_ML = (best_side.sort_values(["ev","true_edge"], ascending=False)
                    .drop_duplicates(subset=["home_team","away_team"], keep="first")
                    .head(10)
                    .reset_index(drop=True))

print("✅ TOP10_ML rebuilt:", TOP10_ML.shape)
print(TOP10_ML[["home_team","away_team","team","book","american_odds_norm","price_decimal","model_prob_ml","consensus_prob","true_edge","ev","units"]])

In [ ]:
def discord_block(df):
    out = []
    for _, r in df.iterrows():
        out.append(
f"""{r['home_team']} vs {r['away_team']}
Bet: {r['team']} ML ({r['book']}) {fmt_amer(r['american_odds_norm'])}
Model: {r['model_prob_ml']:.3f} | Cons: {r['consensus_prob']:.3f}
EV: {r['ev']:.3f} | Units (½ Kelly): {r['units']:.2f}u
"""
        )
    return "\n".join(out).strip()

print(discord_block(TOP10_ML))

In [ ]:
# ===== HARD DIAGNOSTIC: PROBABILITY ALIGNMENT CHECK =====

test_game = "Troy Trojans"

check = ml_model[ml_model["home_team"] == test_game][
    ["home_team","away_team","team","model_prob"]
].copy()

print(check)
print("\nSum by game:")
print(check.groupby(["home_team","away_team"])["model_prob"].sum())

In [ ]:
# Compare model prob vs implied prob side by side for one game

game = ml[(ml["home_team"]=="Troy Trojans")].copy()

print(game[["team","american_odds_norm","implied_prob","model_prob_ml"]])

In [ ]:
ml_model["model_prob"].describe()

In [ ]:
# ===== CLEAN ML MODEL LAYER (NO OLD VARIABLES) =====

MODEL_ML_DF = ml_model.copy()

# Normalize team strings (important)
for col in ["home_team","away_team","team"]:
    MODEL_ML_DF[col] = MODEL_ML_DF[col].str.strip()

MODEL_ML_DF = MODEL_ML_DF[["home_team","away_team","team","model_prob"]]
MODEL_ML_DF = MODEL_ML_DF.rename(columns={"model_prob":"model_prob_ml"})

print("Shape:", MODEL_ML_DF.shape)
print("\nPer-game probability sums (should be 1.00):")
print(
    MODEL_ML_DF.groupby(["home_team","away_team"])["model_prob_ml"]
    .sum()
    .describe()
)

In [ ]:
# ===== STRICT ML MERGE =====

ml_odds = odds_long.copy()
ml_odds["market"] = ml_odds["market"].str.lower()
ml_odds = ml_odds[ml_odds["market"].isin(["h2h","moneyline","ml"])].copy()

for col in ["home_team","away_team","team"]:
    ml_odds[col] = ml_odds[col].str.strip()

ml_merged = ml_odds.merge(
    MODEL_ML_DF,
    on=["home_team","away_team","team"],
    how="inner"
)

print("Merged shape:", ml_merged.shape)
print("\nCheck one extreme dog:")
print(
    ml_merged.sort_values("american_odds", ascending=False)[
        ["home_team","away_team","team","american_odds","model_prob_ml"]
    ].head(10)
)

In [ ]:
# ==========================================
# FIX: Resolve model_prob_spread column collision after merge
# ==========================================
import numpy as np

print("Columns now:", list(sp.columns))

# If merge created suffix columns, consolidate them
if "model_prob_spread" not in sp.columns:
    candidates = [c for c in sp.columns if "model_prob_spread" in c]
    print("Found spread prob candidates:", candidates)

    # Prefer the merged column (usually _y), fall back to _x
    y = "model_prob_spread_y" if "model_prob_spread_y" in sp.columns else None
    x = "model_prob_spread_x" if "model_prob_spread_x" in sp.columns else None

    if y and x:
        sp["model_prob_spread"] = sp[y].combine_first(sp[x])
    elif y:
        sp["model_prob_spread"] = sp[y]
    elif x:
        sp["model_prob_spread"] = sp[x]
    else:
        raise ValueError("No model_prob_spread columns found after merge.")

    # Optional: drop the suffix cols to keep things clean
    drop_cols = [c for c in ["model_prob_spread_x","model_prob_spread_y"] if c in sp.columns]
    sp = sp.drop(columns=drop_cols)

# sanity prints
print("✅ model_prob_spread attached. Nulls:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))
print(sp[["home_team","away_team","team","book","spread","price_decimal","model_prob_spread"]].head(10))

In [ ]:
ml_merged["implied_prob"] = np.where(
    ml_merged["american_odds"] > 0,
    100/(ml_merged["american_odds"]+100),
    (-ml_merged["american_odds"])/((-ml_merged["american_odds"])+100)
)

ml_merged["gap"] = ml_merged["model_prob_ml"] - ml_merged["implied_prob"]

print(ml_merged[["team","american_odds","model_prob_ml","implied_prob","gap"]]
      .sort_values("gap", ascending=False)
      .head(10))

In [ ]:
assert MODEL_ML_DF["model_prob_ml"].std() > 0.05
assert abs(
    MODEL_ML_DF.groupby(["home_team","away_team"])["model_prob_ml"].sum().mean() - 1
) < 0.01

In [ ]:
print("Model ML std:", MODEL_ML_DF["model_prob_ml"].std())
print("Per-game prob sum mean:",
      MODEL_ML_DF.groupby(["home_team","away_team"])["model_prob_ml"].sum().mean())

In [ ]:
# =====================================================
# STABLE v3 — STRICT RUN (NCAAB) — TODAY (2/27)
# Requires: odds_long (normalized), cons_ml/cons_sp builders,
#          ml_model (game/team model probs), model_sp (spread probs) OR model_prob_spread column
# Output: Top 10 ML + Top 10 Spread (no dupes), best book, +EV, ½-Kelly units
# =====================================================

RUN_DATE = "2026-02-27"
TOP_N = 10

# Filters (STRICT but realistic)
MIN_EV = 0.02          # +2% EV
MIN_TRUE_EDGE = 0.02   # model_prob - implied_prob >= 2%
MAX_DOG_AMER = 600     # optional: keep dogs reasonable (set None to disable)
MAX_UNITS = 1.5        # cap ½-Kelly units
MIN_BOOKS = 2          # require >=2 books in consensus (stability)

def american_to_decimal(a):
    a = float(a)
    if a > 0: return 1.0 + a/100.0
    return 1.0 + 100.0/abs(a)

def decimal_to_american(d):
    d = float(d)
    if d >= 2: return int(round((d-1)*100))
    return int(round(-100/(d-1)))

def implied_prob_from_decimal(d):
    d = float(d)
    return 1.0 / d

def half_kelly_units(p, dec, cap=MAX_UNITS):
    # full Kelly f* = (bp - q)/b where b = dec-1
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b
    f = max(0.0, f)          # no negative Kelly
    f = 0.5 * f              # half Kelly
    return min(cap, f)

# ---------- 1) Normalize odds (use your existing normalized DF if present) ----------
df = normalized.copy() if "normalized" in globals() else odds_long.copy()

# Ensure american_odds is in AMERICAN (not decimal)
# If you have both price_decimal and american_odds already, we trust those.
if "price_decimal" not in df.columns:
    df["price_decimal"] = df["american_odds"].apply(american_to_decimal)

df["implied_prob"] = df["price_decimal"].apply(implied_prob_from_decimal)

# ---------- 2) Build consensus for ML + Spread ----------
# ML consensus: average implied prob across books per game/team (market=h2h)
ml = df[df["market"].eq("h2h")].copy()
sp = df[df["market"].eq("spreads")].copy()

cons_ml = (ml.groupby(["home_team","away_team","team"], as_index=False)
             .agg(consensus_prob=("implied_prob","mean"),
                  book_count=("book","nunique")))

cons_sp = (sp.groupby(["home_team","away_team","team","spread"], as_index=False)
             .agg(consensus_prob=("implied_prob","mean"),
                  book_count=("book","nunique")))

# ---------- 3) Attach model probabilities (STRICT: use your real model DF, not any df with model_prob=0.99) ----------
# ML model source: prefer ml_model (124x9) or your known-good ML table
# expects cols: home_team, away_team, team, model_prob
if "ml_model" not in globals():
    raise ValueError("ml_model not found. Use your ML model output DF with real probabilities (mean ~0.50).")

ml_model_src = ml_model.copy()
ml_model_src = ml_model_src[["home_team","away_team","team","model_prob"]].copy()

ml_join = cons_ml.merge(ml_model_src, on=["home_team","away_team","team"], how="inner")

# Spread model source: use model_sp if available; otherwise require model_prob_spread already on sp rows.
# expected cols: home_team, away_team, team, spread, model_prob_spread
if "model_sp" in globals():
    sp_model_src = model_sp.copy()
    # try to detect columns
    cand_cols = [c for c in sp_model_src.columns if c in ["model_prob_spread","model_prob_cover","sim_prob_cover"]]
    if not cand_cols:
        raise ValueError("model_sp exists but no spread prob column found (need model_prob_spread / sim_prob_cover).")
    sp_prob_col = cand_cols[0]
    sp_model_src = sp_model_src.rename(columns={sp_prob_col:"model_prob_spread"})
    key_cols = ["home_team","away_team","team","spread","model_prob_spread"]
    sp_model_src = sp_model_src[[c for c in key_cols if c in sp_model_src.columns]].copy()
    sp_join = cons_sp.merge(sp_model_src, on=["home_team","away_team","team","spread"], how="inner")
else:
    # fallback: if you previously created model_prob_spread in sp rows
    if "model_prob_spread" not in sp.columns:
        raise ValueError("No spread model probs. Provide model_sp OR create model_prob_spread on spread rows.")
    sp_tmp = sp[["home_team","away_team","team","spread","model_prob_spread"]].drop_duplicates()
    sp_join = cons_sp.merge(sp_tmp, on=["home_team","away_team","team","spread"], how="inner")

# ---------- 4) Reattach BEST book line (best price) per market ----------
# ML best price: maximize decimal_odds for the bet team
ml_best = (ml.sort_values("price_decimal", ascending=False)
             .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
             [["home_team","away_team","team","book","american_odds","price_decimal"]])

# Spread best price: maximize decimal_odds per game/team/spread
sp_best = (sp.sort_values("price_decimal", ascending=False)
             .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first")
             [["home_team","away_team","team","spread","book","american_odds","price_decimal"]])

# ---------- 5) EV / Edge / Units ----------
def score_market(df_in, market_name):
    out = df_in.copy()
    out = out.merge(ml_best, on=["home_team","away_team","team"], how="left") if market_name=="ML" else \
          out.merge(sp_best, on=["home_team","away_team","team","spread"], how="left")

    out["implied_from_price"] = out["price_decimal"].apply(implied_prob_from_decimal)
    prob_col = "model_prob" if market_name=="ML" else "model_prob_spread"
    out["true_edge"] = out[prob_col] - out["implied_from_price"]
    out["ev"] = out[prob_col]*out["price_decimal"] - 1.0
    out["units"] = out.apply(lambda r: half_kelly_units(r[prob_col], r["price_decimal"]), axis=1)

    # Filters
    out = out.dropna(subset=["book","american_odds","price_decimal",prob_col])
    out = out[out["book_count"] >= MIN_BOOKS]
    out = out[out["ev"] >= MIN_EV]
    out = out[out["true_edge"] >= MIN_TRUE_EDGE]
    out = out[out["units"] > 0]

    if MAX_DOG_AMER is not None:
        # allow any negative (fav), but cap big dogs
        out = out[(out["american_odds"] < 0) | (out["american_odds"] <= MAX_DOG_AMER)]

    # No dupes: 1 bet per game (pick best EV)
    out["game_key"] = out["home_team"] + " vs " + out["away_team"]
    out = (out.sort_values(["ev","true_edge","units"], ascending=False)
              .drop_duplicates(subset=["game_key"], keep="first"))

    return out.sort_values(["ev","true_edge"], ascending=False).head(TOP_N)

top_ml = score_market(ml_join, "ML")
top_sp = score_market(sp_join, "SPREAD")

print(f"\n✅ TOP {len(top_ml)} ML (STRICT | +EV | NO DUPES | ½-KELLY)\n")
display(top_ml[["home_team","away_team","team","book","american_odds","price_decimal",
                "model_prob","consensus_prob","book_count","true_edge","ev","units"]])

print(f"\n✅ TOP {len(top_sp)} SPREAD (STRICT | +EV | NO DUPES | ½-KELLY)\n")
display(top_sp[["home_team","away_team","team","spread","book","american_odds","price_decimal",
                "model_prob_spread","consensus_prob","book_count","true_edge","ev","units"]])

# Keep these for Discord formatting
top_ml_df = top_ml.copy()
top_sp_df = top_sp.copy()

print("\nSaved: top_ml_df, top_sp_df")

In [ ]:
# ===========================
# FIX: Detect + standardize spread model prob column
# ===========================

import pandas as pd
import numpy as np

if "model_sp" not in globals():
    raise ValueError("model_sp not found in globals(). Build/define your spread-model DF first.")

print("model_sp columns:")
print(list(model_sp.columns))

# Heuristic: find the column that looks like a probability series (0..1) and is not consensus/implied
exclude = set([
    "home_team","away_team","team","spread","book","market","commence_time",
    "american_odds","price_decimal","implied_prob","consensus_prob","market_prob",
    "edge","true_edge","ev","units","book_count"
])

prob_candidates = []
for c in model_sp.columns:
    if c in exclude:
        continue
    s = pd.to_numeric(model_sp[c], errors="coerce")
    if s.notna().sum() == 0:
        continue
    mn, mx = float(s.min()), float(s.max())
    if 0.0 <= mn and mx <= 1.0 and s.nunique(dropna=True) > 1:
        prob_candidates.append((c, s.notna().sum(), s.mean(), s.std(), mn, mx))

prob_candidates = sorted(prob_candidates, key=lambda x: (x[1], x[3]), reverse=True)

print("\nSpread prob candidates (col, n, mean, std, min, max):")
for row in prob_candidates[:15]:
    print(row)

if not prob_candidates:
    raise ValueError(
        "Couldn't auto-detect a spread probability column in model_sp.\n"
        "Tell me the spread prob column name from the printed list above and I'll map it."
    )

best_col = prob_candidates[0][0]
print(f"\n✅ Using spread prob column: {best_col}  -> renaming to model_prob_spread")

# Standardize
model_sp = model_sp.rename(columns={best_col: "model_prob_spread"}).copy()

# Ensure required keys exist (some notebooks store spread as 'line' or 'point')
if "spread" not in model_sp.columns:
    # try common alternatives
    for alt in ["point","line","handicap","spread_line"]:
        if alt in model_sp.columns:
            model_sp = model_sp.rename(columns={alt: "spread"})
            print(f"✅ Renamed {alt} -> spread")
            break

missing_keys = [k for k in ["home_team","away_team","team","spread","model_prob_spread"] if k not in model_sp.columns]
if missing_keys:
    raise ValueError(f"model_sp is missing required columns after standardization: {missing_keys}")

print("\n✅ model_sp standardized. Head:")
display(model_sp[["home_team","away_team","team","spread","model_prob_spread"]].head())
print("\nmodel_prob_spread stats:")
display(model_sp["model_prob_spread"].describe())

In [ ]:
# ===========================
# FIX: model_sp missing 'spread' (it's stored as 'cons_spread')
# ===========================
import pandas as pd

# rename spread column
if "spread" not in model_sp.columns:
    if "cons_spread" in model_sp.columns:
        model_sp = model_sp.rename(columns={"cons_spread": "spread"}).copy()
        print("✅ Renamed cons_spread -> spread")
    else:
        raise ValueError(f"❌ No spread column found. Columns are: {list(model_sp.columns)}")

# confirm
print("model_sp columns now:", list(model_sp.columns))
display(model_sp[["home_team","away_team","team","spread","model_prob_spread"]].head())
display(model_sp["model_prob_spread"].describe())

In [ ]:
# ==========================================
# FIX: Ensure odds_long has price_decimal + proper american_odds
# ==========================================
import numpy as np
import pandas as pd

def american_to_decimal(odds):
    odds = float(odds)
    if odds > 0:
        return 1.0 + odds/100.0
    else:
        return 1.0 + 100.0/abs(odds)

def decimal_to_american(dec):
    dec = float(dec)
    if dec <= 1:
        return np.nan
    if dec >= 2:
        return int(round((dec - 1) * 100))
    else:
        return int(round(-100 / (dec - 1)))

# If odds_long doesn't exist, stop early
if "odds_long" not in globals():
    raise ValueError("odds_long not found. Run your odds pull/parse cell first.")

odds_long = odds_long.copy()

# Normalize: sometimes american_odds column is actually decimal (you had 2.65, 1.53 earlier)
# We'll detect: values between (1.01 and 30) are almost certainly decimal odds.
if "american_odds" not in odds_long.columns:
    raise ValueError("odds_long missing 'american_odds' column.")

odds_long["american_odds"] = pd.to_numeric(odds_long["american_odds"], errors="coerce")

# Create price_decimal
odds_long["price_decimal"] = np.where(
    odds_long["american_odds"].between(1.01, 30.0),  # treat as decimal if it looks decimal
    odds_long["american_odds"],
    np.where(
        odds_long["american_odds"].notna(),
        odds_long["american_odds"].apply(american_to_decimal),
        np.nan
    )
)

# Create/repair american_odds if some rows were decimal
odds_long["american_odds"] = np.where(
    odds_long["american_odds"].between(1.01, 30.0),
    odds_long["price_decimal"].apply(decimal_to_american),
    odds_long["american_odds"]
)

# Final sanity
print("✅ Added/validated price_decimal.")
print("Columns now:", list(odds_long.columns))
print("Sample:")
print(odds_long[["market","book","team","american_odds","price_decimal"]].head(10))
print("Null price_decimal:", int(odds_long["price_decimal"].isna().sum()), "of", len(odds_long))

In [ ]:
# ==========================================
# RUN: TOP 10 ML + TOP 10 SPREAD (NO DUPES | +EV | 1/2 KELLY)
# Uses:
#   - odds_long (normalized odds long)
#   - model_sp (spread model table) with model_prob_spread + spread
#   - ml_f (ML model odds table) with model_prob_ml (NOT constant 0.99)
# ==========================================
import numpy as np
import pandas as pd

# ---------- helpers ----------
def dec_to_implied(dec):
    return 1.0 / dec

def kelly_half(p, dec):
    # half kelly fraction of bankroll (unitless). clip at 0.
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b
    return max(0.0, 0.5 * f)

def fmt_amer(x):
    x = float(x)
    return f"+{int(x)}" if x > 0 else f"{int(x)}"

# ---------- choose source DFs ----------
# odds_long must have: home_team, away_team, book, market, team, price_decimal, american_odds, spread
src = odds_long.copy()

# safety: ensure required columns exist
need_cols = {"home_team","away_team","book","market","team","price_decimal","american_odds"}
missing = need_cols - set(src.columns)
if missing:
    raise ValueError(f"odds_long missing cols: {missing}")

# ---------- ML lines ----------
ml_lines = src[src["market"].eq("h2h")].copy()
ml_lines = ml_lines.dropna(subset=["price_decimal","american_odds"])

# attach ML model probs from ml_f (variable column!)
if "model_prob_ml" not in ml_f.columns:
    raise ValueError("ml_f missing model_prob_ml (this is the non-constant ML prob column).")

ml_model = ml_f[["home_team","away_team","team","model_prob_ml"]].drop_duplicates()

ml = ml_lines.merge(ml_model, on=["home_team","away_team","team"], how="inner")

# EV vs price + 1/2 Kelly
ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units"] = ml.apply(lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"]), axis=1)

# keep best book per game/team by highest EV (then units)
ml_best = (ml.sort_values(["ev","units"], ascending=False)
             .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

# filter: +EV only, no dupes already
top10_ml = (ml_best[ml_best["ev"] > 0]
              .sort_values(["ev","units"], ascending=False)
              .head(10)
              .reset_index(drop=True))

# ---------- SPREAD lines ----------
sp_lines = src[src["market"].eq("spreads")].copy()
if "spread" not in sp_lines.columns:
    raise ValueError("odds_long spreads missing 'spread' column")

sp_lines = sp_lines.dropna(subset=["spread","price_decimal","american_odds"])

# attach spread model probs from model_sp (already fixed)
sp_model = model_sp[["home_team","away_team","team","spread","model_prob_spread"]].drop_duplicates()

# NOTE: spreads can differ by book (4.0 vs 4.5). We merge on exact spread number.
sp = sp_lines.merge(sp_model, on=["home_team","away_team","team","spread"], how="inner")

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units"] = sp.apply(lambda r: kelly_half(r["model_prob_spread"], r["price_decimal"]), axis=1)

# keep best book per game/team/spread
sp_best = (sp.sort_values(["ev","units"], ascending=False)
             .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first"))

top10_sp = (sp_best[sp_best["ev"] > 0]
              .sort_values(["ev","units"], ascending=False)
              .head(10)
              .reset_index(drop=True))

# ---------- pretty print ----------
def print_top_ml(df):
    print("🔥 NCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)\n")
    for i, r in df.iterrows():
        print(f"{i+1}) {r['home_team']} vs {r['away_team']}")
        print(f"Bet: {r['team']} ML")
        print(f"Book: {r['book']} | Odds: {fmt_amer(r['american_odds'])} (Dec {r['price_decimal']:.2f})")
        print(f"Model: {r['model_prob_ml']:.3f} | Implied: {r['implied_prob']:.3f}")
        print(f"True Edge: {r['true_edge']:.3f} | EV: {r['ev']:.3f}")
        print(f"Units (½ Kelly): {r['units']:.2f}u\n")

def print_top_sp(df):
    print("🔥 NCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)\n")
    for i, r in df.iterrows():
        sign_line = f"{r['spread']:+g}"
        print(f"{i+1}) {r['home_team']} vs {r['away_team']}")
        print(f"Bet: {r['team']} {sign_line}")
        print(f"Book: {r['book']} | Odds: {fmt_amer(r['american_odds'])} (Dec {r['price_decimal']:.2f})")
        print(f"Model: {r['model_prob_spread']:.3f} | Implied: {r['implied_prob']:.3f}")
        print(f"True Edge: {r['true_edge']:.3f} | EV: {r['ev']:.3f}")
        print(f"Units (½ Kelly): {r['units']:.2f}u\n")

print(f"✅ Top10 ML rows: {len(top10_ml)} | ✅ Top10 Spread rows: {len(top10_sp)}\n")
print_top_ml(top10_ml)
print("="*60)
print_top_sp(top10_sp)

# keep for downstream
top10_ml_df = top10_ml
top10_sp_df = top10_sp

In [ ]:
# ==========================================
# FIX: Attach model_prob_spread onto spread lines (sp) from model_sp
# ==========================================
import pandas as pd
import numpy as np

# Required objects
if "odds_long" not in globals():
    raise ValueError("odds_long not found.")
if "model_sp" not in globals():
    raise ValueError("model_sp not found. You should have model_sp with model_prob_spread already built.")

# Build spreads-only book lines
sp = odds_long[odds_long["market"].isin(["spreads","spread"])].copy()

# Make sure the key columns exist
need_sp_cols = {"home_team","away_team","team","spread","book","price_decimal"}
missing = need_sp_cols - set(sp.columns)
if missing:
    raise ValueError(f"Spread lines (sp) missing cols: {missing}")

need_model_cols = {"home_team","away_team","team","spread","model_prob_spread"}
missing_m = need_model_cols - set(model_sp.columns)
if missing_m:
    raise ValueError(f"model_sp missing cols: {missing_m}")

# Normalize spread numeric types for a clean merge
sp["spread"] = pd.to_numeric(sp["spread"], errors="coerce")
msp = model_sp.copy()
msp["spread"] = pd.to_numeric(msp["spread"], errors="coerce")

# Merge model spread probs onto each book line
sp = sp.merge(
    msp[["home_team","away_team","team","spread","model_prob_spread"]],
    on=["home_team","away_team","team","spread"],
    how="left"
)

print("✅ Spread lines after attaching model_prob_spread:", sp.shape)
print("Null model_prob_spread:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))
print(sp[["home_team","away_team","team","book","spread","price_decimal","model_prob_spread"]].head(10))

In [ ]:
# ==========================================
# CALIBRATION PATCH — blend model with market
# ==========================================

BLEND_WEIGHT = 0.5  # 0.5 = 50% model, 50% market

sp2["p_blended"] = (
    BLEND_WEIGHT * sp2["model_prob_spread"]
    + (1 - BLEND_WEIGHT) * sp2["implied_prob"]
)

# Recalculate using blended probability
sp2["true_edge"] = sp2["p_blended"] - sp2["implied_prob"]
sp2["ev"] = sp2["p_blended"] * sp2["price_decimal"] - 1.0
sp2["units"] = sp2.apply(lambda r: kelly_half(r["p_blended"], r["price_decimal"]), axis=1)

In [ ]:
# ==========================================
# SPREAD: compute EV + 1/2 Kelly + Top10 (NO DUPES)
# ==========================================
import numpy as np
import pandas as pd

def dec_to_implied(d):
    return 1.0 / d

def kelly_half(p, dec):
    # p = win prob, dec = decimal odds
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b  # full Kelly fraction
    f = max(0.0, f)    # no negative Kelly
    return 0.5 * f     # half-Kelly

sp2 = sp.copy()

# implied from price + edges
sp2["implied_prob"] = sp2["price_decimal"].apply(dec_to_implied)
sp2["true_edge"] = sp2["model_prob_spread"] - sp2["implied_prob"]
sp2["ev"] = sp2["model_prob_spread"] * sp2["price_decimal"] - 1.0
sp2["units"] = sp2.apply(lambda r: kelly_half(r["model_prob_spread"], r["price_decimal"]), axis=1)

# Keep only +EV (you can loosen this if you want)
sp2 = sp2.dropna(subset=["ev","units","true_edge","model_prob_spread","price_decimal"])
sp2 = sp2[sp2["ev"] > 0].copy()

# BEST book per (game, team, spread) by EV then units
sp_best = (
    sp2.sort_values(["ev","units"], ascending=False)
       .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first")
)

# NO DUPES across the card: keep only one bet per (game, team)
top_spreads_df = (
    sp_best.sort_values(["ev","units"], ascending=False)
           .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
           .head(10)
           .reset_index(drop=True)
)

print("✅ +EV spread candidates:", len(sp2))
print("✅ Best-book spread rows:", len(sp_best))
print("✅ Top10 spreads (no dupes):", len(top_spreads_df))

cols = ["home_team","away_team","team","book","spread","american_odds","price_decimal",
        "model_prob_spread","implied_prob","true_edge","ev","units"]
print(top_spreads_df[cols])

In [ ]:
# ==========================================
# DISCORD OUTPUT (SPREAD TOP 10)
# ==========================================
def fmt_amer(x):
    try:
        x = float(x)
    except:
        return str(x)
    if x > 0:
        return f"+{int(round(x))}"
    return f"{int(round(x))}"

msg = "NCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)\n"
for i, r in top_spreads_df.iterrows():
    matchup = f"{r['home_team']} vs {r['away_team']}"
    bet = f"{r['team']} {r['spread']:+g}"
    msg += f"\n{i+1}) {matchup}\n"
    msg += f"Bet: {bet}\n"
    msg += f"Book: {r['book']} | Odds: {fmt_amer(r['american_odds'])} (Dec {r['price_decimal']:.2f})\n"
    msg += f"Model: {r['model_prob_spread']:.3f} | Implied: {r['implied_prob']:.3f}\n"
    msg += f"True Edge: {r['true_edge']:.3f} | EV: {r['ev']:.3f}\n"
    msg += f"Units (½ Kelly): {r['units']:.2f}u\n"

print(msg)

In [ ]:
# =========================================
# STEP NEXT: MARKET-ANCHORED + MODEL BLEND
# (No projections required)
# Produces: top10_ml, top10_sp, and Discord-ready output
# =========================================

import numpy as np
import pandas as pd

# ---------- helpers ----------
def dec_to_implied(d):
    d = float(d)
    return 1.0 / d if d > 0 else np.nan

def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1.0 + (a/100.0)
    else:
        return 1.0 + (100.0/abs(a))

def kelly_half(p, dec):
    # half-kelly fraction, clipped to [0, 0.5] for sanity
    p = float(p); dec = float(dec)
    b = dec - 1.0
    if b <= 0:
        return 0.0
    f = (p*dec - 1.0) / b
    f = 0.5 * f
    return float(np.clip(f, 0.0, 0.5))

def zscore(s):
    s = s.astype(float)
    if s.std(ddof=0) < 1e-12:
        return s*0.0
    return (s - s.mean()) / s.std(ddof=0)

def ensure_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}\nHave: {list(df.columns)}")

def pick_best_book(df, key_cols, score_col):
    # pick best book per bet key
    return (df.sort_values(score_col, ascending=False)
              .drop_duplicates(subset=key_cols, keep="first")
              .reset_index(drop=True))

# ---------- REQUIRED INPUT DFS ----------
# odds_long: from odds api parsing (already in your notebook)
# cons_ml, cons_sp: consensus tables
# ml_model: NON-CONSTANT ML model probs (0-1) per team/game
# model_sp: spread model probs per team/game/spread (or per team/game with "spread" = cons_spread)

ensure_cols(odds_long, ["home_team","away_team","commence_time","book","market","team","american_odds","spread"], "odds_long")

# normalize price_decimal + implied_prob if needed
od = odds_long.copy()

# If american_odds is already decimal in some rows, fix it:
# Rule: if abs(american_odds) < 20 and american_odds not in typical american range, treat as decimal
def normalize_price(row):
    a = row["american_odds"]
    try:
        a = float(a)
    except:
        return np.nan
    # treat as decimal if between 1.01 and 20
    if 1.01 <= a <= 20:
        return a
    return american_to_decimal(a)

od["price_decimal"] = od.apply(normalize_price, axis=1)
od["implied_prob_price"] = od["price_decimal"].apply(dec_to_implied)

# split markets
ml_lines = od[od["market"].isin(["h2h","ml","moneyline"])].copy()
sp_lines = od[od["market"].isin(["spreads","spread"])].copy()

print(f"✅ markets: ml_lines={ml_lines.shape} | sp_lines={sp_lines.shape}")

# ---------- consensus ----------
ensure_cols(cons_ml, ["home_team","away_team","team","consensus_prob","book_count"], "cons_ml")
ensure_cols(cons_sp, ["home_team","away_team","team","consensus_prob","book_count"], "cons_sp")

# ---------- model sources ----------
# ML model df should be the non-constant one (in your notebook it's named ml_model with mean ~0.5 / std ~0.235)
# If you have multiple, use the one with UNIQUE > 1.
ensure_cols(ml_model, ["home_team","away_team","team","model_prob"], "ml_model")

# Spread model df should be standardized to have spread + model_prob_spread
ensure_cols(model_sp, ["home_team","away_team","team","spread","model_prob_spread"], "model_sp")

# ---------- sanity checks ----------
ml_std = float(ml_model["model_prob"].astype(float).std(ddof=0))
if ml_std < 1e-6:
    raise ValueError(f"❌ ml_model.model_prob looks constant (std={ml_std:.2e}). Use the non-constant ML model df (mean ~0.5).")

sp_std = float(model_sp["model_prob_spread"].astype(float).std(ddof=0))
if sp_std < 1e-6:
    raise ValueError(f"❌ model_sp.model_prob_spread looks constant (std={sp_std:.2e}).")

print(f"✅ ML model std: {ml_std:.3f} | ✅ SP model std: {sp_std:.3f}")

# =========================================
# 1) BUILD ML BET TABLE (best book per team/game)
# =========================================

# Best price per bet key (team/game) = max decimal odds
ml_best_price = (ml_lines.dropna(subset=["price_decimal"])
                .sort_values("price_decimal", ascending=False)
                .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
                [["home_team","away_team","team","book","price_decimal","american_odds","commence_time"]]
                .reset_index(drop=True))

ml = (ml_best_price
      .merge(cons_ml[["home_team","away_team","team","consensus_prob","book_count"]], on=["home_team","away_team","team"], how="left")
      .merge(ml_model[["home_team","away_team","team","model_prob"]], on=["home_team","away_team","team"], how="left"))

ml = ml.dropna(subset=["price_decimal","model_prob"]).copy()
ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)

# BLEND (market anchored)
W_ML = 0.60   # model weight (raise if you trust model more)
ml["blended_prob"] = np.clip(W_ML*ml["model_prob"] + (1-W_ML)*ml["consensus_prob"], 0.01, 0.99)

# Edge/EV/Kelly
ml["true_edge"] = ml["blended_prob"] - ml["implied_prob"]
ml["ev"] = ml["blended_prob"]*ml["price_decimal"] - 1.0
ml["units"] = ml.apply(lambda r: kelly_half(r["blended_prob"], r["price_decimal"]), axis=1)

# rank score (prefers EV + true_edge + unit)
ml["rank_score"] = 0.45*zscore(ml["ev"]) + 0.35*zscore(ml["true_edge"]) + 0.20*zscore(ml["units"])

# filters
MIN_EV = 0.01
MIN_EDGE = 0.01
MIN_UNITS = 0.01
ml_f = ml[(ml["ev"] >= MIN_EV) & (ml["true_edge"] >= MIN_EDGE) & (ml["units"] >= MIN_UNITS)].copy()

# NO DUPES: one bet per game (keep highest rank_score)
ml_f = pick_best_book(ml_f, key_cols=["home_team","away_team"], score_col="rank_score")

top10_ml = ml_f.sort_values("rank_score", ascending=False).head(10).reset_index(drop=True)

print(f"✅ ML candidates after filters: {len(ml_f)} | top10_ml: {len(top10_ml)}")

# =========================================
# 2) BUILD SPREAD BET TABLE (best book per team/game/spread)
# =========================================

# attach model_spread probabilities by (home,away,team,spread)
sp = sp_lines.dropna(subset=["price_decimal","spread"]).copy()

# best price per exact spread line per team/game (max decimal)
sp_best_price = (sp.sort_values("price_decimal", ascending=False)
                   .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first")
                   [["home_team","away_team","team","spread","book","price_decimal","american_odds","commence_time"]]
                   .reset_index(drop=True))

spm = (sp_best_price
       .merge(cons_sp[["home_team","away_team","team","consensus_prob","book_count"]], on=["home_team","away_team","team"], how="left")
       .merge(model_sp[["home_team","away_team","team","spread","model_prob_spread"]], on=["home_team","away_team","team","spread"], how="left"))

spm = spm.dropna(subset=["price_decimal","model_prob_spread"]).copy()
spm["implied_prob"] = spm["price_decimal"].apply(dec_to_implied)

W_SP = 0.60
spm["blended_prob"] = np.clip(W_SP*spm["model_prob_spread"] + (1-W_SP)*spm["consensus_prob"], 0.01, 0.99)

spm["true_edge"] = spm["blended_prob"] - spm["implied_prob"]
spm["ev"] = spm["blended_prob"]*spm["price_decimal"] - 1.0
spm["units"] = spm.apply(lambda r: kelly_half(r["blended_prob"], r["price_decimal"]), axis=1)

spm["rank_score"] = 0.45*zscore(spm["ev"]) + 0.35*zscore(spm["true_edge"]) + 0.20*zscore(spm["units"])

# filters
sp_f = spm[(spm["ev"] >= MIN_EV) & (spm["true_edge"] >= MIN_EDGE) & (spm["units"] >= MIN_UNITS)].copy()

# NO DUPES: one spread bet per game (highest rank_score)
sp_f = pick_best_book(sp_f, key_cols=["home_team","away_team"], score_col="rank_score")

top10_sp = sp_f.sort_values("rank_score", ascending=False).head(10).reset_index(drop=True)
print(f"✅ SP candidates after filters: {len(sp_f)} | top10_sp: {len(top10_sp)}")

# =========================================
# 3) DISCORD OUTPUT
# =========================================

def fmt_amer(a):
    try:
        a = float(a)
        if 1.01 <= a <= 20:
            # if it was decimal, don't show as american
            return ""
        sign = "+" if a > 0 else ""
        return f"{sign}{int(a)}"
    except:
        return str(a)

def discord_ml(df):
    lines = []
    lines.append("NCAAB — TOP 10 ML (NO DUPES | BLEND | ½-KELLY)")
    if len(df)==0:
        lines.append("No ML plays passed filters (EV/Edge/Units). Loosen MIN_EV / MIN_EDGE / MIN_UNITS.")
        return "\n".join(lines)

    for i, r in df.reset_index(drop=True).iterrows():
        lines.append(f"\n{i+1}) {r['home_team']} vs {r['away_team']}")
        lines.append(f"Bet: {r['team']} ML")
        amer = fmt_amer(r["american_odds"])
        lines.append(f"Book: {r['book']} | Odds: {amer} (Dec {r['price_decimal']:.2f})")
        lines.append(f"Model: {r['model_prob']:.3f} | Cons: {r['consensus_prob']:.3f} | Blend: {r['blended_prob']:.3f}")
        lines.append(f"Implied: {r['implied_prob']:.3f} | True Edge: {r['true_edge']:.3f} | EV: {r['ev']:.3f}")
        lines.append(f"Units (½ Kelly): {r['units']:.2f}u")
    return "\n".join(lines)

def discord_sp(df):
    lines = []
    lines.append("NCAAB — TOP 10 SPREAD (NO DUPES | BLEND | ½-KELLY)")
    if len(df)==0:
        lines.append("No spread plays passed filters (EV/Edge/Units). Loosen MIN_EV / MIN_EDGE / MIN_UNITS.")
        return "\n".join(lines)

    for i, r in df.reset_index(drop=True).iterrows():
        lines.append(f"\n{i+1}) {r['home_team']} vs {r['away_team']}")
        s = float(r["spread"])
        sign = "+" if s > 0 else ""
        lines.append(f"Bet: {r['team']} {sign}{s:g}")
        amer = fmt_amer(r["american_odds"])
        lines.append(f"Book: {r['book']} | Odds: {amer} (Dec {r['price_decimal']:.2f})")
        lines.append(f"Model: {r['model_prob_spread']:.3f} | Cons: {r['consensus_prob']:.3f} | Blend: {r['blended_prob']:.3f}")
        lines.append(f"Implied: {r['implied_prob']:.3f} | True Edge: {r['true_edge']:.3f} | EV: {r['ev']:.3f}")
        lines.append(f"Units (½ Kelly): {r['units']:.2f}u")
    return "\n".join(lines)

print("\n" + discord_ml(top10_ml))
print("\n" + "="*48 + "\n")
print(discord_sp(top10_sp))

# outputs available:
# top10_ml, top10_sp, ml, spm

In [ ]:
[x for x in globals().keys() if isinstance(globals()[x], pd.DataFrame)]

In [ ]:
import os, requests
import pandas as pd
import numpy as np

# =========================
# CONFIG (edit if needed)
# =========================
TODAY = "2026-02-27"                 # you said today is the 27th
SPORT = "basketball_ncaab"
REGION = "us"
ODDS_FORMAT = "american"             # set to american so prices come in correctly
MARKETS = "h2h,spreads"
ODDS_API_KEY = os.getenv("ODDS_API_KEY", "").strip()

# If you stored your key in a variable earlier, paste it here:
# ODDS_API_KEY = "YOUR_KEY_HERE"

if not ODDS_API_KEY:
    raise ValueError("Missing ODDS_API_KEY. Set env var ODDS_API_KEY or paste it into this cell.")

# =========================
# ODDS API PULL
# =========================
def pull_odds_raw(date_yyyy_mm_dd: str):
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": REGION,
        "markets": MARKETS,
        "oddsFormat": ODDS_FORMAT,
        "dateFormat": "iso",
    }
    r = requests.get(url, params=params, timeout=30)
    print("Status:", r.status_code)
    r.raise_for_status()
    return r.json()

def parse_to_odds_long(raw):
    rows = []
    for game in raw:
        home = game.get("home_team")
        away = game.get("away_team")
        t = game.get("commence_time")
        for bm in game.get("bookmakers", []):
            book = bm.get("key")
            for mkt in bm.get("markets", []):
                market = mkt.get("key")
                for out in mkt.get("outcomes", []):
                    team = out.get("name")
                    price = out.get("price")
                    point = out.get("point", np.nan)
                    rows.append({
                        "home_team": home,
                        "away_team": away,
                        "commence_time": t,
                        "book": book,
                        "market": market,
                        "team": team,
                        "american_odds": price,
                        "spread": point if market == "spreads" else np.nan
                    })
    df = pd.DataFrame(rows)
    return df

# =========================
# NORMALIZERS
# =========================
def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1.0 + (a / 100.0)
    else:
        return 1.0 + (100.0 / abs(a))

def decimal_to_implied(d):
    d = float(d)
    return 1.0 / d

def clean_book_dupes(df):
    # drop exact dupes
    return df.drop_duplicates(subset=["home_team","away_team","commence_time","book","market","team","american_odds","spread"]).copy()

# =========================
# BUILD
# =========================
raw_json = pull_odds_raw(TODAY)
odds_long = parse_to_odds_long(raw_json)
odds_long = clean_book_dupes(odds_long)

# normalize decimal + implied
odds_long["price_decimal"] = odds_long["american_odds"].apply(american_to_decimal)
odds_long["implied_prob"] = odds_long["price_decimal"].apply(decimal_to_implied)

print("✅ odds_long:", odds_long.shape)
print("Markets:", odds_long["market"].value_counts().to_dict())
display(odds_long.head(10))

# split markets
ml_lines = odds_long[odds_long["market"] == "h2h"].copy()
sp_lines = odds_long[odds_long["market"] == "spreads"].copy()

# consensus by avg implied prob (per game/team)
cons_ml = (ml_lines.groupby(["home_team","away_team","team"], as_index=False)
          .agg(consensus_prob=("implied_prob","mean"),
               book_count=("book","nunique")))

cons_sp = (sp_lines.groupby(["home_team","away_team","team"], as_index=False)
          .agg(consensus_prob=("implied_prob","mean"),
               book_count=("book","nunique"),
               cons_spread=("spread","mean")))

print("✅ ml_lines:", ml_lines.shape, "| ✅ sp_lines:", sp_lines.shape)
print("✅ cons_ml:", cons_ml.shape, "| ✅ cons_sp:", cons_sp.shape)
display(cons_ml.head(10))
display(cons_sp.head(10))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

# quick sanity check (should print True)
print("ODDS_API_KEY set:", bool(os.getenv("ODDS_API_KEY")))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
import numpy as np
import pandas as pd

# =========================
# 0) SAFETY CHECKS
# =========================
if "odds_long" not in globals():
    raise ValueError("odds_long not found. Re-run your odds ingestion cell first.")
if odds_long.empty:
    raise ValueError("odds_long is empty. Odds pull returned 0 rows.")

need = {"home_team","away_team","commence_time","book","market","team","american_odds","spread","price_decimal","implied_prob"}
missing = need - set(odds_long.columns)
if missing:
    raise ValueError(f"odds_long missing cols: {missing}")

# =========================
# 1) HELPERS
# =========================
def kelly_half(p, dec):
    # half-kelly fraction of bankroll; return "units" as fraction*100 if you want,
    # but we keep it as "u" scale by multiplying later
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b if b > 0 else 0.0
    return max(0.0, 0.5*f)

def fmt_amer(x):
    x = float(x)
    if x >= 100: return f"+{int(round(x))}"
    if x <= -100: return f"{int(round(x))}"
    return str(x)

def pick_best_line(df):
    # best payout for the bettor: maximize decimal odds
    return df.sort_values("price_decimal", ascending=False).head(1)

def safe_find_model_df():
    """
    Finds a DF in globals() that looks like ML model probs:
    needs columns: home_team, away_team, team, and one prob col.
    Prefers:
      - model_prob_ml
      - model_prob
    and rejects constant-prob DFs.
    """
    items = list(globals().items())  # snapshot to avoid "dict changed size" error
    cands = []
    for name, obj in items:
        if not isinstance(obj, pd.DataFrame):
            continue
        cols = set(obj.columns.astype(str))
        if not {"home_team","away_team","team"} <= cols:
            continue
        prob_cols = [c for c in obj.columns if c in ["model_prob_ml","model_prob"]]
        if not prob_cols:
            continue
        # pick best prob col (prefer model_prob_ml if present)
        prob_col = "model_prob_ml" if "model_prob_ml" in prob_cols else "model_prob"
        s = pd.to_numeric(obj[prob_col], errors="coerce").dropna()
        if len(s) < 10:
            continue
        # reject constant 0.99 bug
        if s.std() < 1e-6:
            continue
        cands.append((name, prob_col, len(s), float(s.std())))
    # choose candidate with largest n, then higher std
    if not cands:
        return None, None
    cands = sorted(cands, key=lambda x: (x[2], x[3]), reverse=True)
    best = cands[0]
    return best[0], best[1]

# =========================
# 2) SPLIT MARKETS + "BEST LINE" PER TEAM/GAME
# =========================
ml = odds_long[odds_long["market"].eq("h2h")].copy()
sp = odds_long[odds_long["market"].eq("spreads")].copy()

# best ML line per team/game
ml_best = (
    ml.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
      .apply(pick_best_line)
      .reset_index(drop=True)
)

# best SP line per team/game/spread number (keep spread in key)
sp_best = (
    sp.groupby(["home_team","away_team","team","spread"], as_index=False, group_keys=False)
      .apply(pick_best_line)
      .reset_index(drop=True)
)

# =========================
# 3) CONSENSUS (already computed in your notebook, but rebuild if missing)
# =========================
if "cons_ml" not in globals():
    # simple consensus: average implied probs across books (NOT vig-free; but ok baseline)
    cons_ml = (
        ml.groupby(["home_team","away_team","team"], as_index=False)
          .agg(consensus_prob=("implied_prob","mean"), book_count=("book","nunique"))
    )
if "cons_sp" not in globals():
    # consensus spread number = average point, consensus_prob = avg implied
    cons_sp = (
        sp.groupby(["home_team","away_team","team"], as_index=False)
          .agg(consensus_prob=("implied_prob","mean"),
               book_count=("book","nunique"),
               cons_spread=("spread","mean"))
    )

# =========================
# 4) ATTACH YOUR MODEL PROBS (AUTO-DETECT)
# =========================
model_df_name, prob_col = safe_find_model_df()
if model_df_name is None:
    print("⚠️ No NON-CONSTANT model probability DF found in globals().")
    print("    I will set model_prob = consensus_prob for now (edges ~ 0) so the pipeline runs.")
    model_probs = None
else:
    print(f"✅ Using model probs from DF: {model_df_name} | column: {prob_col}")
    model_src = globals()[model_df_name][["home_team","away_team","team", prob_col]].copy()
    model_src = model_src.rename(columns={prob_col:"model_prob"})
    model_probs = model_src

# ML attach
ml_out = ml_best.merge(cons_ml, on=["home_team","away_team","team"], how="left")
if model_probs is not None:
    ml_out = ml_out.merge(model_probs, on=["home_team","away_team","team"], how="left")
else:
    ml_out["model_prob"] = ml_out["consensus_prob"]

# SP attach: use the consensus spread NUMBER only (closest to cons_spread), then attach prob
sp_tmp = sp_best.merge(cons_sp[["home_team","away_team","team","cons_spread","consensus_prob","book_count"]],
                       on=["home_team","away_team","team"], how="left")

# keep the line whose spread is closest to consensus spread for that team (so we don't mix alt-lines)
sp_tmp["spread_diff"] = (sp_tmp["spread"] - sp_tmp["cons_spread"]).abs()
sp_out = (
    sp_tmp.sort_values(["home_team","away_team","team","spread_diff","price_decimal"], ascending=[True,True,True,True,False])
          .drop_duplicates(["home_team","away_team","team"], keep="first")
          .drop(columns=["spread_diff"])
)

# model spread probs: try to find a spread model DF if you have one (e.g., model_sp)
sp_model_name, sp_prob_col = None, None
items = list(globals().items())
sp_cands = []
for name, obj in items:
    if not isinstance(obj, pd.DataFrame):
        continue
    cols = set(obj.columns.astype(str))
    # spread model usually has home_team, away_team, team, and either 'spread' or 'cons_spread'
    if not {"home_team","away_team","team"} <= cols:
        continue
    if "model_prob_spread" in cols:
        s = pd.to_numeric(obj["model_prob_spread"], errors="coerce").dropna()
        if len(s) >= 10 and s.std() >= 1e-6:
            sp_cands.append((name, "model_prob_spread", len(s), float(s.std())))
    elif "model_prob" in cols and ("spread" in cols or "cons_spread" in cols):
        s = pd.to_numeric(obj["model_prob"], errors="coerce").dropna()
        if len(s) >= 10 and s.std() >= 1e-6:
            sp_cands.append((name, "model_prob", len(s), float(s.std())))
if sp_cands:
    sp_cands = sorted(sp_cands, key=lambda x: (x[2], x[3]), reverse=True)
    sp_model_name, sp_prob_col = sp_cands[0][0], sp_cands[0][1]
    print(f"✅ Using SPREAD model probs from DF: {sp_model_name} | column: {sp_prob_col}")
    sp_model_src = globals()[sp_model_name][["home_team","away_team","team", sp_prob_col]].copy()
    sp_model_src = sp_model_src.rename(columns={sp_prob_col:"model_prob_spread"})
    sp_out = sp_out.merge(sp_model_src, on=["home_team","away_team","team"], how="left")
else:
    print("⚠️ No NON-CONSTANT spread model DF found. Using consensus_prob as model_prob_spread for now.")
    sp_out["model_prob_spread"] = sp_out["consensus_prob"]

# =========================
# 5) EV + TRUE EDGE (VS PRICE) + ½-KELLY UNITS + FILTERS
# =========================
# ML
ml_out["implied_from_price"] = ml_out["implied_prob"]              # already computed from price
ml_out["true_edge"] = ml_out["model_prob"] - ml_out["implied_from_price"]
ml_out["ev"] = ml_out["model_prob"] * ml_out["price_decimal"] - 1.0
ml_out["units"] = ml_out.apply(lambda r: kelly_half(r["model_prob"], r["price_decimal"]), axis=1)

# Spread
sp_out["implied_from_price"] = sp_out["implied_prob"]
sp_out["true_edge"] = sp_out["model_prob_spread"] - sp_out["implied_from_price"]
sp_out["ev"] = sp_out["model_prob_spread"] * sp_out["price_decimal"] - 1.0
sp_out["units"] = sp_out.apply(lambda r: kelly_half(r["model_prob_spread"], r["price_decimal"]), axis=1)

# practical unit scale: cap + map to u's
def frac_to_units(f):  # f is half-kelly fraction
    # convert to units scale; 1.0u ~ 1% bankroll by default
    # tweak this if you want different mapping
    u = 100.0 * float(f)
    return max(0.0, min(5.0, u/1.0))  # cap at 5u

ml_out["units_u"] = ml_out["units"].apply(frac_to_units)
sp_out["units_u"] = sp_out["units"].apply(frac_to_units)

# Filters: +EV only + remove dupes per game (keep best bet per game by EV)
ml_f = ml_out.dropna(subset=["model_prob","price_decimal","ev"]).copy()
sp_f = sp_out.dropna(subset=["model_prob_spread","price_decimal","ev"]).copy()

# +EV only
ml_f = ml_f[ml_f["ev"] > 0].copy()
sp_f = sp_f[sp_f["ev"] > 0].copy()

# no dupes (one bet per game)
ml_f["game_key"] = ml_f["home_team"] + " vs " + ml_f["away_team"]
sp_f["game_key"] = sp_f["home_team"] + " vs " + sp_f["away_team"]

ml_top = (ml_f.sort_values(["ev","true_edge","units_u"], ascending=False)
            .drop_duplicates(["game_key"], keep="first")
            .head(10)
         )

sp_top = (sp_f.sort_values(["ev","true_edge","units_u"], ascending=False)
            .drop_duplicates(["game_key"], keep="first")
            .head(10)
         )

print(f"✅ ML top: {len(ml_top)} | ✅ SP top: {len(sp_top)}")

# =========================
# 6) DISCORD-FRIENDLY OUTPUT (NO DUPES, HAS UNITS)
# =========================
def print_ml_block(df):
    print("\nNCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)\n")
    if df.empty:
        print("No ML plays passed +EV filter.")
        return
    for i, r in enumerate(df.itertuples(index=False), 1):
        print(f"{i}) {r.home_team} vs {r.away_team}")
        print(f"Bet: {r.team} ML")
        print(f"Book: {r.book} | Odds (Amer): {fmt_amer(r.american_odds)} (Dec {r.price_decimal:.2f})")
        print(f"Model: {r.model_prob:.3f} | Implied: {r.implied_from_price:.3f} | Cons: {r.consensus_prob:.3f}")
        print(f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f} | Units (½ Kelly): {r.units_u:.2f}u\n")

def print_sp_block(df):
    print("\nNCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)\n")
    if df.empty:
        print("No spread plays passed +EV filter.")
        return
    for i, r in enumerate(df.itertuples(index=False), 1):
        line = r.spread
        print(f"{i}) {r.home_team} vs {r.away_team}")
        print(f"Bet: {r.team} {line:+g}")
        print(f"Book: {r.book} | Odds (Amer): {fmt_amer(r.american_odds)} (Dec {r.price_decimal:.2f})")
        print(f"Model(Cover): {r.model_prob_spread:.3f} | Implied: {r.implied_from_price:.3f} | Cons: {r.consensus_prob:.3f}")
        print(f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f} | Units (½ Kelly): {r.units_u:.2f}u\n")

print_ml_block(ml_top)
print_sp_block(sp_top)

# keep vars for next cells
top10_ml = ml_top.copy()
top10_sp = sp_top.copy()
print("✅ Saved: top10_ml, top10_sp, ml_out, sp_out, ml_best, sp_best")

In [ ]:
import numpy as np
import pandas as pd

# -------------------------
# REQUIRED INPUTS
# -------------------------
# odds_long: from odds-api ingest (has markets h2h/spreads + price_decimal + implied_prob)
# ml_model: game/team ML win probs (columns: home_team, away_team, team, model_prob)
# model_sp: spread cover probs (columns: home_team, away_team, team, spread, model_prob_spread)

for name in ["odds_long","ml_model","model_sp"]:
    if name not in globals():
        raise ValueError(f"Missing {name}. It exists earlier in your notebook—make sure that cell ran.")

# -------------------------
# HELPERS
# -------------------------
def kelly_half(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    f = (b*p - q) / b
    return max(0.0, 0.5*f)

def frac_to_units(f):
    # 1u = 1% bankroll; cap 5u
    u = 100.0 * float(f)
    return max(0.0, min(5.0, u))

def pick_best_line(df):
    return df.sort_values("price_decimal", ascending=False).head(1)

def fmt_amer(x):
    x = float(x)
    if x >= 100: return f"+{int(round(x))}"
    if x <= -100: return f"{int(round(x))}"
    return str(x)

# -------------------------
# 1) SPLIT MARKETS + BEST LINE
# -------------------------
ml_lines = odds_long[odds_long["market"].eq("h2h")].copy()
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()

ml_best = (ml_lines.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
                  .apply(pick_best_line).reset_index(drop=True))

sp_best = (sp_lines.groupby(["home_team","away_team","team","spread"], as_index=False, group_keys=False)
                  .apply(pick_best_line).reset_index(drop=True))

# -------------------------
# 2) CONSENSUS (price-based; keep it for context)
# -------------------------
cons_ml = (ml_lines.groupby(["home_team","away_team","team"], as_index=False)
                  .agg(consensus_prob=("implied_prob","mean"), book_count=("book","nunique")))

# For spreads: consensus spread number per team + implied prob avg
cons_sp = (sp_lines.groupby(["home_team","away_team","team"], as_index=False)
                  .agg(consensus_prob=("implied_prob","mean"),
                       book_count=("book","nunique"),
                       cons_spread=("spread","mean")))

# -------------------------
# 3) ATTACH *REAL* MODEL PROBS
# -------------------------
ml_model_use = ml_model[["home_team","away_team","team","model_prob"]].copy()

# spread model must have: home_team away_team team spread model_prob_spread
need_sp = {"home_team","away_team","team","spread","model_prob_spread"}
miss_sp = need_sp - set(model_sp.columns)
if miss_sp:
    raise ValueError(f"model_sp missing columns: {miss_sp}")

sp_model_use = model_sp[["home_team","away_team","team","spread","model_prob_spread"]].copy()

# -------------------------
# 4) BUILD ML OUTPUT (NO DUPES PER GAME)
# -------------------------
ml_out = (ml_best
          .merge(cons_ml, on=["home_team","away_team","team"], how="left")
          .merge(ml_model_use, on=["home_team","away_team","team"], how="left"))

ml_out = ml_out.dropna(subset=["model_prob","price_decimal","implied_prob"]).copy()
ml_out["true_edge"] = ml_out["model_prob"] - ml_out["implied_prob"]
ml_out["ev"] = ml_out["model_prob"] * ml_out["price_decimal"] - 1.0
ml_out["units_u"] = ml_out.apply(lambda r: frac_to_units(kelly_half(r["model_prob"], r["price_decimal"])), axis=1)
ml_out["game_key"] = ml_out["home_team"] + " vs " + ml_out["away_team"]

ml_top = (ml_out[ml_out["ev"] > 0]
          .sort_values(["ev","true_edge","units_u"], ascending=False)
          .drop_duplicates(["game_key"], keep="first")
          .head(10)
          .reset_index(drop=True))

# -------------------------
# 5) BUILD SPREAD OUTPUT (use closest line to model_sp spread)
# -------------------------
# attach model_spread by exact spread where possible
sp_out = (sp_best
          .merge(cons_sp[["home_team","away_team","team","cons_spread","consensus_prob","book_count"]],
                 on=["home_team","away_team","team"], how="left")
          .merge(sp_model_use, on=["home_team","away_team","team","spread"], how="left"))

# if some spreads didn't match exactly, fall back to closest spread per game/team
sp_out["abs_diff"] = np.where(sp_out["model_prob_spread"].isna(),
                              (sp_out["spread"] - sp_out["cons_spread"]).abs(),
                              0.0)

sp_out = (sp_out.sort_values(["home_team","away_team","team","abs_diff","price_decimal"],
                             ascending=[True,True,True,True,False])
                .drop_duplicates(["home_team","away_team","team"], keep="first")
                .drop(columns=["abs_diff"])
                .reset_index(drop=True))

sp_out = sp_out.dropna(subset=["model_prob_spread","price_decimal","implied_prob"]).copy()
sp_out["true_edge"] = sp_out["model_prob_spread"] - sp_out["implied_prob"]
sp_out["ev"] = sp_out["model_prob_spread"] * sp_out["price_decimal"] - 1.0
sp_out["units_u"] = sp_out.apply(lambda r: frac_to_units(kelly_half(r["model_prob_spread"], r["price_decimal"])), axis=1)
sp_out["game_key"] = sp_out["home_team"] + " vs " + sp_out["away_team"]

sp_top = (sp_out[sp_out["ev"] > 0]
          .sort_values(["ev","true_edge","units_u"], ascending=False)
          .drop_duplicates(["game_key"], keep="first")
          .head(10)
          .reset_index(drop=True))

# -------------------------
# 6) PRINT DISCORD OUTPUT
# -------------------------
print("✅ Using TRUE model sources: ml_model + model_sp")
print(f"ML model std: {ml_model_use['model_prob'].std():.6f} | SP model std: {sp_model_use['model_prob_spread'].std():.6f}")

print("\nNCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)\n")
for i, r in enumerate(ml_top.itertuples(index=False), 1):
    print(f"{i}) {r.home_team} vs {r.away_team}")
    print(f"Bet: {r.team} ML")
    print(f"Book: {r.book} | Odds: {fmt_amer(r.american_odds)} (Dec {r.price_decimal:.2f})")
    print(f"Model: {r.model_prob:.3f} | Implied: {r.implied_prob:.3f} | Cons: {r.consensus_prob:.3f}")
    print(f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f} | Units (½ Kelly): {r.units_u:.2f}u\n")

print("\nNCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)\n")
for i, r in enumerate(sp_top.itertuples(index=False), 1):
    print(f"{i}) {r.home_team} vs {r.away_team}")
    print(f"Bet: {r.team} {r.spread:+g}")
    print(f"Book: {r.book} | Odds: {fmt_amer(r.american_odds)} (Dec {r.price_decimal:.2f})")
    print(f"Model(Cover): {r.model_prob_spread:.3f} | Implied: {r.implied_prob:.3f} | Cons: {r.consensus_prob:.3f}")
    print(f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f} | Units (½ Kelly): {r.units_u:.2f}u\n")

# save
top10_ml = ml_top
top10_sp = sp_top
print("✅ Saved: top10_ml, top10_sp, ml_out, sp_out")

In [ ]:
import pandas as pd
import numpy as np

# -------------------------
# Find ML model DF candidate
# -------------------------
cands = []
for k, v in list(globals().items()):
    if isinstance(v, pd.DataFrame):
        cols = set(map(str, v.columns))
        if {"home_team","away_team","team"}.issubset(cols) and ("model_prob" in cols or "model_prob_ml" in cols):
            # stats for whichever prob col exists
            prob_col = "model_prob" if "model_prob" in cols else "model_prob_ml"
            s = v[prob_col].dropna()
            if len(s) > 0:
                cands.append((k, v.shape[0], prob_col, float(s.std()), float(s.min()), float(s.max()), int(s.nunique())))

# sort: prefer non-constant + more rows
cands = sorted(cands, key=lambda x: (x[3] < 1e-6, -x[1]))
print("🔎 ML model candidates (name, rows, prob_col, std, min, max, unique):")
for row in cands[:20]:
    print(" -", row)

if not cands:
    raise ValueError(
        "No DataFrame found that looks like your ML model output.\n"
        "I need a DF with columns [home_team, away_team, team] + a probability column (model_prob or model_prob_ml)."
    )

best_name, _, best_prob_col, best_std, best_min, best_max, best_unique = cands[0]
src = globals()[best_name].copy()

# -------------------------
# Standardize -> ml_model
# -------------------------
src = src.rename(columns={best_prob_col: "model_prob"})
ml_model = src[["home_team","away_team","team","model_prob"]].copy()

# Hard safety: drop NaNs + enforce bounds
ml_model = ml_model.dropna(subset=["model_prob"]).copy()
ml_model["model_prob"] = ml_model["model_prob"].clip(0.0001, 0.9999)

print(f"\n✅ Using ML model DF: {best_name} | prob_col={best_prob_col}")
print("model_prob stats:", ml_model["model_prob"].describe())
print("per-game prob sum mean:",
      ml_model.groupby(["home_team","away_team"])["model_prob"].sum().mean())

# Fail if constant (prevents '0.99 everywhere' bug)
if ml_model["model_prob"].std() < 1e-6:
    raise ValueError(
        "❌ model_prob is still CONSTANT after standardization.\n"
        "That means the model isn't running—it's being hard-set somewhere.\n"
        "Find the cell where model_prob is created and remove any hard-coded 0.99 / consensus override."
    )

In [ ]:
import pandas as pd

if "model_sp" not in globals():
    raise ValueError("Missing model_sp. You had it earlier—run that cell again.")

# If it's using model_prob, rename to model_prob_spread
if "model_prob_spread" not in model_sp.columns and "model_prob" in model_sp.columns:
    model_sp = model_sp.rename(columns={"model_prob":"model_prob_spread"})

# If spread column is missing but cons_spread exists, rename
if "spread" not in model_sp.columns and "cons_spread" in model_sp.columns:
    model_sp = model_sp.rename(columns={"cons_spread":"spread"})

need = {"home_team","away_team","team","spread","model_prob_spread"}
missing = need - set(model_sp.columns)
if missing:
    raise ValueError(f"model_sp still missing columns: {missing}")

print("✅ model_sp ready.", model_sp[["home_team","away_team","team","spread","model_prob_spread"]].head())
print("model_prob_spread std:", float(model_sp["model_prob_spread"].std()))

In [ ]:
import pandas as pd
import numpy as np

# Hard requirements
if "odds_long" not in globals():
    raise ValueError("Missing odds_long. Run the odds pull/parse cell first.")
if "cons_sp" not in globals():
    raise ValueError("Missing cons_sp. Run the consensus build cell first (the one that prints cons_sp).")

# 1) Build a spread model table from consensus (proxy until you plug in true sim cover probs)
#    cons_sp has: home_team, away_team, team, consensus_prob, book_count, cons_spread
model_sp = cons_sp.copy()

# Rename to standardized names
if "cons_spread" in model_sp.columns and "spread" not in model_sp.columns:
    model_sp = model_sp.rename(columns={"cons_spread": "spread"})
if "consensus_prob" not in model_sp.columns:
    raise ValueError("cons_sp missing consensus_prob column. Rebuild cons_sp correctly.")

# TEMP: model_prob_spread proxy = consensus_prob (replace later with your real cover sim probs)
model_sp["model_prob_spread"] = model_sp["consensus_prob"].astype(float)

# Keep only what we need
model_sp = model_sp[["home_team","away_team","team","spread","model_prob_spread"]].copy()

print("✅ model_sp rebuilt from cons_sp (TEMP proxy).")
print("shape:", model_sp.shape)
print("model_prob_spread stats:", model_sp["model_prob_spread"].describe())
print(model_sp.head(10))

# 2) Attach model_prob_spread onto spread betting lines (odds_long spreads)
sp = odds_long[odds_long["market"].eq("spreads")].copy()
if "price_decimal" not in sp.columns:
    raise ValueError("odds_long is missing price_decimal — rerun your normalization cell.")

# Merge on home/away/team (spread line can vary by book, so we DO NOT merge on spread value)
sp = sp.merge(model_sp[["home_team","away_team","team","model_prob_spread"]],
              on=["home_team","away_team","team"], how="left")

# Resolve possible suffixes if you already had columns
cands = [c for c in sp.columns if c.startswith("model_prob_spread")]
if "model_prob_spread" not in sp.columns and cands:
    # pick first candidate and rename
    sp = sp.rename(columns={cands[0]: "model_prob_spread"})

if "model_prob_spread" not in sp.columns:
    raise ValueError("Failed to attach model_prob_spread onto spread lines.")

nulls = int(sp["model_prob_spread"].isna().sum())
print(f"\n✅ Spread lines ready: rows={len(sp)} | null model_prob_spread={nulls}")

# 3) Quick +EV calc for spreads so you can move forward
def dec_to_implied(d):
    return 1.0 / d

def half_kelly(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b
    return max(0.0, 0.5*f)

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units"] = sp.apply(lambda r: half_kelly(r["model_prob_spread"], r["price_decimal"]), axis=1)

# No dupes: keep best book per (game, team) by EV, then top 10 overall
top10_sp = (
    sp.sort_values("ev", ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
      .head(10)
      .reset_index(drop=True)
)

print("\n✅ top10_sp built:", top10_sp.shape)
print(top10_sp[["home_team","away_team","team","book","spread","american_odds","price_decimal","model_prob_spread","implied_prob","true_edge","ev","units"]])

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# ================================
# HYBRID STABLE v3 — MARGIN ENGINE
# ================================

SIGMA_NCAAB = 11.5        # margin std deviation
EDGE_TO_MARGIN = 14.0     # converts win-prob edge into margin points (tunable)

# -------------------------
# Build game-level baseline from consensus spread
# -------------------------

# cons_sp has: home_team, away_team, team, cons_spread
game_margin = (
    cons_sp.groupby(["home_team","away_team"], as_index=False)
    .agg(avg_spread=("cons_spread","mean"))
)

# avg_spread is already signed from each team's perspective
# We'll compute margin from home team perspective

home_rows = cons_sp.copy()
home_rows = home_rows[home_rows["team"] == home_rows["home_team"]]
home_rows = home_rows[["home_team","away_team","cons_spread"]]
home_rows = home_rows.rename(columns={"cons_spread":"market_margin"})

# -------------------------
# Attach ML model probability (your real one)
# -------------------------

if "ml_model" not in globals():
    raise ValueError("ml_model not found. You must standardize ML model first.")

ml_game = ml_model.copy()

# Compute market implied win prob (vigged) from consensus ML
market_ml = cons_ml.copy()

ml_game = ml_game.merge(
    market_ml[["home_team","away_team","team","consensus_prob"]],
    on=["home_team","away_team","team"],
    how="left"
)

ml_game["win_edge"] = ml_game["model_prob"] - ml_game["consensus_prob"]

# -------------------------
# Convert win-edge → margin adjustment
# -------------------------

ml_game["margin_adjustment"] = ml_game["win_edge"] * EDGE_TO_MARGIN

# Keep home-team row only to anchor margin shift
home_adj = ml_game[ml_game["team"] == ml_game["home_team"]]
home_adj = home_adj[["home_team","away_team","margin_adjustment"]]

# -------------------------
# Final projected margin
# -------------------------

hybrid_margin = home_rows.merge(home_adj, on=["home_team","away_team"], how="left")
hybrid_margin["margin_adjustment"] = hybrid_margin["margin_adjustment"].fillna(0)

hybrid_margin["projected_margin"] = (
    hybrid_margin["market_margin"] + hybrid_margin["margin_adjustment"]
)

# -------------------------
# Convert margin → probabilities
# -------------------------

# ML win probability (home perspective)
hybrid_margin["home_win_prob"] = 1 - norm.cdf(
    0,
    loc=hybrid_margin["projected_margin"],
    scale=SIGMA_NCAAB
)

# Cover probability for market spread (home perspective)
hybrid_margin["home_cover_prob"] = 1 - norm.cdf(
    hybrid_margin["market_margin"],
    loc=hybrid_margin["projected_margin"],
    scale=SIGMA_NCAAB
)

print("✅ Hybrid margin engine built.")
print(hybrid_margin.head())

print("\nProjected margin stats:")
print(hybrid_margin["projected_margin"].describe())

print("\nHome win prob stats:")
print(hybrid_margin["home_win_prob"].describe())

In [ ]:
print("ML model std:", ml_model["model_prob"].std())
print("\nWin edge preview:")

tmp = ml_model.merge(
    cons_ml[["home_team","away_team","team","consensus_prob"]],
    on=["home_team","away_team","team"],
    how="left"
)

tmp["edge"] = tmp["model_prob"] - tmp["consensus_prob"]

print(tmp[["home_team","away_team","team","model_prob","consensus_prob","edge"]].head(10))
print("\nEdge stats:")
print(tmp["edge"].describe())

In [ ]:
import requests, pandas as pd

def espn_ncaab_scoreboard(date_yyyymmdd: str):
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
    r = requests.get(url, params={"dates": date_yyyymmdd}, timeout=30)
    r.raise_for_status()
    return r.json()

def build_slate_from_scoreboard(sb_json):
    events = sb_json.get("events", [])
    rows = []
    for ev in events:
        comp = (ev.get("competitions") or [{}])[0]
        comps = comp.get("competitors") or []
        if len(comps) != 2:
            continue

        # ESPN labels "home"/"away"
        by_ha = {c.get("homeAway"): c for c in comps}
        home = by_ha.get("home", {})
        away = by_ha.get("away", {})

        def pick_team(c):
            t = c.get("team") or {}
            return {
                "team_name": t.get("displayName"),
                "team_abbr": t.get("abbreviation"),
                "team_id": t.get("id"),
            }

        rows.append({
            "event_id": ev.get("id"),
            "commence_time": ev.get("date"),
            "home_team": pick_team(home)["team_name"],
            "away_team": pick_team(away)["team_name"],
            "home_team_id": pick_team(home)["team_id"],
            "away_team_id": pick_team(away)["team_id"],
        })

    return pd.DataFrame(rows).dropna(subset=["event_id","home_team","away_team"])

TODAY = "20260227"  # <-- change as needed
sb = espn_ncaab_scoreboard(TODAY)
slate_df = build_slate_from_scoreboard(sb)

print("✅ Slate rows:", slate_df.shape)
slate_df.head(10)

In [ ]:
import time

def espn_event_summary(event_id: str):
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary"
    r = requests.get(url, params={"event": event_id}, timeout=30)
    r.raise_for_status()
    return r.json()

# Pull summaries (light rate limit)
summaries = []
for eid in slate_df["event_id"].astype(str).tolist():
    try:
        j = espn_event_summary(eid)
        summaries.append({
            "event_id": eid,
            "has_boxscore": "boxscore" in j,
            "has_header": "header" in j,
        })
    except Exception as e:
        summaries.append({"event_id": eid, "error": str(e)})
    time.sleep(0.25)

summaries_df = pd.DataFrame(summaries)
print("✅ summaries_df:", summaries_df.shape)
summaries_df.head(10)

In [ ]:
import requests, pandas as pd, numpy as np, time, re
from datetime import datetime

ESPN_BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball"

def espn_get_json(url, params=None, timeout=20):
    r = requests.get(url, params=params, timeout=timeout, headers={"User-Agent":"Mozilla/5.0"})
    r.raise_for_status()
    return r.json()

def yyyymmdd(date_str):
    # accepts "2026-02-27" or "20260227"
    if "-" in date_str:
        return date_str.replace("-", "")
    return date_str

def pull_scoreboard_events(date_str="2026-02-27"):
    date_key = yyyymmdd(date_str)
    url = f"{ESPN_BASE}/scoreboard"
    js = espn_get_json(url, params={"dates": date_key})
    events = js.get("events", []) or []
    rows = []
    for ev in events:
        eid = ev.get("id")
        comp = (ev.get("competitions") or [{}])[0]
        comps = comp.get("competitors") or []
        if len(comps) >= 2:
            home = next((c for c in comps if c.get("homeAway")=="home"), comps[0])
            away = next((c for c in comps if c.get("homeAway")=="away"), comps[1])
            rows.append({
                "event_id": eid,
                "date": date_str,
                "commence_time": comp.get("date"),
                "home_team": (home.get("team") or {}).get("displayName"),
                "away_team": (away.get("team") or {}).get("displayName"),
            })
    return pd.DataFrame(rows).dropna(subset=["event_id"])

# If you only got 2 events before, this will usually fix it:
events_df = pull_scoreboard_events("2026-02-27")
print("✅ events_df:", events_df.shape)
display(events_df.head(10))

# If you already have summaries_df with 2 rows, we’ll also keep it:
if "summaries_df" in globals() and isinstance(summaries_df, pd.DataFrame) and "event_id" in summaries_df.columns:
    print("✅ summaries_df found:", summaries_df.shape)

In [ ]:
def pull_event_summary(event_id):
    url = f"{ESPN_BASE}/summary"
    js = espn_get_json(url, params={"event": str(event_id)})
    return js

def extract_pickcenter(summary_js):
    """
    ESPN summary usually includes pickcenter (spread, over/under, moneyline, predictor, etc.)
    We’ll pull what we can reliably find.
    """
    pc_list = summary_js.get("pickcenter") or []
    if not pc_list:
        return {}
    pc = pc_list[0] or {}

    # predictor is sometimes nested
    predictor = pc.get("predictor")
    if isinstance(predictor, dict):
        home_win_pct = predictor.get("homeTeam", {}).get("gameProjection")
        away_win_pct = predictor.get("awayTeam", {}).get("gameProjection")
    else:
        home_win_pct = None
        away_win_pct = None

    return {
        "espn_spread": pc.get("spread"),
        "espn_over_under": pc.get("overUnder"),
        "espn_home_ml": pc.get("homeTeamOdds"),
        "espn_away_ml": pc.get("awayTeamOdds"),
        "espn_home_win_pct": home_win_pct,
        "espn_away_win_pct": away_win_pct,
    }

def extract_teams(summary_js):
    # From header -> competitions -> competitors
    header = summary_js.get("header") or {}
    comp = (header.get("competitions") or [{}])[0]
    competitors = comp.get("competitors") or []

    home = next((c for c in competitors if c.get("homeAway")=="home"), None)
    away = next((c for c in competitors if c.get("homeAway")=="away"), None)

    def team_name(c):
        if not c: return None
        t = c.get("team") or {}
        return t.get("displayName") or t.get("name")

    def score(c):
        if not c: return None
        try:
            return float(c.get("score")) if c.get("score") is not None else None
        except:
            return None

    return {
        "home_team": team_name(home),
        "away_team": team_name(away),
        "home_score": score(home),
        "away_score": score(away),
        "commence_time": comp.get("date"),
    }

def build_espn_projection_layer(event_ids, sleep_s=0.15):
    rows = []
    for eid in event_ids:
        try:
            js = pull_event_summary(eid)
            t = extract_teams(js)
            pc = extract_pickcenter(js)

            # Build per-team rows (so we can merge to odds_long on home/away/team)
            # If predictor exists, use that. If not, leave as NaN.
            for side in ["home","away"]:
                team = t["home_team"] if side=="home" else t["away_team"]
                wpct = pc.get("espn_home_win_pct") if side=="home" else pc.get("espn_away_win_pct")

                rows.append({
                    "event_id": str(eid),
                    "commence_time": t["commence_time"],
                    "home_team": t["home_team"],
                    "away_team": t["away_team"],
                    "team": team,
                    "espn_spread": pc.get("espn_spread"),
                    "espn_over_under": pc.get("espn_over_under"),
                    "espn_home_win_pct": pc.get("espn_home_win_pct"),
                    "espn_away_win_pct": pc.get("espn_away_win_pct"),
                    "espn_model_prob_ml": (wpct/100.0) if isinstance(wpct,(int,float)) else np.nan,
                })
            time.sleep(sleep_s)
        except Exception as e:
            rows.append({"event_id": str(eid), "error": str(e)})
            continue

    df = pd.DataFrame(rows)
    if "error" in df.columns:
        bad = df[df["error"].notna()]
        if len(bad):
            print("⚠️ Some events failed:", len(bad))
            display(bad[["event_id","error"]].head(10))

    df = df[df.get("team").notna()] if "team" in df.columns else df
    return df

# Choose event_ids:
# Prefer full slate from events_df; fallback to summaries_df.
if "events_df" in globals() and len(events_df):
    event_ids = events_df["event_id"].astype(str).tolist()
elif "summaries_df" in globals() and len(summaries_df):
    event_ids = summaries_df["event_id"].astype(str).tolist()
else:
    raise ValueError("No event_ids found. Run Cell 1 first.")

espn_layer_df = build_espn_projection_layer(event_ids)
print("✅ espn_layer_df:", espn_layer_df.shape)
display(espn_layer_df.head(10))

print("\nESPN ML model prob stats (non-null):")
if "espn_model_prob_ml" in espn_layer_df.columns:
    s = espn_layer_df["espn_model_prob_ml"].dropna()
    print(s.describe())
    print("unique probs:", s.nunique())

In [ ]:
def pull_event_summary(event_id):
    url = f"{ESPN_BASE}/summary"
    js = espn_get_json(url, params={"event": str(event_id)})
    return js

def extract_pickcenter(summary_js):
    """
    ESPN summary usually includes pickcenter (spread, over/under, moneyline, predictor, etc.)
    We’ll pull what we can reliably find.
    """
    pc_list = summary_js.get("pickcenter") or []
    if not pc_list:
        return {}
    pc = pc_list[0] or {}

    # predictor is sometimes nested
    predictor = pc.get("predictor")
    if isinstance(predictor, dict):
        home_win_pct = predictor.get("homeTeam", {}).get("gameProjection")
        away_win_pct = predictor.get("awayTeam", {}).get("gameProjection")
    else:
        home_win_pct = None
        away_win_pct = None

    return {
        "espn_spread": pc.get("spread"),
        "espn_over_under": pc.get("overUnder"),
        "espn_home_ml": pc.get("homeTeamOdds"),
        "espn_away_ml": pc.get("awayTeamOdds"),
        "espn_home_win_pct": home_win_pct,
        "espn_away_win_pct": away_win_pct,
    }

def extract_teams(summary_js):
    # From header -> competitions -> competitors
    header = summary_js.get("header") or {}
    comp = (header.get("competitions") or [{}])[0]
    competitors = comp.get("competitors") or []

    home = next((c for c in competitors if c.get("homeAway")=="home"), None)
    away = next((c for c in competitors if c.get("homeAway")=="away"), None)

    def team_name(c):
        if not c: return None
        t = c.get("team") or {}
        return t.get("displayName") or t.get("name")

    def score(c):
        if not c: return None
        try:
            return float(c.get("score")) if c.get("score") is not None else None
        except:
            return None

    return {
        "home_team": team_name(home),
        "away_team": team_name(away),
        "home_score": score(home),
        "away_score": score(away),
        "commence_time": comp.get("date"),
    }

def build_espn_projection_layer(event_ids, sleep_s=0.15):
    rows = []
    for eid in event_ids:
        try:
            js = pull_event_summary(eid)
            t = extract_teams(js)
            pc = extract_pickcenter(js)

            # Build per-team rows (so we can merge to odds_long on home/away/team)
            # If predictor exists, use that. If not, leave as NaN.
            for side in ["home","away"]:
                team = t["home_team"] if side=="home" else t["away_team"]
                wpct = pc.get("espn_home_win_pct") if side=="home" else pc.get("espn_away_win_pct")

                rows.append({
                    "event_id": str(eid),
                    "commence_time": t["commence_time"],
                    "home_team": t["home_team"],
                    "away_team": t["away_team"],
                    "team": team,
                    "espn_spread": pc.get("espn_spread"),
                    "espn_over_under": pc.get("espn_over_under"),
                    "espn_home_win_pct": pc.get("espn_home_win_pct"),
                    "espn_away_win_pct": pc.get("espn_away_win_pct"),
                    "espn_model_prob_ml": (wpct/100.0) if isinstance(wpct,(int,float)) else np.nan,
                })
            time.sleep(sleep_s)
        except Exception as e:
            rows.append({"event_id": str(eid), "error": str(e)})
            continue

    df = pd.DataFrame(rows)
    if "error" in df.columns:
        bad = df[df["error"].notna()]
        if len(bad):
            print("⚠️ Some events failed:", len(bad))
            display(bad[["event_id","error"]].head(10))

    df = df[df.get("team").notna()] if "team" in df.columns else df
    return df

# Choose event_ids:
# Prefer full slate from events_df; fallback to summaries_df.
if "events_df" in globals() and len(events_df):
    event_ids = events_df["event_id"].astype(str).tolist()
elif "summaries_df" in globals() and len(summaries_df):
    event_ids = summaries_df["event_id"].astype(str).tolist()
else:
    raise ValueError("No event_ids found. Run Cell 1 first.")

espn_layer_df = build_espn_projection_layer(event_ids)
print("✅ espn_layer_df:", espn_layer_df.shape)
display(espn_layer_df.head(10))

print("\nESPN ML model prob stats (non-null):")
if "espn_model_prob_ml" in espn_layer_df.columns:
    s = espn_layer_df["espn_model_prob_ml"].dropna()
    print(s.describe())
    print("unique probs:", s.nunique())

In [ ]:
def ensure_cols(df, cols, name="df"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

ensure_cols(odds_long, ["home_team","away_team","book","market","team","price_decimal","implied_prob"], "odds_long")
ensure_cols(espn_layer_df, ["home_team","away_team","team","espn_model_prob_ml"], "espn_layer_df")

# Merge ESPN projection layer onto odds_long
od = odds_long.merge(
    espn_layer_df[["home_team","away_team","team","espn_model_prob_ml","espn_spread","espn_over_under"]],
    on=["home_team","away_team","team"],
    how="left"
)

# --- ML model probs ---
ml = od[od["market"]=="h2h"].copy()
ml["model_prob_ml"] = ml["espn_model_prob_ml"]  # use ESPN predictor when present

# Safety: if ESPN predictor is missing for a game, we can fall back to consensus or skip.
# Here we SKIP missing model probs (you can change to fallback later).
ml = ml.dropna(subset=["model_prob_ml"])

# Compute edge/EV/half-kelly vs PRICE (not vs consensus)
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0

def kelly_half(p, dec_odds):
    b = dec_odds - 1.0
    q = 1.0 - p
    f = (b*p - q) / b if b > 0 else 0.0
    f = max(0.0, f)
    return 0.5 * f

ml["units"] = ml.apply(lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"]), axis=1)

# No dupes: keep best book per (home,away,team) by EV (or units)
ml_best = (ml.sort_values(["ev","units"], ascending=False)
             .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

top10_ml = ml_best.sort_values(["ev","units"], ascending=False).head(10).copy()

print("✅ top10_ml:", top10_ml.shape)
display(top10_ml[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","true_edge","ev","units"]])

# --- Spread model probs ---
# ESPN predictor does NOT directly give cover probability.
# If you ALSO have a real spread simulator layer, plug it in here.
# For now, we’ll just compute +EV on spread only if you already have model_prob_spread in your notebook.
sp = od[od["market"]=="spreads"].copy()

if "model_prob_spread" in sp.columns:
    sp = sp.dropna(subset=["model_prob_spread"])
    sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
    sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
    sp["units"] = sp.apply(lambda r: kelly_half(r["model_prob_spread"], r["price_decimal"]), axis=1)

    sp_best = (sp.sort_values(["ev","units"], ascending=False)
                 .drop_duplicates(subset=["home_team","away_team","team","spread"], keep="first"))
    top10_sp = sp_best.sort_values(["ev","units"], ascending=False).head(10).copy()

    print("✅ top10_sp:", top10_sp.shape)
    display(top10_sp[["home_team","away_team","team","spread","book","american_odds","price_decimal","model_prob_spread","implied_prob","true_edge","ev","units"]])
else:
    print("⚠️ No model_prob_spread found. ML is now real (ESPN predictor).")
    print("   Next step: we build cover-prob from projected margin + sigma, OR scrape a free projection source for spreads.")

In [ ]:
# DIAG: how many ESPN model probs are non-null?
print("espn_layer_df shape:", espn_layer_df.shape)
print("Non-null espn_model_prob_ml:", int(espn_layer_df["espn_model_prob_ml"].notna().sum()), "of", len(espn_layer_df))

# show a few rows where it's missing
display(espn_layer_df[espn_layer_df["espn_model_prob_ml"].isna()][
    ["event_id","home_team","away_team","espn_spread","espn_over_under","espn_home_win_pct","espn_away_win_pct","espn_model_prob_ml"]
].head(15))

# Pull 1 raw summary json and show keys so we know what's actually inside
test_eid = str(espn_layer_df["event_id"].iloc[0])
raw_js = pull_event_summary(test_eid)
print("Top-level keys:", list(raw_js.keys())[:30])

pc = raw_js.get("pickcenter")
print("pickcenter type:", type(pc), "| len:", (len(pc) if isinstance(pc,list) else None))
if isinstance(pc, list) and len(pc):
    print("pickcenter[0] keys:", list(pc[0].keys())[:50])
    pred = pc[0].get("predictor")
    print("predictor:", pred)
else:
    print("No pickcenter in this summary payload.")

In [ ]:
# Build a game-level projected margin from consensus spread
# convention: negative spread means HOME is favored by abs(spread)

# You already have cons_sp from Odds API pipeline.
# Required columns: home_team, away_team, cons_spread (or spread), consensus_prob (optional)

if "cons_sp" not in globals():
    raise ValueError("Missing cons_sp. You printed it earlier. Make sure the consensus spread cell ran.")

# Ensure cons_spread column exists
if "cons_spread" not in cons_sp.columns and "spread" in cons_sp.columns:
    cons_sp = cons_sp.rename(columns={"spread":"cons_spread"})

need = ["home_team","away_team","cons_spread"]
for c in need:
    if c not in cons_sp.columns:
        raise ValueError(f"cons_sp missing {c}. Columns: {list(cons_sp.columns)}")

game_margin = (cons_sp
    .drop_duplicates(subset=["home_team","away_team"])  # game-level
    [["home_team","away_team","cons_spread"]]
    .copy()
)

# Projected margin (HOME - AWAY) points
# If cons_spread is negative for home favorite, then home margin is -cons_spread.
game_margin["proj_margin_home"] = -game_margin["cons_spread"]

print("✅ game_margin:", game_margin.shape)
display(game_margin.head(10))

In [ ]:
import math

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

SIGMA = 11.5  # tune later (10.5–12.5 typical starting range for NCAAB)

# Create per-team ML probs from projected margin
ml_model = game_margin.copy()
ml_model["home_win_prob"] = ml_model["proj_margin_home"].apply(lambda m: norm_cdf(m / SIGMA))
ml_model["away_win_prob"] = 1.0 - ml_model["home_win_prob"]

home_rows = ml_model[["home_team","away_team","home_win_prob"]].rename(columns={"home_win_prob":"model_prob_ml"})
home_rows["team"] = home_rows["home_team"]

away_rows = ml_model[["home_team","away_team","away_win_prob"]].rename(columns={"away_win_prob":"model_prob_ml"})
away_rows["team"] = away_rows["away_team"]

ml_model_long = pd.concat([home_rows, away_rows], ignore_index=True)

print("✅ ml_model_long:", ml_model_long.shape)
print("ML prob stats:", ml_model_long["model_prob_ml"].describe())
display(ml_model_long.head(10))

In [ ]:
def kelly_half(p, dec_odds):
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    f = (b*p - q) / b
    f = max(0.0, f)
    return 0.5 * f

# Use Odds API moneyline rows
ml_lines = odds_long[odds_long["market"]=="h2h"].copy()

# Merge our derived model probs
ml = ml_lines.merge(
    ml_model_long,
    on=["home_team","away_team","team"],
    how="left"
)

# Drop if still missing (should be rare)
ml = ml.dropna(subset=["model_prob_ml","price_decimal","implied_prob"])

ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units"] = ml.apply(lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"]), axis=1)

# Best book per team per game by EV (then units)
ml_best = (ml.sort_values(["ev","units"], ascending=False)
             .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

top10_ml = ml_best.sort_values(["ev","units"], ascending=False).head(10).copy()

print("✅ top10_ml:", top10_ml.shape)
display(top10_ml[["home_team","away_team","team","book","american_odds","price_decimal",
                  "model_prob_ml","implied_prob","true_edge","ev","units"]])

In [ ]:
# Rebuild ML model cleanly and explicitly

import math

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

SIGMA = 11.5  # NCAAB baseline

# Make sure spread is numeric
game_margin["cons_spread"] = pd.to_numeric(game_margin["cons_spread"], errors="coerce")

# IMPORTANT:
# If spread is -14, that means HOME favored by 14.
# Therefore projected margin (HOME - AWAY) = -spread

game_margin["proj_margin_home"] = -game_margin["cons_spread"]

# Compute probabilities correctly
game_margin["home_win_prob"] = game_margin["proj_margin_home"].apply(lambda m: norm_cdf(m / SIGMA))
game_margin["away_win_prob"] = 1.0 - game_margin["home_win_prob"]

# Melt to long format
home_rows = game_margin[["home_team","away_team","home_win_prob"]].copy()
home_rows["team"] = home_rows["home_team"]
home_rows["model_prob_ml"] = home_rows["home_win_prob"]

away_rows = game_margin[["home_team","away_team","away_win_prob"]].copy()
away_rows["team"] = away_rows["away_team"]
away_rows["model_prob_ml"] = away_rows["away_win_prob"]

ml_model_long = pd.concat([home_rows, away_rows], ignore_index=True)

print("ML model std:", ml_model_long["model_prob_ml"].std())
print("Min prob:", ml_model_long["model_prob_ml"].min())
print("Max prob:", ml_model_long["model_prob_ml"].max())
display(ml_model_long.head(10))

In [ ]:
# Merge with one ML line per game to sanity check alignment
check = odds_long[odds_long["market"]=="h2h"].merge(
    ml_model_long,
    on=["home_team","away_team","team"],
    how="left"
)

# Sort by biggest dogs
check_sorted = check.sort_values("american_odds", ascending=False)

display(check_sorted[[
    "home_team","away_team","team","american_odds","model_prob_ml"
]].head(15))

In [ ]:
# Check one problematic game explicitly
test = game_margin[
    (game_margin["home_team"]=="Wyoming Cowboys") &
    (game_margin["away_team"]=="Air Force Falcons")
]

display(test)

In [ ]:
# Check odds rows for same game
check_game = odds_long[
    (odds_long["home_team"]=="Wyoming Cowboys") &
    (odds_long["away_team"]=="Air Force Falcons") &
    (odds_long["market"]=="h2h")
]

display(check_game[["home_team","away_team","team","american_odds"]])

In [ ]:
import numpy as np
import pandas as pd
from math import erf, sqrt

# ----------------------------
# helper: normal CDF (no scipy needed)
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

# ----------------------------
# REQUIRE: cons_sp exists with columns:
# ['home_team','away_team','team','cons_spread', ...]
# cons_sp has 2 rows per game (home team + away team), with opposite spread signs.

need = {"home_team","away_team","team","cons_spread"}
missing = need - set(cons_sp.columns)
if missing:
    raise ValueError(f"cons_sp missing required cols: {missing}")

# 1) get one home-spread per game (home team row)
home_sp = cons_sp[cons_sp["team"] == cons_sp["home_team"]].copy()
home_sp = home_sp[["home_team","away_team","cons_spread"]].drop_duplicates()

# 2) FIX: spread -> projected margin (home)
# if home is -6.5, projected margin should be +6.5
home_sp["projected_margin_home"] = -home_sp["cons_spread"]

# 3) choose sigma (tune later; 11 is a reasonable NCAAB starting point)
SIGMA = 11.0

home_sp["home_win_prob"] = home_sp["projected_margin_home"].apply(lambda m: norm_cdf(m / SIGMA))
home_sp["away_win_prob"] = 1.0 - home_sp["home_win_prob"]

# 4) build team-level ML probabilities (2 rows per game)
home_rows = home_sp[["home_team","away_team","home_win_prob"]].copy()
home_rows["team"] = home_rows["home_team"]
home_rows = home_rows.rename(columns={"home_win_prob":"model_prob_ml"})[["home_team","away_team","team","model_prob_ml"]]

away_rows = home_sp[["home_team","away_team","away_win_prob"]].copy()
away_rows["team"] = away_rows["away_team"]
away_rows = away_rows.rename(columns={"away_win_prob":"model_prob_ml"})[["home_team","away_team","team","model_prob_ml"]]

ml_model = pd.concat([home_rows, away_rows], ignore_index=True)

print("✅ ml_model built:", ml_model.shape)
print("Model ML std:", float(ml_model["model_prob_ml"].std()))
display(ml_model.head(10))

# 5) sanity check that Wyoming is huge favorite if spread is huge
chk = ml_model[(ml_model["home_team"]=="Wyoming Cowboys") & (ml_model["away_team"]=="Air Force Falcons")]
print("\n🔎 Wyoming vs Air Force probs:")
display(chk)

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# REQUIRE: odds_long (normalized) already exists with:
# ['home_team','away_team','commence_time','book','market','team',
#  'american_odds','price_decimal','implied_prob']
#
# REQUIRE: ml_model exists with:
# ['home_team','away_team','team','model_prob_ml']

# 1) isolate ML lines
ml = odds_long[odds_long["market"] == "h2h"].copy()

need_ml_cols = {"home_team","away_team","team","book","american_odds","price_decimal","implied_prob"}
missing = need_ml_cols - set(ml.columns)
if missing:
    raise ValueError(f"odds_long ML missing cols: {missing}")

# 2) attach model probs
ml = ml.merge(ml_model, on=["home_team","away_team","team"], how="left")

if ml["model_prob_ml"].isna().any():
    n = int(ml["model_prob_ml"].isna().sum())
    raise ValueError(f"Missing model_prob_ml on {n} ML rows — key mismatch (team names).")

# 3) EV + half Kelly
def kelly_half(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b if b > 0 else 0.0
    f = max(0.0, f)
    return 0.5 * f

ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units"] = ml.apply(lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"]), axis=1)

# 4) filters: +EV only, remove ridiculous juice if you want
# (optional) set MAX_JUICE = -200 means exclude favorites worse than -200
MAX_JUICE = -200
ml_f = ml[(ml["ev"] > 0)].copy()
ml_f = ml_f[~((ml_f["american_odds"] < 0) & (ml_f["american_odds"] < MAX_JUICE))]

# 5) no dupes: keep best book per game/team by EV (tie-breaker: units then price)
ml_f = (ml_f.sort_values(["ev","units","price_decimal"], ascending=False)
            .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

# 6) top 10
top10_ml = ml_f.sort_values(["ev","units"], ascending=False).head(10).reset_index(drop=True)

print("✅ top10_ml:", top10_ml.shape)
display(top10_ml[["home_team","away_team","team","book","american_odds","price_decimal",
                  "model_prob_ml","implied_prob","true_edge","ev","units"]])

In [ ]:
# ============================
# DISCORD OUTPUT (TOP10 ML)
# ============================

def fmt_prob(x):
    return f"{x*100:.1f}%"

def fmt_edge(x):
    return f"{x*100:.1f}%"

def fmt_units(u):
    return f"{u:.2f}u"

def fmt_amer(a):
    a = int(a)
    return f"+{a}" if a > 0 else str(a)

def make_discord_top10_ml(df):
    lines = []
    lines.append("NCAAB — TOP 10 ML (NO DUPES | +EV | ½-KELLY)\n")

    for i, r in df.reset_index(drop=True).iterrows():
        matchup = f"{r['home_team']} vs {r['away_team']}"
        bet = f"{r['team']} ML"
        book = str(r["book"])
        amer = fmt_amer(r["american_odds"])
        dec = float(r["price_decimal"])
        mp = float(r["model_prob_ml"])
        imp = float(r["implied_prob"])
        te = float(r["true_edge"])
        ev = float(r["ev"])
        u = float(r["units"])

        lines.append(
            f"{i+1}) {matchup}\n"
            f"Bet: {bet}\n"
            f"Book: {book} | Odds: {amer} (Dec {dec:.2f})\n"
            f"Model: {fmt_prob(mp)} | Implied: {fmt_prob(imp)}\n"
            f"True Edge: {fmt_edge(te)} | EV: {ev:.3f}\n"
            f"Units (½ Kelly): {fmt_units(u)}\n"
        )

    return "\n".join(lines)

print(make_discord_top10_ml(top10_ml))

In [ ]:
import numpy as np
import pandas as pd

# ------------------------
# REQUIRE:
# odds_long
# ml_model (with model_prob_ml)
# model_sp (with model_prob_spread)
# ------------------------

def kelly_half(p, dec, cap=None):
    b = dec - 1
    q = 1 - p
    f = (b*p - q) / b if b > 0 else 0
    f = max(0, f) * 0.5
    if cap:
        f = min(f, cap)
    return f

# ======================
# ---- ML SECTION -----
# ======================

ml = odds_long[odds_long["market"] == "h2h"].copy()
ml = ml.merge(ml_model, on=["home_team","away_team","team"], how="left")

ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1
ml["units"] = ml.apply(lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"], cap=1.0), axis=1)

# ---- Tier A ML ----
ml_core = ml[
    (ml["model_prob_ml"] >= 0.40) &
    (ml["true_edge"] >= 0.05) &
    (ml["american_odds"] >= -200) &
    (ml["american_odds"] <= 400)
].copy()

ml_core = ml_core.sort_values("ev", ascending=False)\
                 .drop_duplicates(["home_team","away_team","team"])\
                 .head(10)

# ---- Tier B ML ----
ml_dogs = ml[
    (ml["american_odds"] >= 300) &
    (ml["american_odds"] <= 800) &
    (ml["true_edge"] >= 0.06)
].copy()

ml_dogs["units"] = ml_dogs.apply(
    lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"], cap=0.4),
    axis=1
)

ml_dogs = ml_dogs.sort_values("ev", ascending=False)\
                 .drop_duplicates(["home_team","away_team","team"])\
                 .head(4)

# ---- Tier C Bomb ----
ml_bomb = ml[
    (ml["american_odds"] >= 800) &
    (ml["true_edge"] >= 0.08)
].copy()

ml_bomb["units"] = ml_bomb.apply(
    lambda r: kelly_half(r["model_prob_ml"], r["price_decimal"], cap=0.10),
    axis=1
)

ml_bomb = ml_bomb.sort_values("ev", ascending=False)\
                 .drop_duplicates(["home_team","away_team","team"])\
                 .head(1)

# ======================
# ---- SPREAD SECTION ---
# ======================

sp = odds_long[odds_long["market"] == "spreads"].copy()
sp = sp.merge(model_sp, on=["home_team","away_team","team","spread"], how="left")

sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1
sp["units"] = sp.apply(lambda r: kelly_half(r["model_prob_spread"], r["price_decimal"], cap=1.0), axis=1)

sp_core = sp[
    (sp["model_prob_spread"] >= 0.54) &
    (sp["true_edge"] >= 0.05)
].copy()

sp_core = sp_core.sort_values("ev", ascending=False)\
                 .drop_duplicates(["home_team","away_team","team"])\
                 .head(10)

# ======================
# FINAL CARD
# ======================

tierA = pd.concat([ml_core, sp_core]).sort_values("ev", ascending=False)
tierB = ml_dogs
tierC = ml_bomb

print("Tier A (Core):", tierA.shape)
print("Tier B (Dogs):", tierB.shape)
print("Tier C (Bomb):", tierC.shape)

display(tierA[["home_team","away_team","team","book","american_odds","model_prob_ml","true_edge","units"]])
display(tierB[["home_team","away_team","team","book","american_odds","model_prob_ml","true_edge","units"]])
display(tierC[["home_team","away_team","team","book","american_odds","model_prob_ml","true_edge","units"]])

In [ ]:
import pandas as pd

# Convert commence_time to datetime
odds_long["commence_time"] = pd.to_datetime(odds_long["commence_time"], utc=True)

# Define target slate date (adjust if needed)
TARGET_DATE = pd.Timestamp("2026-02-28", tz="UTC")

# Keep only games that start on that date (UTC-safe)
odds_long = odds_long[
    odds_long["commence_time"].dt.date == TARGET_DATE.date()
].copy()

print("Filtered odds_long:", odds_long.shape)
print("Unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

In [ ]:
import pandas as pd

odds_long["commence_time"] = pd.to_datetime(odds_long["commence_time"], utc=True)

# Define U.S. Eastern time window for Saturday slate
start = pd.Timestamp("2026-02-28 05:00:00", tz="UTC")  # midnight ET approx
end   = pd.Timestamp("2026-03-01 05:00:00", tz="UTC")  # midnight ET next day

odds_long = odds_long[
    (odds_long["commence_time"] >= start) &
    (odds_long["commence_time"] < end)
].copy()

print("Filtered odds_long:", odds_long.shape)
print("Unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

In [ ]:
import pandas as pd

# 1) Confirm date window + list a few earliest/latest tip times
print("odds_long rows:", len(odds_long))
print("unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

print("\nEarliest tips (UTC):")
print(odds_long[["commence_time","home_team","away_team"]].drop_duplicates().sort_values("commence_time").head(10))

print("\nLatest tips (UTC):")
print(odds_long[["commence_time","home_team","away_team"]].drop_duplicates().sort_values("commence_time").tail(10))

# 2) Quick “did any 2/27 game bleed in?” check (by ET date view)
et = odds_long.copy()
et["commence_time_et"] = et["commence_time"].dt.tz_convert("America/New_York")
et_games = et[["home_team","away_team","commence_time_et"]].drop_duplicates()
et_games["et_date"] = et_games["commence_time_et"].dt.date

print("\nET date counts (games):")
print(et_games.groupby("et_date").size().sort_index())

# 3) Spot check for Western Michigan if you want
wm = odds_long[
    odds_long["home_team"].str.contains("Western Michigan", case=False, na=False) |
    odds_long["away_team"].str.contains("Western Michigan", case=False, na=False)
][["home_team","away_team","commence_time","market","book"]].drop_duplicates()

print("\nWestern Michigan games found:", len(wm))
if len(wm):
    print(wm.sort_values("commence_time").head(20))

In [ ]:
import numpy as np
import pandas as pd

# -----------------------
# REQUIRED INPUT CHECKS
# -----------------------
def _need(df_name, cols):
    if df_name not in globals():
        raise ValueError(f"Missing {df_name}. Run the cell that builds it first.")
    df = globals()[df_name]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} missing columns: {missing}\nHave: {list(df.columns)}")
    return df

odds_long = _need("odds_long", ["home_team","away_team","commence_time","book","market","team","american_odds","price_decimal"])
ml_model  = _need("ml_model",  ["home_team","away_team","team","model_prob_ml"])
model_sp  = _need("model_sp",  ["home_team","away_team","team","spread","model_prob_spread"])

# -----------------------
# HELPERS
# -----------------------
def dec_to_implied(d):
    return 1.0 / d

def half_kelly(p, dec, cap_u=0.75):
    # Kelly fraction for decimal odds: f* = (bp - q)/b , b=dec-1
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b if b > 0 else 0.0
    f = max(0.0, f)
    # half-kelly and cap
    u = 0.5 * f
    return min(u, cap_u)

def best_line_by_game(df, group_cols, score_col="ev"):
    # pick one row per group (highest score)
    return (df.sort_values(score_col, ascending=False)
              .drop_duplicates(subset=group_cols, keep="first"))

# -----------------------
# 1) SPLIT MARKETS
# -----------------------
ml_lines = odds_long[odds_long["market"].eq("h2h")].copy()
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()

# normalize spreads: ensure spread exists only for spreads
if "spread" not in ml_lines.columns:
    ml_lines["spread"] = np.nan
if "spread" not in sp_lines.columns:
    raise ValueError("odds_long spreads market missing 'spread' column")

# -----------------------
# 2) ATTACH MODEL PROBS
# -----------------------
ml = ml_lines.merge(ml_model, on=["home_team","away_team","team"], how="left")
ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)

# EV = p*dec - 1  (per 1 unit staked)
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["units"] = ml.apply(lambda r: half_kelly(r["model_prob_ml"], r["price_decimal"], cap_u=0.75), axis=1)

# spreads: attach model_spread by matching (team, spread) at game level
sp = sp_lines.merge(
    model_sp,
    on=["home_team","away_team","team","spread"],
    how="left"
)
sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["units"] = sp.apply(lambda r: half_kelly(r["model_prob_spread"], r["price_decimal"], cap_u=0.75), axis=1)

# Fail loudly if model not attached
if ml["model_prob_ml"].isna().mean() > 0.05:
    bad = ml[ml["model_prob_ml"].isna()][["home_team","away_team","team"]].drop_duplicates().head(10)
    raise ValueError(f"Too many ML rows missing model_prob_ml. Sample missing:\n{bad}")

if sp["model_prob_spread"].isna().mean() > 0.05:
    bad = sp[sp["model_prob_spread"].isna()][["home_team","away_team","team","spread"]].drop_duplicates().head(10)
    raise ValueError(f"Too many SP rows missing model_prob_spread. Sample missing:\n{bad}")

# -----------------------
# 3) FILTERS (Tier logic)
# -----------------------
# You said: don’t want full card long shots.
# Tier A = core win-ish bets: cap dogs, require decent win prob.
TIERA_MAX_DOG = 300     # +300 or shorter
TIERA_MIN_P   = 0.52    # must be >= 52% win/cover
TIERA_MIN_EV  = 0.02    # small positive EV floor

# Tier B = dogs you’re okay mixing in
TIERB_DOG_MIN = 301
TIERB_DOG_MAX = 900
TIERB_MIN_P   = 0.18
TIERB_MIN_EV  = 0.04

# Tier C = bombs (keep very small list)
TIERC_DOG_MIN = 901
TIERC_MIN_P   = 0.08
TIERC_MIN_EV  = 0.10

def _tier_ml(df):
    df = df.copy()
    # core
    A = df[(df["american_odds"] <= TIERA_MAX_DOG) & (df["model_prob_ml"] >= TIERA_MIN_P) & (df["ev"] >= TIERA_MIN_EV)]
    B = df[(df["american_odds"] >= TIERB_DOG_MIN) & (df["american_odds"] <= TIERB_DOG_MAX) & (df["model_prob_ml"] >= TIERB_MIN_P) & (df["ev"] >= TIERB_MIN_EV)]
    C = df[(df["american_odds"] >= TIERC_DOG_MIN) & (df["model_prob_ml"] >= TIERC_MIN_P) & (df["ev"] >= TIERC_MIN_EV)]
    return A, B, C

def _tier_sp(df):
    df = df.copy()
    # spreads don’t have the same "dog" concept; use probability + EV filters
    A = df[(df["model_prob_spread"] >= 0.54) & (df["ev"] >= 0.02)]
    B = df[(df["model_prob_spread"] >= 0.52) & (df["ev"] >= 0.01)]
    C = df[(df["model_prob_spread"] >= 0.50) & (df["ev"] >= 0.00)]
    return A, B, C

mlA, mlB, mlC = _tier_ml(ml)
spA, spB, spC = _tier_sp(sp)

# -----------------------
# 4) NO DUPES RULES
# -----------------------
# a) pick BEST book line per (game, team, market, spread) by EV
ml_best = best_line_by_game(mlA, ["home_team","away_team","team"], score_col="ev")
sp_best = best_line_by_game(spA, ["home_team","away_team","team","spread"], score_col="ev")

# b) prevent BOTH sides in same game (keep top EV per game per market)
ml_best = best_line_by_game(ml_best, ["home_team","away_team"], score_col="ev")
sp_best = best_line_by_game(sp_best, ["home_team","away_team"], score_col="ev")

# -----------------------
# 5) PICK TOP LISTS
# -----------------------
TOP_CORE_TARGET_MIN = 5
TOP_CORE_TARGET_MAX = 10

core_all = pd.concat([
    ml_best.assign(card_market="ML", card_prob=ml_best["model_prob_ml"]),
    sp_best.assign(card_market="SPREAD", card_prob=sp_best["model_prob_spread"])
], ignore_index=True)

core_all = core_all.sort_values("ev", ascending=False).head(TOP_CORE_TARGET_MAX)

# If we got fewer than 5, backfill from Tier B (still no dupes)
if len(core_all) < TOP_CORE_TARGET_MIN:
    # Tier B backfill pool (ML + SP)
    mlB_best = best_line_by_game(mlB, ["home_team","away_team","team"], score_col="ev")
    spB_best = best_line_by_game(spB, ["home_team","away_team","team","spread"], score_col="ev")
    mlB_best = best_line_by_game(mlB_best, ["home_team","away_team"], score_col="ev")
    spB_best = best_line_by_game(spB_best, ["home_team","away_team"], score_col="ev")

    backfill = pd.concat([
        mlB_best.assign(card_market="ML", card_prob=mlB_best["model_prob_ml"]),
        spB_best.assign(card_market="SPREAD", card_prob=spB_best["model_prob_spread"])
    ], ignore_index=True).sort_values("ev", ascending=False)

    used_games = set(map(tuple, core_all[["home_team","away_team"]].values.tolist()))
    bf = backfill[~backfill.apply(lambda r: (r["home_team"], r["away_team"]) in used_games, axis=1)]
    need = TOP_CORE_TARGET_MIN - len(core_all)
    core_all = pd.concat([core_all, bf.head(need)], ignore_index=True)

# -----------------------
# 6) PRINT DISCORD-STYLE CORE CARD
# -----------------------
def fmt_amer(a):
    a = int(round(a))
    return f"+{a}" if a > 0 else str(a)

def fmt_units(u):
    return f"{u:.2f}u"

print(f"\n✅ Tier A Core Card (ML+Spread) rows: {len(core_all)}\n")
for i, r in enumerate(core_all.itertuples(index=False), 1):
    matchup = f"{r.home_team} vs {r.away_team}"
    if r.card_market == "ML":
        bet = f"{r.team} ML"
        prob = r.model_prob_ml
    else:
        bet = f"{r.team} {r.spread:+g}"
        prob = r.model_prob_spread

    print(f"{i}) {matchup}")
    print(f"Bet: {bet}")
    print(f"Book: {r.book} | Odds: {fmt_amer(r.american_odds)} (Dec {r.price_decimal:.2f})")
    print(f"Model Prob: {prob:.3f} | Implied: {r.implied_prob:.3f}")
    print(f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f}")
    print(f"Units (½ Kelly, capped): {fmt_units(r.units)}\n")

# Optional: save outputs
tierA_core = core_all.copy()

In [ ]:
import numpy as np
import pandas as pd

# --- BUILD a lookup table of (game, team) -> model spread + model cover prob ---
sp_lookup = model_sp[["home_team","away_team","team","spread","model_prob_spread"]].copy()
sp_lookup = sp_lookup.rename(columns={"spread":"cons_spread"})

# --- Take spreads market lines ---
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()

# Attach consensus spread + model_prob_spread by (game, team) only
sp = sp_lines.merge(
    sp_lookup[["home_team","away_team","team","cons_spread","model_prob_spread"]],
    on=["home_team","away_team","team"],
    how="left"
)

# Keep only the lines that match the consensus spread (within tolerance)
# (Odds feeds sometimes use halves; this tolerance avoids float issues)
TOL = 0.001
sp = sp[np.abs(sp["spread"] - sp["cons_spread"]) <= TOL].copy()

print("✅ Spread lines at consensus spread:", sp.shape)
print("Null model_prob_spread:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))

# If still too many nulls, show sample
if sp["model_prob_spread"].isna().mean() > 0.05:
    bad = sp[sp["model_prob_spread"].isna()][["home_team","away_team","team","spread","cons_spread"]].drop_duplicates().head(15)
    raise ValueError(f"Still missing model_prob_spread after consensus filter. Sample:\n{bad}")

# Now compute implied/edge/EV/units like before
def dec_to_implied(d):
    return 1.0 / d

def half_kelly(p, dec, cap_u=0.75):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b if b > 0 else 0.0
    f = max(0.0, f)
    u = 0.5 * f
    return min(u, cap_u)

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["units"] = sp.apply(lambda r: half_kelly(r["model_prob_spread"], r["price_decimal"], cap_u=0.75), axis=1)

print("\n✅ Spread EV layer ready:", sp.shape)
print(sp[["home_team","away_team","team","book","spread","price_decimal","model_prob_spread","implied_prob","true_edge","ev","units"]].head(10))

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# REQUIRED: odds_long + model_sp must exist
# odds_long already filtered to 2026-02-28
# model_sp must have: home_team, away_team, team, spread, model_prob_spread
# ----------------------------

# 1) Build lookup: (game, team) -> consensus spread + model cover prob
sp_lookup = model_sp[["home_team","away_team","team","spread","model_prob_spread"]].copy()
sp_lookup = sp_lookup.rename(columns={"spread":"cons_spread"})

# 2) Pull spread lines for the slate
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()

# 3) Attach consensus spread + model_prob_spread using ONLY (home,away,team)
sp = sp_lines.merge(
    sp_lookup,
    on=["home_team","away_team","team"],
    how="left"
)

print("✅ Spread rows after attaching cons_spread + model_prob_spread:", sp.shape)
print("Null model_prob_spread:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))

# If any missing, show a sample (but DO NOT STOP)
if sp["model_prob_spread"].isna().any():
    bad = sp[sp["model_prob_spread"].isna()][["home_team","away_team","team","spread"]].drop_duplicates().head(15)
    print("⚠️ Missing model_prob_spread for these teams/games (sample):")
    print(bad)

# 4) Keep ONLY the book lines that match the model's consensus spread for that game/team
# (this is the key fix — prevents Houston 18.5/19.5 etc from being unmatched)
TOL = 1e-6
sp = sp[(sp["cons_spread"].notna()) & (np.abs(sp["spread"] - sp["cons_spread"]) <= TOL)].copy()

print("\n✅ Spread rows at consensus spread ONLY:", sp.shape)
print("Null model_prob_spread after filter:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))

# 5) Add EV + 1/2 Kelly (units capped)
def dec_to_implied(d):
    return 1.0 / d

def half_kelly(p, dec, cap_u=0.75):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q) / b
    f = max(0.0, f)
    u = 0.5 * f
    return float(min(u, cap_u))

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units"] = sp.apply(lambda r: half_kelly(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ SP EV layer built:", sp.shape)
display(sp[["home_team","away_team","team","book","spread","american_odds","price_decimal",
            "model_prob_spread","implied_prob","true_edge","ev","units"]].head(12))

In [ ]:
import numpy as np
import pandas as pd

# =========================
# ONE-SHOT SPREAD FIX + TOP 10 (2/28)
# Requires: odds_long (already filtered to 2/28) AND model_sp
# model_sp must include: home_team, away_team, team, spread(or cons_spread), model_prob_spread(or model_prob)
# =========================

# ---------- safety: locate odds_long ----------
if "odds_long" not in globals():
    raise ValueError("odds_long not found. Run your OddsAPI pull + date filter cell first.")

# ---------- standardize model_sp ----------
if "model_sp" not in globals():
    raise ValueError("model_sp not found. Run the ESPN spread model build cell first (or the hybrid margin engine cell that builds model_sp).")

ms = model_sp.copy()

# If model_sp has cons_spread instead of spread, rename it
if "spread" not in ms.columns and "cons_spread" in ms.columns:
    ms = ms.rename(columns={"cons_spread":"spread"})

# If model_sp has model_prob instead of model_prob_spread, rename it
if "model_prob_spread" not in ms.columns and "model_prob" in ms.columns:
    ms = ms.rename(columns={"model_prob":"model_prob_spread"})

need_ms = {"home_team","away_team","team","spread","model_prob_spread"}
missing_ms = need_ms - set(ms.columns)
if missing_ms:
    raise ValueError(f"model_sp missing required columns: {missing_ms}")

# ---------- build spread lines from odds_long ----------
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()
need_sp = {"home_team","away_team","team","book","spread","american_odds","price_decimal"}
missing_sp = need_sp - set(sp_lines.columns)
if missing_sp:
    raise ValueError(f"odds_long(spreads) missing required columns: {missing_sp}")

# ---------- attach model prob + consensus spread using ONLY game/team keys ----------
# (this avoids the alternates mismatch that caused your missing model_prob_spread error)
ms_lookup = ms[["home_team","away_team","team","spread","model_prob_spread"]].copy()
ms_lookup = ms_lookup.rename(columns={"spread":"cons_spread"})

sp = sp_lines.merge(
    ms_lookup,
    on=["home_team","away_team","team"],
    how="left"
)

print("✅ Spread rows after attaching cons_spread + model_prob_spread:", sp.shape)
print("Null model_prob_spread:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))

# ---------- keep ONLY the consensus spread line (drop alternates like -18.5 / -19.5, etc.) ----------
TOL = 1e-6
sp = sp[(sp["cons_spread"].notna()) & (np.abs(sp["spread"] - sp["cons_spread"]) <= TOL)].copy()

print("✅ Spread rows at CONSENSUS spread only:", sp.shape)
print("Null model_prob_spread after filter:", int(sp["model_prob_spread"].isna().sum()), "of", len(sp))

# ---------- EV + 1/2 Kelly ----------
def dec_to_implied(d):
    return 1.0 / float(d)

def half_kelly_units(p, dec, cap_u=0.75):
    p = float(p); dec = float(dec)
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q) / b
    f = max(0.0, f)
    u = 0.5 * f
    return float(min(u, cap_u))

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units"] = sp.apply(lambda r: half_kelly_units(r["model_prob_spread"], r["price_decimal"]), axis=1)

# ---------- no dupes: best book per game+team at the consensus spread ----------
# Score favors EV, then edge, then units
sp["rank_score"] = (sp["ev"] * 1.0) + (sp["true_edge"] * 0.25) + (sp["units"] * 0.10)

sp_best = (
    sp.sort_values("rank_score", ascending=False)
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

# ---------- Top 10 (positive EV only) ----------
top10_sp = sp_best[sp_best["ev"] > 0].sort_values("rank_score", ascending=False).head(10).copy()

print("\n✅ top10_sp:", top10_sp.shape)
display(top10_sp[["home_team","away_team","team","book","spread","american_odds","price_decimal",
                  "model_prob_spread","implied_prob","true_edge","ev","units"]])

# ---------- Discord-style printout ----------
def fmt_amer(x):
    x = int(x)
    return f"+{x}" if x > 0 else str(x)

print("\nNCAAB — TOP 10 SPREAD (NO DUPES | +EV | ½-KELLY)\n")
for i, r in enumerate(top10_sp.itertuples(index=False), 1):
    print(f"{i}) {r.home_team} vs {r.away_team}")
    sign = "" if r.spread < 0 else "+"
    print(f"Bet: {r.team} {sign}{r.spread:g}")
    print(f"Book: {r.book} | Odds: {fmt_amer(r.american_odds)} (Dec {r.price_decimal:.2f})")
    print(f"Model(Cover): {r.model_prob_spread:.3f} | Implied: {r.implied_prob:.3f}")
    print(f"True Edge: {r.true_edge:.3f} | EV: {r.ev:.3f}")
    print(f"Units (½ Kelly): {r.units:.2f}u\n")

In [ ]:
import pandas as pd
import numpy as np

# --- SAFETY: confirm odds_long is 2/28 only ---
print("Unique games in odds_long:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

# --- isolate ML lines ---
ml_lines = odds_long[odds_long["market"] == "h2h"].copy()

# attach model probs
ml = ml_lines.merge(
    ml_model[["home_team","away_team","team","model_prob_ml"]],
    on=["home_team","away_team","team"],
    how="left"
)

# drop any rows that somehow didn't match
ml = ml[ml["model_prob_ml"].notna()].copy()

# EV + edge
ml["implied_prob"] = 1 / ml["price_decimal"]
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1

# half kelly (capped 0.75u)
def half_kelly(p, dec, cap=0.75):
    b = dec - 1
    q = 1 - p
    f = (b*p - q) / b if b > 0 else 0
    f = max(0, f) * 0.5
    return min(f, cap)

ml["units"] = ml.apply(lambda r: half_kelly(r["model_prob_ml"], r["price_decimal"]), axis=1)

# keep best book per team/game
ml_best = (
    ml.sort_values("ev", ascending=False)
      .drop_duplicates(["home_team","away_team","team"])
)

# remove both sides of same game (keep highest EV)
ml_best = (
    ml_best.sort_values("ev", ascending=False)
           .drop_duplicates(["home_team","away_team"])
)

# final top 10
top10_ml = ml_best[ml_best["ev"] > 0].head(10).copy()

print("\n✅ Fresh 2/28 ML Top 10:", top10_ml.shape)
display(top10_ml[["home_team","away_team","team","book","american_odds","model_prob_ml","true_edge","ev","units"]])

In [ ]:
import numpy as np
import pandas as pd

# =========================
# TIER A CARD BUILDER (8–10)
# =========================

# ---------- helpers ----------
def dec_to_implied(dec):
    return 1.0 / float(dec)

def half_kelly(p, dec, cap=0.25):
    """
    Half-Kelly fraction with cap.
    p: win probability
    dec: decimal odds
    """
    b = float(dec) - 1.0
    if b <= 0:
        return 0.0
    f = (p * b - (1 - p)) / b
    f = max(0.0, f)
    f *= 0.5
    return float(min(f, cap))

def ensure_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

def best_line_by_market(df, market):
    """
    Keep BEST line per (home, away, team) for a given market.
    For ML: maximize price_decimal.
    For spreads: maximize price_decimal (same concept: best payout).
    """
    d = df[df["market"] == market].copy()
    if d.empty:
        return d
    d = d.sort_values(["home_team","away_team","team","price_decimal"], ascending=[True,True,True,False])
    d = d.drop_duplicates(subset=["home_team","away_team","team"], keep="first")
    return d

# ---------- input checks ----------
ensure_cols(odds_long, ["home_team","away_team","commence_time","book","market","team","american_odds","spread","price_decimal"], "odds_long")
ensure_cols(ml_model, ["home_team","away_team","team","model_prob_ml"], "ml_model")

HAS_SPREAD_MODEL = ("model_sp" in globals()) and isinstance(model_sp, pd.DataFrame) and (len(model_sp) > 0)
if HAS_SPREAD_MODEL:
    ensure_cols(model_sp, ["home_team","away_team","team","spread","model_prob_spread"], "model_sp")

# ---------- ML market ----------
ml_best = best_line_by_market(odds_long, "h2h")
ml = ml_best.merge(ml_model, on=["home_team","away_team","team"], how="left")

ml = ml.dropna(subset=["model_prob_ml","price_decimal"]).copy()
ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units"] = ml.apply(lambda r: half_kelly(r["model_prob_ml"], r["price_decimal"]), axis=1)

# Tier A ML filters (avoid longshots as "full card long shots")
ML_MAX_DOG = 450         # <= +450 only
ML_MAX_JUICE = -200      # >= -200 only
ML_MIN_TRUE_EDGE = 0.012 # ~1.2%+
ML_MIN_EV = 0.020        # >= 0.02
ML_MIN_UNITS = 0.008     # avoid dust

ml = ml[
    (ml["american_odds"].between(ML_MAX_JUICE, ML_MAX_DOG)) &
    (ml["true_edge"] >= ML_MIN_TRUE_EDGE) &
    (ml["ev"] >= ML_MIN_EV) &
    (ml["units"] >= ML_MIN_UNITS)
].copy()

ml["market"] = "ML"
ml["line"] = np.nan
ml["model_prob"] = ml["model_prob_ml"]

# ---------- Spread market ----------
sp_rows = pd.DataFrame()
if HAS_SPREAD_MODEL:
    sp_best = best_line_by_market(odds_long, "spreads")

    # attach consensus spread model at GAME level spread (from model_sp), then filter to that spread only
    # (this prevents missing model probs due to alternate spread points)
    sp = sp_best.merge(model_sp, on=["home_team","away_team","team"], how="left", suffixes=("","_m"))

    sp = sp.dropna(subset=["model_prob_spread","price_decimal","spread"]).copy()

    # Tier A Spread filters
    SP_MAX_JUICE = -200
    SP_MIN_TRUE_EDGE = 0.010
    SP_MIN_EV = 0.015
    SP_MIN_UNITS = 0.006

    sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
    sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
    sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
    sp["units"] = sp.apply(lambda r: half_kelly(r["model_prob_spread"], r["price_decimal"]), axis=1)

    sp = sp[
        (sp["american_odds"] >= SP_MAX_JUICE) &
        (sp["true_edge"] >= SP_MIN_TRUE_EDGE) &
        (sp["ev"] >= SP_MIN_EV) &
        (sp["units"] >= SP_MIN_UNITS)
    ].copy()

    sp["market"] = "SPREAD"
    sp["line"] = sp["spread"]
    sp["model_prob"] = sp["model_prob_spread"]

    sp_rows = sp

# ---------- combine + dedupe rules ----------
card = pd.concat([ml, sp_rows], ignore_index=True)

if card.empty:
    print("No Tier A plays passed filters.\nTry loosening ML_MAX_DOG / MIN_TRUE_EDGE / MIN_EV, or confirm model_sp exists for spreads.")
else:
    # Rank score: prioritize EV and units, then true_edge
    card["rank_score"] = (
        0.50 * card["ev"].clip(lower=0) +
        0.35 * card["units"].clip(lower=0) +
        0.15 * card["true_edge"].clip(lower=0)
    )

    # no dupes per game+market (keep best side)
    card = (card.sort_values("rank_score", ascending=False)
                 .drop_duplicates(subset=["home_team","away_team","market"], keep="first")
                 .copy())

    # build 8–10 plays, balanced ML + spreads if possible
    target_n = 10

    ml_pick = card[card["market"]=="ML"].head(6)
    sp_pick = card[card["market"]=="SPREAD"].head(6)

    # interleave to avoid all-ML or all-SP
    picks = pd.concat([ml_pick.head(5), sp_pick.head(5)], ignore_index=True)
    if len(picks) < 8:
        picks = pd.concat([picks, card[~card.index.isin(picks.index)].head(target_n - len(picks))], ignore_index=True)

    picks = picks.sort_values("rank_score", ascending=False).head(target_n).reset_index(drop=True)

    # ---------- print card ----------
    print(f"NCAAB — 2/28 TIER A CARD (ML + SPREAD) | {len(picks)} plays | ½-KELLY\n")
    for i, r in picks.iterrows():
        matchup = f"{r['home_team']} vs {r['away_team']}"
        if r["market"] == "ML":
            bet = f"{r['team']} ML"
        else:
            # spread sign: show from TEAM perspective already in odds_long
            bet = f"{r['team']} {r['line']:+g}"
        print(f"{i+1}) {matchup}")
        print(f"Bet: {bet}")
        print(f"Book: {r['book']} | Odds: {int(r['american_odds']) if float(r['american_odds']).is_integer() else r['american_odds']} (Dec {r['price_decimal']:.2f})")
        print(f"Model Prob: {r['model_prob']:.3f} | Implied: {r['implied_prob']:.3f}")
        print(f"True Edge: {r['true_edge']:.3f} | EV: {r['ev']:.3f}")
        print(f"Units (½ Kelly): {r['units']:.2f}u")
        print("")

    # Keep for later
    tierA_card = picks.copy()
    print("✅ Saved: tierA_card")
    if not HAS_SPREAD_MODEL:
        print("⚠️ Note: model_sp not found, so this run is ML-only.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# =========================
# SIMULATION-BASED SPREAD MODEL
# =========================

assert "ml_model" in globals(), "Run ML model first."
assert "odds_long" in globals(), "Need odds_long."

SIGMA = 11.0  # NCAAB scoring std dev (tunable 10.5–11.5)

# --- 1. Build projected margin from win prob ---
ml_tmp = ml_model.copy()

# home team rows only to compute game-level margin
home_rows = ml_tmp.merge(
    odds_long[["home_team","away_team"]].drop_duplicates(),
    on=["home_team","away_team"],
    how="inner"
)

SIGMA = 9.0  # reduced sigma

home_rows = ml_model.copy()

# Clamp probabilities to prevent explosion
home_rows["p_clamped"] = home_rows["model_prob_ml"].clip(0.03, 0.97)

home_rows["proj_margin_home"] = SIGMA * norm.ppf(home_rows["p_clamped"])

proj_margin = home_rows[[
    "home_team","away_team","proj_margin_home"
]].drop_duplicates()

proj_margin = home_rows[["home_team","away_team","proj_margin_home"]].drop_duplicates()

# --- 2. Attach to spread lines ---
sp = odds_long[odds_long["market"] == "spreads"].copy()

sp = sp.merge(proj_margin, on=["home_team","away_team"], how="left")

# remove rows missing projected margin
sp = sp.dropna(subset=["proj_margin_home"])

# --- 3. Compute cover probability ---
def cover_prob(row):
    mu = row["proj_margin_home"]
    spread = row["spread"]

    # If team is home
    if row["team"] == row["home_team"]:
        return 1 - norm.cdf((spread - mu) / SIGMA)
    else:
        # away margin is -mu
        mu_away = -mu
        return 1 - norm.cdf((spread - mu_away) / SIGMA)

sp["model_prob_spread"] = sp.apply(cover_prob, axis=1)

# --- 4. sanity checks ---
print("Spread model std:", sp["model_prob_spread"].std())
print("Spread prob min/max:", sp["model_prob_spread"].min(), sp["model_prob_spread"].max())

model_sp = sp[[
    "home_team","away_team","team","spread","model_prob_spread"
]].copy()

print("✅ Simulation spread model built:", model_sp.shape)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# =========================
# SPREAD SIM ENGINE (ROBUST)
# =========================
# Requires:
# - odds_long (filtered to 2/28 already)
# - cons_sp (spread consensus table) with columns:
#     home_team, away_team, team, consensus_prob, cons_spread
# - cons_ml (ML consensus table) with columns:
#     home_team, away_team, team, consensus_prob
# - ml_model with columns:
#     home_team, away_team, team, model_prob_ml  (ESPN predictor-derived, non-constant)

assert "odds_long" in globals() and len(odds_long) > 0, "odds_long missing/empty"
assert "cons_sp" in globals() and len(cons_sp) > 0, "cons_sp missing/empty"
assert "cons_ml" in globals() and len(cons_ml) > 0, "cons_ml missing/empty"
assert "ml_model" in globals() and len(ml_model) > 0, "ml_model missing/empty"

# ---------- helpers ----------
def dec_to_implied(dec):
    return 1.0 / dec

def kelly_half(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    f = (p * (b + 1) - 1) / b
    return max(0.0, 0.5 * f)

# ---------- build game-level home win prob model ----------
# get home team per game
games = odds_long[["home_team","away_team"]].drop_duplicates().copy()

# attach model home win prob using ml_model
home_model = (
    ml_model.merge(games, on=["home_team","away_team"], how="inner")
            .loc[lambda d: d["team"] == d["home_team"], ["home_team","away_team","model_prob_ml"]]
            .rename(columns={"model_prob_ml":"p_home_model"})
)

# attach market home win prob using cons_ml (consensus probs)
home_mkt = (
    cons_ml.merge(games, on=["home_team","away_team"], how="inner")
          .loc[lambda d: d["team"] == d["home_team"], ["home_team","away_team","consensus_prob"]]
          .rename(columns={"consensus_prob":"p_home_mkt"})
)

# merge those
p_home = games.merge(home_model, on=["home_team","away_team"], how="left") \
              .merge(home_mkt, on=["home_team","away_team"], how="left")

# sanity
if p_home["p_home_model"].isna().mean() > 0.05:
    print("⚠️ Missing p_home_model examples:")
    print(p_home[p_home["p_home_model"].isna()][["home_team","away_team"]].head(10))

# clamp BOTH to avoid extreme behavior
p_home["p_home_model"] = p_home["p_home_model"].clip(0.05, 0.95)
p_home["p_home_mkt"]   = p_home["p_home_mkt"].clip(0.05, 0.95)

# ---------- base spread mean from consensus spread (home perspective) ----------
# cons_sp has cons_spread per team; grab HOME team row only
home_spread = (
    cons_sp.merge(games, on=["home_team","away_team"], how="inner")
           .loc[lambda d: d["team"] == d["home_team"], ["home_team","away_team","cons_spread"]]
           .rename(columns={"cons_spread":"market_spread_home"})
)

# market_spread_home is already "home spread" (typically negative for favored)
engine = p_home.merge(home_spread, on=["home_team","away_team"], how="left")

# ---------- adjustment from ML edge vs market ----------
# If model likes home more than market, shift spread slightly more negative (home stronger)
K_EDGE_TO_POINTS = 10.0   # 0.05 win-prob edge ~ 0.5 points
engine["edge_home"] = (engine["p_home_model"] - engine["p_home_mkt"]).fillna(0.0)

engine["proj_margin_home"] = (-engine["market_spread_home"]) + (K_EDGE_TO_POINTS * engine["edge_home"])
# Explanation: margin is home - away.
# If spread is -6.5, then base margin is +6.5 for home.
# market_spread_home is negative for home favorite; hence -market_spread_home

# ---------- simulate cover prob at consensus line ----------
SIGMA_MARGIN = 11.0  # realistic-ish CBB margin sd; not tiny
# For a given home line L_home (spread), home covers if (margin + L_home) > 0.
# So P(cover_home) = P(margin > -L_home)
engine["home_cover_prob"] = 1.0 - norm.cdf((-engine["market_spread_home"] - engine["proj_margin_home"]) / SIGMA_MARGIN)

# clamp cover probs (final guardrail)
engine["home_cover_prob"] = engine["home_cover_prob"].clip(0.03, 0.97)

# ---------- expand to team-level spread model df ----------
# Build team-level cover prob:
# - If team == home: use home_cover_prob
# - If team == away: cover prob = 1 - home_cover_prob at flipped spread
model_sp = pd.concat([
    engine.assign(team=engine["home_team"], spread=engine["market_spread_home"], model_prob_spread=engine["home_cover_prob"])[
        ["home_team","away_team","team","spread","model_prob_spread"]
    ],
    engine.assign(team=engine["away_team"], spread=-engine["market_spread_home"], model_prob_spread=1.0-engine["home_cover_prob"])[
        ["home_team","away_team","team","spread","model_prob_spread"]
    ],
], ignore_index=True)

print("✅ model_sp (simulation) built:", model_sp.shape)
print("Spread model std:", float(model_sp["model_prob_spread"].std()))
print("Spread prob min/max:", float(model_sp["model_prob_spread"].min()), float(model_sp["model_prob_spread"].max()))

# ---------- attach to actual spread lines (all books) ----------
sp = odds_long[odds_long["market"]=="spreads"].copy()

# merge model_sp onto each spread line by game + team (NOT spread) then evaluate at that line
# We'll compute cover prob for each book line using the same proj_margin_home but evaluated at that line.
# Build a map of proj_margin_home per game:
gm = engine[["home_team","away_team","proj_margin_home"]].copy()

sp = sp.merge(gm, on=["home_team","away_team"], how="left")

# compute cover probability for each row's line:
# if row team is home: P(margin + spread > 0)
# if row team is away: P(-margin + spread > 0) equivalent to P(margin < spread)
z_home = (0 - (-(sp["spread"]) - sp["proj_margin_home"])) / SIGMA_MARGIN
# easier explicit:
# For home team row: cover if margin + spread > 0 => margin > -spread
# P = 1 - CDF((-spread - mu)/sigma)
p_home_cover_at_line = 1.0 - norm.cdf((-sp["spread"] - sp["proj_margin_home"]) / SIGMA_MARGIN)

# for away team row: away covers if (-margin) + spread > 0 => margin < spread
p_away_cover_at_line = norm.cdf((sp["spread"] - sp["proj_margin_home"]) / SIGMA_MARGIN)

sp["model_prob_spread"] = np.where(sp["team"]==sp["home_team"], p_home_cover_at_line, p_away_cover_at_line).clip(0.03,0.97)

print("✅ sp lines with model_prob_spread:", sp.shape, "| nulls:", int(sp["model_prob_spread"].isna().sum()))
print("Spread line prob std:", float(sp["model_prob_spread"].std()))

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

assert "odds_long" in globals() and len(odds_long) > 0, "odds_long missing/empty"
assert "ml_model" in globals() and len(ml_model) > 0, "ml_model missing/empty"

# ---------- params (tunable) ----------
SIGMA_WIN   = 11.0   # maps ML win prob -> implied margin (bigger = more extreme margins)
SIGMA_COVER = 11.0   # spread cover noise
P_CLAMP_LO, P_CLAMP_HI = 0.05, 0.95

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    f = (p * (b + 1) - 1) / b
    return max(0.0, f)

# ---------- game list ----------
games = odds_long[["home_team","away_team"]].drop_duplicates().copy()

# ---------- get home win prob from ESPN predictor (ml_model) ----------
home_p = (
    ml_model.merge(games, on=["home_team","away_team"], how="inner")
            .loc[lambda d: d["team"] == d["home_team"], ["home_team","away_team","model_prob_ml"]]
            .rename(columns={"model_prob_ml":"p_home"})
            .copy()
)

# clamp to avoid infinities
home_p["p_home"] = home_p["p_home"].clip(P_CLAMP_LO, P_CLAMP_HI)

# implied margin mean from win prob
home_p["mu_margin_home"] = SIGMA_WIN * norm.ppf(home_p["p_home"])   # home - away

# ---------- attach to every spread line ----------
sp = odds_long[odds_long["market"]=="spreads"].copy()
sp = sp.merge(home_p[["home_team","away_team","mu_margin_home"]], on=["home_team","away_team"], how="left")

# home covers if margin + spread > 0  -> margin > -spread
p_home_cover = 1.0 - norm.cdf((-sp["spread"] - sp["mu_margin_home"]) / SIGMA_COVER)

# away cover is the complement at its own line (since spread is from that team's perspective in odds_long)
# but easier: compute directly based on whether row team is home or away
p_away_cover = norm.cdf((sp["spread"] - sp["mu_margin_home"]) / SIGMA_COVER)

sp["model_prob_spread"] = np.where(sp["team"]==sp["home_team"], p_home_cover, p_away_cover).clip(0.03, 0.97)

print("✅ Spread line prob std:", float(sp["model_prob_spread"].std()))
print("✅ Spread prob min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))
print("Rows:", sp.shape)

# ---------- score spreads (+EV + FULL Kelly) ----------
# ensure price_decimal exists
assert "price_decimal" in sp.columns, "price_decimal missing (should already be in odds_long)"

sp["implied_prob"] = 1.0 / sp["price_decimal"]
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

# best book per team/game by EV (no dupes)
sp_best = (
    sp.sort_values(["home_team","away_team","team","ev"], ascending=[True,True,True,False])
      .drop_duplicates(subset=["home_team","away_team","team"], keep="first")
)

# Top 10 spreads
top10_sp = sp_best.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp:", top10_sp.shape)
display(top10_sp[["home_team","away_team","team","book","spread","american_odds","price_decimal",
                  "model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"]])

In [ ]:
import numpy as np
import pandas as pd

# ========= helpers =========
def dec_to_implied(dec):
    return 1.0 / dec if dec and dec > 0 else np.nan

def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1.0 + a/100.0
    else:
        return 1.0 + 100.0/abs(a)

def kelly_full(p, dec):
    # dec includes stake, so b = dec-1
    b = dec - 1.0
    q = 1.0 - p
    if b <= 0 or p <= 0 or p >= 1:
        return 0.0
    f = (b*p - q) / b
    return max(0.0, f)

def ensure_decimal_cols(df):
    df = df.copy()
    if "price_decimal" not in df.columns:
        if "american_odds" in df.columns:
            df["price_decimal"] = df["american_odds"].apply(american_to_decimal)
        else:
            raise ValueError("Need price_decimal or american_odds.")
    return df

def pick_best_line(df, score_col="ev"):
    # higher EV wins; tie-break by higher true_edge then best price
    df = df.copy()
    for c in [score_col, "true_edge", "price_decimal"]:
        if c not in df.columns:
            df[c] = np.nan
    return (
        df.sort_values([score_col, "true_edge", "price_decimal"], ascending=[False, False, False])
          .head(1)
    )

# ========= REQUIRED INPUTS =========
# Expect these to already exist from your pipeline:
# odds_long (filtered to 2/28 already)
# ml_model with: home_team, away_team, team, model_prob_ml
# model_sp with: home_team, away_team, team, spread, model_prob_spread

for name in ["odds_long", "ml_model", "model_sp"]:
    if name not in globals():
        raise ValueError(f"Missing {name}. Run the cell that creates it first.")

odds_long = odds_long.copy()

# ========= build ML lines =========
ml_lines = odds_long[odds_long["market"].eq("h2h")].copy()
ml_lines = ensure_decimal_cols(ml_lines)

ml = ml_lines.merge(
    ml_model[["home_team","away_team","team","model_prob_ml"]],
    on=["home_team","away_team","team"],
    how="left"
)

ml = ml.dropna(subset=["model_prob_ml","price_decimal"])
ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units_full_kelly"] = ml.apply(lambda r: kelly_full(r["model_prob_ml"], r["price_decimal"]), axis=1)

# Keep best book per team per game
ml_best = (
    ml.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
      .apply(pick_best_line, score_col="ev")
)

# Filter: only +EV
ml_best = ml_best[ml_best["ev"] > 0].copy()

top10_ml = (
    ml_best.sort_values(["ev","true_edge","units_full_kelly"], ascending=False)
           .head(10)
           .reset_index(drop=True)
)

# ========= build SPREAD lines (consensus spread only, then best book at that spread) =========
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()
sp_lines = ensure_decimal_cols(sp_lines)

# Attach model spread prob by matching the SAME spread number used in model_sp
sp = sp_lines.merge(
    model_sp[["home_team","away_team","team","spread","model_prob_spread"]],
    on=["home_team","away_team","team","spread"],
    how="left"
)

sp = sp.dropna(subset=["model_prob_spread","price_decimal"])
sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

# Keep best book per team/spread/game
sp_best = (
    sp.groupby(["home_team","away_team","team","spread"], as_index=False, group_keys=False)
      .apply(pick_best_line, score_col="ev")
)

# Filter: only +EV
sp_best = sp_best[sp_best["ev"] > 0].copy()

# No-dupes at the GAME level: keep best single spread bet per game (whichever side/line is best)
sp_best_game = (
    sp_best.sort_values(["ev","true_edge","units_full_kelly"], ascending=False)
           .drop_duplicates(subset=["home_team","away_team"], keep="first")
)

top10_sp = (
    sp_best_game.sort_values(["ev","true_edge","units_full_kelly"], ascending=False)
               .head(10)
               .reset_index(drop=True)
)

# ========= format DISCORD card =========
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_u(x): return f"{x:.2f}u"

def american_fmt(a):
    a = float(a)
    if a > 0: return f"+{int(a)}"
    return f"{int(a)}"

def discord_card_ml(df):
    lines = []
    lines.append("NCAAB — 2/28 TOP 10 ML (NO DUPES | +EV | FULL KELLY)")
    for i, r in df.iterrows():
        lines.append("")
        lines.append(f"{i+1}) {r['home_team']} vs {r['away_team']}")
        lines.append(f"Bet: {r['team']} ML")
        lines.append(f"Book: {r['book']} | Odds: {american_fmt(r['american_odds'])} (Dec {r['price_decimal']:.2f})")
        lines.append(f"Model: {fmt_pct(r['model_prob_ml'])} | Implied: {fmt_pct(r['implied_prob'])}")
        lines.append(f"True Edge: {fmt_pct(r['true_edge'])} | EV: {r['ev']:.3f}")
        lines.append(f"Units (Full Kelly): {fmt_u(r['units_full_kelly'])}")
    return "\n".join(lines)

def discord_card_sp(df):
    lines = []
    lines.append("NCAAB — 2/28 TOP 10 SPREAD (NO DUPES | +EV | FULL KELLY)")
    for i, r in df.iterrows():
        s = float(r["spread"])
        side = r["team"]
        line = f"{s:+g}".replace("+","+")  # keep sign
        lines.append("")
        lines.append(f"{i+1}) {r['home_team']} vs {r['away_team']}")
        lines.append(f"Bet: {side} {line}")
        lines.append(f"Book: {r['book']} | Odds: {american_fmt(r['american_odds'])} (Dec {r['price_decimal']:.2f})")
        lines.append(f"Model(Cover): {fmt_pct(r['model_prob_spread'])} | Implied: {fmt_pct(r['implied_prob'])}")
        lines.append(f"True Edge: {fmt_pct(r['true_edge'])} | EV: {r['ev']:.3f}")
        lines.append(f"Units (Full Kelly): {fmt_u(r['units_full_kelly'])}")
    return "\n".join(lines)

print(discord_card_sp(top10_sp))
print("\n" + "="*40 + "\n")
print(discord_card_ml(top10_ml))

# Save outputs
tier_top10_sp = top10_sp.copy()
tier_top10_ml = top10_ml.copy()
print("\n✅ Saved: tier_top10_sp, tier_top10_ml")

In [ ]:
import numpy as np
import pandas as pd

# --- REQUIRE these from your prior run cell ---
for name in ["ml_best","sp_best"]:
    if name not in globals():
        raise ValueError(f"Missing {name}. Run the Top10 build cell first so ml_best/sp_best exist.")

# =========================
# Tier A filters (tweakable)
# =========================
MIN_ML_TRUE_EDGE = 0.03          # 3%+
MIN_SP_TRUE_EDGE = 0.02          # 2%+
MIN_UNITS = 0.05                 # at least 0.05u full Kelly
MAX_CHALK = -200                 # don't take worse than -200
MAX_DOG = 350                    # don't take longer than +350
MAX_ABS_SPREAD = 12              # avoid huge spreads

def american_val(a):
    return float(a)

def american_fmt(a):
    a = float(a)
    return f"+{int(a)}" if a > 0 else f"{int(a)}"

def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_u(x): return f"{x:.2f}u"

# --- Tier A ML ---
mlA = ml_best.copy()
mlA = mlA[mlA["ev"] > 0].copy()
mlA = mlA[mlA["true_edge"] >= MIN_ML_TRUE_EDGE].copy()
mlA = mlA[mlA["units_full_kelly"] >= MIN_UNITS].copy()

mlA = mlA[(mlA["american_odds"].apply(american_val) >= MAX_CHALK) &
          (mlA["american_odds"].apply(american_val) <= MAX_DOG)].copy()

# keep best ML per game (no dupes at game level)
mlA = (mlA.sort_values(["ev","true_edge","units_full_kelly"], ascending=False)
          .drop_duplicates(subset=["home_team","away_team"], keep="first")
          .head(10)
          .reset_index(drop=True))

# --- Tier A SPREAD ---
spA = sp_best.copy()
spA = spA[spA["ev"] > 0].copy()
spA = spA[spA["true_edge"] >= MIN_SP_TRUE_EDGE].copy()
spA = spA[spA["units_full_kelly"] >= MIN_UNITS].copy()
spA = spA[spA["spread"].abs() <= MAX_ABS_SPREAD].copy()

# keep best spread bet per game
spA = (spA.sort_values(["ev","true_edge","units_full_kelly"], ascending=False)
          .drop_duplicates(subset=["home_team","away_team"], keep="first")
          .head(10)
          .reset_index(drop=True))

# --- Combine to 5-10 plays total (prefer higher units then EV) ---
combo = []
if len(mlA):
    mlA_out = mlA.assign(market="ML", line=np.nan, prob=mlA["model_prob_ml"])
    combo.append(mlA_out)
if len(spA):
    spA_out = spA.assign(market="SPREAD", line=spA["spread"], prob=spA["model_prob_spread"])
    combo.append(spA_out)

if not combo:
    raise ValueError("No Tier A plays passed. Lower MIN_TRUE_EDGE / MIN_UNITS or widen odds/spread filters.")

card = pd.concat(combo, ignore_index=True)

card = (card.sort_values(["units_full_kelly","ev","true_edge"], ascending=False)
            .head(10))

# Try to keep 5–10 total; if more than 8, trim to 8 (clean Discord card)
if len(card) > 8:
    card = card.head(8)

# --- Discord output ---
lines = []
lines.append(f"NCAAB — 2/28 TIER A CARD (ML + SPREAD) | {len(card)} plays | FULL KELLY")
for i, r in enumerate(card.itertuples(index=False), start=1):
    lines.append("")
    lines.append(f"{i}) {r.home_team} vs {r.away_team}")
    if r.market == "ML":
        lines.append(f"Bet: {r.team} ML")
        lines.append(f"Book: {r.book} | Odds: {american_fmt(r.american_odds)} (Dec {r.price_decimal:.2f})")
        lines.append(f"Model: {fmt_pct(r.model_prob_ml)} | Implied: {fmt_pct(r.implied_prob)}")
    else:
        s = float(r.spread)
        line = f"{s:+g}"
        lines.append(f"Bet: {r.team} {line}")
        lines.append(f"Book: {r.book} | Odds: {american_fmt(r.american_odds)} (Dec {r.price_decimal:.2f})")
        lines.append(f"Model(Cover): {fmt_pct(r.model_prob_spread)} | Implied: {fmt_pct(r.implied_prob)}")

    lines.append(f"True Edge: {fmt_pct(r.true_edge)} | EV: {r.ev:.3f}")
    lines.append(f"Units (Full Kelly): {fmt_u(r.units_full_kelly)}")

print("\n".join(lines))

tierA_card = card.copy()
print("\n✅ Saved: tierA_card")

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Tier A Auto-Loosen Builder
# Requires: ml_best, sp_best
# -----------------------------

assert "ml_best" in globals() and isinstance(ml_best, pd.DataFrame) and len(ml_best) > 0, "Missing ml_best. Run your ML build cell first."
assert "sp_best" in globals() and isinstance(sp_best, pd.DataFrame) and len(sp_best) > 0, "Missing sp_best. Run your Spread build cell first."

def build_tierA(ml_best, sp_best,
                MIN_TRUE_EDGE_ML=0.015,
                MIN_TRUE_EDGE_SP=0.012,
                MIN_UNITS=0.02,
                MAX_JUICE_ML=-220,
                MAX_JUICE_SP=-120,
                SPREAD_MAX_ABS=15.5,
                TARGET_MIN=5,
                TARGET_MAX=10):

    # --- ML prep ---
    ml = ml_best.copy()
    # normalize column names we rely on
    if "model_prob_ml" not in ml.columns and "model_prob" in ml.columns:
        ml = ml.rename(columns={"model_prob":"model_prob_ml"})
    # ensure implied_prob exists
    if "implied_prob" not in ml.columns and "price_decimal" in ml.columns:
        ml["implied_prob"] = 1.0 / ml["price_decimal"]
    # ensure true_edge exists
    if "true_edge" not in ml.columns and ("model_prob_ml" in ml.columns) and ("implied_prob" in ml.columns):
        ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
    # ensure ev exists
    if "ev" not in ml.columns and ("model_prob_ml" in ml.columns) and ("price_decimal" in ml.columns):
        ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0

    # filter ML (tier A = "not crazy dogs" + edge + some size)
    ml_f = ml.copy()
    if "american_odds" in ml_f.columns:
        ml_f = ml_f[(ml_f["american_odds"] >= -5000)]  # safety
        ml_f = ml_f[(ml_f["american_odds"] <= MAX_JUICE_ML)]  # exclude heavy chalk beyond threshold
        ml_f = ml_f[(ml_f["american_odds"] >= -MAX_JUICE_ML)] if MAX_JUICE_ML > 0 else ml_f  # no-op normally

    # allow dogs but not bombs: keep >= -220 and <= +450 by default
    # If american_odds missing, we won't block
    if "american_odds" in ml_f.columns:
        ml_f = ml_f[(ml_f["american_odds"] <= 450)]
    ml_f = ml_f.dropna(subset=["true_edge","ev"])
    if "units_full_kelly" in ml_f.columns:
        ml_f["units"] = ml_f["units_full_kelly"]
    elif "units" not in ml_f.columns:
        # fallback: create a "size proxy" from EV (not ideal, but keeps pipeline moving)
        ml_f["units"] = ml_f["ev"].clip(lower=0)

    ml_f = ml_f[(ml_f["true_edge"] >= MIN_TRUE_EDGE_ML) & (ml_f["units"] >= MIN_UNITS)]

    # best one per game (ML)
    if {"home_team","away_team"}.issubset(ml_f.columns):
        ml_f = (ml_f.sort_values(["ev","true_edge"], ascending=False)
                    .drop_duplicates(subset=["home_team","away_team"], keep="first"))

    # --- SP prep ---
    sp = sp_best.copy()
    # normalize name
    if "model_prob_spread" not in sp.columns and "model_prob" in sp.columns:
        sp = sp.rename(columns={"model_prob":"model_prob_spread"})
    # ensure implied_prob exists
    if "implied_prob" not in sp.columns and "price_decimal" in sp.columns:
        sp["implied_prob"] = 1.0 / sp["price_decimal"]
    # true edge
    if "true_edge" not in sp.columns and ("model_prob_spread" in sp.columns) and ("implied_prob" in sp.columns):
        sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
    # ev
    if "ev" not in sp.columns and ("model_prob_spread" in sp.columns) and ("price_decimal" in sp.columns):
        sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

    if "units_full_kelly" in sp.columns:
        sp["units"] = sp["units_full_kelly"]
    elif "units" not in sp.columns:
        sp["units"] = sp["ev"].clip(lower=0)

    sp_f = sp.dropna(subset=["true_edge","ev"]).copy()

    # spread sanity: keep within abs threshold if we have spreads
    if "spread" in sp_f.columns:
        sp_f = sp_f[sp_f["spread"].abs() <= SPREAD_MAX_ABS]

    # juice sanity
    if "american_odds" in sp_f.columns:
        sp_f = sp_f[sp_f["american_odds"] >= MAX_JUICE_SP]  # exclude -130, -150 etc if MAX_JUICE_SP=-120

    sp_f = sp_f[(sp_f["true_edge"] >= MIN_TRUE_EDGE_SP) & (sp_f["units"] >= MIN_UNITS)]

    # best one per game (SP)
    if {"home_team","away_team"}.issubset(sp_f.columns):
        sp_f = (sp_f.sort_values(["ev","true_edge"], ascending=False)
                    .drop_duplicates(subset=["home_team","away_team"], keep="first"))

    # combine
    card = pd.concat([ml_f.assign(market="ML"), sp_f.assign(market="SPREAD")], ignore_index=True)
    card = card.sort_values(["ev","true_edge"], ascending=False)

    # limit to 10 but keep at least 5 if possible
    if len(card) > TARGET_MAX:
        card = card.head(TARGET_MAX)

    return card, ml_f, sp_f


# -----------------------------
# Auto-loosen ladder (stops when 5-10 plays found)
# -----------------------------
ladder = [
    # (MIN_TRUE_EDGE_ML, MIN_TRUE_EDGE_SP, MIN_UNITS, MAX_JUICE_ML, MAX_JUICE_SP, SPREAD_MAX_ABS)
    (0.015, 0.012, 0.02, -220, -120, 15.5),
    (0.012, 0.010, 0.015, -220, -120, 16.5),
    (0.010, 0.008, 0.01, -200, -125, 17.5),
    (0.008, 0.006, 0.005, -200, -130, 18.5),
    (0.006, 0.005, 0.003, -180, -135, 20.0),
]

found = None
used = None

for params in ladder:
    card, ml_f, sp_f = build_tierA(
        ml_best, sp_best,
        MIN_TRUE_EDGE_ML=params[0],
        MIN_TRUE_EDGE_SP=params[1],
        MIN_UNITS=params[2],
        MAX_JUICE_ML=params[3],
        MAX_JUICE_SP=params[4],
        SPREAD_MAX_ABS=params[5],
        TARGET_MIN=5,
        TARGET_MAX=10,
    )
    if 5 <= len(card) <= 10:
        found = card
        used = params
        break

# If still nothing, return the best few anyway (for visibility)
if found is None:
    card, ml_f, sp_f = build_tierA(
        ml_best, sp_best,
        MIN_TRUE_EDGE_ML=0.006,
        MIN_TRUE_EDGE_SP=0.005,
        MIN_UNITS=0.0,
        MAX_JUICE_ML=-180,
        MAX_JUICE_SP=-140,
        SPREAD_MAX_ABS=22.0,
        TARGET_MIN=1,
        TARGET_MAX=10,
    )
    found = card
    used = ("FALLBACK", 0.006, 0.005, 0.0, -180, -140, 22.0)

print("✅ Tier A auto-loosen finished. Params used:", used)
print("Plays found:", len(found))
display_cols = ["market","home_team","away_team","team","book","american_odds","spread","price_decimal","model_prob_ml","model_prob_spread","implied_prob","true_edge","ev","units"]
display_cols = [c for c in display_cols if c in found.columns]
found_display = found[display_cols].copy()

# Pretty % formatting if present
for c in ["model_prob_ml","model_prob_spread","implied_prob","true_edge"]:
    if c in found_display.columns:
        found_display[c] = found_display[c].astype(float)

print(found_display.to_string(index=False))

In [ ]:
# -----------------------------
# DISCORD CARD (Tier A | Full Kelly)
# Requires: found_display OR found (from the auto-loosen cell)
# -----------------------------

import pandas as pd
import numpy as np

# Grab the DF the last cell printed
if "found" in globals() and isinstance(found, pd.DataFrame) and len(found) > 0:
    card_df = found.copy()
elif "found_display" in globals() and isinstance(found_display, pd.DataFrame) and len(found_display) > 0:
    card_df = found_display.copy()
else:
    raise ValueError("Couldn't find Tier A dataframe. Run the auto-loosen cell first.")

def fmt_pct(x):
    if pd.isna(x): return "—"
    return f"{100*float(x):.1f}%"

def fmt_odds(a):
    if pd.isna(a): return "—"
    a = int(a)
    return f"{a:+d}" if a > 0 else str(a)

def fmt_units(u):
    if pd.isna(u): return "—"
    return f"{float(u):.2f}u"

lines = []
lines.append("NCAAB — 2/28 TIER A (ML + SPREAD) | FULL KELLY\n")

for i, r in enumerate(card_df.itertuples(index=False), start=1):
    d = r._asdict()

    market = d.get("market","")
    home = d.get("home_team","")
    away = d.get("away_team","")
    team = d.get("team","")
    book = d.get("book","")
    amer = d.get("american_odds", np.nan)
    spread = d.get("spread", np.nan)

    price_dec = d.get("price_decimal", np.nan)
    implied = d.get("implied_prob", np.nan)
    true_edge = d.get("true_edge", np.nan)
    ev = d.get("ev", np.nan)
    units = d.get("units", np.nan)

    if market == "SPREAD":
        bet_line = f"{team} {spread:+g}"
        model_p = d.get("model_prob_spread", np.nan)
        model_label = f"Model(Cover): {fmt_pct(model_p)}"
    else:
        bet_line = f"{team} ML"
        model_p = d.get("model_prob_ml", np.nan)
        model_label = f"Model: {fmt_pct(model_p)}"

    lines.append(
        f"{i}) {home} vs {away}\n"
        f"Bet: {bet_line}\n"
        f"Book: {book} | Odds: {fmt_odds(amer)} (Dec {float(price_dec):.2f})\n"
        f"{model_label} | Implied: {fmt_pct(implied)}\n"
        f"True Edge: {fmt_pct(true_edge)} | EV: {float(ev):.3f}\n"
        f"Units (Full Kelly): {fmt_units(units)}\n"
    )

discord_card = "\n".join(lines)
print(discord_card)

In [ ]:
import numpy as np
import pandas as pd
from math import erf, sqrt

# =========================
# CONFIG: pick the bet
# =========================
HOME = "South Dakota Coyotes"
AWAY = "South Dakota St Jackrabbits"
BET_TEAM = "South Dakota St Jackrabbits"
BET_SPREAD = -2.5  # SDST -2.5
N_SIMS = 200000
SEED = 42

# =========================
# Helpers
# =========================
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def kelly_full(p, dec_odds):
    b = dec_odds - 1.0
    q = 1.0 - p
    f = (b*p - q) / b
    return max(0.0, f)

def spread_cover_prob_from_margin(mu, sigma, spread_for_team):
    """
    margin = (BET_TEAM points - opponent points)
    spread_for_team: negative if BET_TEAM favored (ex -2.5), positive if dog (+2.5)
    Cover condition:
      margin > (-spread_for_team)
      ex spread=-2.5 -> margin > 2.5
         spread=+2.5 -> margin > -2.5
    """
    threshold = -float(spread_for_team)
    # P(margin > threshold) where margin ~ N(mu, sigma)
    z = (threshold - mu) / sigma
    p = 1.0 - norm_cdf(z)
    return p, threshold, z

# =========================
# Locate the spread line row (odds + model_prob_spread)
# =========================
# We expect your spread lines DF to exist as either sp_out / sp_best / sp / etc.
spread_df_candidates = ["sp_out", "sp_best", "sp_lines", "sp", "tier_top10_sp", "top10_sp"]
spread_df = None
for name in spread_df_candidates:
    if name in globals() and isinstance(globals()[name], pd.DataFrame) and len(globals()[name]) > 0:
        df = globals()[name]
        need = {"home_team","away_team","team","spread","price_decimal"}
        if need.issubset(set(df.columns)):
            spread_df = df
            spread_df_name = name
            break

if spread_df is None:
    raise ValueError("Couldn't find your spread lines DF. Expected one of: " + ", ".join(spread_df_candidates))

# Filter to our exact bet
row = spread_df[
    (spread_df["home_team"] == HOME) &
    (spread_df["away_team"] == AWAY) &
    (spread_df["team"] == BET_TEAM) &
    (spread_df["spread"].astype(float) == float(BET_SPREAD))
].copy()

if row.empty:
    # fallback: sometimes home/away orientation flips — try both orientations
    row = spread_df[
        (spread_df["home_team"] == AWAY) &
        (spread_df["away_team"] == HOME) &
        (spread_df["team"] == BET_TEAM) &
        (spread_df["spread"].astype(float) == float(BET_SPREAD))
    ].copy()

if row.empty:
    # last fallback: same matchup, closest spread
    tmp = spread_df[
        ((spread_df["home_team"] == HOME) & (spread_df["away_team"] == AWAY)) |
        ((spread_df["home_team"] == AWAY) & (spread_df["away_team"] == HOME))
    ].copy()
    tmp["spread_dist"] = (tmp["spread"].astype(float) - float(BET_SPREAD)).abs()
    tmp = tmp[tmp["team"] == BET_TEAM].sort_values("spread_dist").head(5)
    raise ValueError("Exact spread line not found. Closest lines:\n" + str(tmp[["home_team","away_team","team","spread","book","american_odds","price_decimal"]]))

row = row.iloc[0].to_dict()

dec = float(row["price_decimal"])
amer = row.get("american_odds", None)
book = row.get("book", None)

# =========================
# Locate mu + sigma used by your sim engine
# =========================
# The sim engine typically creates something like:
#  - projected_margin or mu_margin per game/team
#  - sigma or spread_sigma
# We'll search for a DF in globals that has the matchup and a margin column.
mu = None
sigma = None
source_name = None

margin_df_candidates = [
    "hybrid_margin_engine", "margin_engine", "engine", "hybrid", "proj", "projected",
    "model_sp", "model_sp_sim", "sim_spread_df", "spread_sim_df", "sim_df"
]

possible_dfs = []
for k, v in globals().items():
    if isinstance(v, pd.DataFrame) and len(v) > 0:
        cols = set(v.columns)
        if {"home_team","away_team"}.issubset(cols):
            possible_dfs.append((k, v))

def find_mu_sigma(df):
    # Common names we might have
    mu_cols = [c for c in df.columns if c in ["mu","mu_margin","projected_margin","expected_margin","margin","proj_margin"]]
    sig_cols = [c for c in df.columns if c in ["sigma","margin_sigma","spread_sigma","sd_margin","stdev_margin"]]
    if not mu_cols:
        # sometimes it's just "projected_margin" but stored for HOME only
        return None, None, None, None
    mu_col = mu_cols[0]
    sig_col = sig_cols[0] if sig_cols else None
    return mu_col, sig_col, mu_cols, sig_cols

# try to find a df containing our matchup + a mu column
for name, df in possible_dfs:
    mu_col, sig_col, _, _ = find_mu_sigma(df)
    if mu_col is None:
        continue

    g = df[
        ((df["home_team"] == HOME) & (df["away_team"] == AWAY)) |
        ((df["home_team"] == AWAY) & (df["away_team"] == HOME))
    ].copy()

    if g.empty:
        continue

    # If the df is team-level, it might include "team"
    if "team" in g.columns:
        g = g[g["team"] == BET_TEAM].copy()
        if g.empty:
            continue

    mu_val = float(g.iloc[0][mu_col])
    sig_val = float(g.iloc[0][sig_col]) if (sig_col is not None and pd.notna(g.iloc[0][sig_col])) else None

    mu = mu_val
    sigma = sig_val
    source_name = name
    mu_source_col = mu_col
    sigma_source_col = sig_col
    break

# If sigma not found, choose a sensible default (you can replace later)
if sigma is None:
    # typical college hoops margin sigma range ~ 10–13; your sim earlier produced realistic prob range using a sigma like this
    sigma = 11.5
    sigma_source_col = "DEFAULT(11.5)"

# IMPORTANT: mu must be BET_TEAM margin, i.e., (BET_TEAM - opponent).
# If your stored mu is HOME margin (home - away), convert if needed.
# We'll attempt to detect if mu is home margin by checking if BET_TEAM == home_team.
if source_name is not None:
    # if stored at game-level with no "team", it may be HOME margin.
    df_src = globals()[source_name]
    is_team_level = "team" in df_src.columns
    if not is_team_level:
        # mu is likely home margin: (HOME - AWAY)
        # convert to BET_TEAM margin:
        if BET_TEAM == HOME:
            mu = mu
        elif BET_TEAM == AWAY:
            mu = -mu

# =========================
# Compute analytic cover prob + Monte Carlo verification
# =========================
p_analytic, threshold, z = spread_cover_prob_from_margin(mu, sigma, BET_SPREAD)

rng = np.random.default_rng(SEED)
margins = rng.normal(loc=mu, scale=sigma, size=N_SIMS)
p_mc = float(np.mean(margins > threshold))

implied = 1.0/dec
true_edge = p_mc - implied
ev = p_mc*dec - 1.0
kelly = kelly_full(p_mc, dec)

print("========== SPREAD MODEL INSIDES ==========")
print(f"Matchup: {HOME} vs {AWAY}")
print(f"Bet: {BET_TEAM} {BET_SPREAD:+g}")
print(f"Book: {book} | Amer: {amer} | Dec: {dec:.3f}")
print("")
print("---- Inputs (mu/sigma) ----")
print(f"mu (expected margin for BET_TEAM): {mu:.3f}")
print(f"sigma (margin volatility): {sigma:.3f}   [source: {sigma_source_col}]")
if source_name is not None:
    print(f"mu source DF: {source_name} | column: {mu_source_col}")
print("")
print("---- Cover condition ----")
print(f"Cover threshold: margin > {threshold:.3f}")
print(f"z = (threshold - mu)/sigma = {z:.3f}")
print("")
print("---- Probabilities ----")
print(f"Analytic P(cover): {100*p_analytic:.2f}%")
print(f"Monte Carlo P(cover) (N={N_SIMS:,}): {100*p_mc:.2f}%")
print("")
print("---- Price / Edge / EV / Kelly ----")
print(f"Implied prob from price: {100*implied:.2f}%")
print(f"True edge (MC - implied): {100*true_edge:.2f}%")
print(f"EV = p*dec - 1: {ev:.3f}")
print(f"Full Kelly fraction: {kelly:.4f}  =>  {100*kelly:.2f}% of bankroll")
print("=========================================")

# Optional: quick simulation summary (no charts)
qs = np.quantile(margins, [0.05, 0.25, 0.50, 0.75, 0.95])
print("\nSimulated margin distribution (BET_TEAM - opp):")
print(f"  mean={margins.mean():.2f} | std={margins.std():.2f}")
print(f"  5%={qs[0]:.2f} | 25%={qs[1]:.2f} | 50%={qs[2]:.2f} | 75%={qs[3]:.2f} | 95%={qs[4]:.2f}")

In [ ]:
print("Check stored model_prob_spread:")
print(row_df[["model_prob_spread"]])

print("\nCheck hybrid margin table for this game:")
print(
    hybrid_margin[
        (hybrid_margin["home_team"]=="South Dakota Coyotes") &
        (hybrid_margin["away_team"]=="South Dakota St Jackrabbits")
    ][["projected_margin"]]
)

In [ ]:
# =========================
# SPREAD PIPELINE DIAGNOSTIC
# =========================

HOME = "South Dakota Coyotes"
AWAY = "South Dakota St Jackrabbits"
BET_TEAM = "South Dakota St Jackrabbits"
BET_SPREAD = -2.5

# 1️⃣ Find spread row from sp_out / sp_best / etc
spread_df_candidates = ["sp_out", "sp_best", "sp", "top10_sp"]

spread_df = None
for name in spread_df_candidates:
    if name in globals():
        df = globals()[name]
        if {"home_team","away_team","team","spread"}.issubset(df.columns):
            spread_df = df
            spread_df_name = name
            break

if spread_df is None:
    raise ValueError("Could not locate spread dataframe.")

row = spread_df[
    (spread_df["home_team"] == HOME) &
    (spread_df["away_team"] == AWAY) &
    (spread_df["team"] == BET_TEAM) &
    (spread_df["spread"].astype(float) == float(BET_SPREAD))
].copy()

if row.empty:
    raise ValueError("Exact spread row not found.")

print("===== STORED SPREAD ROW =====")
print(row[["home_team","away_team","team","spread","model_prob_spread","price_decimal"]])

# 2️⃣ Check hybrid margin table
if "hybrid_margin" in globals():
    hm = hybrid_margin[
        (hybrid_margin["home_team"] == HOME) &
        (hybrid_margin["away_team"] == AWAY)
    ]
    print("\n===== HYBRID MARGIN =====")
    print(hm[["projected_margin"]])
else:
    print("\nNo hybrid_margin dataframe found.")

# 3️⃣ Quick consistency check
if not row.empty and "hybrid_margin" in globals() and not hm.empty:
    stored_prob = float(row.iloc[0]["model_prob_spread"])
    projected_margin = float(hm.iloc[0]["projected_margin"])

    print("\n===== CONSISTENCY CHECK =====")
    print(f"Stored model_prob_spread: {stored_prob:.4f}")
    print(f"Hybrid projected_margin: {projected_margin:.4f}")
    print("If projected_margin == -2.5 and model_prob_spread ≈ 0.51, they are NOT connected.")

In [ ]:
import numpy as np
import pandas as pd
import math

# -----------------------
# SETTINGS
# -----------------------
SIGMA_DEFAULT = 11.5  # if you have a better per-league/per-team sigma later, swap it here

# must exist
assert "odds_long" in globals() and len(odds_long) > 0, "odds_long missing"
assert "hybrid_margin" in globals() and len(hybrid_margin) > 0, "hybrid_margin missing"

# -----------------------
# HELPERS
# -----------------------
def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def cover_prob_from_mu_spread(mu, spread, sigma):
    # Cover if (margin + spread > 0)
    # margin ~ N(mu, sigma)
    # P = Φ((mu + spread)/sigma)
    return norm_cdf((mu + spread) / sigma)

# -----------------------
# BUILD SPREAD LINES (fresh)
# -----------------------
sp = odds_long[odds_long["market"].eq("spreads")].copy()

need = {"home_team","away_team","team","spread","price_decimal","american_odds","book","commence_time"}
missing = need - set(sp.columns)
if missing:
    raise ValueError(f"sp missing cols: {missing}")

# -----------------------
# ATTACH HYBRID MARGIN
# projected_margin is assumed HOME - AWAY
# -----------------------
hm = hybrid_margin[["home_team","away_team","projected_margin"]].copy()
sp = sp.merge(hm, on=["home_team","away_team"], how="left")

if sp["projected_margin"].isna().mean() > 0.01:
    bad = sp[sp["projected_margin"].isna()][["home_team","away_team"]].drop_duplicates().head(10)
    raise ValueError(f"Missing projected_margin for too many rows. Sample:\n{bad}")

# -----------------------
# CONVERT TO TEAM-SPECIFIC mu
# If team == home => mu_team = +projected_margin
# If team == away => mu_team = -projected_margin
# -----------------------
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["projected_margin"],
    -sp["projected_margin"]
).astype(float)

# sigma (constant for now)
sp["sigma"] = SIGMA_DEFAULT

# -----------------------
# COMPUTE MODEL COVER PROB
# -----------------------
sp["model_prob_spread"] = sp.apply(
    lambda r: cover_prob_from_mu_spread(r["mu_team"], float(r["spread"]), float(r["sigma"])),
    axis=1
)

# sanity
print("✅ Rebuilt model_prob_spread from hybrid_margin (home-away) + sigma.")
print("Spread prob std:", float(sp["model_prob_spread"].std()))
print("Spread prob min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))
print("Rows:", sp.shape)

# -----------------------
# OPTIONAL: overwrite your global sp_out/sp_best if you use them later
# -----------------------
sp_out = sp.copy()

# -----------------------
# QUICK CHECK: the game you audited
# -----------------------
HOME = "South Dakota Coyotes"
AWAY = "South Dakota St Jackrabbits"
BET_TEAM = "South Dakota St Jackrabbits"
BET_SPREAD = -2.5

chk = sp_out[
    (sp_out["home_team"]==HOME) &
    (sp_out["away_team"]==AWAY) &
    (sp_out["team"]==BET_TEAM) &
    (sp_out["spread"].astype(float)==float(BET_SPREAD))
][["home_team","away_team","team","spread","projected_margin","mu_team","sigma","model_prob_spread","price_decimal"]].head(1)

print("\n===== CHECK (South Dakota St -2.5) =====")
print(chk)

In [ ]:
import numpy as np
import pandas as pd

# -----------------------
# Helpers
# -----------------------
def dec_to_implied(d):
    d = float(d)
    return 1.0 / d

def kelly_full(p, dec):
    # Kelly fraction for decimal odds
    # f* = (bp - q)/b where b = dec-1, q = 1-p
    p = float(p); dec = float(dec)
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q) / b
    return max(0.0, f)

def pick_best_line(g, score_col="ev"):
    g = g.sort_values(score_col, ascending=False)
    return g.iloc[0]

# -----------------------
# Build ML + Spread clean sets from odds_long
# -----------------------
assert "odds_long" in globals() and len(odds_long)>0, "odds_long missing"
assert "ml_model" in globals() and len(ml_model)>0, "ml_model missing"
assert "sp_out" in globals() and len(sp_out)>0, "sp_out missing (run spread rebuild cell first)"

# ----- ML -----
ml = odds_long[odds_long["market"].eq("h2h")].copy()
ml = ml.merge(ml_model[["home_team","away_team","team","model_prob_ml"]],
              on=["home_team","away_team","team"], how="left")

ml = ml.dropna(subset=["model_prob_ml","price_decimal"]).copy()
ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units_full_kelly"] = ml.apply(lambda r: kelly_full(r["model_prob_ml"], r["price_decimal"]), axis=1)

# filters (adjust if you want)
ML_MAX_JUICE = -200
ml = ml[ml["american_odds"].astype(float) >= ML_MAX_JUICE].copy()

# NO DUPES: best bet per team per game
ml_best = (
    ml.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
      .apply(pick_best_line, score_col="ev")
)

top10_ml = ml_best.sort_values("ev", ascending=False).head(10).copy()

# ----- SPREAD -----
sp = sp_out.copy()
sp = sp.dropna(subset=["model_prob_spread","price_decimal","spread"]).copy()

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

SP_MAX_JUICE = -200
sp = sp[sp["american_odds"].astype(float) >= SP_MAX_JUICE].copy()

# NO DUPES: best bet per team per game
sp_best = (
    sp.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
      .apply(pick_best_line, score_col="ev")
)

top10_sp = sp_best.sort_values("ev", ascending=False).head(10).copy()

print("✅ top10_ml:", top10_ml.shape, "| ✅ top10_sp:", top10_sp.shape)

# -----------------------
# Pretty print (Discord-ready style)
# -----------------------
def fmt_pct(x):
    return f"{100*float(x):.1f}%"

def fmt_u(x):
    return f"{float(x):.2f}u"

print("\nNCAAB — 2/28 TOP 10 SPREAD (NO DUPES | +EV | FULL KELLY)\n")
for i, r in enumerate(top10_sp.itertuples(index=False), 1):
    print(f"{i}) {r.home_team} vs {r.away_team}")
    print(f"Bet: {r.team} {r.spread:+g}")
    print(f"Book: {r.book} | Odds: {int(r.american_odds)} (Dec {float(r.price_decimal):.3f})")
    print(f"Model(Cover): {fmt_pct(r.model_prob_spread)} | Implied: {fmt_pct(r.implied_prob)}")
    print(f"True Edge: {fmt_pct(r.true_edge)} | EV: {float(r.ev):.3f}")
    print(f"Units (Full Kelly): {fmt_u(r.units_full_kelly)}\n")

print("========================================\n")
print("NCAAB — 2/28 TOP 10 ML (NO DUPES | +EV | FULL KELLY)\n")
for i, r in enumerate(top10_ml.itertuples(index=False), 1):
    print(f"{i}) {r.home_team} vs {r.away_team}")
    print(f"Bet: {r.team} ML")
    print(f"Book: {r.book} | Odds: {int(r.american_odds)} (Dec {float(r.price_decimal):.3f})")
    print(f"Model: {fmt_pct(r.model_prob_ml)} | Implied: {fmt_pct(r.implied_prob)}")
    print(f"True Edge: {fmt_pct(r.true_edge)} | EV: {float(r.ev):.3f}")
    print(f"Units (Full Kelly): {fmt_u(r.units_full_kelly)}\n")

# Save for later cells
tier_top10_sp = top10_sp
tier_top10_ml = top10_ml

In [ ]:
import numpy as np
import pandas as pd
from math import erf, sqrt

assert "top10_sp" in globals() and len(top10_sp)>0, "top10_sp missing"
assert "sp_out" in globals() and len(sp_out)>0, "sp_out missing"

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def mc_cover_prob(mu, sigma, threshold, n=100000, seed=42):
    rng = np.random.default_rng(seed)
    sims = rng.normal(mu, sigma, n)
    return float((sims > threshold).mean())

print("========== SPREAD SANITY CHECK ==========")
print("Using sigma column if present; else default 11.5.\n")

check_rows = top10_sp.copy()

# Ensure we have mu_team / sigma if they exist in sp_out
cols_available = set(sp_out.columns)
use_sigma_col = "sigma" in cols_available
use_mu_col = "mu_team" in cols_available

for i, r in enumerate(check_rows.itertuples(index=False), 1):
    # match the exact row in sp_out by keys
    row = sp_out[
        (sp_out["home_team"]==r.home_team) &
        (sp_out["away_team"]==r.away_team) &
        (sp_out["team"]==r.team) &
        (sp_out["spread"]==r.spread) &
        (sp_out["book"]==r.book)
    ].copy()

    if row.empty:
        print(f"{i}) {r.home_team} vs {r.away_team} | {r.team} {r.spread:+g} — ❌ row not found in sp_out")
        continue

    row = row.iloc[0]

    # Pull mu/sigma
    mu = float(row["mu_team"]) if use_mu_col else np.nan
    sigma = float(row["sigma"]) if use_sigma_col else 11.5

    # For spread bet TEAM at spread s:
    # define margin = (TEAM points - OPP points)
    # cover condition for TEAM at spread s: margin + s > 0  => margin > -s
    threshold = float(-row["spread"])

    stored_p = float(row["model_prob_spread"])

    # If mu is missing, we can't recompute correctly.
    if np.isnan(mu):
        print(f"{i}) {r.home_team} vs {r.away_team}")
        print(f"   Bet: {r.team} {r.spread:+g} | Stored P: {stored_p:.4f}")
        print("   ⚠️ mu_team missing on sp_out row. Need mu_team attached to validate.\n")
        continue

    z = (threshold - mu) / sigma
    p_analytic = 1.0 - norm_cdf(z)
    p_mc = mc_cover_prob(mu, sigma, threshold, n=150000, seed=100+i)

    delta = stored_p - p_mc
    ok = abs(delta) <= 0.03  # 3% tolerance

    print(f"{i}) {r.home_team} vs {r.away_team}")
    print(f"   Bet: {r.team} {float(r.spread):+g} | Book: {r.book} | Dec: {float(r.price_decimal):.3f}")
    print(f"   mu_team={mu:.3f} | sigma={sigma:.3f} | threshold={threshold:.3f}")
    print(f"   Stored P={stored_p:.4f} | Analytic P={p_analytic:.4f} | MC P={p_mc:.4f} | Δ(stored-MC)={delta:+.4f}")
    print(f"   SANITY: {'✅ PASS' if ok else '❌ FAIL'}\n")

print("========================================")

In [ ]:
import numpy as np
import pandas as pd
from math import log
from scipy.stats import norm

# -----------------------------
# REQUIREMENTS CHECK
# -----------------------------
need = ["odds_long","cons_ml","cons_sp","ml_model"]
missing = [x for x in need if x not in globals()]
if missing:
    raise ValueError(f"Missing required DF(s): {missing}. Make sure those cells ran first.")

# -----------------------------
# Helpers
# -----------------------------
def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1.0 + a/100.0
    else:
        return 1.0 + 100.0/abs(a)

def dec_to_implied(d):
    d = float(d)
    return 1.0/d if d > 0 else np.nan

def kelly_full(p, dec):
    # full Kelly on decimal odds
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b if b > 0 else 0.0
    return max(0.0, f)

def pick_best_line(group, score_col="ev"):
    # pick row with max score_col; break ties with higher true_edge
    g = group.copy()
    g = g.sort_values([score_col, "true_edge"], ascending=[False, False])
    return g.iloc[0]

# -----------------------------
# 1) Build GAME-LEVEL market "home win prob" + home spread
#    (Used ONLY to calibrate global scale s)
# -----------------------------
# Market home win prob from consensus ML
market_home = cons_ml.copy()
market_home = market_home.rename(columns={"consensus_prob":"p_mkt"})
market_home["is_home_team_row"] = (market_home["team"] == market_home["home_team"])
market_home = market_home[market_home["is_home_team_row"]].copy()
market_home = market_home[["home_team","away_team","p_mkt"]]

# Market home spread from consensus spread
mkt_sp = cons_sp.copy()
mkt_sp = mkt_sp.rename(columns={"cons_spread":"spread_home"})
mkt_sp["is_home_team_row"] = (mkt_sp["team"] == mkt_sp["home_team"])
mkt_sp = mkt_sp[mkt_sp["is_home_team_row"]].copy()
mkt_sp = mkt_sp[["home_team","away_team","spread_home"]]

cal = market_home.merge(mkt_sp, on=["home_team","away_team"], how="inner")

# Filter to stable calibration set (avoid extreme probs & near-zero logit instability)
cal = cal[(cal["p_mkt"] > 0.08) & (cal["p_mkt"] < 0.92)].copy()
cal["x"] = logit(cal["p_mkt"])
cal = cal[cal["x"].abs() > 0.15].copy()  # avoid tiny logits

# If spread_home is for HOME team, it should be negative when home favored.
# Model: p_home ≈ logistic(margin / s)  <=> margin = s*logit(p_home)
# Market relation: spread_home ≈ -margin  =>  -spread_home ≈ s*logit(p_home)
# Fit s via least squares: y = s*x where y = -spread_home
cal["y"] = -cal["spread_home"]

if len(cal) < 20:
    raise ValueError(f"Not enough games to calibrate mapping (n={len(cal)}). Check cons_ml/cons_sp for 2/28.")

s = (cal["x"] @ cal["y"]) / (cal["x"] @ cal["x"])
print(f"✅ Calibrated global spread scale s = {s:.3f} points per logit unit (from market ML↔spread calibration)")
print("Calibration sample size:", len(cal))

# -----------------------------
# 2) Build projected margin from YOUR model ML probabilities
# -----------------------------
# Need home win prob from ml_model (team==home_team)
mm = ml_model.copy()
mm = mm.rename(columns={"model_prob_ml":"p_model"})
mm["is_home_team_row"] = (mm["team"] == mm["home_team"])
home_model = mm[mm["is_home_team_row"]][["home_team","away_team","p_model"]].copy()

home_model["projected_margin_home_minus_away"] = s * logit(home_model["p_model"])

hybrid_margin_v2 = home_model[["home_team","away_team","projected_margin_home_minus_away"]].copy()
hybrid_margin_v2 = hybrid_margin_v2.rename(columns={"projected_margin_home_minus_away":"projected_margin"})
print("✅ hybrid_margin_v2 built:", hybrid_margin_v2.shape)

# Quick sanity: margins should not just mirror market spreads
print("\nProjected margin stats:")
print(hybrid_margin_v2["projected_margin"].describe())

# -----------------------------
# 3) Rebuild spread probabilities from projected margin + sigma (NO circular spread anchoring)
# -----------------------------
SIGMA_DEFAULT = 11.5

sp = odds_long[odds_long["market"].eq("spreads")].copy()

# Ensure decimals/implied exist (you already have them, but safe)
if "price_decimal" not in sp.columns:
    sp["price_decimal"] = sp["american_odds"].apply(american_to_decimal)
if "implied_prob" not in sp.columns:
    sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)

# Attach projected_margin
sp = sp.merge(hybrid_margin_v2, on=["home_team","away_team"], how="left")
if sp["projected_margin"].isna().mean() > 0.05:
    bad = sp[sp["projected_margin"].isna()][["home_team","away_team"]].drop_duplicates().head(10)
    raise ValueError(f"Too many spread rows missing projected_margin. Sample:\n{bad}")

# Convert projected_margin (home-away) -> mu_team (team-opp)
sp["mu_team"] = np.where(sp["team"].eq(sp["home_team"]),
                         sp["projected_margin"],
                         -sp["projected_margin"])

# Sigma (use column if exists else default)
if "sigma" in sp.columns:
    sp["sigma_used"] = sp["sigma"].fillna(SIGMA_DEFAULT)
else:
    sp["sigma_used"] = SIGMA_DEFAULT

# Cover probability: P( Normal(mu_team, sigma) + spread > 0 ) = Phi((mu_team + spread)/sigma)
sp["model_prob_spread"] = norm.cdf((sp["mu_team"] + sp["spread"]) / sp["sigma_used"])

# EV + Kelly
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread model rebuilt from ML-derived margin (non-circular).")
print("Spread prob std:", float(sp["model_prob_spread"].std()))
print("Spread prob min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))
print("Rows:", sp.shape)

# -----------------------------
# 4) TOP 10 SPREADS (NO DUPES | +EV) using best line per game+side
# -----------------------------
# No dupes by matchup+team
sp_pos = sp[sp["ev"] > 0].copy()

# pick best line per (home,away,team) by EV
sp_best = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

# Then NO DUPES by game (take max EV among the two sides)
sp_game_best = (
    sp_best.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp = sp_game_best.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp:", top10_sp.shape)
display_cols = ["home_team","away_team","team","book","spread","american_odds","price_decimal",
                "model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"]
print(top10_sp[display_cols].to_string(index=False))

# Save for next cells
globals()["sp_out"] = sp
globals()["top10_sp"] = top10_sp
globals()["hybrid_margin_v2"] = hybrid_margin_v2
globals()["SPREAD_SCALE_S"] = s

In [ ]:
import numpy as np
import pandas as pd

def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1.0 + a/100.0
    else:
        return 1.0 + 100.0/abs(a)

def dec_to_implied(d):
    d = float(d)
    return 1.0/d if d > 0 else np.nan

def kelly_full(p, dec):
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q)/b if b > 0 else 0.0
    return max(0.0, f)

def pick_best_line(group, score_col="ev"):
    g = group.copy().sort_values([score_col, "true_edge"], ascending=[False, False])
    return g.iloc[0]

# ML lines
ml = odds_long[odds_long["market"].eq("h2h")].copy()
if "price_decimal" not in ml.columns:
    ml["price_decimal"] = ml["american_odds"].apply(american_to_decimal)
if "implied_prob" not in ml.columns:
    ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)

# attach model probs
mm = ml_model.copy()
mm = mm.rename(columns={"model_prob_ml":"model_prob_ml"})
ml = ml.merge(mm[["home_team","away_team","team","model_prob_ml"]],
              on=["home_team","away_team","team"], how="left")

if ml["model_prob_ml"].isna().mean() > 0.05:
    bad = ml[ml["model_prob_ml"].isna()][["home_team","away_team","team"]].drop_duplicates().head(10)
    raise ValueError(f"Too many ML rows missing model_prob_ml. Sample:\n{bad}")

ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units_full_kelly"] = ml.apply(lambda r: kelly_full(r["model_prob_ml"], r["price_decimal"]), axis=1)

ml_pos = ml[ml["ev"] > 0].copy()

ml_best = (
    ml_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

ml_game_best = (
    ml_best.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_ml = ml_game_best.sort_values("ev", ascending=False).head(10).copy()

print("✅ top10_ml:", top10_ml.shape)
print(top10_ml[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","true_edge","ev","units_full_kelly"]].to_string(index=False))

globals()["top10_ml"] = top10_ml

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# =========================
# REQUIREMENTS
# =========================
for name in ["odds_long", "ml_model"]:
    if name not in globals():
        raise ValueError(f"Missing {name}. Run the Odds API cell and ESPN ML cell first.")

need_odds_cols = {"home_team","away_team","book","market","team","american_odds","price_decimal","implied_prob"}
missing = need_odds_cols - set(odds_long.columns)
if missing:
    raise ValueError(f"odds_long missing cols: {missing}")

need_ml_cols = {"home_team","away_team","team","model_prob_ml"}
missing = need_ml_cols - set(ml_model.columns)
if missing:
    raise ValueError(f"ml_model missing cols: {missing}")

BASE_SIGMA = 11.5  # (we'll replace with sigma_game later using totals)

# =========================
# HELPERS
# =========================
def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col, "true_edge"], ascending=[False, False])
    return g.iloc[0]

# =========================
# 1) CONSENSUS SPREAD PER TEAM (median across books)
# =========================
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()
if sp_lines.empty:
    raise ValueError("No spread rows found in odds_long (market=='spreads').")

cons_sp = (
    sp_lines.groupby(["home_team","away_team","team"], as_index=False)
    .agg(cons_spread=("spread","median"), book_count=("book","nunique"))
)

print("✅ cons_sp built:", cons_sp.shape)

# Keep only lines at the consensus spread for each team/game
sp = sp_lines.merge(cons_sp[["home_team","away_team","team","cons_spread"]],
                    on=["home_team","away_team","team"], how="left")
sp = sp[sp["spread"].round(3).eq(sp["cons_spread"].round(3))].copy()

print("✅ Spread rows at CONSENSUS spread only:", sp.shape)

# =========================
# 2) CALIBRATE: points per logit unit (s)
# Use games with both: model ML prob for home team + a market spread (consensus)
# =========================
# Build "market margin" from consensus spread (home team line is negative when home favored)
# We'll use the HOME team's consensus spread as the market margin signal:
home_cons_sp = cons_sp[cons_sp["team"].eq(cons_sp["home_team"])].copy()
home_cons_sp = home_cons_sp.rename(columns={"cons_spread":"market_margin"})[["home_team","away_team","market_margin"]]

# Get model home win prob from ml_model
home_ml = ml_model[ml_model["team"].eq(ml_model["home_team"])].copy()
home_ml = home_ml.rename(columns={"model_prob_ml":"home_win_prob"})[["home_team","away_team","home_win_prob"]]

cal = home_cons_sp.merge(home_ml, on=["home_team","away_team"], how="inner").dropna()

if len(cal) < 20:
    raise ValueError(f"Not enough calibration games with both spread+ESPN ML prob (n={len(cal)}).")

# Fit s via least squares: market_margin ≈ - s * logit(home_win_prob)
x = -logit(cal["home_win_prob"].values)  # predictor
y = cal["market_margin"].values          # response (points)
s = (x @ y) / (x @ x)

print(f"✅ Calibrated global spread scale s = {s:.3f} points per logit unit")
print("Calibration sample size:", len(cal))

# =========================
# 3) BUILD ML-DERIVED PROJECTED MARGIN (NON-CIRCULAR)
# margin (home-away) = - s * logit(P_home)
# =========================
hybrid_margin_v2 = home_ml.copy()
hybrid_margin_v2["projected_margin"] = -s * logit(hybrid_margin_v2["home_win_prob"].values)
hybrid_margin_v2 = hybrid_margin_v2[["home_team","away_team","projected_margin"]]

print("✅ hybrid_margin_v2 built:", hybrid_margin_v2.shape)
print("\nProjected margin stats:")
print(hybrid_margin_v2["projected_margin"].describe())

# =========================
# 4) COMPUTE COVER PROB FOR EACH CONSENSUS SPREAD LINE
# For a team with spread s_line:
# cover if (team_margin > -spread)
# team_margin ~ Normal(mu_team, sigma)
# => P = Phi( (mu_team + spread) / sigma )
# =========================
sp = sp.merge(hybrid_margin_v2, on=["home_team","away_team"], how="left")
if sp["projected_margin"].isna().any():
    bad = sp[sp["projected_margin"].isna()][["home_team","away_team"]].drop_duplicates().head(10)
    raise ValueError(f"Missing projected_margin for some spread rows. Sample:\n{bad}")

sp["mu_team"] = np.where(
    sp["team"].eq(sp["home_team"]),
    sp["projected_margin"],
    -sp["projected_margin"]
)
sp["sigma"] = BASE_SIGMA

sp["model_prob_spread"] = norm.cdf((sp["mu_team"] + sp["spread"]) / sp["sigma"])

# EV / Edge / Kelly
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread model rebuilt from ML-derived margin (non-circular).")
print("Spread prob std:", float(sp["model_prob_spread"].std()))
print("Spread prob min/max:",
      float(sp["model_prob_spread"].min()),
      float(sp["model_prob_spread"].max()))

# =========================
# 5) TOP 10 SPREADS (+EV, NO DUPES)
# =========================
sp_pos = sp[sp["ev"] > 0].copy()

sp_best = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_game_best = (
    sp_best.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp = sp_game_best.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp:", top10_sp.shape)
print(top10_sp[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# Save outputs
globals()["cons_sp"] = cons_sp
globals()["hybrid_margin_v2"] = hybrid_margin_v2
globals()["sp_out"] = sp
globals()["top10_sp"] = top10_sp
globals()["s_calibrated"] = s

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# =========================
# REQUIREMENTS
# =========================
for name in ["odds_long", "ml_model", "cons_sp"]:
    if name not in globals():
        raise ValueError(f"Missing {name}. Run Odds API + ESPN ML + cons_sp cells first.")

BASE_SIGMA = 11.5

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col, "true_edge"], ascending=[False, False])
    return g.iloc[0]

# =========================
# 1) SPREAD LINES @ CONSENSUS SPREAD ONLY
# =========================
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()
sp = sp_lines.merge(
    cons_sp[["home_team","away_team","team","cons_spread"]],
    on=["home_team","away_team","team"],
    how="left"
)
sp = sp[sp["spread"].round(3).eq(sp["cons_spread"].round(3))].copy()
print("✅ Spread rows at CONSENSUS spread only:", sp.shape)

# =========================
# 2) CALIBRATION (FIXED SIGN)
# We want:
#   expected_home_away_pts  ≈  s * logit(P_home)
#
# Market spread for home team is negative when home favored.
# So a proxy for expected home-away points is:  (-home_cons_spread)
# =========================
home_cons_sp = cons_sp[cons_sp["team"].eq(cons_sp["home_team"])].copy()
home_cons_sp = home_cons_sp.rename(columns={"cons_spread":"home_spread"})[
    ["home_team","away_team","home_spread"]
]

home_ml = ml_model[ml_model["team"].eq(ml_model["home_team"])].copy()
home_ml = home_ml.rename(columns={"model_prob_ml":"p_home"})[
    ["home_team","away_team","p_home"]
]

cal = home_cons_sp.merge(home_ml, on=["home_team","away_team"], how="inner").dropna()
if len(cal) < 20:
    raise ValueError(f"Not enough calibration games with both spread+ESPN ML prob (n={len(cal)}).")

x = logit(cal["p_home"].values)                  # logit(P_home)
y = (-cal["home_spread"].values)                 # expected home-away points proxy
s = (x @ y) / (x @ x)

print(f"✅ Recalibrated spread scale s = {s:.3f} points per logit unit (SIGN FIX)")
print("Calibration sample size:", len(cal))

# =========================
# 3) ML-DERIVED PROJECTED MARGIN (HOME - AWAY, in points)
# =========================
hybrid_margin_v2 = home_ml.copy()
hybrid_margin_v2["proj_margin_home_away"] = s * logit(hybrid_margin_v2["p_home"].values)  # + => home favored
hybrid_margin_v2 = hybrid_margin_v2[["home_team","away_team","proj_margin_home_away"]]

print("✅ hybrid_margin_v2 rebuilt:", hybrid_margin_v2.shape)
print(hybrid_margin_v2["proj_margin_home_away"].describe())

# =========================
# 4) COVER PROB FOR EACH TEAM/LINE
# Let team margin be (team - opponent).
# If bet team is home: mu = +proj_margin_home_away
# If bet team is away: mu = -proj_margin_home_away
#
# Spread is already signed for that team (favorite negative, dog positive).
# Cover condition: (mu + spread) > 0
# => P = Phi( (mu + spread)/sigma )
# =========================
sp = sp.merge(hybrid_margin_v2, on=["home_team","away_team"], how="left")
if sp["proj_margin_home_away"].isna().any():
    bad = sp[sp["proj_margin_home_away"].isna()][["home_team","away_team"]].drop_duplicates().head(10)
    raise ValueError(f"Missing proj_margin_home_away for some spread rows. Sample:\n{bad}")

sp["mu_team"] = np.where(
    sp["team"].eq(sp["home_team"]),
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)
sp["sigma"] = BASE_SIGMA
sp["model_prob_spread"] = norm.cdf((sp["mu_team"] + sp["spread"]) / sp["sigma"])

# Edge / EV / Kelly
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread probs after sign fix:")
print("std:", float(sp["model_prob_spread"].std()),
      "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))

# =========================
# 5) TOP 10 SPREADS (+EV, NO DUPES)
# =========================
sp_pos = sp[sp["ev"] > 0].copy()

sp_best = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_game_best = (
    sp_best.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp = sp_game_best.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp:", top10_sp.shape)
print(top10_sp[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "mu_team","sigma","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# =========================
# QUICK SANITY CHECK: WYOMING vs AIR FORCE
# =========================
chk = sp[(sp["home_team"].str.contains("Wyoming", na=False)) & (sp["away_team"].str.contains("Air Force", na=False))].copy()
if len(chk):
    print("\n===== WYOMING/AIR FORCE CHECK (sample) =====")
    print(chk[["team","spread","mu_team","model_prob_spread","implied_prob","ev"]].head(8).to_string(index=False))

# Save
globals()["hybrid_margin_v2"] = hybrid_margin_v2
globals()["sp_out_v2"] = sp
globals()["top10_sp_v2"] = top10_sp
globals()["s_calibrated_v2"] = s

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# =============== REQUIREMENTS ===============
for name in ["odds_long", "sp_out_v2", "hybrid_margin_v2"]:
    if name not in globals():
        raise ValueError(f"Missing {name}. Run the Spread v2 rebuild cell first.")

if "market" not in odds_long.columns:
    raise ValueError("odds_long missing 'market' column.")

if not (odds_long["market"] == "totals").any():
    raise ValueError(
        "No totals found in odds_long. Re-pull Odds API with markets=h2h,spreads,totals "
        "and ensure totals rows include a 'total' number."
    )

BASE_SIGMA = 11.5
TOTAL_REF = 145.0   # typical NCAAB total baseline (tunable)

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

# =============== 1) BUILD CONSENSUS TOTAL PER GAME ===============
tot = odds_long[odds_long["market"].eq("totals")].copy()

# Expect a numeric column named 'total' (or sometimes 'points').
# Try to auto-detect if needed.
total_col = None
for c in ["total", "points", "line", "total_points"]:
    if c in tot.columns:
        total_col = c
        break
if total_col is None:
    raise ValueError(f"Totals rows found, but no total column detected. Columns: {list(tot.columns)}")

tot[total_col] = pd.to_numeric(tot[total_col], errors="coerce")
tot = tot.dropna(subset=[total_col])

# consensus total = mean across books (you can use median if preferred)
cons_tot = (
    tot.groupby(["home_team","away_team"], as_index=False)[total_col]
       .mean()
       .rename(columns={total_col:"cons_total"})
)

print("✅ cons_tot built:", cons_tot.shape)
print(cons_tot["cons_total"].describe())

# =============== 2) PACE FACTOR + SIGMA_GAME ===============
# pace_factor scales volatility with total (higher total = more possessions/variance)
cons_tot["pace_factor"] = np.sqrt(cons_tot["cons_total"] / TOTAL_REF)
cons_tot["sigma_game"] = BASE_SIGMA * cons_tot["pace_factor"]

print("\n✅ sigma_game stats:")
print(cons_tot["sigma_game"].describe())

# =============== 3) REBUILD SPREAD PROBS USING SIGMA_GAME ===============
sp2 = sp_out_v2.copy()

sp2 = sp2.merge(cons_tot[["home_team","away_team","cons_total","sigma_game"]],
                on=["home_team","away_team"], how="left")

# fallback if totals missing for a game (rare)
sp2["sigma_game"] = sp2["sigma_game"].fillna(BASE_SIGMA)

# recompute cover prob using sigma_game
sp2["model_prob_spread_sigma"] = norm.cdf((sp2["mu_team"] + sp2["spread"]) / sp2["sigma_game"])

# recompute EV + Kelly
sp2["true_edge_sigma"] = sp2["model_prob_spread_sigma"] - sp2["implied_prob"]
sp2["ev_sigma"] = sp2["model_prob_spread_sigma"] * sp2["price_decimal"] - 1.0
sp2["units_full_kelly_sigma"] = sp2.apply(lambda r: kelly_full(r["model_prob_spread_sigma"], r["price_decimal"]), axis=1)

print("\n✅ Spread probs w/ sigma_game:")
print("std:", float(sp2["model_prob_spread_sigma"].std()),
      "| min/max:", float(sp2["model_prob_spread_sigma"].min()), float(sp2["model_prob_spread_sigma"].max()))

# =============== 4) TOP 10 SPREADS (NO DUPES) USING SIGMA VERSION ===============
def pick_best_line(group, score_col="ev_sigma"):
    g = group.sort_values([score_col, "true_edge_sigma"], ascending=[False, False])
    return g.iloc[0]

sp_pos = sp2[sp2["ev_sigma"] > 0].copy()

sp_best = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev_sigma")
)

sp_game_best = (
    sp_best.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev_sigma")
)

top10_sp_sigma = sp_game_best.sort_values("ev_sigma", ascending=False).head(10).copy()

print("\n✅ top10_sp_sigma:", top10_sp_sigma.shape)
print(top10_sp_sigma[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","model_prob_spread_sigma","implied_prob","true_edge_sigma","ev_sigma","units_full_kelly_sigma"
]].to_string(index=False))

# save
globals()["sp_out_v2_sigma"] = sp2
globals()["top10_sp_v2_sigma"] = top10_sp_sigma

In [ ]:
import os, requests, math
import numpy as np
import pandas as pd
from scipy.stats import norm

# =========================
# CONFIG
# =========================
ODDS_API_KEY = os.getenv("ODDS_API_KEY", "") or "10ceac94f9b9cb76f8c65510ca6df18f"

SPORT = "basketball_ncaab"
REGIONS = "us"
BOOKMAKERS = "draftkings,fanduel,betmgm,bovada,betonlineag,lowvig,betrivers,betus"
MARKETS = "h2h,spreads,totals"     # <-- totals added
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

TARGET_DATE_UTC = pd.Timestamp("2026-02-28", tz="UTC")  # change if needed

BASE_SIGMA = 11.5
TOTAL_REF = 145.0  # NCAAB baseline (tunable)

# =========================
# HELPERS
# =========================
def amer_to_dec(a):
    a = float(a)
    if a > 0:
        return 1 + a/100
    return 1 + 100/abs(a)

def dec_to_implied(d):
    d = float(d)
    return 1.0/d if d > 0 else np.nan

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col], ascending=[False])
    return g.iloc[0]

# =========================
# 1) PULL ODDS (WITH TOTALS)
# =========================
if not ODDS_API_KEY:
    raise ValueError("Missing ODDS_API_KEY. Put it in env var ODDS_API_KEY or set it in this cell.")

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": REGIONS,
    "markets": MARKETS,
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
    "bookmakers": BOOKMAKERS,
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
events = r.json()

rows = []
for ev in events:
    home = ev.get("home_team")
    away = ev.get("away_team")
    commence = ev.get("commence_time")
    for bk in ev.get("bookmakers", []):
        book = bk.get("key")
        for m in bk.get("markets", []):
            market = m.get("key")
            for out in m.get("outcomes", []):
                team = out.get("name")
                price = out.get("price")
                spread = out.get("point", np.nan) if market == "spreads" else np.nan
                total = out.get("point", np.nan) if market == "totals" else np.nan
                rows.append([home, away, commence, book, market, team, price, spread, total])

odds_long = pd.DataFrame(rows, columns=[
    "home_team","away_team","commence_time","book","market","team","american_odds","spread","total"
])

# normalize
odds_long["commence_time"] = pd.to_datetime(odds_long["commence_time"], utc=True, errors="coerce")
odds_long["american_odds"] = pd.to_numeric(odds_long["american_odds"], errors="coerce")
odds_long["spread"] = pd.to_numeric(odds_long["spread"], errors="coerce")
odds_long["total"] = pd.to_numeric(odds_long["total"], errors="coerce")

odds_long["price_decimal"] = odds_long["american_odds"].apply(amer_to_dec)
odds_long["implied_prob"] = odds_long["price_decimal"].apply(dec_to_implied)

# filter to target slate date (UTC)
odds_long = odds_long[odds_long["commence_time"].dt.date == TARGET_DATE_UTC.date()].copy()

print("✅ odds_long:", odds_long.shape)
print("Markets:", odds_long["market"].value_counts().to_dict())
print("Unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

if not (odds_long["market"] == "totals").any():
    raise ValueError("Totals still not present after pull. Check MARKETS / bookmakers / sport key.")

# =========================
# 2) CONSENSUS ML + CONSENSUS SPREAD + CONSENSUS TOTAL
# =========================
ml_lines = odds_long[odds_long["market"].eq("h2h")].copy()
sp_lines = odds_long[odds_long["market"].eq("spreads")].copy()
tot_lines = odds_long[odds_long["market"].eq("totals")].copy()

cons_ml = (
    ml_lines.groupby(["home_team","away_team","team"], as_index=False)
    .agg(consensus_prob=("implied_prob","mean"), book_count=("book","nunique"))
)

cons_sp = (
    sp_lines.groupby(["home_team","away_team","team"], as_index=False)
    .agg(consensus_prob=("implied_prob","mean"),
         book_count=("book","nunique"),
         cons_spread=("spread","mean"))
)

# consensus total (one per game; average of totals across books/outcomes)
# totals outcomes usually are "Over" and "Under" at same point, so averaging is fine.
cons_tot = (
    tot_lines.groupby(["home_team","away_team"], as_index=False)
    .agg(cons_total=("total","mean"))
)

print("✅ cons_ml:", cons_ml.shape, "| ✅ cons_sp:", cons_sp.shape, "| ✅ cons_tot:", cons_tot.shape)

# =========================
# 3) REQUIRE ML MODEL PROBS (FROM ESPN CELL)
# =========================
# Expecting: ml_model with columns [home_team, away_team, team, model_prob_ml]
if "ml_model" not in globals():
    raise ValueError("Missing ml_model (ESPN predictor). Run your ESPN predictor/ML model cell first.")

need = {"home_team","away_team","team","model_prob_ml"}
missing = need - set(ml_model.columns)
if missing:
    raise ValueError(f"ml_model missing columns: {missing}")

# =========================
# 4) CALIBRATE SPREAD SCALE s (ML↔market spread)
# =========================
# merge market spread (from cons_sp, use HOME row)
home_sp = cons_sp[cons_sp["team"] == cons_sp["home_team"]].copy()
home_sp = home_sp.merge(
    ml_model[ml_model["team"] == ml_model["home_team"]][["home_team","away_team","model_prob_ml"]],
    on=["home_team","away_team"],
    how="inner"
)

# logit
eps = 1e-6
p = home_sp["model_prob_ml"].clip(eps, 1-eps)
home_sp["logit_p"] = np.log(p/(1-p))

# market margin is negative for home favorite (home spread)
home_sp["market_margin"] = home_sp["cons_spread"]

# calibrate s via least squares: market_margin ≈ -s * logit_p  (SIGN FIX)
x = home_sp["logit_p"].values
y = home_sp["market_margin"].values
s = - (x*y).sum() / (x*x).sum()
print(f"✅ Recalibrated spread scale s = {s:.3f} points per logit unit (SIGN FIX)")
print("Calibration sample size:", len(home_sp))

# =========================
# 5) HYBRID_MARGIN_V2 (ML → projected margin)
# =========================
# projected home-away margin:
# proj_margin_home_away = -s * logit(p_home)
all_games = ml_model[ml_model["team"] == ml_model["home_team"]][["home_team","away_team","model_prob_ml"]].copy()
p_home = all_games["model_prob_ml"].clip(eps, 1-eps)
all_games["proj_margin_home_away"] = -s * np.log(p_home/(1-p_home))

hybrid_margin_v2 = all_games[["home_team","away_team","proj_margin_home_away"]].copy()
print("✅ hybrid_margin_v2 rebuilt:", hybrid_margin_v2.shape)
print(hybrid_margin_v2["proj_margin_home_away"].describe())

# =========================
# 6) SIGMA_GAME from totals (pace proxy)
# =========================
cons_tot["pace_factor"] = np.sqrt(cons_tot["cons_total"] / TOTAL_REF)
cons_tot["sigma_game"] = BASE_SIGMA * cons_tot["pace_factor"]
print("\n✅ sigma_game stats:")
print(cons_tot["sigma_game"].describe())

# =========================
# 7) BUILD SPREAD LINE PROBS @ CONSENSUS SPREAD ONLY
# =========================
# attach consensus spread to each spread line, then keep only lines at consensus spread point
sp = sp_lines.merge(
    cons_sp[["home_team","away_team","team","cons_spread"]],
    on=["home_team","away_team","team"],
    how="left"
)

sp = sp[sp["spread"].round(3) == sp["cons_spread"].round(3)].copy()
print("\n✅ Spread rows at CONSENSUS spread only:", sp.shape)

# attach projected margin + sigma_game
sp = sp.merge(hybrid_margin_v2, on=["home_team","away_team"], how="left")
sp = sp.merge(cons_tot[["home_team","away_team","cons_total","sigma_game"]], on=["home_team","away_team"], how="left")

sp["sigma_game"] = sp["sigma_game"].fillna(BASE_SIGMA)

# mu for a given team:
# if betting HOME: mu_team = proj_margin_home_away
# if betting AWAY: mu_team = -proj_margin_home_away
sp["mu_team"] = np.where(sp["team"] == sp["home_team"], sp["proj_margin_home_away"], -sp["proj_margin_home_away"])

# cover prob: P(mu + spread > 0) under Normal(mu, sigma_game)
sp["model_prob_spread"] = norm.cdf((sp["mu_team"] + sp["spread"]) / sp["sigma_game"])

# edges
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0
sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread probs after totals-sigma:")
print("std:", float(sp["model_prob_spread"].std()), "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))

# =========================
# 8) TOP 10 SPREADS (NO DUPES)
# =========================
sp_pos = sp[sp["ev"] > 0].copy()

sp_best_team = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp:", top10_sp.shape)
print(top10_sp[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# =========================
# 9) TOP 10 ML (NO DUPES)
# =========================
ml = ml_lines.merge(
    ml_model[["home_team","away_team","team","model_prob_ml"]],
    on=["home_team","away_team","team"],
    how="left"
)

ml = ml.dropna(subset=["model_prob_ml"]).copy()
ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0
ml["units_full_kelly"] = ml.apply(lambda r: kelly_full(r["model_prob_ml"], r["price_decimal"]), axis=1)

ml_pos = ml[ml["ev"] > 0].copy()

ml_best_team = (
    ml_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

ml_best_game = (
    ml_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_ml = ml_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_ml:", top10_ml.shape)
print(top10_ml[[
    "home_team","away_team","team","book","american_odds","price_decimal",
    "model_prob_ml","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save globals
globals()["odds_long"] = odds_long
globals()["cons_ml"] = cons_ml
globals()["cons_sp"] = cons_sp
globals()["cons_tot"] = cons_tot
globals()["hybrid_margin_v2"] = hybrid_margin_v2
globals()["sp_out_v2"] = sp
globals()["top10_sp_v2"] = top10_sp
globals()["top10_ml"] = top10_ml
print("\n✅ Saved: odds_long, cons_ml, cons_sp, cons_tot, hybrid_margin_v2, sp_out_v2, top10_sp_v2, top10_ml")

In [ ]:
import numpy as np
from scipy.stats import norm

# sp_out_v2 exists from the prior cell
assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "Run the Spread v2 build cell first."

sp = sp_out_v2.copy()

# Ensure sigma_game exists
if "sigma_game" not in sp.columns:
    sp["sigma_game"] = 11.5
sp["sigma_game"] = sp["sigma_game"].fillna(11.5)

# mu_team should be expected margin (TEAM - OPP)
# In your existing sp_out_v2, mu_team is already set that way. We'll trust it.
assert "mu_team" in sp.columns, "mu_team missing. Re-run the v2 build cell."

# ---- CORRECT COVER PROB ----
# Cover if (TEAM margin) > -spread
# p = 1 - Phi( (-spread - mu) / sigma )
sp["model_prob_spread"] = 1.0 - norm.cdf(((-sp["spread"]) - sp["mu_team"]) / sp["sigma_game"])

# Recompute edge/EV/Kelly
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("✅ FIXED spread probs:")
print("std:", float(sp["model_prob_spread"].std()),
      "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))

# rebuild top10_sp (no dupes)
sp_pos = sp[sp["ev"] > 0].copy()

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col], ascending=[False])
    return g.iloc[0]

sp_best_team = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp_v2 = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp_v2 (FIXED):", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
globals()["top10_sp_v2"] = top10_sp_v2
print("\n✅ Saved corrected: sp_out_v2, top10_sp_v2")

In [ ]:
import numpy as np
from scipy.stats import norm

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "Run your Spread v2 build cell first."
assert "hybrid_margin_v2" in globals() and len(hybrid_margin_v2) > 0, "hybrid_margin_v2 missing."
assert "cons_tot" in globals() and len(cons_tot) > 0, "cons_tot missing (totals sigma)."

sp = sp_out_v2.copy()

# --- Ensure we have proj margin (HOME - AWAY) on every row ---
# hybrid_margin_v2 should have: home_team, away_team, proj_margin_home_away (or similar)
hm = hybrid_margin_v2.copy()

# detect margin col
margin_col = None
for c in ["proj_margin_home_away", "projected_margin", "proj_margin"]:
    if c in hm.columns:
        margin_col = c
        break
if margin_col is None:
    raise ValueError("Couldn't find margin column in hybrid_margin_v2 (need proj_margin_home_away / projected_margin).")

hm = hm.rename(columns={margin_col: "proj_margin_home_away"})

sp = sp.merge(
    hm[["home_team","away_team","proj_margin_home_away"]],
    on=["home_team","away_team"],
    how="left"
)

if sp["proj_margin_home_away"].isna().any():
    bad = sp[sp["proj_margin_home_away"].isna()][["home_team","away_team"]].drop_duplicates().head(10)
    raise ValueError(f"Missing proj_margin_home_away for some games. Sample:\n{bad}")

# --- Attach sigma_game from totals (already built in your totals cell) ---
sp = sp.merge(cons_tot[["home_team","away_team","cons_total"]], on=["home_team","away_team"], how="left")

# If you already computed sigma_game earlier, keep it; else compute quick proxy
# (same general shape you had: small variation around ~11.5)
if "sigma_game" not in sp.columns or sp["sigma_game"].isna().all():
    base_sigma = 11.5
    league_total = float(cons_tot["cons_total"].median())
    sp["sigma_game"] = base_sigma * np.sqrt((sp["cons_total"] / league_total).clip(0.85, 1.20))

sp["sigma_game"] = sp["sigma_game"].fillna(11.5)

# --- CRITICAL FIX: mu_team must be TEAM - OPP ---
# If team == home_team => mu_team = (home - away)
# If team == away_team => mu_team = -(home - away)
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)

# --- Correct cover probability: TEAM covers if margin > -spread ---
sp["model_prob_spread"] = 1.0 - norm.cdf(((-sp["spread"]) - sp["mu_team"]) / sp["sigma_game"])

# Recompute EV/Kelly
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("✅ Spread probs AFTER mu_team sign fix:")
print("std:", float(sp["model_prob_spread"].std()),
      "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))

# Quick sanity spot-check for UConn/Seton Hall (should flip)
chk = sp[(sp["home_team"].str.contains("UConn", na=False)) & (sp["away_team"].str.contains("Seton Hall", na=False))][
    ["team","spread","proj_margin_home_away","mu_team","sigma_game","model_prob_spread","implied_prob","ev"]
].head(6)
print("\n===== UConn/Seton Hall check =====")
print(chk.to_string(index=False))

# rebuild top10 spreads (no dupes)
sp_pos = sp[sp["ev"] > 0].copy()

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col], ascending=[False])
    return g.iloc[0]

sp_best_team = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp_v2 = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp_v2 (AFTER FIX):", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
globals()["top10_sp_v2"] = top10_sp_v2
print("\n✅ Saved corrected: sp_out_v2, top10_sp_v2")

In [ ]:
import numpy as np
from scipy.stats import norm

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "Run your Spread v2 build cell first."
assert "hybrid_margin_v2" in globals() and len(hybrid_margin_v2) > 0, "hybrid_margin_v2 missing."
assert "cons_tot" in globals() and len(cons_tot) > 0, "cons_tot missing (totals sigma)."

sp = sp_out_v2.copy()
hm = hybrid_margin_v2.copy()

print("hybrid_margin_v2 columns:", list(hm.columns))

# ---- find the margin column in hybrid_margin_v2 ----
# We want the HOME - AWAY projected margin column.
cand = []
for c in hm.columns:
    lc = c.lower()
    if ("margin" in lc) and ("proj" in lc or "project" in lc or "hybrid" in lc):
        cand.append(c)

# also allow a generic "projected_margin" if that's all we have
fallback = [c for c in hm.columns if c.lower() in ["projected_margin", "proj_margin", "market_margin"]]
cand = cand + [c for c in fallback if c not in cand]

if not cand:
    raise ValueError(
        "Couldn't find any projected margin column in hybrid_margin_v2.\n"
        f"Columns were: {list(hm.columns)}"
    )

# pick the best candidate (prefer explicit home-away naming)
priority = []
for c in cand:
    lc = c.lower()
    score = 0
    if "home" in lc and "away" in lc:
        score += 3
    if "proj" in lc or "project" in lc:
        score += 2
    if "margin" in lc:
        score += 1
    priority.append((score, c))
priority.sort(reverse=True)
margin_col = priority[0][1]

print("✅ Using margin col from hybrid_margin_v2:", margin_col)

hm = hm.rename(columns={margin_col: "proj_margin_home_away"})

need_cols = ["home_team", "away_team", "proj_margin_home_away"]
missing = [c for c in need_cols if c not in hm.columns]
if missing:
    raise ValueError(f"hybrid_margin_v2 missing required columns after rename: {missing}")

# ---- merge margin into spreads ----
sp = sp.merge(
    hm[["home_team", "away_team", "proj_margin_home_away"]],
    on=["home_team", "away_team"],
    how="left"
)

# Handle accidental duplicate columns (proj_margin_home_away_x/y)
margin_candidates = [c for c in sp.columns if "proj_margin_home_away" in c]
if "proj_margin_home_away" not in sp.columns and margin_candidates:
    # choose non-null
    for c in margin_candidates:
        if sp[c].notna().any():
            sp["proj_margin_home_away"] = sp[c]
            break

if "proj_margin_home_away" not in sp.columns:
    raise ValueError(f"After merge, could not create proj_margin_home_away. Found: {margin_candidates}")

if sp["proj_margin_home_away"].isna().any():
    bad = sp[sp["proj_margin_home_away"].isna()][["home_team", "away_team"]].drop_duplicates().head(10)
    raise ValueError(f"Missing proj_margin_home_away for some games. Sample:\n{bad}")

# ---- totals + sigma_game ----
sp = sp.merge(cons_tot[["home_team", "away_team", "cons_total"]],
              on=["home_team", "away_team"], how="left")

if "sigma_game" not in sp.columns or sp["sigma_game"].isna().all():
    base_sigma = 11.5
    league_total = float(cons_tot["cons_total"].median())
    sp["sigma_game"] = base_sigma * np.sqrt((sp["cons_total"] / league_total).clip(0.85, 1.20))

sp["sigma_game"] = sp["sigma_game"].fillna(11.5)

# ---- CRITICAL: mu_team = TEAM - OPP ----
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)

# ---- cover prob: TEAM covers if margin > -spread ----
sp["model_prob_spread"] = 1.0 - norm.cdf(((-sp["spread"]) - sp["mu_team"]) / sp["sigma_game"])

# ---- EV / Kelly ----
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread probs AFTER mu_team sign fix:")
print("std:", float(sp["model_prob_spread"].std()),
      "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))

# ---- sanity check: pick one known game if present ----
sample = sp.head(8)[["home_team","away_team","team","spread","proj_margin_home_away","mu_team","sigma_game","model_prob_spread","implied_prob","ev"]]
print("\n===== SAMPLE ROWS =====")
print(sample.to_string(index=False))

# ---- rebuild top10 spreads (no dupes) ----
sp_pos = sp[sp["ev"] > 0].copy()

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col], ascending=[False])
    return g.iloc[0]

sp_best_team = (
    sp_pos.groupby(["home_team", "away_team", "team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team", "away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp_v2 = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp_v2 (AFTER FIX):", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
globals()["top10_sp_v2"] = top10_sp_v2
print("\n✅ Saved corrected: sp_out_v2, top10_sp_v2")

In [ ]:
import numpy as np
from scipy.stats import norm

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0
assert "hybrid_margin_v2" in globals() and len(hybrid_margin_v2) > 0
assert "cons_tot" in globals() and len(cons_tot) > 0

MODE = "STRICT"   # "STRICT" (drop missing) or "FILL" (use market spread for missing)
BASE_SIGMA = 11.5

sp = sp_out_v2.copy()
hm = hybrid_margin_v2.copy()

# merge margin
sp = sp.merge(
    hm[["home_team","away_team","proj_margin_home_away"]],
    on=["home_team","away_team"],
    how="left"
)

missing_games = sp[sp["proj_margin_home_away"].isna()][["home_team","away_team"]].drop_duplicates()
print(f"Missing proj_margin_home_away games: {len(missing_games)}")

if len(missing_games) > 0:
    print("Sample missing:")
    print(missing_games.head(12).to_string(index=False))

if MODE.upper() == "STRICT":
    before = len(sp)
    sp = sp[sp["proj_margin_home_away"].notna()].copy()
    after = len(sp)
    print(f"✅ STRICT: dropped rows without margin -> {before} → {after} rows")
else:
    # FILL: infer home-away margin from consensus spread (circular fallback)
    # cons_spread is HOME spread (negative means home favored)
    if "cons_sp" not in globals() or len(cons_sp) == 0:
        raise ValueError("MODE=FILL requires cons_sp (consensus spread per game/team). Build cons_sp first.")
    # build game-level cons_home_spread
    home_sp = (
        cons_sp[cons_sp["team"] == cons_sp["home_team"]][["home_team","away_team","cons_spread"]]
        .rename(columns={"cons_spread":"cons_home_spread"})
        .drop_duplicates()
    )
    sp = sp.merge(home_sp, on=["home_team","away_team"], how="left")
    # projected margin home-away ≈ -home_spread
    sp["proj_margin_home_away"] = sp["proj_margin_home_away"].fillna(-sp["cons_home_spread"])
    print("✅ FILL: backfilled missing margins from market spread where needed.")

# totals + sigma_game
sp = sp.merge(cons_tot[["home_team","away_team","cons_total"]],
              on=["home_team","away_team"], how="left")

league_total = float(cons_tot["cons_total"].median())
sp["sigma_game"] = BASE_SIGMA * np.sqrt((sp["cons_total"] / league_total).clip(0.85, 1.20))
sp["sigma_game"] = sp["sigma_game"].fillna(BASE_SIGMA)

# mu_team: TEAM - OPP
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)

# cover prob: TEAM covers if margin > -spread
sp["model_prob_spread"] = 1.0 - norm.cdf(((-sp["spread"]) - sp["mu_team"]) / sp["sigma_game"])

# EV / Kelly
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread probs AFTER rebuild:")
print("std:", float(sp["model_prob_spread"].std()),
      "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()),
      "| rows:", len(sp),
      "| unique games:", sp[["home_team","away_team"]].drop_duplicates().shape[0])

# rebuild top10 spreads (no dupes)
sp_pos = sp[sp["ev"] > 0].copy()

def pick_best_line(group, score_col="ev"):
    g = group.sort_values([score_col], ascending=[False])
    return g.iloc[0]

sp_best_team = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp_v2 = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp_v2:", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
globals()["top10_sp_v2"] = top10_sp_v2
print("\n✅ Saved: sp_out_v2, top10_sp_v2")

In [ ]:
import numpy as np
from scipy.stats import norm

# ---------- guards ----------
assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "sp_out_v2 missing. Run your spreads build cell first."
assert "hybrid_margin_v2" in globals() and len(hybrid_margin_v2) > 0, "hybrid_margin_v2 missing. Run the ML→margin (v2) cell first."
assert "cons_tot" in globals() and len(cons_tot) > 0, "cons_tot missing. Run totals consensus cell first."

MODE = "STRICT"   # STRICT drops games with no margin; FILL backfills from market spread (circular)
BASE_SIGMA = 11.5

sp = sp_out_v2.copy()
hm = hybrid_margin_v2.copy()

print("hybrid_margin_v2 columns:", list(hm.columns))

# ---------- auto-detect margin column ----------
margin_candidates = [c for c in hm.columns if c.lower() in ["proj_margin_home_away", "projected_margin", "proj_margin"]]
if not margin_candidates:
    # last resort: any column containing 'margin'
    margin_candidates = [c for c in hm.columns if "margin" in c.lower() and c not in ["home_team","away_team"]]

if not margin_candidates:
    raise ValueError("Could not find a margin column in hybrid_margin_v2. Expected proj_margin_home_away or projected_margin.")

margin_col = margin_candidates[0]
print(f"✅ Using margin col from hybrid_margin_v2: {margin_col}")

# standardize to proj_margin_home_away
hm2 = hm[["home_team","away_team", margin_col]].rename(columns={margin_col: "proj_margin_home_away"}).copy()

# ---------- merge margin ----------
sp = sp.merge(hm2, on=["home_team","away_team"], how="left")

if "proj_margin_home_away" not in sp.columns:
    raise ValueError("Merge failed: proj_margin_home_away not present after merge (unexpected).")

missing_games = sp.loc[sp["proj_margin_home_away"].isna(), ["home_team","away_team"]].drop_duplicates()
print(f"Missing proj_margin_home_away games: {len(missing_games)}")
if len(missing_games) > 0:
    print("Sample missing:")
    print(missing_games.head(12).to_string(index=False))

# ---------- handle missing ----------
if MODE.upper() == "STRICT":
    before = len(sp)
    sp = sp[sp["proj_margin_home_away"].notna()].copy()
    after = len(sp)
    print(f"✅ STRICT: dropped rows without margin -> {before} → {after} rows")
else:
    # FILL: infer home-away margin from market consensus spread (circular fallback)
    assert "cons_sp" in globals() and len(cons_sp) > 0, "MODE=FILL requires cons_sp built first."
    home_sp = (
        cons_sp[cons_sp["team"] == cons_sp["home_team"]][["home_team","away_team","cons_spread"]]
        .rename(columns={"cons_spread":"cons_home_spread"})
        .drop_duplicates()
    )
    sp = sp.merge(home_sp, on=["home_team","away_team"], how="left")
    sp["proj_margin_home_away"] = sp["proj_margin_home_away"].fillna(-sp["cons_home_spread"])
    print("✅ FILL: backfilled missing margins from market spread where needed.")

# ---------- totals + sigma_game ----------
sp = sp.merge(cons_tot[["home_team","away_team","cons_total"]], on=["home_team","away_team"], how="left")

league_total = float(cons_tot["cons_total"].median())
sp["sigma_game"] = BASE_SIGMA * np.sqrt((sp["cons_total"] / league_total).clip(0.85, 1.20))
sp["sigma_game"] = sp["sigma_game"].fillna(BASE_SIGMA)

# ---------- mu_team (TEAM - OPP) ----------
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)

# ---------- cover prob: TEAM covers if margin > -spread ----------
sp["model_prob_spread"] = 1.0 - norm.cdf(((-sp["spread"]) - sp["mu_team"]) / sp["sigma_game"])

# ---------- EV / Kelly ----------
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread probs AFTER rebuild:")
print(
    "std:", float(sp["model_prob_spread"].std()),
    "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()),
    "| rows:", len(sp),
    "| unique games:", sp[["home_team","away_team"]].drop_duplicates().shape[0]
)

# ---------- top10 spreads (no dupes) ----------
sp_pos = sp[sp["ev"] > 0].copy()

def pick_best_line(group, score_col="ev"):
    return group.sort_values(score_col, ascending=False).iloc[0]

sp_best_team = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp_v2 = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp_v2:", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
globals()["top10_sp_v2"] = top10_sp_v2
print("\n✅ Saved: sp_out_v2, top10_sp_v2")

In [ ]:
import numpy as np
from scipy.stats import norm

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "sp_out_v2 missing."
assert "hybrid_margin_v2" in globals() and len(hybrid_margin_v2) > 0, "hybrid_margin_v2 missing."
assert "cons_tot" in globals() and len(cons_tot) > 0, "cons_tot missing."

BASE_SIGMA = 11.5
MODE = "STRICT"   # STRICT drops games with no margin; FILL uses market fallback (circular)

sp = sp_out_v2.copy()
hm = hybrid_margin_v2.copy()[["home_team","away_team","proj_margin_home_away"]].copy()

# --- prevent column collision ---
for c in ["proj_margin_home_away", "proj_margin_home_away_x", "proj_margin_home_away_y"]:
    if c in sp.columns:
        sp = sp.drop(columns=[c])

# --- merge margin safely ---
sp = sp.merge(hm, on=["home_team","away_team"], how="left")

# if suffixes still somehow happened, resolve
cands = [c for c in sp.columns if c.startswith("proj_margin_home_away")]
if "proj_margin_home_away" not in sp.columns and len(cands) > 0:
    # pick the first candidate and standardize name
    keep = cands[0]
    sp = sp.rename(columns={keep: "proj_margin_home_away"})
    # drop any other duplicates
    drop_others = [c for c in cands if c != keep]
    if drop_others:
        sp = sp.drop(columns=drop_others)

if "proj_margin_home_away" not in sp.columns:
    raise ValueError(f"Still no proj_margin_home_away after merge. Candidates found: {cands}")

missing_games = sp.loc[sp["proj_margin_home_away"].isna(), ["home_team","away_team"]].drop_duplicates()
print(f"Missing proj_margin_home_away games: {len(missing_games)}")
if len(missing_games) > 0:
    print(missing_games.head(12).to_string(index=False))

# --- handle missing ---
if MODE.upper() == "STRICT":
    before = len(sp)
    sp = sp[sp["proj_margin_home_away"].notna()].copy()
    print(f"✅ STRICT drop -> {before} → {len(sp)} rows")
else:
    assert "cons_sp" in globals() and len(cons_sp) > 0, "MODE=FILL requires cons_sp."
    home_sp = (
        cons_sp[cons_sp["team"] == cons_sp["home_team"]][["home_team","away_team","cons_spread"]]
        .rename(columns={"cons_spread":"cons_home_spread"})
        .drop_duplicates()
    )
    sp = sp.merge(home_sp, on=["home_team","away_team"], how="left")
    sp["proj_margin_home_away"] = sp["proj_margin_home_away"].fillna(-sp["cons_home_spread"])
    print("✅ FILL backfilled missing margins from market spread where needed.")

# --- totals + sigma_game ---
sp = sp.merge(cons_tot[["home_team","away_team","cons_total"]], on=["home_team","away_team"], how="left")
league_total = float(cons_tot["cons_total"].median())

sp["sigma_game"] = BASE_SIGMA * np.sqrt((sp["cons_total"] / league_total).clip(0.85, 1.20))
sp["sigma_game"] = sp["sigma_game"].fillna(BASE_SIGMA)

# --- mu_team (TEAM - OPP) ---
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)

# --- cover prob: TEAM covers if margin > -spread ---
sp["model_prob_spread"] = 1.0 - norm.cdf(((-sp["spread"]) - sp["mu_team"]) / sp["sigma_game"])

# --- EV / Kelly ---
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread prob summary:",
      "std=", float(sp["model_prob_spread"].std()),
      "min/max=", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()),
      "rows=", len(sp))

# --- Top 10 spreads, no dupes ---
sp_pos = sp[sp["ev"] > 0].copy()

def pick_best_line(group, score_col="ev"):
    return group.sort_values(score_col, ascending=False).iloc[0]

sp_best_team = (
    sp_pos.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

sp_best_game = (
    sp_best_team.groupby(["home_team","away_team"], as_index=False, group_keys=False)
    .apply(pick_best_line, score_col="ev")
)

top10_sp_v2 = sp_best_game.sort_values("ev", ascending=False).head(10).copy()

print("\n✅ top10_sp_v2:", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

# save
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
globals()["top10_sp_v2"] = top10_sp_v2
print("\n✅ Saved: sp_out_v2, top10_sp_v2")

In [ ]:
import pandas as pd
import numpy as np

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "sp_out_v2 missing."
assert "cons_tot" in globals() and len(cons_tot) > 0, "cons_tot missing."

sp = sp_out_v2.copy()

# ---- find totals column in cons_tot ----
tot_cands = [c for c in cons_tot.columns if c.lower() in ["cons_total","consensus_total","total","points_total","tot"]]
if not tot_cands:
    # fallback: any numeric col besides keys
    num_cols = [c for c in cons_tot.columns if c not in ["home_team","away_team"] and pd.api.types.is_numeric_dtype(cons_tot[c])]
    if len(num_cols) == 1:
        tot_cands = num_cols
    else:
        raise ValueError(f"Couldn't identify totals column in cons_tot. Columns: {list(cons_tot.columns)}")

tot_col = tot_cands[0]
ct = cons_tot[["home_team","away_team", tot_col]].rename(columns={tot_col:"cons_total"}).drop_duplicates()

# ---- merge totals ----
# prevent collisions
for c in ["cons_total","cons_total_x","cons_total_y"]:
    if c in sp.columns:
        sp = sp.drop(columns=[c])

sp = sp.merge(ct, on=["home_team","away_team"], how="left")

# resolve suffixes if any
cands = [c for c in sp.columns if c.startswith("cons_total")]
if "cons_total" not in sp.columns and len(cands) > 0:
    keep = cands[0]
    sp = sp.rename(columns={keep:"cons_total"})
    drop_others = [c for c in cands if c != keep]
    if drop_others:
        sp = sp.drop(columns=drop_others)

# ---- diagnose if still missing ----
if "cons_total" not in sp.columns:
    raise ValueError(f"Totals merge failed: cons_total not present. cons_tot cols={list(cons_tot.columns)}")

missing_tot = sp["cons_total"].isna().mean()
print(f"Totals attached. Null cons_total rate: {missing_tot:.3%}")

if missing_tot > 0.05:
    miss = sp.loc[sp["cons_total"].isna(), ["home_team","away_team"]].drop_duplicates().head(12)
    print("\nSample games missing totals (keys mismatch):")
    print(miss.to_string(index=False))

# ---- build sigma_game (fallback to base sigma if missing totals) ----
BASE_SIGMA = 11.5
league_total = float(ct["cons_total"].median())

sp["sigma_game"] = BASE_SIGMA * np.sqrt((sp["cons_total"] / league_total).clip(0.85, 1.20))
sp["sigma_game"] = sp["sigma_game"].fillna(BASE_SIGMA)

print("\n✅ sigma_game stats:")
print(sp["sigma_game"].describe())

# save back
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2
print("\n✅ Updated sp_out_v2 now includes cons_total + sigma_game")

In [ ]:
import numpy as np
import pandas as pd
import math

# =========================
# SPREAD v2 REBUILD (mu_team + sigma_game -> cover prob -> top10)
# Requires: sp_out_v2, hybrid_margin_v2
# =========================

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "sp_out_v2 missing."
assert "hybrid_margin_v2" in globals() and len(hybrid_margin_v2) > 0, "hybrid_margin_v2 missing."

sp = sp_out_v2.copy()

# ---- identify margin column ----
margin_col_cands = [c for c in hybrid_margin_v2.columns if c.lower() in ["proj_margin_home_away","projected_margin","proj_margin","margin"]]
if not margin_col_cands:
    # fallback: any numeric col besides keys
    num_cols = [c for c in hybrid_margin_v2.columns if c not in ["home_team","away_team"] and pd.api.types.is_numeric_dtype(hybrid_margin_v2[c])]
    if len(num_cols) == 1:
        margin_col_cands = num_cols
    else:
        raise ValueError(f"Can't identify margin col in hybrid_margin_v2. cols={list(hybrid_margin_v2.columns)}")

MARGIN_COL = margin_col_cands[0]
hm = hybrid_margin_v2[["home_team","away_team", MARGIN_COL]].rename(columns={MARGIN_COL:"proj_margin_home_away"}).drop_duplicates()

# ---- attach margin ----
for c in ["proj_margin_home_away","proj_margin_home_away_x","proj_margin_home_away_y"]:
    if c in sp.columns:
        sp = sp.drop(columns=[c])

sp = sp.merge(hm, on=["home_team","away_team"], how="left")

# STRICT drop if some games don't have margin
missing = sp["proj_margin_home_away"].isna().mean()
print(f"Margin attached. Null proj_margin_home_away rate: {missing:.3%}")
if missing > 0:
    miss_games = sp.loc[sp["proj_margin_home_away"].isna(), ["home_team","away_team"]].drop_duplicates().head(15)
    print("\nSample missing margin games (dropping STRICT):")
    print(miss_games.to_string(index=False))
    sp = sp[sp["proj_margin_home_away"].notna()].copy()

# ---- make sure sigma_game exists (fallback) ----
if "sigma_game" not in sp.columns:
    BASE_SIGMA = 11.5
    sp["sigma_game"] = BASE_SIGMA

# ---- build mu_team for EACH ROW'S team given home/away margin ----
# proj_margin_home_away = (HOME - AWAY)
# If betting HOME team: mu_team = proj_margin_home_away
# If betting AWAY team: mu_team = -proj_margin_home_away
sp["mu_team"] = np.where(sp["team"] == sp["home_team"], sp["proj_margin_home_away"], -sp["proj_margin_home_away"])

# ---- cover probability for THIS row's spread ----
# Convention: spread is line for "team" (negative = favored, positive = dog)
# Bet wins if (team_margin + spread) > 0  i.e. team_margin > -spread
# So threshold = -spread
sp["threshold"] = -sp["spread"]

# Normal CDF without scipy
def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

# P(cover) = P( margin > threshold ) = 1 - Phi((threshold - mu)/sigma)
z = (sp["threshold"] - sp["mu_team"]) / sp["sigma_game"]
sp["model_prob_spread"] = 1.0 - z.apply(norm_cdf)

# ---- betting metrics ----
def dec_to_implied(dec):
    return 1.0/dec if dec and dec > 0 else np.nan

sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1.0

def kelly_full(p, dec):
    # b = dec - 1
    b = dec - 1.0
    q = 1.0 - p
    f = (b*p - q) / b if b > 0 else 0.0
    return max(0.0, f)

sp["units_full_kelly"] = sp.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread prob sanity:")
print("std:", float(sp["model_prob_spread"].std()), "| min/max:", float(sp["model_prob_spread"].min()), float(sp["model_prob_spread"].max()))
print("Rows:", sp.shape)

# ---- pick best line per game (NO DUPES) by EV ----
def pick_best_line(df, score_col="ev"):
    # highest EV, tie-breaker higher prob then better price
    df = df.sort_values([score_col, "model_prob_spread", "price_decimal"], ascending=[False, False, False])
    return df.iloc[0]

top10_sp_v2 = (
    sp.groupby(["home_team","away_team"], as_index=False, group_keys=False)
      .apply(pick_best_line, score_col="ev")
      .sort_values("ev", ascending=False)
      .head(10)
      .reset_index(drop=True)
)

globals()["sp_out_v2"] = sp
globals()["top10_sp_v2"] = top10_sp_v2

print("\n✅ top10_sp_v2:", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import math

# -----------------------------
# Helpers
# -----------------------------
def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def dec_to_implied(dec):
    return 1.0/dec if dec and dec > 0 else np.nan

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1.0 - p
    f = (b*p - q)/b
    return float(max(0.0, f))

def pick_best_line(df, score_col="ev"):
    df = df.sort_values([score_col, "model_prob_spread", "price_decimal"], ascending=[False, False, False])
    return df.iloc[0]

# -----------------------------
# Preconditions
# -----------------------------
for name in ["odds_long","cons_ml","cons_sp","cons_tot"]:
    assert name in globals(), f"Missing {name}."

# pick ML model df name
MLDF = None
for cand in ["ml_model","model_ml","model_ml_df"]:
    if cand in globals() and len(globals()[cand]) > 0:
        MLDF = globals()[cand].copy()
        break

# If no ml_model, we will just use consensus
if MLDF is None:
    print("⚠️ No ml_model found; using cons_ml only for margins.")
    MLDF = cons_ml.rename(columns={"consensus_prob":"model_prob_ml"})[["home_team","away_team","team","model_prob_ml"]].copy()

# standardize MLDF columns
if "model_prob_ml" not in MLDF.columns:
    # try common alt
    alt = [c for c in MLDF.columns if c in ["model_prob","model_prob_win","win_prob","prob"]]
    if not alt:
        raise ValueError(f"ml_model found but no prob col. cols={list(MLDF.columns)}")
    MLDF = MLDF.rename(columns={alt[0]:"model_prob_ml"})

MLDF = MLDF[["home_team","away_team","team","model_prob_ml"]].dropna().copy()

# -----------------------------
# Build p_home for ALL games in slate
# -----------------------------
games = (
    odds_long[odds_long["market"]=="spreads"][["home_team","away_team"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# p_home from model (team==home_team)
p_home = MLDF.merge(games, on=["home_team","away_team"], how="inner")
p_home = p_home[p_home["team"] == p_home["home_team"]][["home_team","away_team","model_prob_ml"]]
p_home = p_home.rename(columns={"model_prob_ml":"p_home_model"})

# fallback p_home from consensus
p_home_cons = cons_ml.merge(games, on=["home_team","away_team"], how="inner")
p_home_cons = p_home_cons[p_home_cons["team"] == p_home_cons["home_team"]][["home_team","away_team","consensus_prob"]]
p_home_cons = p_home_cons.rename(columns={"consensus_prob":"p_home_cons"})

pm = games.merge(p_home, on=["home_team","away_team"], how="left").merge(p_home_cons, on=["home_team","away_team"], how="left")
pm["p_home"] = pm["p_home_model"].fillna(pm["p_home_cons"])

null_rate = pm["p_home"].isna().mean()
print(f"p_home coverage: {(1-null_rate):.1%} | missing: {null_rate:.1%}")

# if still missing, fill with 0.5 (rare)
pm["p_home"] = pm["p_home"].fillna(0.5)

# -----------------------------
# Calibrate s from market spreads vs logit(p_home)
# We need market "cons_spread" for HOME team: negative when home is favorite.
# Approx relationship: expected_margin_home ≈ -cons_spread_home.
# So: -cons_spread_home ≈ s * logit(p_home)
# => s ≈ median( (-cons_spread_home) / logit(p_home) )
# -----------------------------
home_sp = cons_sp[["home_team","away_team","team","cons_spread"]].copy()
home_sp = home_sp[home_sp["team"] == home_sp["home_team"]][["home_team","away_team","cons_spread"]]
cal = pm.merge(home_sp, on=["home_team","away_team"], how="inner").copy()
cal["x"] = logit(cal["p_home"])
cal = cal[np.abs(cal["x"]) > 0.10].copy()  # avoid near 0 logits
cal["s_est"] = (-cal["cons_spread"]) / cal["x"]
cal = cal.replace([np.inf,-np.inf], np.nan).dropna(subset=["s_est"])

if len(cal) < 20:
    raise ValueError(f"Not enough calibration points for s (n={len(cal)}).")

s = float(cal["s_est"].median())
print(f"✅ Calibrated spread scale s = {s:.3f} points per logit (n={len(cal)})")

# -----------------------------
# Build hybrid_margin_all (HOME - AWAY) guaranteed orientation
# -----------------------------
pm["proj_margin_home_away"] = s * logit(pm["p_home"])
hybrid_margin_all = pm[["home_team","away_team","proj_margin_home_away"]].copy()

print("\nProjected margin stats (HOME-AWAY):")
print(hybrid_margin_all["proj_margin_home_away"].describe())

# -----------------------------
# Build sigma_game from totals (already free via Odds API)
# sigma_game = BASE_SIGMA * sqrt(total / league_median_total), clipped
# -----------------------------
BASE_SIGMA = 11.5
league_total = float(cons_tot["cons_total"].median())

tot = cons_tot[["home_team","away_team","cons_total"]].drop_duplicates()
sig = games.merge(tot, on=["home_team","away_team"], how="left")
sig["sigma_game"] = BASE_SIGMA * np.sqrt((sig["cons_total"] / league_total).clip(0.85, 1.20))
sig["sigma_game"] = sig["sigma_game"].fillna(BASE_SIGMA)

print("\n✅ sigma_game stats:")
print(sig["sigma_game"].describe())

# -----------------------------
# Rebuild spread lines at consensus spread only
# -----------------------------
sp_lines = odds_long[odds_long["market"]=="spreads"].copy()

# attach cons_spread for each row team
sp_lines = sp_lines.merge(
    cons_sp[["home_team","away_team","team","cons_spread"]],
    on=["home_team","away_team","team"],
    how="left"
)

# keep only rows exactly at consensus spread (within tolerance)
TOL = 1e-6
sp_lines = sp_lines[np.isfinite(sp_lines["cons_spread"])].copy()
sp_lines = sp_lines[np.isfinite(sp_lines["spread"])].copy()
sp_lines = sp_lines[np.abs(sp_lines["spread"] - sp_lines["cons_spread"]) <= TOL].copy()

print(f"\n✅ Spread rows at CONSENSUS spread only: {sp_lines.shape}")

# attach margin + totals/sigma
sp_lines = sp_lines.merge(hybrid_margin_all, on=["home_team","away_team"], how="left")
sp_lines = sp_lines.merge(sig[["home_team","away_team","cons_total","sigma_game"]], on=["home_team","away_team"], how="left")

# mu_team for each row team (team - opp)
sp_lines["mu_team"] = np.where(sp_lines["team"] == sp_lines["home_team"],
                               sp_lines["proj_margin_home_away"],
                               -sp_lines["proj_margin_home_away"])

sp_lines["threshold"] = -sp_lines["spread"]
z = (sp_lines["threshold"] - sp_lines["mu_team"]) / sp_lines["sigma_game"]
sp_lines["model_prob_spread"] = 1.0 - z.apply(norm_cdf)

# metrics
sp_lines["implied_prob"] = sp_lines["price_decimal"].apply(dec_to_implied)
sp_lines["true_edge"] = sp_lines["model_prob_spread"] - sp_lines["implied_prob"]
sp_lines["ev"] = sp_lines["model_prob_spread"] * sp_lines["price_decimal"] - 1.0
sp_lines["units_full_kelly"] = sp_lines.apply(lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread prob sanity after rebuild:")
print("std:", float(sp_lines["model_prob_spread"].std()),
      "| min/max:", float(sp_lines["model_prob_spread"].min()), float(sp_lines["model_prob_spread"].max()))

# -----------------------------
# top10_sp_v2 (best line per game by EV)
# -----------------------------
top10_sp_v2 = (
    sp_lines.groupby(["home_team","away_team"], as_index=False, group_keys=False)
            .apply(pick_best_line, score_col="ev")
            .sort_values("ev", ascending=False)
            .head(10)
            .reset_index(drop=True)
)

globals()["hybrid_margin_all"] = hybrid_margin_all
globals()["sp_out_v2"] = sp_lines
globals()["top10_sp_v2"] = top10_sp_v2

print("\n✅ top10_sp_v2:", top10_sp_v2.shape)
print(top10_sp_v2[[
    "home_team","away_team","team","book","spread","american_odds","price_decimal",
    "cons_total","sigma_game","mu_team","model_prob_spread","implied_prob","true_edge","ev","units_full_kelly"
]].to_string(index=False))

In [ ]:
import numpy as np
import math

def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

assert "sp_out_v2" in globals() and len(sp_out_v2) > 0, "sp_out_v2 missing."

sp = sp_out_v2.copy()

# Strict definition:
# M = TEAM - OPP
# Cover if M > -spread
sp["threshold_check"] = -sp["spread"]
sp["z_check"] = (sp["threshold_check"] - sp["mu_team"]) / sp["sigma_game"]
sp["model_prob_spread_check"] = 1.0 - sp["z_check"].apply(norm_cdf)

sp["delta"] = sp["model_prob_spread_check"] - sp["model_prob_spread"]

print("=== GLOBAL CHECK ===")
print("max |delta|:", float(np.max(np.abs(sp["delta"]))), "mean delta:", float(sp["delta"].mean()))
print("prob std (check):", float(sp["model_prob_spread_check"].std()),
      "min/max:", float(sp["model_prob_spread_check"].min()), float(sp["model_prob_spread_check"].max()))

# Pull the top row from your top10_sp_v2 (Northwestern St +7) and show all internals
row = sp[
    (sp["home_team"]=="Texas A&M-CC Islanders") &
    (sp["away_team"]=="Northwestern St Demons") &
    (sp["team"]=="Northwestern St Demons") &
    (np.isclose(sp["spread"], 7.0))
].head(1)

print("\n=== NWST +7 INSIDES ===")
if len(row)==0:
    print("Row not found (book/line might differ). Showing closest match instead.")
    row = sp[(sp["home_team"]=="Texas A&M-CC Islanders") &
             (sp["away_team"]=="Northwestern St Demons") &
             (sp["team"]=="Northwestern St Demons")].sort_values("ev", ascending=False).head(1)

r = row.iloc[0]
print(f"Game: {r['home_team']} vs {r['away_team']}")
print(f"Bet: {r['team']} {r['spread']} | book={r['book']} dec={r['price_decimal']:.4f}")
print(f"mu_team (TEAM-OPP): {r['mu_team']:.6f}")
print(f"threshold = -spread: {r['threshold_check']:.6f}")
print(f"sigma_game: {r['sigma_game']:.6f}")
print(f"z = (thr - mu)/sigma: {r['z_check']:.6f}")
print(f"Pcover stored: {r['model_prob_spread']:.6f}")
print(f"Pcover check : {r['model_prob_spread_check']:.6f}")
print(f"delta: {r['delta']:+.6f}")

# save back if you want the corrected column
sp_out_v2 = sp
globals()["sp_out_v2"] = sp_out_v2

In [ ]:
import numpy as np
import pandas as pd

# =========================
# SPREAD v2 DIAGNOSTICS CELL
# Drop-in (does NOT replace prior cells)
# Requires: cons_sp (has cons_spread), hybrid_margin_v2 (has proj_margin_home_away),
#          cons_ml (optional), odds_long (for books/lines)
# =========================

# --- Helpers ---
def require(name):
    if name not in globals():
        raise ValueError(f"Missing {name}. Run the v2 build cell that creates it.")
    return globals()[name]

cons_sp = require("cons_sp")
hybrid_margin_v2 = require("hybrid_margin_v2")
odds_long = require("odds_long")

# -------- 1) Build a GAME-LEVEL table (one row per matchup) --------
# cons_sp is TEAM-level (two rows per game). We want home-team spread per game.
# By your convention: cons_spread is from team perspective.
# We'll pivot to get home spread and away spread.

sp_piv = cons_sp.pivot_table(
    index=["home_team","away_team"],
    columns="team",
    values="cons_spread",
    aggfunc="mean"
).reset_index()

# Determine the home team spread column dynamically
# (home spread should be the column with name == home_team)
def get_home_spread(row):
    ht = row["home_team"]
    at = row["away_team"]
    home_sp = row.get(ht, np.nan)
    away_sp = row.get(at, np.nan)
    return home_sp, away_sp

home_spreads = []
away_spreads = []
for _, r in sp_piv.iterrows():
    hs, aws = get_home_spread(r)
    home_spreads.append(hs)
    away_spreads.append(aws)

game_sp = sp_piv[["home_team","away_team"]].copy()
game_sp["market_spread_home"] = home_spreads
game_sp["market_spread_away"] = away_spreads

# -------- 2) Attach MODEL projected margin (HOME - AWAY) --------
m = hybrid_margin_v2.copy()

# Find margin col (you already have proj_margin_home_away)
margin_col = None
for c in ["proj_margin_home_away", "projected_margin", "proj_margin"]:
    if c in m.columns:
        margin_col = c
        break
if margin_col is None:
    raise ValueError(f"hybrid_margin_v2 has no margin column. Columns: {list(m.columns)}")

m = m.rename(columns={margin_col: "model_margin_home"})

game = game_sp.merge(
    m[["home_team","away_team","model_margin_home"]],
    on=["home_team","away_team"],
    how="left"
)

# -------- 3) Compute disagreement metrics --------
# Market implied margin for home is approximately -market_spread_home
# (if home is -7, home is expected +7 vs away)
game["market_margin_home"] = -game["market_spread_home"]
game["margin_diff"] = game["model_margin_home"] - game["market_margin_home"]

# Useful magnitudes
game["abs_margin_diff"] = game["margin_diff"].abs()

# -------- 4) Summaries --------
print("\n===== SPREAD v2 DIAGNOSTICS =====")
print("Games in cons_sp:", game_sp.shape[0])
print("Games with model margin:", int(game["model_margin_home"].notna().sum()), "of", len(game))
print("Missing model margin:", int(game["model_margin_home"].isna().sum()))

print("\nTop 15 disagreements (|model_margin - market_margin|):")
display_cols = ["home_team","away_team","market_spread_home","market_margin_home","model_margin_home","margin_diff","abs_margin_diff"]
print(game.sort_values("abs_margin_diff", ascending=False)[display_cols].head(15).to_string(index=False))

# -------- 5) Deep-dive function for any matchup --------
def spread_v2_insides(home_team, away_team):
    row = game[(game["home_team"]==home_team) & (game["away_team"]==away_team)]
    if row.empty:
        print("No game found in game table.")
        return
    row = row.iloc[0]
    print("\n========== GAME INSIDES ==========")
    print(f"{home_team} vs {away_team}")
    print(f"Market spread (home): {row['market_spread_home']}")
    print(f"Market implied margin (home): {row['market_margin_home']:+.3f}")
    print(f"Model margin (home-away): {row['model_margin_home']:+.3f}")
    print(f"DIFF (model - market): {row['margin_diff']:+.3f}")
    print("=================================")

    # Show all consensus spread rows for this game
    cs = cons_sp[(cons_sp["home_team"]==home_team) & (cons_sp["away_team"]==away_team)].copy()
    if not cs.empty:
        print("\n--- cons_sp rows (team spreads) ---")
        print(cs[["team","cons_spread","consensus_prob","book_count"]].to_string(index=False))

    # Show example best book lines at consensus spread (from odds_long)
    sp_lines = odds_long[(odds_long["market"]=="spreads") &
                         (odds_long["home_team"]==home_team) &
                         (odds_long["away_team"]==away_team)].copy()
    if not sp_lines.empty:
        # attach cons spread per team
        sp_lines = sp_lines.merge(
            cs[["team","cons_spread"]].rename(columns={"cons_spread":"cons_spread_team"}),
            on="team",
            how="left"
        )
        sp_lines_cons = sp_lines[np.isclose(sp_lines["spread"], sp_lines["cons_spread_team"], atol=1e-6)]
        print("\n--- sample book spread lines at consensus spread ---")
        if sp_lines_cons.empty:
            print("(No exact-match lines at consensus spread; showing closest 10 by |spread-cons_spread|)")
            sp_lines["abs_diff"] = (sp_lines["spread"] - sp_lines["cons_spread_team"]).abs()
            print(sp_lines.sort_values("abs_diff")[["book","team","spread","american_odds","price_decimal","cons_spread_team","abs_diff"]].head(10).to_string(index=False))
        else:
            print(sp_lines_cons[["book","team","spread","american_odds","price_decimal"]].head(15).to_string(index=False))

# -------- 6) Default deep-dive (edit these names if you want) --------
# If your earlier "NWST +7" game exists in this slate, set it here.
# Otherwise, pick any matchup from the Top 15 disagreements printed above.

# Example (comment out if not present):
# spread_v2_insides("Texas A&M-CC Islanders", "Northwestern St Demons")

print("\n✅ Diagnostic table built: `game` (per-game spreads + margins + diff).")
print("Use: spread_v2_insides(home_team, away_team) to drill into any matchup.\n")

In [ ]:
import numpy as np
import pandas as pd

# =========================
# FULL ML -> MARGIN REBUILD (SIGN FIXED)
# =========================

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

# --- Preconditions ---
for name in ["odds_long","cons_ml","cons_sp"]:
    if name not in globals():
        raise ValueError(f"Missing {name}")

# ---------- 1) Build p_home ----------
games = (
    odds_long[odds_long["market"]=="spreads"][["home_team","away_team"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Try to use model ML first
MLDF = None
for cand in ["ml_model","model_ml","model_ml_df"]:
    if cand in globals():
        MLDF = globals()[cand].copy()
        break

if MLDF is None:
    print("⚠️ No ml_model found. Using consensus ML only.")
    MLDF = cons_ml.rename(columns={"consensus_prob":"model_prob_ml"})

if "model_prob_ml" not in MLDF.columns:
    alt = [c for c in MLDF.columns if c in ["model_prob","win_prob","prob"]]
    if not alt:
        raise ValueError("No ML probability column found.")
    MLDF = MLDF.rename(columns={alt[0]:"model_prob_ml"})

MLDF = MLDF[["home_team","away_team","team","model_prob_ml"]]

p_home_model = (
    MLDF[MLDF["team"]==MLDF["home_team"]]
    [["home_team","away_team","model_prob_ml"]]
    .rename(columns={"model_prob_ml":"p_home_model"})
)

p_home_cons = (
    cons_ml[cons_ml["team"]==cons_ml["home_team"]]
    [["home_team","away_team","consensus_prob"]]
    .rename(columns={"consensus_prob":"p_home_cons"})
)

pm = games.merge(p_home_model, on=["home_team","away_team"], how="left")
pm = pm.merge(p_home_cons, on=["home_team","away_team"], how="left")

pm["p_home"] = pm["p_home_model"].fillna(pm["p_home_cons"])
pm["p_home"] = pm["p_home"].fillna(0.5)

print("p_home coverage:", round(100*(1-pm["p_home"].isna().mean()),2), "%")

# ---------- 2) Calibrate s ----------
home_sp = cons_sp[cons_sp["team"]==cons_sp["home_team"]][
    ["home_team","away_team","cons_spread"]
]

cal = pm.merge(home_sp, on=["home_team","away_team"], how="inner").copy()

cal["x"] = logit(cal["p_home"])
cal = cal[np.abs(cal["x"]) > 0.10].copy()

# IMPORTANT: SIGN FIX
cal["s_est"] = (cal["cons_spread"]) / cal["x"]

cal = cal.replace([np.inf,-np.inf], np.nan).dropna(subset=["s_est"])

if len(cal) < 20:
    raise ValueError("Not enough calibration points.")

s = float(cal["s_est"].median())

print(f"✅ Recalibrated spread scale s = {s:.3f} points per logit (SIGN FIXED)")

# ---------- 3) Build Corrected Margin ----------
pm["proj_margin_home_away"] = s * logit(pm["p_home"])

hybrid_margin_v2 = pm[["home_team","away_team","proj_margin_home_away"]].copy()

globals()["hybrid_margin_v2"] = hybrid_margin_v2

print("\nProjected margin stats (HOME - AWAY):")
print(hybrid_margin_v2["proj_margin_home_away"].describe())

In [ ]:
import numpy as np
import pandas as pd

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

# --- requires: odds_long, cons_sp, pm with p_home OR rebuild p_home quickly ---
if "pm" not in globals() or "p_home" not in pm.columns:
    # rebuild minimal pm if needed
    games = odds_long[odds_long["market"]=="spreads"][["home_team","away_team"]].drop_duplicates()
    p_home_cons = (
        cons_ml[cons_ml["team"]==cons_ml["home_team"]][["home_team","away_team","consensus_prob"]]
        .rename(columns={"consensus_prob":"p_home"})
    )
    pm = games.merge(p_home_cons, on=["home_team","away_team"], how="left")
    pm["p_home"] = pm["p_home"].fillna(0.5)

# Home spread consensus (negative = home favored)
home_sp = cons_sp[cons_sp["team"]==cons_sp["home_team"]][["home_team","away_team","cons_spread"]].copy()

cal = pm.merge(home_sp, on=["home_team","away_team"], how="inner").copy()
cal["x"] = logit(cal["p_home"])

# Use MARKET MARGIN (HOME-AWAY) instead of spread:
cal["market_margin_home"] = -cal["cons_spread"]

# filter weak logits
cal = cal[np.abs(cal["x"]) > 0.10].copy()

# positive scale now
cal["s_est"] = cal["market_margin_home"] / cal["x"]
cal = cal.replace([np.inf,-np.inf], np.nan).dropna(subset=["s_est"])

if len(cal) < 20:
    raise ValueError("Not enough calibration points after filtering.")

s = float(cal["s_est"].median())
print(f"✅ Recalibrated spread scale s = {s:.3f} points per logit (correct sign)")

# Build corrected projected margin (HOME - AWAY)
pm["proj_margin_home_away"] = s * logit(pm["p_home"])

hybrid_margin_v2 = pm[["home_team","away_team","proj_margin_home_away"]].copy()
globals()["hybrid_margin_v2"] = hybrid_margin_v2

print("\nProjected margin stats (HOME - AWAY):")
print(hybrid_margin_v2["proj_margin_home_away"].describe())

In [ ]:
# Force rebuild hybrid_margin_v2 from current pm (with correct sign)
assert "pm" in globals(), "pm not found. Re-run the calibration cell first."

hybrid_margin_v2 = pm[["home_team","away_team","proj_margin_home_away"]].copy()

# Overwrite globally
globals()["hybrid_margin_v2"] = hybrid_margin_v2

print("✅ hybrid_margin_v2 overwritten.")
print(hybrid_margin_v2["proj_margin_home_away"].describe())

# Quick spot check: Wyoming
test = hybrid_margin_v2[
    (hybrid_margin_v2["home_team"]=="Wyoming Cowboys") &
    (hybrid_margin_v2["away_team"]=="Air Force Falcons")
]

print("\nWyoming test margin:")
print(test)

In [ ]:
import numpy as np
import pandas as pd

assert "cons_sp" in globals()
assert "hybrid_margin_v2" in globals()

# Build home-level consensus spread table
home_sp = cons_sp[cons_sp["team"] == cons_sp["home_team"]][
    ["home_team","away_team","cons_spread"]
].copy()

home_sp["market_margin_home"] = -home_sp["cons_spread"]

# Merge current corrected margin
game = home_sp.merge(
    hybrid_margin_v2,
    on=["home_team","away_team"],
    how="left"
)

game = game.rename(columns={
    "proj_margin_home_away":"model_margin_home"
})

game["margin_diff"] = game["model_margin_home"] - game["market_margin_home"]
game["abs_margin_diff"] = game["margin_diff"].abs()

print("\n===== CLEAN DIAGNOSTICS (USING CURRENT MARGIN) =====")
print("Games with model margin:", game["model_margin_home"].notna().sum(), "of", len(game))

print("\nTop 10 disagreements:")
print(
    game.sort_values("abs_margin_diff", ascending=False)[
        ["home_team","away_team",
         "market_margin_home","model_margin_home",
         "margin_diff","abs_margin_diff"]
    ].head(10).to_string(index=False)
)

# Wyoming sanity check
wy = game[
    (game["home_team"]=="Wyoming Cowboys") &
    (game["away_team"]=="Air Force Falcons")
]

print("\nWyoming diagnostic row:")
print(wy)

In [ ]:
import numpy as np
import pandas as pd
import math

# ==============================
# CLEAN SPREAD v2 REBUILD
# ==============================

def norm_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def dec_to_implied(dec):
    return 1.0 / dec if dec and dec > 0 else np.nan

def kelly_full(p, dec):
    b = dec - 1.0
    if b <= 0:
        return 0.0
    q = 1 - p
    return max(0.0, (b*p - q) / b)

assert "odds_long" in globals()
assert "cons_sp" in globals()
assert "hybrid_margin_v2" in globals()

# 1️⃣ Keep only spreads
sp = odds_long[odds_long["market"]=="spreads"].copy()

# 2️⃣ Attach consensus spread (team-level)
sp = sp.merge(
    cons_sp[["home_team","away_team","team","cons_spread"]],
    on=["home_team","away_team","team"],
    how="left"
)

# Keep only rows exactly at consensus spread
sp = sp[np.isclose(sp["spread"], sp["cons_spread"], atol=1e-6)].copy()

print("Consensus spread rows:", sp.shape)

# 3️⃣ Attach corrected margin (HOME - AWAY)
sp = sp.merge(
    hybrid_margin_v2,
    on=["home_team","away_team"],
    how="left"
)

if sp["proj_margin_home_away"].isna().any():
    raise ValueError("Missing projected margin for some games.")

# 4️⃣ Attach sigma_game if exists, else fallback
if "sigma_game" in globals():
    sig = sigma_game[["home_team","away_team","sigma_game"]] \
          if "sigma_game" in globals() else None

if "cons_tot" in globals():
    BASE_SIGMA = 11.5
    league_total = float(cons_tot["cons_total"].median())
    sp = sp.merge(
        cons_tot[["home_team","away_team","cons_total"]],
        on=["home_team","away_team"],
        how="left"
    )
    sp["sigma_game"] = BASE_SIGMA * np.sqrt(
        (sp["cons_total"] / league_total).clip(0.85,1.20)
    )
else:
    sp["sigma_game"] = 11.5

# 5️⃣ Compute mu_team (TEAM - OPP)
sp["mu_team"] = np.where(
    sp["team"] == sp["home_team"],
    sp["proj_margin_home_away"],
    -sp["proj_margin_home_away"]
)

# 6️⃣ Cover condition: TEAM margin > -spread
sp["threshold"] = -sp["spread"]
z = (sp["threshold"] - sp["mu_team"]) / sp["sigma_game"]
sp["model_prob_spread"] = 1 - z.apply(norm_cdf)

# 7️⃣ Betting metrics
sp["implied_prob"] = sp["price_decimal"].apply(dec_to_implied)
sp["true_edge"] = sp["model_prob_spread"] - sp["implied_prob"]
sp["ev"] = sp["model_prob_spread"] * sp["price_decimal"] - 1
sp["units_full_kelly"] = sp.apply(
    lambda r: kelly_full(r["model_prob_spread"], r["price_decimal"]),
    axis=1
)

print("\nSpread prob sanity:")
print("std:", float(sp["model_prob_spread"].std()),
      "| min/max:",
      float(sp["model_prob_spread"].min()),
      float(sp["model_prob_spread"].max()))

# 8️⃣ Best line per game (no dupes)
def pick_best(df):
    df = df.sort_values(["ev","model_prob_spread","price_decimal"],
                        ascending=[False,False,False])
    return df.iloc[0]

top10_sp_v2 = (
    sp.groupby(["home_team","away_team"], as_index=False, group_keys=False)
      .apply(pick_best)
      .sort_values("ev", ascending=False)
      .head(10)
      .reset_index(drop=True)
)

globals()["sp_out_v2"] = sp
globals()["top10_sp_v2"] = top10_sp_v2

print("\nTOP 10 SPREADS:")
print(top10_sp_v2[[
    "home_team","away_team","team","book",
    "spread","american_odds","model_prob_spread",
    "implied_prob","ev","units_full_kelly"
]].to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

assert "sp_out_v2" in globals()
sp = sp_out_v2.copy()

# market spread from cons_sp (home perspective)
market_home = cons_sp[["home_team","away_team","cons_spread"]].drop_duplicates().copy()
market_home["market_spread_home"] = market_home["cons_spread"]  # home team line
market_home["market_margin_home"] = -market_home["market_spread_home"]

# attach market margin + model margin
sp = sp.merge(
    hybrid_margin_v2[["home_team","away_team","proj_margin_home_away"]],
    on=["home_team","away_team"],
    how="left"
).merge(
    market_home[["home_team","away_team","market_spread_home","market_margin_home"]],
    on=["home_team","away_team"],
    how="left"
)

# diagnostics: neutral model margin with big market spread
sp["abs_model_margin"] = sp["proj_margin_home_away"].abs()
sp["abs_market_spread"] = sp["market_spread_home"].abs()

neutral = sp[
    (sp["abs_model_margin"] <= 0.50) &  # model basically says "pick'em"
    (sp["abs_market_spread"] >= 5.0)    # market says it's not
][["home_team","away_team","proj_margin_home_away","market_spread_home","market_margin_home"]].drop_duplicates()

print("Neutral-margin / big-spread games:", len(neutral))
display(neutral.head(20))

# ✅ Guardrail filter (recommended):
# Drop games where model margin is near 0 but market spread is large (usually missing/weak signal)
DROP_NEUTRAL = True

if DROP_NEUTRAL:
    before = len(sp)
    sp = sp[~(
        (sp["abs_model_margin"] <= 0.50) &
        (sp["abs_market_spread"] >= 5.0)
    )].copy()
    print(f"Applied guardrail: {before} -> {len(sp)} rows")

# rebuild top10 spreads after guardrail
def pick_best(df):
    return df.sort_values(["ev","model_prob_spread","price_decimal"],
                          ascending=[False,False,False]).iloc[0]

top10_sp_v2_guard = (
    sp.groupby(["home_team","away_team"], as_index=False, group_keys=False)
      .apply(pick_best)
      .sort_values("ev", ascending=False)
      .head(10)
      .reset_index(drop=True)
)

globals()["top10_sp_v2_guard"] = top10_sp_v2_guard
globals()["sp_out_v2_guard"] = sp

print("\n✅ TOP 10 SPREADS (GUARDED):")
print(top10_sp_v2_guard[[
    "home_team","away_team","team","book","spread","american_odds",
    "model_prob_spread","implied_prob","ev","units_full_kelly"
]].to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

assert "sp_out_v2" in globals(), "Missing sp_out_v2"
assert "cons_sp" in globals(), "Missing cons_sp"
assert "hybrid_margin_v2" in globals(), "Missing hybrid_margin_v2"

sp = sp_out_v2.copy()

# --- market spread home / margin home ---
market_home = cons_sp[["home_team","away_team","cons_spread"]].drop_duplicates().copy()
market_home["market_spread_home"] = market_home["cons_spread"]
market_home["market_margin_home"] = -market_home["market_spread_home"]

# --- merge model margin + market margin ---
sp = sp.merge(
    hybrid_margin_v2[["home_team","away_team","proj_margin_home_away"]],
    on=["home_team","away_team"],
    how="left"
).merge(
    market_home[["home_team","away_team","market_spread_home","market_margin_home"]],
    on=["home_team","away_team"],
    how="left"
)

# --- normalize margin col name (handle suffixes) ---
margin_candidates = [c for c in sp.columns if "proj_margin_home_away" in c]
if not margin_candidates:
    raise ValueError(f"Could not find proj_margin_home_away after merge. Columns: {list(sp.columns)}")

# prefer exact name if present, else take first candidate and rename
if "proj_margin_home_away" not in sp.columns:
    sp = sp.rename(columns={margin_candidates[0]: "proj_margin_home_away"})

# --- diagnostics: neutral model margin with big market spread ---
sp["abs_model_margin"] = sp["proj_margin_home_away"].abs()
sp["abs_market_spread"] = sp["market_spread_home"].abs()

neutral = sp[
    (sp["abs_model_margin"] <= 0.50) &
    (sp["abs_market_spread"] >= 5.0)
][["home_team","away_team","proj_margin_home_away","market_spread_home","market_margin_home"]].drop_duplicates()

print("Neutral-margin / big-spread games:", len(neutral))
display(neutral.head(25))

# --- guardrail filter ---
DROP_NEUTRAL = True
if DROP_NEUTRAL:
    before = len(sp)
    sp = sp[~(
        (sp["abs_model_margin"] <= 0.50) &
        (sp["abs_market_spread"] >= 5.0)
    )].copy()
    print(f"Applied guardrail: {before} -> {len(sp)} rows")

# --- rebuild top10 spreads (no dupes) ---
def pick_best(df):
    return df.sort_values(["ev","model_prob_spread","price_decimal"],
                          ascending=[False,False,False]).iloc[0]

top10_sp_v2_guard = (
    sp.groupby(["home_team","away_team"], as_index=False, group_keys=False)
      .apply(pick_best)
      .sort_values("ev", ascending=False)
      .head(10)
      .reset_index(drop=True)
)

globals()["top10_sp_v2_guard"] = top10_sp_v2_guard
globals()["sp_out_v2_guard"] = sp

print("\n✅ TOP 10 SPREADS (GUARDED):")
print(top10_sp_v2_guard[[
    "home_team","away_team","team","book","spread","american_odds",
    "model_prob_spread","implied_prob","ev","units_full_kelly"
]].to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

# --- REQUIREMENTS ---
assert "top10_sp_v2_guard" in globals(), "Run the guardrail spread cell first (top10_sp_v2_guard)."
assert "top10_ml" in globals() or "top10_ml_v2" in globals(), "Need ML top10 df (top10_ml or top10_ml_v2)."
assert "ml_out" in globals() or "ml_best" in globals() or "ml_lines" in globals(), "Need ML lines output df in memory."

# pick whichever ML top10 exists
ml_top = globals().get("top10_ml_v2", globals().get("top10_ml")).copy()

# ----------------------------
# 1) ML GUARDRAIL (optional)
# ----------------------------
# Drop super-longshot bombs by default; you can change this later.
MAX_DOG_ODDS = 600   # allow dogs up to +600
MIN_FKELLY = 0.01    # minimum full Kelly fraction (1% of bankroll)

ml_top_guard = ml_top.copy()
ml_top_guard = ml_top_guard[ml_top_guard["american_odds"].between(-10000, MAX_DOG_ODDS)].copy()
ml_top_guard = ml_top_guard[ml_top_guard["units_full_kelly"] >= MIN_FKELLY].copy()

# If this is too strict and returns too few, relax automatically
if len(ml_top_guard) < 5:
    MIN_FKELLY = 0.005
    ml_top_guard = ml_top.copy()
    ml_top_guard = ml_top_guard[ml_top_guard["american_odds"].between(-10000, MAX_DOG_ODDS)].copy()
    ml_top_guard = ml_top_guard[ml_top_guard["units_full_kelly"] >= MIN_FKELLY].copy()

ml_top_guard = ml_top_guard.sort_values("ev", ascending=False).head(10).reset_index(drop=True)
globals()["top10_ml_guard"] = ml_top_guard

print("✅ ML guard params:", {"MAX_DOG_ODDS": MAX_DOG_ODDS, "MIN_FKELLY": MIN_FKELLY})
print("✅ TOP 10 ML (GUARDED):")
print(ml_top_guard[[
    "home_team","away_team","team","book","american_odds","model_prob_ml","implied_prob","ev","units_full_kelly"
]].to_string(index=False))

# -------------------------------------------
# 2) BUILD TIER A CARD (5–10 plays, de-dupe)
# -------------------------------------------
sp_top = top10_sp_v2_guard.copy()

# standardize columns so we can concat
sp_card = sp_top.assign(
    market="SPREAD",
    bet=lambda d: d["team"] + " " + d["spread"].astype(str),
    model_prob=lambda d: d["model_prob_spread"],
    units=lambda d: d["units_full_kelly"]
)[["market","home_team","away_team","bet","book","american_odds","price_decimal","model_prob","implied_prob","ev","units"]]

ml_card = ml_top_guard.assign(
    market="ML",
    bet=lambda d: d["team"] + " ML",
    model_prob=lambda d: d["model_prob_ml"],
    units=lambda d: d["units_full_kelly"]
)[["market","home_team","away_team","bet","book","american_odds","price_decimal","model_prob","implied_prob","ev","units"]]

combo = pd.concat([sp_card, ml_card], ignore_index=True)

# de-dupe by game: keep highest EV play per matchup
combo_best = (combo.sort_values(["ev","units"], ascending=False)
                   .groupby(["home_team","away_team"], as_index=False)
                   .head(1))

# Now take top 5–10 overall by EV
TARGET_N = 8
tierA = combo_best.sort_values("ev", ascending=False).head(TARGET_N).reset_index(drop=True)

globals()["tierA_final"] = tierA

print("\n✅ TIER A FINAL (ML + SPREAD) —", len(tierA), "plays")
print(tierA.to_string(index=False))

# ------------------------
# 3) DISCORD CARD OUTPUT
# ------------------------
def fmt_odds(a):
    a = int(a)
    return f"+{a}" if a > 0 else str(a)

lines = []
lines.append("NCAAB — 2/28 TIER A (ML + SPREAD) | FULL KELLY")
lines.append("")
for i, r in tierA.iterrows():
    lines.append(f"{i+1}) {r['home_team']} vs {r['away_team']}")
    lines.append(f"Bet: {r['bet']}")
    lines.append(f"Book: {r['book']} | Odds: {fmt_odds(r['american_odds'])}")
    lines.append(f"Model: {r['model_prob']*100:.1f}% | Implied: {r['implied_prob']*100:.1f}%")
    lines.append(f"EV: {r['ev']:.3f} | Units (Full Kelly): {r['units']:.2f}u")
    lines.append("")

discord_card = "\n".join(lines).strip()
globals()["discord_card_tierA"] = discord_card

print("\n===== DISCORD CARD (COPY) =====\n")
print(discord_card)

In [ ]:
# ==============================
# FORMAT HELPERS (RUN ONCE)
# ==============================

def fmt_american(a):
    a = int(a)
    return f"+{a}" if a > 0 else str(a)

def fmt_spread(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def build_discord_card(df, title):
    lines = []
    lines.append(title)
    lines.append("")

    for i, r in df.iterrows():
        lines.append(f"{i+1}) {r['home_team']} vs {r['away_team']}")

        # Spread or ML formatting
        if r["market"] == "SPREAD":
            bet_text = f"{r['team']} {fmt_spread(r['spread'])}"
        else:
            bet_text = f"{r['team']} ML"

        lines.append(f"Bet: {bet_text}")
        lines.append(f"Book: {r['book']} | Odds: {fmt_american(r['american_odds'])}")
        lines.append(f"Model: {r['model_prob']*100:.1f}% | Implied: {r['implied_prob']*100:.1f}%")
        lines.append(f"EV: {r['ev']:.3f} | Units (Full Kelly): {r['units']:.2f}u")
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
discord_card_clean = build_discord_card(
    tierA_final,
    "NCAAB — 2/28 TIER A (ML + SPREAD) | FULL KELLY"
)

print(discord_card_clean)

In [ ]:
# ==============================
# DISCORD CARD BUILDER (ROBUST)
# ==============================

def fmt_american(a):
    a = int(a)
    return f"+{a}" if a > 0 else str(a)

def fmt_spread(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def _get(row, *keys, default=None):
    for k in keys:
        if k in row and pd.notna(row[k]):
            return row[k]
    return default

def build_discord_card(df, title):
    lines = [title, ""]

    df = df.reset_index(drop=True).copy()

    for i in range(len(df)):
        r = df.iloc[i]

        home = _get(r, "home_team", default="HOME")
        away = _get(r, "away_team", default="AWAY")
        market = _get(r, "market", default="").upper()

        # Prefer 'bet' (already fully formatted); else build from team/spread
        bet_text = _get(r, "bet", default=None)
        if bet_text is None:
            team = _get(r, "team", default=None)
            spread = _get(r, "spread", default=None)
            if team is not None and spread is not None and market == "SPREAD":
                bet_text = f"{team} {fmt_spread(spread)}"
            elif team is not None:
                bet_text = f"{team} ML"
            else:
                bet_text = "—"

        book = _get(r, "book", default="—")
        amer = _get(r, "american_odds", default=None)
        odds_txt = fmt_american(amer) if amer is not None else "—"

        # model prob column varies between builds
        model_prob = _get(r, "model_prob", "model_prob_ml", "model_prob_spread", default=None)
        implied = _get(r, "implied_prob", default=None)
        ev = _get(r, "ev", default=None)
        units = _get(r, "units", "units_full_kelly", default=None)

        model_txt = f"{float(model_prob)*100:.1f}%" if model_prob is not None else "—"
        implied_txt = f"{float(implied)*100:.1f}%" if implied is not None else "—"
        ev_txt = f"{float(ev):.3f}" if ev is not None else "—"
        units_txt = f"{float(units):.2f}u" if units is not None else "—"

        lines.append(f"{i+1}) {home} vs {away}")
        lines.append(f"Bet: {bet_text}")
        lines.append(f"Book: {book} | Odds: {odds_txt}")
        lines.append(f"Model: {model_txt} | Implied: {implied_txt}")
        lines.append(f"EV: {ev_txt} | Units (Full Kelly): {units_txt}")
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
discord_card_clean = build_discord_card(
    tierA_final,
    "NCAAB — 2/28 TIER A (ML + SPREAD) | FULL KELLY"
)
print(discord_card_clean)

In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# Helpers
# -----------------------------
def amer_to_dec(a):
    a = float(a)
    if a > 0:
        return 1.0 + a/100.0
    return 1.0 + 100.0/abs(a)

def dec_to_implied(dec):
    return 1.0/float(dec)

def kelly_full(p, dec):
    # for decimal odds: b = dec-1, q=1-p => f* = (bp - q)/b
    b = float(dec) - 1.0
    if b <= 0:
        return 0.0
    f = (b*float(p) - (1.0-float(p))) / b
    return max(0.0, f)

def parlay_metrics(df):
    p = df["model_prob_ml"].prod()
    dec = df["price_decimal"].prod()
    ev = p*dec - 1.0
    return p, dec, ev

def pick_best_line(group):
    # pick by max EV, tiebreaker by higher price (optional)
    g = group.copy()
    g["ev"] = g["model_prob_ml"] * g["price_decimal"] - 1.0
    g = g.sort_values(["ev","price_decimal"], ascending=[False, False])
    return g.head(1)

# -----------------------------
# Build ML menu (best line per team/game)
# Requires:
#   odds_long with market == 'h2h' rows
#   ml_model with columns: home_team, away_team, team, model_prob_ml
# -----------------------------
assert "odds_long" in globals(), "Missing odds_long"
assert "ml_model" in globals(), "Missing ml_model (needs model_prob_ml per team/game)"

ml = odds_long[odds_long["market"].eq("h2h")].copy()

# normalize decimal if not present
if "price_decimal" not in ml.columns:
    ml["price_decimal"] = ml["american_odds"].apply(amer_to_dec)

# attach model probs
ml = ml.merge(
    ml_model[["home_team","away_team","team","model_prob_ml"]],
    on=["home_team","away_team","team"],
    how="left"
)

# keep only rows with model probs
ml = ml[ml["model_prob_ml"].notna()].copy()

# best book line per (game, team)
ml_best = (
    ml.groupby(["home_team","away_team","team"], as_index=False)
      .apply(pick_best_line)
      .reset_index(drop=True)
)

# compute edges
ml_best["implied_prob"] = ml_best["price_decimal"].apply(dec_to_implied)
ml_best["true_edge"] = ml_best["model_prob_ml"] - ml_best["implied_prob"]
ml_best["ev"] = ml_best["model_prob_ml"] * ml_best["price_decimal"] - 1.0
ml_best["kelly_full"] = ml_best.apply(lambda r: kelly_full(r["model_prob_ml"], r["price_decimal"]), axis=1)

print("✅ ML menu built:", ml_best.shape)
display(ml_best.sort_values("model_prob_ml", ascending=False).head(10))

# -----------------------------
# USER CONTROLS (tune these)
# -----------------------------
MAX_DOG_ODDS = 600      # allow dogs up to +600
MIN_PROB_SAFE = 0.62    # for "most likely" parlay pool
MIN_EDGE_EV = 0.005     # for "+EV" pool
MIN_KELLY = 0.01        # avoid microscopic edges for parlays
N_LEGS_SAFE = 4
N_LEGS_LOTTO = 16

# -----------------------------
# Filter pools
# -----------------------------
ml_best["american_odds"] = ml_best["american_odds"].astype(float)

pool_most_likely = ml_best[
    (ml_best["model_prob_ml"] >= MIN_PROB_SAFE) &
    (ml_best["american_odds"] >= -10000) &   # just a sanity cap
    (ml_best["american_odds"] <= MAX_DOG_ODDS)
].copy()

pool_ev = ml_best[
    (ml_best["true_edge"] >= MIN_EDGE_EV) &
    (ml_best["kelly_full"] >= MIN_KELLY) &
    (ml_best["american_odds"] <= MAX_DOG_ODDS)
].copy()

print("\nPool sizes:",
      "most_likely =", len(pool_most_likely),
      "| ev_pool =", len(pool_ev))

# -----------------------------
# Top singles lists
# -----------------------------
top_prob = ml_best.sort_values("model_prob_ml", ascending=False).head(20)
top_ev = ml_best.sort_values("ev", ascending=False).head(20)

print("\n🏁 TOP 20 MOST PROBABLE ML (singles)")
display(top_prob[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob"]])

print("\n💰 TOP 20 +EV ML (singles)")
display(top_ev[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","true_edge","ev","kelly_full"]])

# -----------------------------
# Build a 4-leg "most likely to hit" parlay (greedy by prob)
# -----------------------------
def build_greedy_parlay(df_pool, n_legs, mode="prob"):
    used_games = set()
    legs = []
    df = df_pool.copy()
    if mode == "prob":
        df = df.sort_values("model_prob_ml", ascending=False)
    else:
        df = df.sort_values("ev", ascending=False)
    for _, r in df.iterrows():
        game_key = (r["home_team"], r["away_team"])
        if game_key in used_games:
            continue
        used_games.add(game_key)
        legs.append(r)
        if len(legs) == n_legs:
            break
    return pd.DataFrame(legs)

parlay4_prob = build_greedy_parlay(pool_most_likely, N_LEGS_SAFE, mode="prob")
p4, d4, ev4 = parlay_metrics(parlay4_prob) if len(parlay4_prob)==N_LEGS_SAFE else (np.nan,np.nan,np.nan)

print(f"\n✅ 4-LEG MOST LIKELY (n={len(parlay4_prob)})  P(hit)={p4:.3f}  Dec={d4:.2f}  EV={ev4:.3f}")
display(parlay4_prob[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","true_edge","ev"]])

# -----------------------------
# Build a 4-leg "+EV" parlay (greedy by EV)
# -----------------------------
parlay4_ev = build_greedy_parlay(pool_ev, N_LEGS_SAFE, mode="ev")
p4e, d4e, ev4e = parlay_metrics(parlay4_ev) if len(parlay4_ev)==N_LEGS_SAFE else (np.nan,np.nan,np.nan)

print(f"\n✅ 4-LEG +EV (n={len(parlay4_ev)})  P(hit)={p4e:.3f}  Dec={d4e:.2f}  EV={ev4e:.3f}")
display(parlay4_ev[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","true_edge","ev"]])

# -----------------------------
# 16-leg lotto builder
# Two modes:
#   - "prob": highest hit rate
#   - "ev": higher EV but still guardrailed
# -----------------------------
def random_search_best_parlay(df_pool, n_legs, iters=5000, mode="prob", seed=7):
    rng = np.random.default_rng(seed)
    df = df_pool.copy()
    # ensure unique games possible
    games = df[["home_team","away_team"]].drop_duplicates()
    if len(games) < n_legs:
        print("Not enough unique games in pool for that many legs.")
        return pd.DataFrame()
    best = None
    best_score = -1e18

    # index by game, then sample one team-leg per game
    by_game = list(df.groupby(["home_team","away_team"]))
    for _ in range(iters):
        sampled_games = rng.choice(len(by_game), size=n_legs, replace=False)
        legs = []
        for gi in sampled_games:
            gdf = by_game[gi][1]
            # choose one leg from that game (bias toward better, but random)
            if mode == "prob":
                weights = np.clip(gdf["model_prob_ml"].values, 1e-6, None)
            else:
                weights = np.clip((gdf["ev"].values - gdf["ev"].min() + 1e-6), 1e-6, None)
            weights = weights / weights.sum()
            pick_idx = rng.choice(len(gdf), p=weights)
            legs.append(gdf.iloc[pick_idx])
        legs_df = pd.DataFrame(legs)

        phit, dec, ev = parlay_metrics(legs_df)
        score = phit if mode=="prob" else ev
        if score > best_score:
            best_score = score
            best = legs_df.copy()

    return best

# Lotto pools: you can pick which pool feeds it
lotto_pool = pool_ev.copy()
if len(lotto_pool) < N_LEGS_LOTTO:
    # fallback if EV pool too small
    lotto_pool = pool_most_likely.copy()

parlay16 = random_search_best_parlay(lotto_pool, N_LEGS_LOTTO, iters=7000, mode="prob")
if len(parlay16)==N_LEGS_LOTTO:
    p16, d16, ev16 = parlay_metrics(parlay16)
    print(f"\n🎰 16-LEG LOTTO (mode=prob)  P(hit)={p16:.6f}  Dec={d16:.2f}  EV={ev16:.3f}")
    display(parlay16[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","true_edge","ev"]])
else:
    print("\nCould not build a full 16-leg with current pool/filters — relax thresholds.")

In [ ]:
# ==============================
# MIXED PARLAY BUILDER
# ==============================

assert "pool_most_likely" in globals()
assert "pool_ev" in globals()
assert "parlay_metrics" in globals()

# ----- Controls -----
ANCHOR_MIN_PROB = 0.65
EV_MIN_EDGE = 0.01
ANCHOR_COUNT_4 = 2
EV_COUNT_4 = 2

ANCHOR_COUNT_8 = 3
EV_COUNT_8 = 5

ANCHOR_COUNT_16 = 5
EV_COUNT_16 = 11

# ----- Build anchor pool -----
anchors = pool_most_likely[
    pool_most_likely["model_prob_ml"] >= ANCHOR_MIN_PROB
].copy().sort_values("model_prob_ml", ascending=False)

ev_legs = pool_ev[
    pool_ev["true_edge"] >= EV_MIN_EDGE
].copy().sort_values("ev", ascending=False)

def build_mixed(anchor_df, ev_df, n_anchor, n_ev):
    used_games = set()
    legs = []

    # take anchors first
    for _, r in anchor_df.iterrows():
        g = (r["home_team"], r["away_team"])
        if g in used_games:
            continue
        legs.append(r)
        used_games.add(g)
        if len(legs) == n_anchor:
            break

    # then EV legs
    for _, r in ev_df.iterrows():
        g = (r["home_team"], r["away_team"])
        if g in used_games:
            continue
        legs.append(r)
        used_games.add(g)
        if len(legs) == n_anchor + n_ev:
            break

    return pd.DataFrame(legs)

# ---- 4 LEG MIX ----
mixed4 = build_mixed(anchors, ev_legs, ANCHOR_COUNT_4, EV_COUNT_4)
p4, d4, ev4 = parlay_metrics(mixed4)

print(f"\n🔥 MIXED 4-LEG (2 anchors + 2 EV)")
print(f"P(hit)={p4:.3f} | Decimal={d4:.2f} | EV={ev4:.3f}")
display(mixed4[["home_team","away_team","team","american_odds","model_prob_ml","true_edge","ev"]])

# ---- 8 LEG MIX ----
mixed8 = build_mixed(anchors, ev_legs, ANCHOR_COUNT_8, EV_COUNT_8)
p8, d8, ev8 = parlay_metrics(mixed8)

print(f"\n🔥 MIXED 8-LEG (3 anchors + 5 EV)")
print(f"P(hit)={p8:.4f} | Decimal={d8:.2f} | EV={ev8:.3f}")
display(mixed8[["home_team","away_team","team","american_odds","model_prob_ml","true_edge","ev"]])

# ---- 16 LEG LOTTO MIX ----
mixed16 = build_mixed(anchors, ev_legs, ANCHOR_COUNT_16, EV_COUNT_16)
p16, d16, ev16 = parlay_metrics(mixed16)

print(f"\n🎰 MIXED 16-LEG LOTTO")
print(f"P(hit)={p16:.6f} | Decimal={d16:.2f} | EV={ev16:.3f}")
display(mixed16[["home_team","away_team","team","american_odds","model_prob_ml","true_edge","ev"]])

In [ ]:
import os

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"  # your key
os.environ["ODDS_API_KEY"] = ODDS_API_KEY
print("✅ ODDS_API_KEY set (length):", len(os.environ["ODDS_API_KEY"]))

In [ ]:
import requests, pandas as pd, numpy as np

ODDS_API_KEY = os.environ.get("ODDS_API_KEY")
if not ODDS_API_KEY:
    raise ValueError("Missing ODDS_API_KEY env var.")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": REGIONS,
    "markets": MARKETS,
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
}

r = requests.get(url, params=params, timeout=30)
print("Status:", r.status_code)
r.raise_for_status()
data = r.json()
print("Games pulled:", len(data))

rows = []
for g in data:
    home = g.get("home_team")
    away = g.get("away_team")
    commence = g.get("commence_time")
    for b in g.get("bookmakers", []):
        book = b.get("key")
        for m in b.get("markets", []):
            mk = m.get("key")
            for o in m.get("outcomes", []):
                row = {
                    "home_team": home,
                    "away_team": away,
                    "commence_time": commence,
                    "book": book,
                    "market": mk,
                    "team": o.get("name"),
                    "american_odds": o.get("price"),
                    "spread": o.get("point") if mk == "spreads" else np.nan,
                    "total": o.get("point") if mk == "totals" else np.nan,
                }
                rows.append(row)

odds_long = pd.DataFrame(rows)

# price_decimal + implied_prob
def amer_to_dec(a):
    a = float(a)
    return (1 + a/100) if a > 0 else (1 + 100/abs(a))

odds_long["price_decimal"] = odds_long["american_odds"].astype(float).apply(amer_to_dec)
odds_long["implied_prob"] = 1 / odds_long["price_decimal"]

print("✅ odds_long:", odds_long.shape)
print("Markets:", odds_long["market"].value_counts().to_dict())
print("Unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

In [ ]:
import pandas as pd

odds_long["commence_time"] = pd.to_datetime(odds_long["commence_time"], utc=True)
TARGET_DATE = pd.Timestamp("2026-02-28", tz="UTC")

odds_long = odds_long[odds_long["commence_time"].dt.date == TARGET_DATE.date()].copy()

print("Filtered odds_long:", odds_long.shape)
print("Unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])
print("Markets after filter:", odds_long["market"].value_counts().to_dict())

In [ ]:
import os, pandas as pd

print("===== RUNTIME SANITY =====")
print("ODDS_API_KEY in env?:", bool(os.environ.get("ODDS_API_KEY")))
if os.environ.get("ODDS_API_KEY"):
    print("ODDS_API_KEY length:", len(os.environ["ODDS_API_KEY"]))

need = ["odds_long", "cons_ml", "cons_sp", "cons_tot", "ml_model", "hybrid_margin_v2", "sp_out_v2", "top10_ml", "top10_sp_v2"]
print("\n===== GLOBALS CHECK =====")
for n in need:
    exists = n in globals()
    if exists:
        obj = globals()[n]
        shape = getattr(obj, "shape", None)
        if shape is None and hasattr(obj, "__len__"):
            shape = (len(obj),)
        print(f"✅ {n}: exists | shape={shape}")
    else:
        print(f"❌ {n}: missing")

print("\n===== odds_long QUICK VIEW (if exists) =====")
if "odds_long" in globals() and isinstance(odds_long, pd.DataFrame) and len(odds_long) > 0:
    print("rows:", len(odds_long), "| markets:", odds_long["market"].value_counts().to_dict())
    if "commence_time" in odds_long.columns:
        try:
            ct = pd.to_datetime(odds_long["commence_time"], utc=True, errors="coerce")
            print("commence_time min/max:", ct.min(), ct.max())
        except Exception as e:
            print("commence_time parse issue:", e)
    print(odds_long.head(3))
else:
    print("odds_long not present or empty.")

In [ ]:
print("hello")

In [ ]:
print("odds_long in globals?", "odds_long" in globals())
print("ml_model in globals?", "ml_model" in globals())

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("key set:", len(os.environ["ODDS_API_KEY"]))

In [ ]:
import sys, platform
print("python:", sys.version.split()[0])
print("platform:", platform.platform()[:60])

In [ ]:
import time, sys
print("START", sys.version)
time.sleep(2)
print("END")

In [ ]:
print("kernel back")

In [ ]:
import sys
sys.stdout.write("stdout test\n")
sys.stderr.write("stderr test\n")

In [ ]:
import time
print("START")
time.sleep(2)
print("END")

In [ ]:
1+1

In [ ]:
print("hello")

In [ ]:
print("kernel alive")

In [ ]:
import numpy as np
import pandas as pd

assert "odds_long" in globals() and "ml_model" in globals(), "Need odds_long + ml_model in memory."
assert (odds_long["market"]=="h2h").any(), "No h2h rows in odds_long."

def dec_to_implied(dec):
    return 1.0 / float(dec)

def american_to_decimal(amer):
    amer = float(amer)
    if amer > 0:
        return 1.0 + amer/100.0
    else:
        return 1.0 + 100.0/abs(amer)

def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))

# --------- ML lines (best line per team per game) ----------
ml = odds_long.loc[odds_long["market"]=="h2h"].copy()

# normalize decimals if not present
if "price_decimal" not in ml.columns:
    ml["price_decimal"] = ml["american_odds"].apply(american_to_decimal)

ml["implied_prob"] = ml["price_decimal"].apply(dec_to_implied)

# attach model prob (ml_model has [home_team, away_team, team, model_prob_ml])
need = {"home_team","away_team","team","model_prob_ml"}
assert need.issubset(set(ml_model.columns)), f"ml_model missing columns: {need - set(ml_model.columns)}"

ml = ml.merge(
    ml_model[["home_team","away_team","team","model_prob_ml"]],
    on=["home_team","away_team","team"],
    how="left"
)

# STRICT: drop if no model prob
ml = ml.dropna(subset=["model_prob_ml"]).copy()

ml["true_edge"] = ml["model_prob_ml"] - ml["implied_prob"]
ml["ev"] = ml["model_prob_ml"] * ml["price_decimal"] - 1.0

# Full Kelly (fraction of bankroll)
# kelly = (bp - q)/b where b=dec-1, p=model_prob, q=1-p
def kelly_full(p, dec):
    p = float(p)
    dec = float(dec)
    b = dec - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    k = (b*p - q)/b
    return max(0.0, k)

ml["units_full_kelly"] = ml.apply(lambda r: kelly_full(r["model_prob_ml"], r["price_decimal"]), axis=1)

# pick best book per team/game by EV
def pick_best_book(df):
    return df.sort_values(["ev","price_decimal"], ascending=[False, False]).head(1)

ml_best = (
    ml.groupby(["home_team","away_team","team"], as_index=False, group_keys=False)
      .apply(pick_best_book)
      .reset_index(drop=True)
)

print("✅ ml_best:", ml_best.shape)
print("Model prob range:", float(ml_best["model_prob_ml"].min()), "→", float(ml_best["model_prob_ml"].max()))
print("Top rows preview:")
print(ml_best[["home_team","away_team","team","book","american_odds","price_decimal","model_prob_ml","implied_prob","ev","units_full_kelly"]].head(10))

In [ ]:
import numpy as np
import pandas as pd

assert "cons_ml" in globals(), "cons_ml missing (need consensus ML)."
assert "hybrid_margin_v2" in globals(), "hybrid_margin_v2 missing."
assert {"home_team","away_team","team","consensus_prob"}.issubset(cons_ml.columns), "cons_ml missing required columns."
assert {"home_team","away_team","proj_margin_home_away"}.issubset(hybrid_margin_v2.columns), "hybrid_margin_v2 missing required columns."

def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def logistic(x):
    return 1/(1+np.exp(-x))

# --- Build p_home from consensus probs (cons_ml has both teams as rows) ---
tmp = cons_ml.copy()

# identify home row vs away row
tmp["is_home_team"] = (tmp["team"] == tmp["home_team"])

home_rows = tmp[tmp["is_home_team"]].copy()
away_rows = tmp[~tmp["is_home_team"]].copy()

p_home = home_rows[["home_team","away_team","consensus_prob"]].rename(columns={"consensus_prob":"p_home"})
p_away = away_rows[["home_team","away_team","consensus_prob"]].rename(columns={"consensus_prob":"p_away"})

p_home = p_home.merge(p_away, on=["home_team","away_team"], how="left")

# Sometimes only one side is available; normalize if needed
p_home["p_away"] = p_home["p_away"].fillna(1 - p_home["p_home"])
p_home["sum_p"] = p_home["p_home"] + p_home["p_away"]
p_home["p_home"] = (p_home["p_home"] / p_home["sum_p"]).clip(1e-6, 1-1e-6)

print("p_home coverage:", f"{p_home['p_home'].notna().mean()*100:.1f}%", "| games:", len(p_home))

# --- Calibrate scale s: margin ~= s * logit(p_home) ---
# Use hybrid_margin_v2's modeled margin as target (HOME-AWAY)
cal = p_home.merge(
    hybrid_margin_v2[["home_team","away_team","proj_margin_home_away"]],
    on=["home_team","away_team"],
    how="inner"
).dropna()

assert len(cal) >= 20, f"Calibration too small (n={len(cal)})."

x = logit(cal["p_home"].values)
y = cal["proj_margin_home_away"].values

# OLS slope through origin: y ≈ s*x  =>  s = (x'y)/(x'x)
s = float((x @ y) / (x @ x))
s = abs(s)  # enforce positive scale magnitude

print(f"✅ Recalibrated spread scale s = {s:.3f} points per logit (correct sign)")
print("Calibration sample size:", len(cal))

# --- Build game-level margins from p_home ---
game_margin = p_home.copy()
game_margin["proj_margin_home_away"] = s * logit(game_margin["p_home"].values)

# overwrite hybrid_margin_v2 with full coverage (114 games etc.)
hybrid_margin_v2 = game_margin[["home_team","away_team","proj_margin_home_away"]].copy()
print("✅ hybrid_margin_v2 overwritten with calibrated margins.")
print(hybrid_margin_v2["proj_margin_home_away"].describe())

# --- Build ml_model for both teams ---
rows = []
for r in game_margin.itertuples(index=False):
    # home team prob
    pH = float(r.p_home)
    rows.append((r.home_team, r.away_team, r.home_team, pH))
    # away team prob
    rows.append((r.home_team, r.away_team, r.away_team, 1-pH))

ml_model = pd.DataFrame(rows, columns=["home_team","away_team","team","model_prob_ml"])
print("✅ ml_model rebuilt:", ml_model.shape)
print("Model ML std:", float(ml_model["model_prob_ml"].std()))

In [ ]:
import numpy as np
import pandas as pd

assert "odds_long" in globals(), "odds_long missing. Re-pull Odds API first."
assert (odds_long["market"]=="h2h").any(), "No h2h rows in odds_long."

# ---- helpers ----
def american_to_decimal(amer):
    amer = float(amer)
    if amer > 0:
        return 1.0 + amer/100.0
    else:
        return 1.0 + 100.0/abs(amer)

def to_decimal(row):
    if "price_decimal" in row and pd.notna(row["price_decimal"]):
        return float(row["price_decimal"])
    return american_to_decimal(row["american_odds"])

# ---- build cons_ml from odds_long h2h ----
ml = odds_long.loc[odds_long["market"]=="h2h"].copy()

# ensure decimal + implied
ml["price_decimal"] = ml.apply(to_decimal, axis=1)
ml["implied_prob"] = 1.0 / ml["price_decimal"]

# consensus by averaging implied probs across books per side
cons_ml = (
    ml.groupby(["home_team","away_team","team"], as_index=False)
      .agg(consensus_prob=("implied_prob","mean"),
           book_count=("implied_prob","size"))
)

# normalize within each game so P(home)+P(away)=1
g = cons_ml.groupby(["home_team","away_team"])["consensus_prob"].transform("sum")
cons_ml["consensus_prob"] = (cons_ml["consensus_prob"] / g).clip(1e-6, 1-1e-6)

print("✅ cons_ml rebuilt:", cons_ml.shape)
print(cons_ml.head(10).to_string(index=False))

# quick sanity: show a few games where sum != 1 (should be ~1)
chk = cons_ml.groupby(["home_team","away_team"])["consensus_prob"].sum().reset_index(name="sum_prob")
print("\nSum prob sanity (min/mean/max):",
      float(chk["sum_prob"].min()),
      float(chk["sum_prob"].mean()),
      float(chk["sum_prob"].max()))

In [ ]:
import os, requests, math
import pandas as pd
import numpy as np

# ====== REQUIRED ======
ODDS_API_KEY = os.getenv("ODDS_API_KEY", "").strip()
if not ODDS_API_KEY:
    ODDS_API_KEY = "PASTE_YOUR_KEY_HERE".strip()

if ODDS_API_KEY in ["", "PASTE_YOUR_KEY_HERE"]:
    raise ValueError("Paste your ODDS_API_KEY into ODDS_API_KEY in this cell.")

# ====== SETTINGS ======
SPORT = "basketball_ncaab"
REGIONS = "us"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"
BOOKMAKERS = ""  # optional comma list; leave "" for all available

# Pull all 3 markets we need
MARKETS = "h2h,spreads,totals"

# ====== HELPERS ======
def american_to_decimal(amer):
    amer = float(amer)
    if amer > 0:
        return 1.0 + amer/100.0
    else:
        return 1.0 + 100.0/abs(amer)

def dec_to_implied(dec):
    dec = float(dec)
    return 1.0 / dec

def normalize_team_name(s):
    return str(s).strip()

# ====== FETCH ======
url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": REGIONS,
    "markets": MARKETS,
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
}
if BOOKMAKERS:
    params["bookmakers"] = BOOKMAKERS

resp = requests.get(url, params=params, timeout=30)
print("Status:", resp.status_code)
if resp.status_code != 200:
    raise ValueError(f"Odds API error: {resp.status_code} | {resp.text[:300]}")

data = resp.json()
if not isinstance(data, list) or len(data) == 0:
    raise ValueError("Odds API returned empty list. Check sport key / params.")

# ====== FLATTEN TO odds_long ======
rows = []
for ev in data:
    home = normalize_team_name(ev.get("home_team"))
    away = normalize_team_name(ev.get("away_team"))
    commence = ev.get("commence_time")
    for bm in ev.get("bookmakers", []):
        book = bm.get("key")
        for mk in bm.get("markets", []):
            market = mk.get("key")  # h2h, spreads, totals
            for out in mk.get("outcomes", []):
                team = normalize_team_name(out.get("name"))
                amer = out.get("price")
                point = out.get("point", np.nan)  # spreads: +/- ; totals: number ; h2h: none
                dec = american_to_decimal(amer) if pd.notna(amer) else np.nan
                imp = dec_to_implied(dec) if pd.notna(dec) else np.nan

                rows.append({
                    "home_team": home,
                    "away_team": away,
                    "commence_time": commence,
                    "book": book,
                    "market": market,
                    "team": team,
                    "american_odds": amer,
                    "point": point,
                    "price_decimal": dec,
                    "implied_prob": imp
                })

odds_long = pd.DataFrame(rows)

# Standardize columns you used previously
# spreads: "spread" = point ; totals: "total" = point
odds_long["spread"] = np.where(odds_long["market"]=="spreads", odds_long["point"], np.nan)
odds_long["total"]  = np.where(odds_long["market"]=="totals", odds_long["point"], np.nan)

# Convert commence_time to datetime
odds_long["commence_time"] = pd.to_datetime(odds_long["commence_time"], utc=True, errors="coerce")

print("✅ odds_long:", odds_long.shape)
print("Markets:", odds_long["market"].value_counts().to_dict())
print("Unique games:", odds_long[["home_team","away_team"]].drop_duplicates().shape[0])

# ====== CONSENSUS: ML ======
ml = odds_long.loc[odds_long["market"]=="h2h"].copy()
cons_ml = (
    ml.groupby(["home_team","away_team","team"], as_index=False)
      .agg(consensus_prob=("implied_prob","mean"), book_count=("implied_prob","size"))
)
g = cons_ml.groupby(["home_team","away_team"])["consensus_prob"].transform("sum")
cons_ml["consensus_prob"] = (cons_ml["consensus_prob"] / g).clip(1e-6, 1-1e-6)

# ====== CONSENSUS: SPREAD (avg spread per game) ======
sp = odds_long.loc[odds_long["market"]=="spreads"].copy()
# Consensus spread is average of the HOME side spread across books (home side is the outcome with name==home_team)
home_sp = sp.loc[sp["team"]==sp["home_team"], ["home_team","away_team","spread"]].copy()
cons_spread_game = home_sp.groupby(["home_team","away_team"], as_index=False)["spread"].mean().rename(columns={"spread":"cons_spread_home"})
# Build cons_sp as a per-team table like you had before
cons_sp = pd.merge(
    cons_spread_game.assign(team=cons_spread_game["home_team"], cons_spread=lambda d: d["cons_spread_home"]),
    cons_spread_game.assign(team=cons_spread_game["away_team"], cons_spread=lambda d: -d["cons_spread_home"]),
    how="outer"
)[["home_team","away_team","team","cons_spread_home","cons_spread"]]

# ====== CONSENSUS: TOTALS ======
tot = odds_long.loc[odds_long["market"]=="totals"].copy()
cons_tot = (
    tot.groupby(["home_team","away_team"], as_index=False)
       .agg(cons_total=("total","mean"))
)

print(f"✅ cons_ml: {cons_ml.shape} | ✅ cons_sp: {cons_sp.shape} | ✅ cons_tot: {cons_tot.shape}")
print("\ncons_ml head:\n", cons_ml.head(6).to_string(index=False))
print("\ncons_sp head:\n", cons_sp.head(6).to_string(index=False))
print("\ncons_tot head:\n", cons_tot.head(6).to_string(index=False))

In [ ]:
import pandas as pd, numpy as np, math

# =========================
# 1) FILTER TO TARGET SLATE
# =========================
assert "odds_long" in globals(), "odds_long missing (re-run Odds API pull cell)."

TARGET_DATE = pd.Timestamp("2026-02-28", tz="UTC")

od = odds_long.copy()
od["commence_time"] = pd.to_datetime(od["commence_time"], utc=True, errors="coerce")
od = od[od["commence_time"].dt.date == TARGET_DATE.date()].copy()

print("✅ Filtered odds_long:", od.shape)
print("✅ Unique games:", od[["home_team","away_team"]].drop_duplicates().shape[0])
print("Markets:", od["market"].value_counts().to_dict())

# =========================
# 2) HELPERS
# =========================
def american_to_decimal(a):
    a = float(a)
    return (1.0 + a/100.0) if a > 0 else (1.0 + 100.0/abs(a))

def dec_to_implied(d):
    return 1.0/float(d)

def logit(p):
    p = float(np.clip(p, 1e-6, 1-1e-6))
    return math.log(p/(1-p))

def norm_cdf(x):
    # standard normal CDF via erf
    return 0.5*(1.0 + math.erf(x/math.sqrt(2.0)))

def p_cover(mu_team, spread, sigma):
    # margin TEAM-OPP must be > -spread to cover
    thr = -float(spread)
    z = (thr - float(mu_team))/float(sigma)
    return 1.0 - norm_cdf(z)

def full_kelly(p, dec):
    # Kelly for decimal odds: f* = (p*dec - 1)/(dec-1)
    p = float(p); dec = float(dec)
    b = dec - 1.0
    if b <= 0:
        return 0.0
    f = (p*dec - 1.0)/b
    return float(np.clip(f, 0.0, 1.0))

def pick_best_line(df, score_col="ev"):
    # choose best line per (home, away, market, team, spread_if_exists)
    return df.sort_values(score_col, ascending=False).head(1)

# =========================
# 3) REBUILD CONSENSUS TABLES (ON THIS SLATE)
# =========================
# ML consensus (devig within game)
ml = od.loc[od["market"]=="h2h"].copy()
assert len(ml) > 0, "No h2h rows in filtered odds_long."

cons_ml = (ml.groupby(["home_team","away_team","team"], as_index=False)
             .agg(consensus_prob=("implied_prob","mean"),
                  book_count=("implied_prob","size")))

# devig per game
g = cons_ml.groupby(["home_team","away_team"])["consensus_prob"].transform("sum")
cons_ml["consensus_prob"] = (cons_ml["consensus_prob"] / g).clip(1e-6, 1-1e-6)

# consensus spread (home-side average)
sp = od.loc[od["market"]=="spreads"].copy()
assert len(sp) > 0, "No spreads rows in filtered odds_long."

home_sp = sp.loc[sp["team"]==sp["home_team"], ["home_team","away_team","spread"]].copy()
cons_spread_game = (home_sp.groupby(["home_team","away_team"], as_index=False)["spread"]
                          .mean()
                          .rename(columns={"spread":"cons_spread_home"}))

cons_sp = pd.concat([
    cons_spread_game.assign(team=cons_spread_game["home_team"], cons_spread=lambda d: d["cons_spread_home"]),
    cons_spread_game.assign(team=cons_spread_game["away_team"], cons_spread=lambda d: -d["cons_spread_home"]),
], ignore_index=True)[["home_team","away_team","team","cons_spread_home","cons_spread"]]

# consensus totals
tot = od.loc[od["market"]=="totals"].copy()
assert len(tot) > 0, "No totals rows in filtered odds_long."
cons_tot = (tot.groupby(["home_team","away_team"], as_index=False)
              .agg(cons_total=("total","mean")))

print(f"✅ cons_ml: {cons_ml.shape} | ✅ cons_sp: {cons_sp.shape} | ✅ cons_tot: {cons_tot.shape}")

# =========================
# 4) ML MODEL (DEVIGGED CONSENSUS) + BEST BOOK LINE
# =========================
ml_model = cons_ml.rename(columns={"consensus_prob":"model_prob_ml"})[["home_team","away_team","team","model_prob_ml","book_count"]].copy()

ml_lines = ml.copy()
ml_lines["price_decimal"] = ml_lines["price_decimal"].astype(float)
ml_lines["implied_prob"] = ml_lines["implied_prob"].astype(float)

# attach model prob to each ML line
ml_out = ml_lines.merge(ml_model[["home_team","away_team","team","model_prob_ml"]],
                        on=["home_team","away_team","team"], how="left")

ml_out["ev"] = ml_out["model_prob_ml"] * ml_out["price_decimal"] - 1.0
ml_out["units_full_kelly"] = ml_out.apply(lambda r: full_kelly(r["model_prob_ml"], r["price_decimal"]), axis=1)

# best ML line per team/game
ml_best = (ml_out.groupby(["home_team","away_team","team"], as_index=False)
                 .apply(lambda g: g.sort_values("ev", ascending=False).head(1))
                 .reset_index(drop=True))

# =========================
# 5) SPREAD v2: CALIBRATE SCALE + BUILD mu + sigma_game (totals)
# =========================
# Build p_home from ml_model
p_home = (ml_model.loc[ml_model["team"]==ml_model["home_team"], ["home_team","away_team","model_prob_ml"]]
                  .rename(columns={"model_prob_ml":"p_home"}))

cal = cons_spread_game.merge(p_home, on=["home_team","away_team"], how="inner").copy()
cal["logit_home"] = cal["p_home"].apply(logit)

# spread_home ≈ -s * logit(p_home)  (favorite => p_home>0.5 logit>0 => spread negative)
num = float(np.sum(cal["cons_spread_home"] * cal["logit_home"]))
den = float(np.sum(cal["logit_home"] * cal["logit_home"]))
s = -num/den if den > 0 else 6.5
s = abs(s)

print(f"✅ Calibrated spread scale s = {s:.3f} points per logit (n={len(cal)})")

# mu = projected margin HOME-AWAY
hybrid_margin_v2 = p_home.copy()
hybrid_margin_v2["proj_margin_home_away"] = -s * hybrid_margin_v2["p_home"].apply(logit)

print("\nProjected margin stats (HOME-AWAY):")
print(hybrid_margin_v2["proj_margin_home_away"].describe())

# sigma_game from totals (pace proxy)
BASE_SIGMA = 11.5
league_total = float(cons_tot["cons_total"].median())
sigma_tbl = cons_tot.copy()
sigma_tbl["sigma_game"] = BASE_SIGMA * np.sqrt((sigma_tbl["cons_total"]/league_total).clip(0.85, 1.20))
sigma_tbl["sigma_game"] = sigma_tbl["sigma_game"].fillna(BASE_SIGMA)

print("\n✅ sigma_game stats:")
print(sigma_tbl["sigma_game"].describe())

# =========================
# 6) BUILD SPREAD LINES @ CONSENSUS SPREAD ONLY (clean, no dupes across alternate spreads)
# =========================
sp_lines = sp.copy()
sp_lines["price_decimal"] = sp_lines["price_decimal"].astype(float)
sp_lines["implied_prob"] = sp_lines["implied_prob"].astype(float)

# attach consensus spread (team-side) to each spread line, then filter to those rows only
sp_lines = sp_lines.merge(cons_sp[["home_team","away_team","team","cons_spread"]], on=["home_team","away_team","team"], how="left")
sp_cons_only = sp_lines[np.isclose(sp_lines["spread"], sp_lines["cons_spread"], atol=1e-6)].copy()

print("\n✅ Spread rows at CONSENSUS spread only:", sp_cons_only.shape)

# attach mu and sigma_game
sp_cons_only = sp_cons_only.merge(hybrid_margin_v2[["home_team","away_team","proj_margin_home_away"]], on=["home_team","away_team"], how="left")
sp_cons_only = sp_cons_only.merge(sigma_tbl[["home_team","away_team","cons_total","sigma_game"]], on=["home_team","away_team"], how="left")

# if any missing mu (should be 0 now), fill 0 as guardrail
sp_cons_only["proj_margin_home_away"] = sp_cons_only["proj_margin_home_away"].fillna(0.0)
sp_cons_only["sigma_game"] = sp_cons_only["sigma_game"].fillna(BASE_SIGMA)

# mu_team = TEAM-OPP
sp_cons_only["mu_team"] = np.where(
    sp_cons_only["team"] == sp_cons_only["home_team"],
    sp_cons_only["proj_margin_home_away"],
    -sp_cons_only["proj_margin_home_away"]
)

sp_cons_only["model_prob_spread"] = sp_cons_only.apply(lambda r: p_cover(r["mu_team"], r["spread"], r["sigma_game"]), axis=1)
sp_cons_only["ev"] = sp_cons_only["model_prob_spread"] * sp_cons_only["price_decimal"] - 1.0
sp_cons_only["units_full_kelly"] = sp_cons_only.apply(lambda r: full_kelly(r["model_prob_spread"], r["price_decimal"]), axis=1)

print("\n✅ Spread prob sanity:")
print("std:", float(sp_cons_only["model_prob_spread"].std()),
      "| min/max:", float(sp_cons_only["model_prob_spread"].min()), float(sp_cons_only["model_prob_spread"].max()))

# best SP line per game+team (already consensus spread so mostly 1, but keep safe)
sp_best = (sp_cons_only.groupby(["home_team","away_team","team","spread"], as_index=False)
                    .apply(lambda g: g.sort_values("ev", ascending=False).head(1))
                    .reset_index(drop=True))

# =========================
# 7) OUTPUTS YOU ASKED FOR (PARLAY BUILDER)
# =========================
# ---- ML ANCHORS (most likely winners) ----
anchors = ml_best.sort_values("model_prob_ml", ascending=False).copy()
# mild guard: avoid insane prices; tune these freely
anchors = anchors[(anchors["price_decimal"] <= 1.55) | (anchors["american_odds"] >= -200)].copy()
anchors = anchors.head(20)

print("\n==============================")
print("MOST PROBABLE ML ANCHORS (Top 20)")
print("==============================")
print(anchors[["home_team","away_team","team","book","american_odds","model_prob_ml","ev","units_full_kelly"]].to_string(index=False))

# ---- BEST +EV ML LEGS (guarded) ----
MAX_DOG = 600
MIN_FK = 0.01
ev_legs = ml_best.copy()
ev_legs = ev_legs[(ev_legs["american_odds"] <= MAX_DOG) & (ev_legs["units_full_kelly"] >= MIN_FK)]
ev_legs = ev_legs.sort_values("ev", ascending=False).head(30)

print("\n==============================")
print("BEST +EV ML LEGS (guarded, Top 30)")
print("==============================")
print(ev_legs[["home_team","away_team","team","book","american_odds","model_prob_ml","ev","units_full_kelly"]].to_string(index=False))

def parlay_summary(df, title):
    df = df.copy()
    df["dec"] = df["american_odds"].apply(american_to_decimal)
    parlay_dec = float(df["dec"].prod())
    parlay_prob = float(df["model_prob_ml"].prod())
    parlay_ev = parlay_prob * parlay_dec - 1.0
    print(f"\n==== {title} ====")
    print(df[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
    print(f"\nParlay decimal: {parlay_dec:.3f}")
    print(f"Parlay hit-prob (model): {parlay_prob:.4f} ({parlay_prob*100:.2f}%)")
    print(f"Parlay EV (model): {parlay_ev:.3f}")
    return parlay_dec, parlay_prob, parlay_ev

# ---- 4-leg ANCHOR PARLAY (highest win prob, unique games) ----
# keep unique games by key
anchors["game_key"] = anchors["home_team"] + "||" + anchors["away_team"]
a4 = anchors.drop_duplicates("game_key").head(4).copy()
parlay_summary(a4, "4-LEG ANCHOR ML PARLAY (max win-prob)")

# ---- 4-leg MIXED PARLAY (2 anchors + 2 best EV legs, no game dupes) ----
a2 = anchors.drop_duplicates("game_key").head(2).copy()
ev_legs["game_key"] = ev_legs["home_team"] + "||" + ev_legs["away_team"]
mix = pd.concat([a2, ev_legs[~ev_legs["game_key"].isin(set(a2["game_key"]))].drop_duplicates("game_key").head(2)], ignore_index=True)
parlay_summary(mix, "4-LEG MIXED ML PARLAY (2 anchors + 2 EV legs)")

# ---- 16-leg LOTTO PARLAY (EV-leaning, guarded) ----
lotto_pool = ml_best.copy()
lotto_pool = lotto_pool[(lotto_pool["american_odds"] >= 120) & (lotto_pool["american_odds"] <= MAX_DOG)]
lotto_pool = lotto_pool.sort_values("ev", ascending=False).drop_duplicates(["home_team","away_team"]).head(16).copy()
if len(lotto_pool) == 16:
    parlay_summary(lotto_pool, "16-LEG LOTTO ML PARLAY (EV-leaning, +120 to +600)")
else:
    print("\n⚠️ Not enough lotto-eligible legs to build 16. Found:", len(lotto_pool))

# =========================
# 8) SAVE OBJECTS
# =========================
odds_long_slate = od
print("\n✅ Saved globals: odds_long_slate, cons_ml, cons_sp, cons_tot, ml_model, ml_best, hybrid_margin_v2, sp_cons_only, sp_best")

In [ ]:
import numpy as np, pandas as pd

assert "ml_best" in globals(), "ml_best missing — run the prior parlay builder cell first."

# =========================
# KNOBS (EDIT THESE)
# =========================
ANCHOR_MIN_PROB   = 0.70     # anchors must be >= this win prob
ANCHOR_MIN_EV     = -0.01    # allow slightly negative EV for anchors (set to 0.00 if you want strict +EV only)
EV_LEG_MIN_EV     = 0.01     # EV legs must clear this (0.01 = +1% ROI in decimal space)
MAX_DOG_ODDS      = 600      # cap longshots in EV pool / lotto
LOTTO_MIN_EV      = 0.005    # lotto legs EV floor
LOTTO_MIN_PROB    = 0.12     # avoid total dust
N_ANCHORS_SHOW    = 15
N_EV_SHOW         = 25

def american_to_decimal(a):
    a = float(a)
    return (1.0 + a/100.0) if a > 0 else (1.0 + 100.0/abs(a))

def parlay_metrics(df):
    df = df.copy()
    df["dec"] = df["american_odds"].apply(american_to_decimal)
    parlay_dec = float(df["dec"].prod())
    parlay_prob = float(df["model_prob_ml"].prod())
    parlay_ev = parlay_prob * parlay_dec - 1.0
    return parlay_dec, parlay_prob, parlay_ev

def print_parlay(df, title):
    df = df.copy()
    dec, prob, ev = parlay_metrics(df)
    print(f"\n==== {title} ====")
    print(df[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
    print(f"Parlay decimal: {dec:.3f}")
    print(f"Hit-prob (model): {prob:.6f} ({prob*100:.4f}%)")
    print(f"Parlay EV (model): {ev:.3f}")
    return dec, prob, ev

# Add a unique game key so we never duplicate same matchup
pool = ml_best.copy()
pool["game_key"] = pool["home_team"] + "||" + pool["away_team"]

# =========================
# 1) "SMART" ANCHORS
# High prob, but NOT insanely overpriced
# =========================
anchors = pool[
    (pool["model_prob_ml"] >= ANCHOR_MIN_PROB) &
    (pool["ev"] >= ANCHOR_MIN_EV)
].copy()

# Rank anchors by a blend: win prob first, then EV
anchors = anchors.sort_values(["model_prob_ml","ev"], ascending=[False, False])
anchors_show = anchors.drop_duplicates("game_key").head(N_ANCHORS_SHOW)

print("\n==============================")
print(f"SMART ML ANCHORS (p>={ANCHOR_MIN_PROB}, EV>={ANCHOR_MIN_EV})")
print("==============================")
if len(anchors_show)==0:
    print("⚠️ No anchors passed. Try ANCHOR_MIN_PROB=0.65 and/or ANCHOR_MIN_EV=-0.02")
else:
    print(anchors_show[["home_team","away_team","team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))

# =========================
# 2) BEST +EV LEGS (GUARDED)
# =========================
ev_legs = pool[
    (pool["american_odds"] <= MAX_DOG_ODDS) &
    (pool["ev"] >= EV_LEG_MIN_EV)
].copy()

ev_legs = ev_legs.sort_values("ev", ascending=False)
ev_show = ev_legs.drop_duplicates("game_key").head(N_EV_SHOW)

print("\n==============================")
print(f"BEST +EV ML LEGS (EV>={EV_LEG_MIN_EV}, dogs<=+{MAX_DOG_ODDS})")
print("==============================")
if len(ev_show)==0:
    print("⚠️ No EV legs passed. Try EV_LEG_MIN_EV=0.005 or raise MAX_DOG_ODDS.")
else:
    print(ev_show[["home_team","away_team","team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))

# =========================
# 3) BUILD PARLAYS
# =========================
# 4-leg anchors: take top 4 unique games from anchors
a4 = anchors.drop_duplicates("game_key").head(4).copy()
if len(a4)==4:
    print_parlay(a4, "4-LEG ANCHOR PARLAY (smart anchors)")
else:
    print("\n⚠️ Not enough anchors for a 4-leg. Lower ANCHOR_MIN_PROB or ANCHOR_MIN_EV.")

# 4-leg mixed: 2 anchors + 2 EV legs, all unique games
a2 = anchors.drop_duplicates("game_key").head(2).copy()
mix = a2.copy()
if len(mix) < 2:
    print("\n⚠️ Not enough anchors for mixed. Lower ANCHOR_MIN_PROB or ANCHOR_MIN_EV.")
else:
    add = ev_legs[~ev_legs["game_key"].isin(set(mix["game_key"]))].drop_duplicates("game_key").head(2).copy()
    mix = pd.concat([mix, add], ignore_index=True)
    if len(mix)==4:
        print_parlay(mix, "4-LEG MIXED PARLAY (2 anchors + 2 EV legs)")
    else:
        print("\n⚠️ Not enough EV legs to complete mixed. Lower EV_LEG_MIN_EV or widen dog cap.")

# 16-leg lotto: EV-leaning + prob floor + dog cap
lotto = pool[
    (pool["american_odds"] >= 120) &
    (pool["american_odds"] <= MAX_DOG_ODDS) &
    (pool["ev"] >= LOTTO_MIN_EV) &
    (pool["model_prob_ml"] >= LOTTO_MIN_PROB)
].copy()

lotto = lotto.sort_values("ev", ascending=False).drop_duplicates("game_key").head(16).copy()
if len(lotto)==16:
    print_parlay(lotto, "16-LEG LOTTO (guarded)")
else:
    print(f"\n⚠️ Lotto pool too small: {len(lotto)} legs. "
          f"Try LOTTO_MIN_PROB=0.10 and/or LOTTO_MIN_EV=0.0 and/or MAX_DOG_ODDS=800.")

# =========================
# 4) SAVE
# =========================
smart_anchors = anchors.drop_duplicates("game_key").copy()
best_ev_legs = ev_legs.drop_duplicates("game_key").copy()
parlay_anchor4 = a4.copy()
parlay_mixed4 = mix.copy() if 'mix' in locals() else None
parlay_lotto16 = lotto.copy()

print("\n✅ Saved: smart_anchors, best_ev_legs, parlay_anchor4, parlay_mixed4, parlay_lotto16")

In [ ]:
# Anchor guard: DO NOT use anchors worse than -400
ANCHOR_MAX_FAV = -400

In [ ]:
anchors2 = pool[
    (pool["model_prob_ml"] >= ANCHOR_MIN_PROB) &
    (pool["american_odds"] >= -400) &
    (pool["american_odds"] <= ANCHOR_MAX_POS)
].copy()

In [ ]:
# Re-define anchor knobs (Colab reset safe)
ANCHOR_MIN_PROB = 0.72
ANCHOR_MAX_FAV  = -400      # hard cap per your rule
ANCHOR_MAX_POS  = 250       # allow small dogs if ever needed
EV_MIN          = 0.01
EV_DOG_CAP      = 600

In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# Helpers
# -----------------------------
def american_to_decimal(odds):
    odds = float(odds)
    if odds > 0:
        return 1 + odds/100.0
    else:
        return 1 + 100.0/abs(odds)

def prob_from_decimal(dec):
    return 1.0/dec

def parlay_decimal(dec_list):
    out = 1.0
    for d in dec_list:
        out *= float(d)
    return out

def parlay_hit_prob(prob_list):
    out = 1.0
    for p in prob_list:
        out *= float(p)
    return out

def parlay_ev(parlay_dec, parlay_p):
    # EV per 1 unit stake: p*(payout) - 1, where payout=decimal
    return parlay_p * parlay_dec - 1

def fmt_odds(o):
    o = int(o)
    return f"+{o}" if o > 0 else f"{o}"

def build_pool_from_ml_best():
    # If you already created ml_best (best line per game), use it.
    if "ml_best" in globals() and isinstance(globals()["ml_best"], pd.DataFrame) and len(globals()["ml_best"]) > 0:
        pool = ml_best.copy()
        # standardize columns
        if "price_decimal" not in pool.columns:
            pool["price_decimal"] = pool["american_odds"].apply(american_to_decimal)
        return pool

    # Otherwise build a pool from odds_long + ml_model (model probabilities)
    assert "odds_long" in globals() or "odds_long_slate" in globals(), "Need odds_long or odds_long_slate."
    assert "ml_model" in globals(), "Need ml_model (home_team/away_team/team/model_prob_ml)."

    ol = globals().get("odds_long_slate", globals().get("odds_long")).copy()
    h2h = ol[ol["market"] == "h2h"].copy()
    h2h["price_decimal"] = h2h.get("price_decimal", np.nan)
    if h2h["price_decimal"].isna().any():
        h2h["price_decimal"] = h2h["american_odds"].apply(american_to_decimal)

    # attach model probs
    pool = h2h.merge(
        ml_model[["home_team","away_team","team","model_prob_ml"]],
        on=["home_team","away_team","team"],
        how="left"
    )
    pool = pool.dropna(subset=["model_prob_ml"]).copy()

    # implied / EV
    pool["implied_prob"] = pool["price_decimal"].apply(prob_from_decimal)
    pool["ev"] = pool["model_prob_ml"] * pool["price_decimal"] - 1

    # pick best line per team/game by EV (or highest decimal if same)
    pool = (pool.sort_values(["home_team","away_team","team","ev","price_decimal"], ascending=[True,True,True,False,False])
                .groupby(["home_team","away_team","team"], as_index=False)
                .head(1)
                .reset_index(drop=True))
    return pool

# -----------------------------
# Build pools
# -----------------------------
pool = build_pool_from_ml_best()

# ensure needed cols
need = {"home_team","away_team","team","american_odds","price_decimal","model_prob_ml","ev"}
missing = need - set(pool.columns)
assert not missing, f"Pool missing columns: {missing}"

# -----------------------------
# Anchors: MOST PROBABLE, BUT <= -400 (your rule)
# -----------------------------
anchors = pool[
    (pool["american_odds"] >= ANCHOR_MAX_FAV) &  # e.g. -400
    (pool["model_prob_ml"] >= ANCHOR_MIN_PROB)
].copy().sort_values("model_prob_ml", ascending=False)

# -----------------------------
# EV legs (guarded)
# -----------------------------
ev_legs = pool[
    (pool["ev"] >= EV_MIN) &
    (pool["american_odds"] <= EV_DOG_CAP) &
    (pool["american_odds"] >= -4000)  # just avoid nonsense; adjust if you want
].copy().sort_values("ev", ascending=False)

print("\n==============================")
print(f"SMART ANCHORS (p>={ANCHOR_MIN_PROB}, odds>={ANCHOR_MAX_FAV})")
print("==============================")
if len(anchors)==0:
    print("⚠️ None found. Lower ANCHOR_MIN_PROB (ex: 0.70) or relax odds cap.")
else:
    print(anchors[["home_team","away_team","team","book","american_odds","model_prob_ml","ev"]].head(20).to_string(index=False))

print("\n==============================")
print(f"BEST +EV ML LEGS (EV>={EV_MIN}, dogs<=+{EV_DOG_CAP})")
print("==============================")
if len(ev_legs)==0:
    print("⚠️ None found. Lower EV_MIN (ex: 0.005) or widen dog cap.")
else:
    print(ev_legs[["home_team","away_team","team","book","american_odds","model_prob_ml","ev"]].head(30).to_string(index=False))

# -----------------------------
# 4-leg anchor parlay (max win-prob)
# -----------------------------
print("\n==== 4-LEG ANCHOR ML PARLAY (max win-prob, <=-400 cap) ====")
if len(anchors) < 4:
    print("⚠️ Not enough anchors for a 4-leg. Try ANCHOR_MIN_PROB=0.70.")
    parlay_anchor4 = pd.DataFrame()
else:
    parlay_anchor4 = anchors.head(4).copy()
    dec = parlay_decimal(parlay_anchor4["price_decimal"])
    p   = parlay_hit_prob(parlay_anchor4["model_prob_ml"])
    evp = parlay_ev(dec, p)
    print(parlay_anchor4[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
    print(f"\nParlay decimal: {dec:.3f}")
    print(f"Parlay hit-prob (model): {p:.4f} ({p*100:.2f}%)")
    print(f"Parlay EV (model): {evp:.3f}")

# -----------------------------
# 4-leg mixed (2 anchors + 2 EV legs)
# -----------------------------
print("\n==== 4-LEG MIXED ML PARLAY (2 anchors + 2 EV legs) ====")
parlay_mixed4 = pd.DataFrame()
if len(anchors) >= 2 and len(ev_legs) >= 2:
    mix = pd.concat([anchors.head(2), ev_legs.head(2)], ignore_index=True)

    # de-dupe same game/team
    mix = (mix.sort_values("ev", ascending=False)
              .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

    if len(mix) >= 4:
        parlay_mixed4 = mix.head(4).copy()
        dec = parlay_decimal(parlay_mixed4["price_decimal"])
        p   = parlay_hit_prob(parlay_mixed4["model_prob_ml"])
        evp = parlay_ev(dec, p)
        print(parlay_mixed4[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
        print(f"\nParlay decimal: {dec:.3f}")
        print(f"Parlay hit-prob (model): {p:.4f} ({p*100:.2f}%)")
        print(f"Parlay EV (model): {evp:.3f}")
    else:
        print("⚠️ Not enough unique legs after de-dupe. Widen EV pool or lower thresholds.")
else:
    print("⚠️ Need at least 2 anchors and 2 EV legs. Lower thresholds.")

# -----------------------------
# 16-leg lotto (EV leaning)
# -----------------------------
print("\n==== 16-LEG LOTTO ML PARLAY (EV-leaning, guarded) ====")
parlay_lotto16 = ev_legs.copy()
parlay_lotto16 = parlay_lotto16.sort_values("ev", ascending=False)

# keep unique games/teams
parlay_lotto16 = parlay_lotto16.drop_duplicates(subset=["home_team","away_team","team"], keep="first").head(16).copy()
if len(parlay_lotto16) < 16:
    print(f"⚠️ Only {len(parlay_lotto16)} legs available with current filters.")
else:
    dec = parlay_decimal(parlay_lotto16["price_decimal"])
    p   = parlay_hit_prob(parlay_lotto16["model_prob_ml"])
    evp = parlay_ev(dec, p)
    print(parlay_lotto16[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
    print(f"\nParlay decimal: {dec:.3f}")
    print(f"Hit-prob (model): {p:.8f} ({p*100:.6f}%)")
    print(f"Parlay EV (model): {evp:.3f}")

# Save globals for later cells
globals()["smart_anchors"]   = anchors
globals()["best_ev_legs"]    = ev_legs
globals()["parlay_anchor4"]  = parlay_anchor4
globals()["parlay_mixed4"]   = parlay_mixed4
globals()["parlay_lotto16"]  = parlay_lotto16

print("\n✅ Saved: smart_anchors, best_ev_legs, parlay_anchor4, parlay_mixed4, parlay_lotto16")import numpy as np
import pandas as pd

# -----------------------------
# Helpers
# -----------------------------
def american_to_decimal(odds):
    odds = float(odds)
    if odds > 0:
        return 1 + odds/100.0
    else:
        return 1 + 100.0/abs(odds)

def prob_from_decimal(dec):
    return 1.0/dec

def parlay_decimal(dec_list):
    out = 1.0
    for d in dec_list:
        out *= float(d)
    return out

def parlay_hit_prob(prob_list):
    out = 1.0
    for p in prob_list:
        out *= float(p)
    return out

def parlay_ev(parlay_dec, parlay_p):
    # EV per 1 unit stake: p*(payout) - 1, where payout=decimal
    return parlay_p * parlay_dec - 1

def fmt_odds(o):
    o = int(o)
    return f"+{o}" if o > 0 else f"{o}"

def build_pool_from_ml_best():
    # If you already created ml_best (best line per game), use it.
    if "ml_best" in globals() and isinstance(globals()["ml_best"], pd.DataFrame) and len(globals()["ml_best"]) > 0:
        pool = ml_best.copy()
        # standardize columns
        if "price_decimal" not in pool.columns:
            pool["price_decimal"] = pool["american_odds"].apply(american_to_decimal)
        return pool

    # Otherwise build a pool from odds_long + ml_model (model probabilities)
    assert "odds_long" in globals() or "odds_long_slate" in globals(), "Need odds_long or odds_long_slate."
    assert "ml_model" in globals(), "Need ml_model (home_team/away_team/team/model_prob_ml)."

    ol = globals().get("odds_long_slate", globals().get("odds_long")).copy()
    h2h = ol[ol["market"] == "h2h"].copy()
    h2h["price_decimal"] = h2h.get("price_decimal", np.nan)
    if h2h["price_decimal"].isna().any():
        h2h["price_decimal"] = h2h["american_odds"].apply(american_to_decimal)

    # attach model probs
    pool = h2h.merge(
        ml_model[["home_team","away_team","team","model_prob_ml"]],
        on=["home_team","away_team","team"],
        how="left"
    )
    pool = pool.dropna(subset=["model_prob_ml"]).copy()

    # implied / EV
    pool["implied_prob"] = pool["price_decimal"].apply(prob_from_decimal)
    pool["ev"] = pool["model_prob_ml"] * pool["price_decimal"] - 1

    # pick best line per team/game by EV (or highest decimal if same)
    pool = (pool.sort_values(["home_team","away_team","team","ev","price_decimal"], ascending=[True,True,True,False,False])
                .groupby(["home_team","away_team","team"], as_index=False)
                .head(1)
                .reset_index(drop=True))
    return pool

# -----------------------------
# Build pools
# -----------------------------
pool = build_pool_from_ml_best()

# ensure needed cols
need = {"home_team","away_team","team","american_odds","price_decimal","model_prob_ml","ev"}
missing = need - set(pool.columns)
assert not missing, f"Pool missing columns: {missing}"

# -----------------------------
# Anchors: MOST PROBABLE, BUT <= -400 (your rule)
# -----------------------------
anchors = pool[
    (pool["american_odds"] >= ANCHOR_MAX_FAV) &  # e.g. -400
    (pool["model_prob_ml"] >= ANCHOR_MIN_PROB)
].copy().sort_values("model_prob_ml", ascending=False)

# -----------------------------
# EV legs (guarded)
# -----------------------------
ev_legs = pool[
    (pool["ev"] >= EV_MIN) &
    (pool["american_odds"] <= EV_DOG_CAP) &
    (pool["american_odds"] >= -4000)  # just avoid nonsense; adjust if you want
].copy().sort_values("ev", ascending=False)

print("\n==============================")
print(f"SMART ANCHORS (p>={ANCHOR_MIN_PROB}, odds>={ANCHOR_MAX_FAV})")
print("==============================")
if len(anchors)==0:
    print("⚠️ None found. Lower ANCHOR_MIN_PROB (ex: 0.70) or relax odds cap.")
else:
    print(anchors[["home_team","away_team","team","book","american_odds","model_prob_ml","ev"]].head(20).to_string(index=False))

print("\n==============================")
print(f"BEST +EV ML LEGS (EV>={EV_MIN}, dogs<=+{EV_DOG_CAP})")
print("==============================")
if len(ev_legs)==0:
    print("⚠️ None found. Lower EV_MIN (ex: 0.005) or widen dog cap.")
else:
    print(ev_legs[["home_team","away_team","team","book","american_odds","model_prob_ml","ev"]].head(30).to_string(index=False))

# -----------------------------
# 4-leg anchor parlay (max win-prob)
# -----------------------------
print("\n==== 4-LEG ANCHOR ML PARLAY (max win-prob, <=-400 cap) ====")
if len(anchors) < 4:
    print("⚠️ Not enough anchors for a 4-leg. Try ANCHOR_MIN_PROB=0.70.")
    parlay_anchor4 = pd.DataFrame()
else:
    parlay_anchor4 = anchors.head(4).copy()
    dec = parlay_decimal(parlay_anchor4["price_decimal"])
    p   = parlay_hit_prob(parlay_anchor4["model_prob_ml"])
    evp = parlay_ev(dec, p)
    print(parlay_anchor4[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
    print(f"\nParlay decimal: {dec:.3f}")
    print(f"Parlay hit-prob (model): {p:.4f} ({p*100:.2f}%)")
    print(f"Parlay EV (model): {evp:.3f}")

# -----------------------------
# 4-leg mixed (2 anchors + 2 EV legs)
# -----------------------------
print("\n==== 4-LEG MIXED ML PARLAY (2 anchors + 2 EV legs) ====")
parlay_mixed4 = pd.DataFrame()
if len(anchors) >= 2 and len(ev_legs) >= 2:
    mix = pd.concat([anchors.head(2), ev_legs.head(2)], ignore_index=True)

    # de-dupe same game/team
    mix = (mix.sort_values("ev", ascending=False)
              .drop_duplicates(subset=["home_team","away_team","team"], keep="first"))

    if len(mix) >= 4:
        parlay_mixed4 = mix.head(4).copy()
        dec = parlay_decimal(parlay_mixed4["price_decimal"])
        p   = parlay_hit_prob(parlay_mixed4["model_prob_ml"])
        evp = parlay_ev(dec, p)
        print(parlay_mixed4[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
        print(f"\nParlay decimal: {dec:.3f}")
        print(f"Parlay hit-prob (model): {p:.4f} ({p*100:.2f}%)")
        print(f"Parlay EV (model): {evp:.3f}")
    else:
        print("⚠️ Not enough unique legs after de-dupe. Widen EV pool or lower thresholds.")
else:
    print("⚠️ Need at least 2 anchors and 2 EV legs. Lower thresholds.")

# -----------------------------
# 16-leg lotto (EV leaning)
# -----------------------------
print("\n==== 16-LEG LOTTO ML PARLAY (EV-leaning, guarded) ====")
parlay_lotto16 = ev_legs.copy()
parlay_lotto16 = parlay_lotto16.sort_values("ev", ascending=False)

# keep unique games/teams
parlay_lotto16 = parlay_lotto16.drop_duplicates(subset=["home_team","away_team","team"], keep="first").head(16).copy()
if len(parlay_lotto16) < 16:
    print(f"⚠️ Only {len(parlay_lotto16)} legs available with current filters.")
else:
    dec = parlay_decimal(parlay_lotto16["price_decimal"])
    p   = parlay_hit_prob(parlay_lotto16["model_prob_ml"])
    evp = parlay_ev(dec, p)
    print(parlay_lotto16[["team","home_team","away_team","book","american_odds","model_prob_ml","ev"]].to_string(index=False))
    print(f"\nParlay decimal: {dec:.3f}")
    print(f"Hit-prob (model): {p:.8f} ({p*100:.6f}%)")
    print(f"Parlay EV (model): {evp:.3f}")

# Save globals for later cells
globals()["smart_anchors"]   = anchors
globals()["best_ev_legs"]    = ev_legs
globals()["parlay_anchor4"]  = parlay_anchor4
globals()["parlay_mixed4"]   = parlay_mixed4
globals()["parlay_lotto16"]  = parlay_lotto16

print("\nSaved: smart_anchors, best_ev_legs, parlay_anchor4, parlay_mixed4, parlay_lotto16")

In [ ]:
import numpy as np
import pandas as pd

def american_to_decimal(o):
    o = float(o)
    return 1 + o/100 if o > 0 else 1 + 100/abs(o)

def parlay_metrics(df):
    df = df.copy()
    df["dec"] = df["american_odds"].apply(american_to_decimal)
    dec = df["dec"].prod()
    p = df["model_prob_ml"].prod()
    ev = p * dec - 1
    return dec, p, ev

# Build pool from ml_best (must exist)
assert "ml_best" in globals(), "ml_best missing"

pool = ml_best.copy()
pool["game_key"] = pool["home_team"] + "||" + pool["away_team"]

# --------------------
# Anchor rules
# --------------------
ANCHOR_MIN_PROB = 0.70
ANCHOR_MAX_FAV  = -400

anchors = pool[
    (pool["model_prob_ml"] >= ANCHOR_MIN_PROB) &
    (pool["american_odds"] >= ANCHOR_MAX_FAV)
].copy().sort_values("model_prob_ml", ascending=False)

# --------------------
# EV legs
# --------------------
EV_MIN = 0.01
EV_DOG_CAP = 600

ev_legs = pool[
    (pool["ev"] >= EV_MIN) &
    (pool["american_odds"] <= EV_DOG_CAP)
].copy().sort_values("ev", ascending=False)

print("\nSMART ANCHORS")
print(anchors.head(15)[["team","american_odds","model_prob_ml","ev"]])

print("\nBEST EV LEGS")
print(ev_legs.head(15)[["team","american_odds","model_prob_ml","ev"]])

# --------------------
# 4-leg mixed (2 anchors + 2 EV)
# --------------------
if len(anchors) >= 2 and len(ev_legs) >= 2:
    mix = pd.concat([anchors.head(2), ev_legs.head(2)], ignore_index=True)
    mix = mix.drop_duplicates("game_key").head(4)

    if len(mix) == 4:
        dec, p, ev = parlay_metrics(mix)
        print("\n4-LEG MIXED PARLAY")
        print(mix[["team","american_odds","model_prob_ml","ev"]])
        print(f"\nDecimal: {dec:.3f}")
        print(f"Hit Prob: {p:.4f} ({p*100:.2f}%)")
        print(f"EV: {ev:.3f}")
    else:
        print("\nNot enough unique legs for mixed parlay.")
else:
    print("\nNot enough anchors or EV legs.")

In [ ]:
import pandas as pd
import numpy as np
import re

# ---------- Helpers ----------
def american_to_implied(odds):
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return abs(odds) / (abs(odds) + 100.0)

def to_float_pct(x):
    # accepts "63.8%" or 63.8 or None
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = str(x).strip().replace("%","")
    if s in ["--", "", "nan", "None"]:
        return np.nan
    try:
        return float(s) / 100.0
    except:
        return np.nan

def parse_line(prop_str):
    # expects "o9.5" or "u14.5" etc.
    if prop_str is None:
        return (None, np.nan)
    s = str(prop_str).strip().lower()
    m = re.match(r"^([ou])\s*([0-9]+(\.[0-9]+)?)$", s)
    if not m:
        return (None, np.nan)
    return (m.group(1), float(m.group(2)))

# ---------- INPUT ----------
# OPTION A (BEST): If you have a CSV, set this to the path and run.
CSV_PATH = None  # e.g. "/content/props.csv"

# OPTION B: If you've already built a dataframe named props_raw, we'll use it.
# props_raw should contain columns similar to:
# ["player","game","prop_type","prop","odds","avg","l5","l10","y2025","h_away","h2h","dvp_l5","dvp_l10","dvp_2025","dvp_h_away"]

if CSV_PATH:
    props_raw = pd.read_csv(CSV_PATH)

assert "props_raw" in globals(), "Create props_raw (dataframe) or set CSV_PATH to your props CSV."

# ---------- STANDARDIZE ----------
rename_map = {
    "PLAYER":"player","Player":"player","player":"player",
    "GAME":"game","Game":"game","game":"game",
    "PROP":"prop_type","Prop":"prop_type","prop_type":"prop_type",
    "OUTCOME":"prop","Outcome":"prop","prop":"prop",
    "ODDS":"odds","Odds":"odds","odds":"odds",
    "AVG":"avg","Avg":"avg","avg":"avg",
    "IM PROB %":"im_prob","IM_PROB":"im_prob","im_prob":"im_prob",
    "L5":"l5","L10":"l10","2025":"y2025","HM/AW":"h_away","H2H":"h2h",
    "DVP L5":"dvp_l5","DVP L10":"dvp_l10","DVP 2025":"dvp_2025","DVP HM/AW":"dvp_h_away"
}
props_df = props_raw.rename(columns={c: rename_map.get(c, c) for c in props_raw.columns}).copy()

# coerce needed cols
needed = ["player","game","prop_type","prop","odds","avg","l5","l10","y2025","h_away","h2h","dvp_l5","dvp_l10","dvp_2025","dvp_h_away"]
for c in needed:
    if c not in props_df.columns:
        props_df[c] = np.nan

# parse o/u + line
props_df["ou"], props_df["line"] = zip(*props_df["prop"].apply(parse_line))

# numeric odds (strip +)
props_df["odds"] = props_df["odds"].astype(str).str.replace("−","-", regex=False)
props_df["odds"] = props_df["odds"].str.replace("+","", regex=False)
props_df["odds"] = pd.to_numeric(props_df["odds"], errors="coerce")

# implied prob from odds (market)
props_df["implied_prob"] = props_df["odds"].apply(lambda x: american_to_implied(x) if pd.notna(x) else np.nan)

# convert hit rates to decimals
for c in ["avg","l5","l10","y2025","h_away","h2h","dvp_l5","dvp_l10","dvp_2025","dvp_h_away","im_prob"]:
    props_df[c] = props_df[c].apply(to_float_pct)

# basic key
props_df["key"] = props_df["player"].astype(str) + " | " + props_df["game"].astype(str) + " | " + props_df["prop_type"].astype(str) + " | " + props_df["prop"].astype(str)

print("✅ props_df built:", props_df.shape)
print("Cols:", list(props_df.columns))
props_df.head(10)

In [ ]:
# ==============================
# NCAAM STABLE — EXPORT DIAGNOSTICS FOR NBA PORT
# Paste at bottom of notebook and run AFTER your model completes.
# ==============================

import pandas as pd
import numpy as np
import json, inspect, re, os, datetime

# ---- 1) Try to locate your main dataframe automatically ----
candidates = ["df", "games", "slate", "projections", "results", "card", "final_df"]
main_name = None
main_df = None

for name in candidates:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        main_name = name
        main_df = globals()[name]
        break

if main_df is None:
    # fallback: pick the largest DataFrame in memory
    dfs = [(k,v) for k,v in globals().items() if isinstance(v, pd.DataFrame)]
    if dfs:
        main_name, main_df = sorted(dfs, key=lambda kv: kv[1].shape[0]*kv[1].shape[1], reverse=True)[0]

if main_df is None:
    raise ValueError("No DataFrame found. Your notebook may store results in a dict/list instead of df.")

print(f"[FOUND MAIN DF] name={main_name} shape={main_df.shape}")

# ---- 2) Save schema + sample rows ----
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = f"ncaam_stable_export_{ts}"
os.makedirs(out_dir, exist_ok=True)

schema = {
    "df_name": main_name,
    "shape": list(main_df.shape),
    "columns": list(map(str, main_df.columns)),
    "dtypes": {str(c): str(main_df[c].dtype) for c in main_df.columns},
}

with open(f"{out_dir}/schema.json", "w") as f:
    json.dump(schema, f, indent=2)

main_df.head(25).to_csv(f"{out_dir}/sample_head25.csv", index=False)
main_df.tail(25).to_csv(f"{out_dir}/sample_tail25.csv", index=False)

print(f"[SAVED] {out_dir}/schema.json")
print(f"[SAVED] {out_dir}/sample_head25.csv")
print(f"[SAVED] {out_dir}/sample_tail25.csv")

# ---- 3) Capture key scalar parameters if they exist ----
param_keys = [
    "SIGMA_TOTAL","SIGMA_SPREAD","sigma_total","sigma_spread",
    "LOGIT_K","logit_k","K_SPREAD","k_spread",
    "HOME_ADV","home_adv","HCA","hca",
    "WEIGHTS","weights","W","w",
    "EDGE_MIN","edge_min","MIN_EDGE","min_edge",
    "PROB_MIN","prob_min","MIN_PROB","min_prob",
    "KELLY_CAP","kelly_cap","MAX_KELLY","max_kelly"
]

params_found = {}
for k in param_keys:
    if k in globals():
        v = globals()[k]
        # JSON-safe conversion
        if isinstance(v, (np.generic,)):
            v = v.item()
        if isinstance(v, (np.ndarray,)):
            v = v.tolist()
        params_found[k] = v

with open(f"{out_dir}/params_found.json", "w") as f:
    json.dump(params_found, f, indent=2)

print(f"[SAVED] {out_dir}/params_found.json (keys={len(params_found)})")

# ---- 4) Save intermediate columns if they exist (projection/probability core) ----
# This helps me replicate your exact pipeline even if function code isn't shown.
key_like = [
    "PROJ","PROJECT","PRED","MEAN","LAMBDA","MU",
    "SPREAD","TOTAL","WIN_PROB","OVER_PROB","UNDER_PROB",
    "FAIR","IMPLIED","BOOK","EDGE","EV","KELLY",
    "PACE","POSS","PPP","ORTG","DRTG","NET"
]

cols_interest = []
for c in main_df.columns:
    cs = str(c).upper()
    if any(k in cs for k in key_like):
        cols_interest.append(c)

main_df[cols_interest].head(200).to_csv(f"{out_dir}/core_cols_head200.csv", index=False)
print(f"[SAVED] {out_dir}/core_cols_head200.csv (cols={len(cols_interest)})")

# ---- 5) OPTIONAL: If you built your model with custom functions, export source when possible ----
# Add function names here if you know them (safe if missing).
function_names = [
    "project_game","project_points","calc_probs","spread_to_winprob","total_to_prob",
    "skellam_prob","poisson_prob","kelly_fraction","run_model","stable_run"
]

fn_dump = {}
for fn in function_names:
    if fn in globals() and callable(globals()[fn]):
        try:
            fn_dump[fn] = inspect.getsource(globals()[fn])
        except:
            fn_dump[fn] = "<source unavailable>"

with open(f"{out_dir}/functions_source.json", "w") as f:
    json.dump(fn_dump, f, indent=2)

print(f"[SAVED] {out_dir}/functions_source.json (fns={len(fn_dump)})")

print("\nDONE. Download the folder files from the Colab file browser:")
print(out_dir)

In [ ]:
# ==============================
# NCAAM STABLE — EXPORTER v2
# Handles DataFrame / dict / list structures
# Paste at bottom and run AFTER model executes
# ==============================

import pandas as pd
import numpy as np
import json, os, datetime, inspect

ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = f"ncaam_stable_export_{ts}"
os.makedirs(out_dir, exist_ok=True)

print("=== INVENTORYING MEMORY ===")

# ----------------------------
# 1) Inventory objects
# ----------------------------
objs = []
for k, v in globals().items():
    if k.startswith("_"):
        continue
    try:
        t = type(v).__name__
        size = None
        if isinstance(v, (list, tuple, set, dict)):
            size = len(v)
        objs.append((k, t, size))
    except:
        continue

inv = pd.DataFrame(objs, columns=["name","type","size"]).sort_values(["type","size"], ascending=[True, False])
inv.to_csv(f"{out_dir}/inventory.csv", index=False)

print(f"[SAVED] {out_dir}/inventory.csv")
display(inv.head(30))

# ----------------------------
# 2) Detect main object
# ----------------------------
main_name = None
main_obj = None

# Priority 1: DataFrame
dfs = [(k,v) for k,v in globals().items() if isinstance(v, pd.DataFrame)]
if dfs:
    main_name, main_obj = sorted(dfs, key=lambda kv: kv[1].shape[0]*kv[1].shape[1], reverse=True)[0]
    print(f"[FOUND DataFrame] {main_name}")

# Priority 2: list of dicts
if main_obj is None:
    lod = [(k,v) for k,v in globals().items()
           if isinstance(v, list) and len(v)>0 and isinstance(v[0], dict)]
    if lod:
        main_name, main_obj = sorted(lod, key=lambda kv: len(kv[1]), reverse=True)[0]
        print(f"[FOUND list of dicts] {main_name}")

# Priority 3: dict of lists
if main_obj is None:
    dol = []
    for k,v in globals().items():
        if isinstance(v, dict) and len(v)>0:
            vals = list(v.values())
            if all(isinstance(x, list) for x in vals):
                dol.append((k,v))
    if dol:
        main_name, main_obj = sorted(dol, key=lambda kv: len(kv[1]), reverse=True)[0]
        print(f"[FOUND dict of lists] {main_name}")

# Priority 4: list of lists
if main_obj is None:
    lol = [(k,v) for k,v in globals().items()
           if isinstance(v, list) and len(v)>0 and isinstance(v[0], (list, tuple))]
    if lol:
        main_name, main_obj = sorted(lol, key=lambda kv: len(kv[1]), reverse=True)[0]
        print(f"[FOUND list of lists] {main_name}")

if main_obj is None:
    raise ValueError("No structured tabular object found. Check inventory.csv.")

# ----------------------------
# 3) Convert to DataFrame
# ----------------------------
if isinstance(main_obj, pd.DataFrame):
    df_export = main_obj.copy()

elif isinstance(main_obj, list) and isinstance(main_obj[0], dict):
    df_export = pd.DataFrame(main_obj)

elif isinstance(main_obj, dict):
    df_export = pd.DataFrame(main_obj)

elif isinstance(main_obj, list):
    df_export = pd.DataFrame(main_obj)

else:
    raise ValueError("Unable to convert detected object into DataFrame.")

print(f"[EXPORTING] {main_name} → shape={df_export.shape}")

# ----------------------------
# 4) Save schema + samples
# ----------------------------
schema = {
    "source_object": main_name,
    "shape": list(df_export.shape),
    "columns": list(map(str, df_export.columns)),
    "dtypes": {str(c): str(df_export[c].dtype) for c in df_export.columns}
}

with open(f"{out_dir}/schema.json", "w") as f:
    json.dump(schema, f, indent=2)

df_export.head(50).to_csv(f"{out_dir}/sample_head50.csv", index=False)

print(f"[SAVED] {out_dir}/schema.json")
print(f"[SAVED] {out_dir}/sample_head50.csv")

# ----------------------------
# 5) Extract core projection columns
# ----------------------------
key_like = [
    "PROJ","PROJECT","PRED","MEAN","LAMBDA","MU",
    "SPREAD","TOTAL","WIN_PROB","OVER_PROB","UNDER_PROB",
    "FAIR","IMPLIED","BOOK","EDGE","EV","KELLY",
    "PACE","POSS","PPP","ORTG","DRTG","NET"
]

cols_interest = []
for c in df_export.columns:
    if any(k in str(c).upper() for k in key_like):
        cols_interest.append(c)

if cols_interest:
    df_export[cols_interest].head(200).to_csv(f"{out_dir}/core_cols_head200.csv", index=False)
    print(f"[SAVED] {out_dir}/core_cols_head200.csv (cols={len(cols_interest)})")
else:
    print("No obvious projection columns detected.")

print("\nDONE. Download the folder from the left Colab file browser:")
print(out_dir)

In [ ]:
# ==============================
# NCAAM STABLE — EXPORTER v2.1
# Fix: iterate over snapshot of globals()
# Paste at bottom and run AFTER model executes
# ==============================

import pandas as pd
import numpy as np
import json, os, datetime

ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
out_dir = f"ncaam_stable_export_{ts}"
os.makedirs(out_dir, exist_ok=True)

print("=== INVENTORYING MEMORY ===")

# Snapshot globals to avoid "dictionary changed size during iteration"
g = dict(globals())

# ----------------------------
# 1) Inventory objects
# ----------------------------
objs = []
for k, v in g.items():
    if str(k).startswith("_"):
        continue
    try:
        t = type(v).__name__
        size = None
        if isinstance(v, (list, tuple, set, dict)):
            size = len(v)
        elif isinstance(v, pd.DataFrame):
            size = int(v.shape[0])
        objs.append((k, t, size))
    except:
        continue

inv = pd.DataFrame(objs, columns=["name","type","size"]).sort_values(["type","size"], ascending=[True, False])
inv.to_csv(f"{out_dir}/inventory.csv", index=False)

print(f"[SAVED] {out_dir}/inventory.csv")
display(inv.head(30))

# ----------------------------
# 2) Detect main object
# ----------------------------
main_name = None
main_obj = None

# Priority 1: DataFrame
dfs = [(k,v) for k,v in g.items() if isinstance(v, pd.DataFrame)]
if dfs:
    main_name, main_obj = sorted(dfs, key=lambda kv: kv[1].shape[0]*kv[1].shape[1], reverse=True)[0]
    print(f"[FOUND DataFrame] {main_name}")

# Priority 2: list of dicts
if main_obj is None:
    lod = [(k,v) for k,v in g.items()
           if isinstance(v, list) and len(v)>0 and isinstance(v[0], dict)]
    if lod:
        main_name, main_obj = sorted(lod, key=lambda kv: len(kv[1]), reverse=True)[0]
        print(f"[FOUND list of dicts] {main_name}")

# Priority 3: dict of lists
if main_obj is None:
    dol = []
    for k,v in g.items():
        if isinstance(v, dict) and len(v)>0:
            vals = list(v.values())
            if all(isinstance(x, list) for x in vals):
                dol.append((k,v))
    if dol:
        main_name, main_obj = sorted(dol, key=lambda kv: len(kv[1]), reverse=True)[0]
        print(f"[FOUND dict of lists] {main_name}")

# Priority 4: list of lists
if main_obj is None:
    lol = [(k,v) for k,v in g.items()
           if isinstance(v, list) and len(v)>0 and isinstance(v[0], (list, tuple))]
    if lol:
        main_name, main_obj = sorted(lol, key=lambda kv: len(kv[1]), reverse=True)[0]
        print(f"[FOUND list of lists] {main_name}")

if main_obj is None:
    raise ValueError("No structured tabular object found. Check inventory.csv.")

# ----------------------------
# 3) Convert to DataFrame
# ----------------------------
if isinstance(main_obj, pd.DataFrame):
    df_export = main_obj.copy()
elif isinstance(main_obj, list) and isinstance(main_obj[0], dict):
    df_export = pd.DataFrame(main_obj)
elif isinstance(main_obj, dict):
    df_export = pd.DataFrame(main_obj)
elif isinstance(main_obj, list):
    df_export = pd.DataFrame(main_obj)
else:
    raise ValueError("Unable to convert detected object into DataFrame.")

print(f"[EXPORTING] {main_name} → shape={df_export.shape}")

# ----------------------------
# 4) Save schema + samples
# ----------------------------
schema = {
    "source_object": main_name,
    "shape": list(df_export.shape),
    "columns": list(map(str, df_export.columns)),
    "dtypes": {str(c): str(df_export[c].dtype) for c in df_export.columns}
}
with open(f"{out_dir}/schema.json", "w") as f:
    json.dump(schema, f, indent=2)

df_export.head(50).to_csv(f"{out_dir}/sample_head50.csv", index=False)

print(f"[SAVED] {out_dir}/schema.json")
print(f"[SAVED] {out_dir}/sample_head50.csv")

# ----------------------------
# 5) Extract core projection columns
# ----------------------------
key_like = [
    "PROJ","PROJECT","PRED","MEAN","LAMBDA","MU",
    "SPREAD","TOTAL","WIN_PROB","OVER_PROB","UNDER_PROB",
    "FAIR","IMPLIED","BOOK","EDGE","EV","KELLY",
    "PACE","POSS","PPP","ORTG","DRTG","NET"
]
cols_interest = [c for c in df_export.columns if any(k in str(c).upper() for k in key_like)]

if cols_interest:
    df_export[cols_interest].head(200).to_csv(f"{out_dir}/core_cols_head200.csv", index=False)
    print(f"[SAVED] {out_dir}/core_cols_head200.csv (cols={len(cols_interest)})")
else:
    print("No obvious projection columns detected.")

print("\nDONE. Download the folder from the left Colab file browser:")
print(out_dir)

In [ ]:
import pandas as pd, numpy as np, inspect

g = dict(globals())
cands = []
for k,v in g.items():
    if k.startswith("_"):
        continue
    if inspect.ismodule(v) or inspect.isfunction(v):
        continue
    info = None
    if isinstance(v, pd.DataFrame):
        info = f"DataFrame {v.shape}"
    elif isinstance(v, pd.Series):
        info = f"Series len={len(v)}"
    elif isinstance(v, np.ndarray):
        info = f"ndarray shape={v.shape}"
    elif isinstance(v, dict) and len(v)>0:
        info = f"dict keys={len(v)}"
    elif isinstance(v, list) and len(v)>0:
        info = f"list len={len(v)} first={type(v[0]).__name__}"
    if info:
        cands.append((k, info))

print("POSSIBLE OUTPUT OBJECTS:")
for k,info in sorted(cands, key=lambda x: x[0])[:120]:
    print("-", k, ":", info)

POSSIBLE OUTPUT OBJECTS:
- In : list len=12 first=str
- book : dict keys=4
- candidates : list len=7 first=str
- g : dict keys=41
- game : dict keys=7
- games_df : DataFrame (68, 6)
- games_raw : list len=68 first=dict
- inv : DataFrame (22, 3)
- market : dict keys=3
- objs : list len=22 first=tuple
- outcome : dict keys=2
- params : dict keys=3


In [ ]:
import pandas as pd, numpy as np, json, os, datetime

if "EXPORT_OBJ" not in globals():
    raise ValueError("Set EXPORT_OBJ = your final output object first.")

obj = EXPORT_OBJ
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
out_dir = f"ncaam_stable_export_{ts}"
os.makedirs(out_dir, exist_ok=True)

def to_df(x):
    if isinstance(x, pd.DataFrame): return x
    if isinstance(x, list) and len(x) and isinstance(x[0], dict): return pd.DataFrame(x)
    if isinstance(x, dict): return pd.DataFrame(x)
    if isinstance(x, list): return pd.DataFrame(x)
    if isinstance(x, np.ndarray): return pd.DataFrame(x)
    raise ValueError(f"Can't convert type={type(x)} to DataFrame")

df_export = to_df(obj)
df_export.to_csv(f"{out_dir}/export.csv", index=False)

meta = {"type": str(type(obj)), "shape": list(df_export.shape), "columns": list(map(str, df_export.columns))}
with open(f"{out_dir}/meta.json","w") as f:
    json.dump(meta, f, indent=2)

print("[SAVED]", f"{out_dir}/export.csv")
print("[SAVED]", f"{out_dir}/meta.json")
display(df_export.head(20))

ValueError: Set EXPORT_OBJ = your final output object first.

In [ ]:
import pandas as pd, numpy as np, inspect

g = dict(globals())
candidates = []

for k,v in g.items():
    if k.startswith("_"):
        continue
    if inspect.ismodule(v) or inspect.isfunction(v):
        continue

    if isinstance(v, pd.DataFrame):
        candidates.append((k, f"DataFrame shape={v.shape}"))
    elif isinstance(v, dict) and len(v)>0:
        candidates.append((k, f"dict keys={len(v)}"))
    elif isinstance(v, list) and len(v)>0:
        candidates.append((k, f"list len={len(v)} first={type(v[0]).__name__}"))
    elif isinstance(v, np.ndarray):
        candidates.append((k, f"ndarray shape={v.shape}"))

print("POSSIBLE MODEL OUTPUT VARIABLES:\n")
for k,info in sorted(candidates):
    print("-", k, ":", info)

POSSIBLE MODEL OUTPUT VARIABLES:

- In : list len=14 first=str
- book : dict keys=4
- candidates : list len=7 first=str
- cands : list len=12 first=tuple
- g : dict keys=74
- game : dict keys=7
- games_df : DataFrame shape=(68, 6)
- games_raw : list len=68 first=dict
- inv : DataFrame shape=(22, 3)
- market : dict keys=3
- objs : list len=22 first=tuple
- outcome : dict keys=2
- params : dict keys=3


In [ ]:
EXPORT_OBJ = games_df

In [ ]:
import pandas as pd, numpy as np, json, os, datetime

if "EXPORT_OBJ" not in globals():
    raise ValueError("Set EXPORT_OBJ = your final output object first.")

obj = EXPORT_OBJ
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
out_dir = f"ncaam_stable_export_{ts}"
os.makedirs(out_dir, exist_ok=True)

def to_df(x):
    if isinstance(x, pd.DataFrame): return x
    if isinstance(x, list) and len(x) and isinstance(x[0], dict): return pd.DataFrame(x)
    if isinstance(x, dict): return pd.DataFrame(x)
    if isinstance(x, list): return pd.DataFrame(x)
    if isinstance(x, np.ndarray): return pd.DataFrame(x)
    raise ValueError(f"Can't convert type={type(x)} to DataFrame")

df_export = to_df(obj)
df_export.to_csv(f"{out_dir}/export.csv", index=False)

meta = {"type": str(type(obj)), "shape": list(df_export.shape), "columns": list(map(str, df_export.columns))}
with open(f"{out_dir}/meta.json","w") as f:
    json.dump(meta, f, indent=2)

print("[SAVED]", f"{out_dir}/export.csv")
print("[SAVED]", f"{out_dir}/meta.json")
display(df_export.head(20))

[SAVED] ncaam_stable_export_20260301_003943/export.csv
[SAVED] ncaam_stable_export_20260301_003943/meta.json


,home_team,away_team,home_ml,away_ml,home_implied_prob,away_implied_prob
0,Grambling St Tigers,Florida A&M Rattlers,9.00,1.06,0.111111,0.943396
1,Western Carolina Catamounts,Furman Paladins,NaN,NaN,NaN,NaN
2,Creighton Bluejays,Providence Friars,2.60,1.48,0.384615,0.675676
3,West Virginia Mountaineers,BYU Cougars,1.11,6.00,0.900901,0.166667
4,Sam Houston St Bearkats,Missouri St Bears,NaN,NaN,NaN,NaN
5,Wake Forest Demon Deacons,Syracuse Orange,1.02,12.00,0.980392,0.083333
6,Princeton Tigers,Dartmouth Big Green,1.01,34.00,0.990099,0.029412
7,Pennsylvania Quakers,Harvard Crimson,1.24,4.00,0.806452,0.250000
8,Idaho State Bengals,Weber State Wildcats,29.00,1.00,0.034483,1.000000
9,LSU Tigers,Oklahoma Sooners,3.90,1.24,0.256410,0.806452


In [ ]:
import pandas as pd, numpy as np

g = dict(globals())

# show likely result objects (bigger than 6 cols, or named like model outputs)
hits = []
for k,v in g.items():
    if isinstance(v, pd.DataFrame):
        hits.append((k, v.shape[0], v.shape[1], list(v.columns)[:12]))
    elif isinstance(v, dict) and len(v)>0 and any(s in k.lower() for s in ["proj","model","result","output","stable","edge","ev"]):
        hits.append((k, "dict", len(v), None))

hits = sorted(hits, key=lambda x: (x[2] if isinstance(x[2], int) else -1), reverse=True)

print("DATAFRAMES IN MEMORY (largest columns first):")
for h in hits[:30]:
    if isinstance(h[1], int):
        print(f"- {h[0]} : shape=({h[1]},{h[2]}) cols_preview={h[3]}")
    else:
        print(f"- {h[0]} : dict keys={h[2]}")

DATAFRAMES IN MEMORY (largest columns first):
- games_df : shape=(68,6) cols_preview=['home_team', 'away_team', 'home_ml', 'away_ml', 'home_implied_prob', 'away_implied_prob']
- EXPORT_OBJ : shape=(68,6) cols_preview=['home_team', 'away_team', 'home_ml', 'away_ml', 'home_implied_prob', 'away_implied_prob']
- obj : shape=(68,6) cols_preview=['home_team', 'away_team', 'home_ml', 'away_ml', 'home_implied_prob', 'away_implied_prob']
- df_export : shape=(68,6) cols_preview=['home_team', 'away_team', 'home_ml', 'away_ml', 'home_implied_prob', 'away_implied_prob']
- inv : shape=(22,3) cols_preview=['name', 'type', 'size']


In [3]:
# =========================
# STABLE v3 — NCAAB RUN
# =========================

SPORT = "ncaab"
SLATE_DATE = "2026-02-29"
TIMEZONE = "America/Indiana/Indianapolis"

# Core Toggles
OUTSIDE_ON = True
STRICT_INJURY_GATE = True
TEMPO_ADJUSTMENTS = True
PUBLIC_SHARP_SPLIT = True
CORRELATION_CHECK = True

# Kelly Settings
KELLY_MODE = "HALF"   # FULL or HALF
MIN_EDGE_PERCENT = 0.05  # 5% minimum for props
MIN_SPREAD_EDGE = 1.5    # 1.5 pts minimum edge

UNIT_CAP = 1.0
UNIT_MIN = 0.25

print("Parameters locked for:", SPORT, SLATE_DATE)

Parameters locked for: ncaab 2026-02-29


In [4]:
import pandas as pd
import numpy as np
import math
from scipy.stats import skellam, poisson
from datetime import datetime

pd.set_option("display.max_columns", None)

print("Environment ready.")

Environment ready.


In [5]:
games_df = pull_market_data(
    sport=SPORT,
    date=SLATE_DATE,
    outside=OUTSIDE_ON
)

print("Games pulled:", len(games_df))
games_df.head()

NameError: name 'pull_market_data' is not defined

In [7]:
# ==============================
# Stable Helper: pull_market_data()
# (Defines the missing function)
# ==============================

import os
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

def _sport_map(s: str) -> str:
    s = (s or "").lower().strip()
    if s in ["ncaab", "college", "cbb", "basketball_ncaab"]:
        return "basketball_ncaab"
    return s  # allow passing already-correct OddsAPI sport key

def _to_utc_window(date_str: str):
    # date_str format: YYYY-MM-DD
    day = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    start_utc = day
    end_utc = day + timedelta(days=1)
    return start_utc, end_utc

def american_to_implied(odds: float):
    if odds is None or pd.isna(odds):
        return None
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return (-odds) / ((-odds) + 100.0)

def pull_market_data(
    sport: str,
    date: str,
    outside: bool = True,
    regions: str = "us",
    markets: str = "h2h,spreads,totals",
    odds_format: str = "american",
    date_filter: bool = True,
    preferred_book: str | None = None,   # e.g. "draftkings", "fanduel" (optional)
):
    """
    Pull NCAAB odds from The Odds API v4 and return a WIDE game table:
    home_team, away_team, commence_time,
    home_ml, away_ml,
    spread_home_line, spread_home_odds, spread_away_line, spread_away_odds,
    total_line, over_odds, under_odds,
    implied probs (ml) convenience fields
    """

    # Resolve API key from env OR an existing API_KEY variable (if your notebook already defines it)
    api_key = os.environ.get("ODDS_API_KEY", None)
    if api_key is None:
        api_key = globals().get("API_KEY", None)
    if not api_key:
        raise ValueError("Missing ODDS API key. Set os.environ['ODDS_API_KEY'] or define API_KEY in a prior cell.")

    sport_key = _sport_map(sport)
    url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds"

    params = {
        "apiKey": api_key,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,   # 'american' recommended
    }

    if date_filter:
        start_utc, end_utc = _to_utc_window(date)
        params["commenceTimeFrom"] = start_utc.isoformat().replace("+00:00", "Z")
        params["commenceTimeTo"] = end_utc.isoformat().replace("+00:00", "Z")

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API Error {r.status_code}: {r.text}")

    games_raw = r.json()

    rows = []
    for g in games_raw:
        home = g.get("home_team")
        away = g.get("away_team")
        commence = g.get("commence_time")

        # choose a book: either first book, or match preferred_book key
        books = g.get("bookmakers", []) or []
        book = None
        if preferred_book:
            for b in books:
                if (b.get("key") or "").lower() == preferred_book.lower():
                    book = b
                    break
        if book is None and books:
            book = books[0]

        # defaults
        home_ml = away_ml = None
        spread_home_line = spread_home_odds = None
        spread_away_line = spread_away_odds = None
        total_line = over_odds = under_odds = None

        if book:
            for m in book.get("markets", []):
                key = m.get("key")

                if key == "h2h":
                    for o in m.get("outcomes", []):
                        if o.get("name") == home:
                            home_ml = o.get("price")
                        elif o.get("name") == away:
                            away_ml = o.get("price")

                elif key == "spreads":
                    for o in m.get("outcomes", []):
                        if o.get("name") == home:
                            spread_home_line = o.get("point")
                            spread_home_odds = o.get("price")
                        elif o.get("name") == away:
                            spread_away_line = o.get("point")
                            spread_away_odds = o.get("price")

                elif key == "totals":
                    for o in m.get("outcomes", []):
                        # totals outcomes typically named "Over"/"Under"
                        nm = (o.get("name") or "").lower()
                        total_line = o.get("point")  # same for both outcomes
                        if "over" in nm:
                            over_odds = o.get("price")
                        elif "under" in nm:
                            under_odds = o.get("price")

        rows.append({
            "home_team": home,
            "away_team": away,
            "commence_time": commence,

            "home_ml": home_ml,
            "away_ml": away_ml,

            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,

            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,

            "home_ml_implied": american_to_implied(home_ml) if odds_format == "american" else (1/float(home_ml) if home_ml else None),
            "away_ml_implied": american_to_implied(away_ml) if odds_format == "american" else (1/float(away_ml) if away_ml else None),
        })

    df = pd.DataFrame(rows)

    # Quick cleanup: drop games missing teams
    df = df.dropna(subset=["home_team", "away_team"]).reset_index(drop=True)

    print(f"✅ pull_market_data: pulled {len(df)} games for {sport_key} on {date} (date_filter={date_filter})")
    return df

In [8]:
games_df = pull_market_data(
    sport=SPORT,          # "ncaab" or "basketball_ncaab"
    date=SLATE_DATE,      # "2026-02-29"
    outside=OUTSIDE_ON
)
games_df.head()

ValueError: day is out of range for month

In [9]:
# 2026 is NOT a leap year → 2/29/2026 is invalid.
# Sunday slate should be 2026-03-01 (Sunday) instead.

SPORT = "ncaab"
SLATE_DATE = "2026-03-01"   # Sunday
OUTSIDE_ON = True

print("Running slate date:", SLATE_DATE)

Running slate date: 2026-03-01


In [10]:
games_df = pull_market_data(
    sport=SPORT,
    date=SLATE_DATE,
    outside=OUTSIDE_ON
)
games_df.head()

✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,-110.0,-120.0,1.5,-154,-1.5,120,174.5,-105,-125,0.523810,0.545455
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-16.5,-106,16.5,-125,145.5,-112,-118,NaN,NaN
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-120.0,-106.0,-0.5,-122,0.5,-108,153.5,-114,-114,0.545455,0.514563
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,NaN,NaN,-8.5,-102,8.5,-130,146.5,-108,-122,NaN,NaN
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-770.0,470.0,-8.5,-110,8.5,-120,127.5,-122,-108,0.885057,0.175439


In [11]:
# Clean slate before modeling

# Drop games missing spreads or totals
games_df = games_df.dropna(subset=["spread_home_line", "total_line"])

# If ML missing, derive pseudo-implied from spread odds (optional)
games_df["home_ml_implied"] = games_df["home_ml_implied"].fillna(0)
games_df["away_ml_implied"] = games_df["away_ml_implied"].fillna(0)

print("Games after cleanup:", len(games_df))
games_df.head()

Games after cleanup: 26


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,-110.0,-120.0,1.5,-154,-1.5,120,174.5,-105,-125,0.523810,0.545455
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-16.5,-106,16.5,-125,145.5,-112,-118,0.000000,0.000000
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-120.0,-106.0,-0.5,-122,0.5,-108,153.5,-114,-114,0.545455,0.514563
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,NaN,NaN,-8.5,-102,8.5,-130,146.5,-108,-122,0.000000,0.000000
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-770.0,470.0,-8.5,-110,8.5,-120,127.5,-122,-108,0.885057,0.175439


In [12]:
# ==============================
# RUN OUR NCAAB ENGINE ON games_df
# ==============================

import pandas as pd
import numpy as np

# 1) Ensure expected columns exist + numeric
num_cols = [
    "home_ml","away_ml",
    "spread_home_line","spread_home_odds","spread_away_line","spread_away_odds",
    "total_line","over_odds","under_odds"
]
for c in num_cols:
    if c in games_df.columns:
        games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

# 2) If your notebook already defined the engine, use it
engine_fn = None
for name in ["run_professional_sim", "run_simulation_pipeline"]:
    if name in globals() and callable(globals()[name]):
        engine_fn = globals()[name]
        print(f"✅ Using engine: {name}")
        break

if engine_fn is None:
    raise NameError("Could not find run_professional_sim or run_simulation_pipeline in this notebook. Scroll up once and run the cell that defines the engine.")

# 3) Run engine (default sims if function supports it)
try:
    model_df = engine_fn(games_df.copy(), sims=5000)
except TypeError:
    model_df = engine_fn(games_df.copy())

print("✅ Model output rows:", len(model_df))
model_df.head()

NameError: Could not find run_professional_sim or run_simulation_pipeline in this notebook. Scroll up once and run the cell that defines the engine.

In [13]:
# ==============================
# INLINE CORE MODEL (Skellam + Poisson Base)
# ==============================

import numpy as np
from scipy.stats import skellam

df = games_df.copy()

# Simple baseline implied from market total & spread
df["proj_home_score"] = (df["total_line"] / 2) - (df["spread_home_line"] / 2)
df["proj_away_score"] = (df["total_line"] / 2) + (df["spread_home_line"] / 2)

df["model_spread"] = df["proj_home_score"] - df["proj_away_score"]
df["model_total"] = df["proj_home_score"] + df["proj_away_score"]

# Skellam distribution for cover probability
df["cover_prob_home"] = 1 - skellam.cdf(
    df["spread_home_line"],
    df["proj_home_score"],
    df["proj_away_score"]
)

# Over probability using normal approx
df["over_prob"] = 1 - skellam.cdf(
    df["total_line"],
    df["proj_home_score"],
    df["proj_away_score"]
)

df.head()


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied,proj_home_score,proj_away_score,model_spread,model_total,cover_prob_home,over_prob
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,-110.0,-120.0,1.5,-154,-1.5,120,174.5,-105,-125,0.523810,0.545455,86.5,88.0,-1.5,174.5,0.410128,0.0
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-16.5,-106,16.5,-125,145.5,-112,-118,0.000000,0.000000,81.0,64.5,16.5,145.5,0.996958,0.0
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-120.0,-106.0,-0.5,-122,0.5,-108,153.5,-114,-114,0.545455,0.514563,77.0,76.5,0.5,153.5,0.532183,0.0
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,NaN,NaN,-8.5,-102,8.5,-130,146.5,-108,-122,0.000000,0.000000,77.5,69.0,8.5,146.5,0.920156,0.0
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-770.0,470.0,-8.5,-110,8.5,-120,127.5,-122,-108,0.885057,0.175439,68.0,59.5,8.5,127.5,0.934178,0.0


In [14]:
# ==============================
# EV + Kelly Calculation
# ==============================

def american_to_prob(odds):
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return -odds / (-odds + 100)

df["implied_spread_prob"] = df["spread_home_odds"].apply(american_to_prob)

df["edge"] = df["cover_prob_home"] - df["implied_spread_prob"]

# Half Kelly
df["kelly_units"] = np.maximum(0, df["edge"] * 2)

df = df.sort_values("edge", ascending=False)

df[[
    "home_team",
    "away_team",
    "spread_home_line",
    "cover_prob_home",
    "implied_spread_prob",
    "edge",
    "kelly_units"
]].head(15)

,home_team,away_team,spread_home_line,cover_prob_home,implied_spread_prob,edge,kelly_units
1,UC San Diego Tritons,Cal Poly Mustangs,-16.5,0.996958,0.514563,0.482395,0.964791
8,South Florida Bulls,Tulane Green Wave,-14.5,0.990103,0.512195,0.477908,0.955816
20,Siena Saints,Rider Broncs,-16.0,0.996365,0.523810,0.472555,0.945111
5,Davidson Wildcats,La Salle Explorers,-11.0,0.968596,0.523810,0.444786,0.889573
3,Utah State Aggies,Grand Canyon Antelopes,-8.5,0.920156,0.504950,0.415205,0.830410
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,-8.5,0.934178,0.523810,0.410369,0.820738
19,Temple Owls,Rice Owls,-7.5,0.896183,0.523810,0.372374,0.744748
13,Florida Atlantic Owls,Charlotte 49ers,-7.0,0.867059,0.523810,0.343250,0.686500
6,Maryland Terrapins,Rutgers Scarlet Knights,-5.5,0.823076,0.512195,0.310881,0.621762
7,UAB Blazers,North Texas Mean Green,-4.5,0.774727,0.523810,0.250918,0.501835


In [15]:
# =========================
# NCAAB STABLE RUN PARAMS
# =========================

SLATE_DATE = "2026-03-01"          # Sunday (2/29 is invalid in 2026)
TIMEZONE = "America/Indiana/Indianapolis"

OUTSIDE_ON = True
STRICT_INJURY_GATE = True          # kept for consistency (we'll enforce via filters if no injury feed)
HALF_KELLY = True

MAX_JUICE = -200                   # exclude heavy-juice lines
UNIT_MIN, UNIT_CAP = 0.25, 1.0

# Stability/Sensitivity
SENS_MARGIN_SHIFT = 1.0            # +/- points on mean margin
SENS_TOTAL_SHIFT  = 1.5            # +/- points on mean total

In [16]:
import pandas as pd
import numpy as np
from math import sqrt
from scipy.stats import norm

df = games_df.copy()

# Ensure numeric
num_cols = [
    "home_ml","away_ml",
    "spread_home_line","spread_home_odds","spread_away_line","spread_away_odds",
    "total_line","over_odds","under_odds"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop games missing spread or total (needed for our score-based engine)
df = df.dropna(subset=["spread_home_line","total_line"]).reset_index(drop=True)

print("✅ Games ready:", len(df))
df.head()

✅ Games ready: 26


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,-110.0,-120.0,1.5,-154,-1.5,120,174.5,-105,-125,0.523810,0.545455
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-16.5,-106,16.5,-125,145.5,-112,-118,0.000000,0.000000
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-120.0,-106.0,-0.5,-122,0.5,-108,153.5,-114,-114,0.545455,0.514563
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,NaN,NaN,-8.5,-102,8.5,-130,146.5,-108,-122,0.000000,0.000000
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-770.0,470.0,-8.5,-110,8.5,-120,127.5,-122,-108,0.885057,0.175439


In [17]:
# =========================
# PHASE 2–3: SCORING MODEL
# =========================

# Derive implied mean scores from market total & spread
# If spread_home_line = -8.5 (home -8.5), home expected to win by 8.5
df["mu_home"] = (df["total_line"] / 2) - (df["spread_home_line"] / 2)
df["mu_away"] = (df["total_line"] / 2) + (df["spread_home_line"] / 2)

# Variance proxy (Poisson sum / Skellam)
df["var_margin"] = (df["mu_home"] + df["mu_away"]).clip(lower=1.0)
df["sd_margin"]  = np.sqrt(df["var_margin"])

df["mean_margin"] = df["mu_home"] - df["mu_away"]          # = -spread_home_line
df["mean_total"]  = df["mu_home"] + df["mu_away"]          # = total_line

# ML win prob: P(margin > 0)
df["p_home_ml"] = 1 - norm.cdf((0 - df["mean_margin"]) / df["sd_margin"])
df["p_away_ml"] = 1 - df["p_home_ml"]

# Spread cover prob for HOME side at spread_home_line:
# Home bet wins if margin > -spread_home_line
df["p_home_cover"] = 1 - norm.cdf(((-df["spread_home_line"]) - df["mean_margin"]) / df["sd_margin"])

# Total: P(Over)
df["p_over"]  = 1 - norm.cdf((df["total_line"] - df["mean_total"]) / df["sd_margin"])
df["p_under"] = 1 - df["p_over"]

df[["home_team","away_team","spread_home_line","total_line","p_home_ml","p_home_cover","p_over"]].head()

,home_team,away_team,spread_home_line,total_line,p_home_ml,p_home_cover,p_over
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1.5,174.5,0.454797,0.5,0.5
1,UC San Diego Tritons,Cal Poly Mustangs,-16.5,145.5,0.914327,0.5,0.5
2,UNLV Rebels,Nevada Wolf Pack,-0.5,153.5,0.516096,0.5,0.5
3,Utah State Aggies,Grand Canyon Antelopes,-8.5,146.5,0.758743,0.5,0.5
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,-8.5,127.5,0.774207,0.5,0.5


In [18]:
# =========================
# PHASE 4: EV + KELLY
# =========================

def american_to_b(odds):
    """Return net decimal payout 'b' for Kelly, e.g. -110 -> 0.9091, +120 -> 1.2"""
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    return (odds/100.0) if odds > 0 else (100.0/abs(odds))

def kelly_fraction(p, b):
    """Kelly fraction f* = (p(b+1)-1)/b"""
    if pd.isna(p) or pd.isna(b) or b <= 0:
        return np.nan
    return (p*(b+1) - 1) / b

def ev_percent(p, b):
    """EV per 1 unit risked"""
    if pd.isna(p) or pd.isna(b) or b <= 0:
        return np.nan
    return p*b - (1-p)

def cap_units(f):
    if pd.isna(f) or f <= 0:
        return 0.0
    if HALF_KELLY:
        f = 0.5 * f
    # scale to "units" with caps
    u = min(UNIT_CAP, max(0.0, f * 4))  # 4x is a pragmatic mapping; keeps most plays in 0.25–1u
    if u > 0 and u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

# Build candidate rows
rows = []

for _, r in df.iterrows():
    home = r["home_team"]; away = r["away_team"]

    # ML home
    if not pd.isna(r.get("home_ml")):
        b = american_to_b(r["home_ml"])
        p = r["p_home_ml"]
        ev = ev_percent(p, b)
        k  = kelly_fraction(p, b)
        rows.append([home, away, "ML", f"{home} ML", r["home_ml"], p, ev, k])

    # Spread home
    if not pd.isna(r.get("spread_home_odds")):
        b = american_to_b(r["spread_home_odds"])
        p = r["p_home_cover"]
        ev = ev_percent(p, b)
        k  = kelly_fraction(p, b)
        rows.append([home, away, "Spread", f"{home} {r['spread_home_line']}", r["spread_home_odds"], p, ev, k])

    # Total over
    if not pd.isna(r.get("over_odds")):
        b = american_to_b(r["over_odds"])
        p = r["p_over"]
        ev = ev_percent(p, b)
        k  = kelly_fraction(p, b)
        rows.append([home, away, "Total", f"Over {r['total_line']}", r["over_odds"], p, ev, k])

    # Total under
    if not pd.isna(r.get("under_odds")):
        b = american_to_b(r["under_odds"])
        p = r["p_under"]
        ev = ev_percent(p, b)
        k  = kelly_fraction(p, b)
        rows.append([home, away, "Total", f"Under {r['total_line']}", r["under_odds"], p, ev, k])

cand = pd.DataFrame(rows, columns=[
    "home_team","away_team","market","bet","odds","model_prob","ev","kelly_f"
])

# Filter juice + NaNs + negative EV
cand["odds_num"] = pd.to_numeric(cand["odds"], errors="coerce")
cand = cand[cand["odds_num"] >= MAX_JUICE].copy()
cand = cand.dropna(subset=["model_prob","ev","kelly_f"])
cand = cand[cand["ev"] > 0].copy()

cand["units"] = cand["kelly_f"].apply(cap_units)
cand = cand[cand["units"] > 0].copy()

cand = cand.sort_values(["ev","model_prob"], ascending=False).reset_index(drop=True)

print("✅ +EV candidates:", len(cand))
cand.head(20)

✅ +EV candidates: 0


,home_team,away_team,market,bet,odds,model_prob,ev,kelly_f,odds_num,units


In [19]:
# =========================
# PHASE 6: SENSITIVITY / STABILITY
# =========================

def prob_spread(mean_margin, sd, spread_home_line):
    # win if margin > -spread_home_line
    return 1 - norm.cdf(((-spread_home_line) - mean_margin) / sd)

def prob_total_over(mean_total, sd, total_line):
    return 1 - norm.cdf((total_line - mean_total) / sd)

def prob_ml(mean_margin, sd):
    return 1 - norm.cdf((0 - mean_margin) / sd)

# map game stats for quick lookup
game_key = df.set_index(["home_team","away_team"])[["mean_margin","sd_margin","mean_total","total_line","spread_home_line"]].to_dict("index")

stable_flags = []
worst_probs = []

for _, r in cand.iterrows():
    k = (r["home_team"], r["away_team"])
    g = game_key.get(k)

    if not g:
        stable_flags.append(False); worst_probs.append(np.nan); continue

    mm = g["mean_margin"]; sd = g["sd_margin"]; mt = g["mean_total"]
    tl = g["total_line"];  sh = g["spread_home_line"]

    # worst-case probability depending on market
    probs = []
    if r["market"] == "ML":
        for dm in [-SENS_MARGIN_SHIFT, +SENS_MARGIN_SHIFT]:
            probs.append(prob_ml(mm + dm, sd))
    elif r["market"] == "Spread":
        for dm in [-SENS_MARGIN_SHIFT, +SENS_MARGIN_SHIFT]:
            probs.append(prob_spread(mm + dm, sd, sh))
    else:  # Total
        for dt in [-SENS_TOTAL_SHIFT, +SENS_TOTAL_SHIFT]:
            if "Over" in r["bet"]:
                probs.append(prob_total_over(mt + dt, sd, tl))
            else:
                probs.append(1 - prob_total_over(mt + dt, sd, tl))

    worst_p = float(np.min(probs))
    worst_probs.append(worst_p)

    # stable if worst-case still positive EV at current odds
    b = (r["odds_num"]/100.0) if r["odds_num"] > 0 else (100.0/abs(r["odds_num"]))
    worst_ev = worst_p*b - (1-worst_p)
    stable_flags.append(worst_ev > 0)

cand["worst_case_prob"] = worst_probs
cand["stable_flag"] = stable_flags

final_card = cand[cand["stable_flag"] == True].copy().reset_index(drop=True)

print("✅ Stable plays:", len(final_card))
final_card.head(20)

✅ Stable plays: 0


,home_team,away_team,market,bet,odds,model_prob,ev,kelly_f,odds_num,units,worst_case_prob,stable_flag


In [20]:
# =========================
# OUTPUTS
# =========================

top20 = final_card.head(20).copy()

def fmt_odds(o):
    if pd.isna(o): return ""
    o = int(float(o))
    return f"+{o}" if o > 0 else str(o)

def to_pct(x):
    return f"{round(float(x)*100,1)}%"

top20["matchup"] = top20["away_team"] + " at " + top20["home_team"]
top20["odds_fmt"] = top20["odds"].apply(fmt_odds)

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " (" + top20["odds_fmt"] + ") — " + top20["units"].astype(str) + "u\n" +
    "Model Prob: " + top20["model_prob"].apply(to_pct) + " | Worst-case: " + top20["worst_case_prob"].apply(to_pct) + "\n" +
    "EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))

# Export
out_path = f"ncaab_stable_{SLATE_DATE}_top20.csv"
top20.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

Empty DataFrame
Columns: [discord_text]
Index: []
✅ Exported: ncaab_stable_2026-03-01_top20.csv


In [21]:
import numpy as np
import pandas as pd
from scipy.stats import norm

df = games_df.copy()

# numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","total_line"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# keep rows where we can do ML-vs-spread comparison
df = df.dropna(subset=["home_ml","away_ml","spread_home_line"]).reset_index(drop=True)

def ml_to_prob(ml):
    if pd.isna(ml):
        return np.nan
    ml = float(ml)
    return (-ml) / (-ml + 100) if ml < 0 else 100 / (ml + 100)

df["home_imp"] = df["home_ml"].apply(ml_to_prob)
df["away_imp"] = df["away_ml"].apply(ml_to_prob)

# De-vig
tot = (df["home_imp"] + df["away_imp"]).replace(0, np.nan)
df["home_mkt"] = df["home_imp"] / tot
df["away_mkt"] = df["away_imp"] / tot

print("✅ Rows with ML+Spread:", len(df))
df[["home_team","away_team","home_ml","away_ml","spread_home_line","home_mkt"]].head()

✅ Rows with ML+Spread: 3


,home_team,away_team,home_ml,away_ml,spread_home_line,home_mkt
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,-110.0,-120.0,1.5,0.489879
1,UNLV Rebels,Nevada Wolf Pack,-120.0,-106.0,-0.5,0.514571
2,UC Irvine Anteaters,UC Santa Barbara Gauchos,-770.0,470.0,-8.5,0.834569


In [22]:
# =========================
# MODEL: Spread → Win Prob
# =========================

# NCAAB margin SD: ~10–12; lightly scale with total if available
# (Keeps behavior consistent across low/high totals)
base_sd = 11.0
df["sd_margin"] = base_sd

# spread_home_line is negative when home favored (e.g. -6.5)
# mean_margin = -spread_home_line (home expected margin)
df["mean_margin"] = -df["spread_home_line"]

# Win prob: P(margin > 0)
df["home_model"] = 1 - norm.cdf((0 - df["mean_margin"]) / df["sd_margin"])

# Edge vs de-vig ML market
df["edge"] = df["home_model"] - df["home_mkt"]

df[["home_team","away_team","spread_home_line","home_model","home_mkt","edge"]].sort_values("edge", ascending=False).head(15)

,home_team,away_team,spread_home_line,home_model,home_mkt,edge
1,UNLV Rebels,Nevada Wolf Pack,-0.5,0.518127,0.514571,0.003556
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1.5,0.445767,0.489879,-0.044112
2,UC Irvine Anteaters,UC Santa Barbara Gauchos,-8.5,0.780158,0.834569,-0.054411


In [23]:
# =========================
# MODEL: Spread → Win Prob
# =========================

# NCAAB margin SD: ~10–12; lightly scale with total if available
# (Keeps behavior consistent across low/high totals)
base_sd = 11.0
df["sd_margin"] = base_sd

# spread_home_line is negative when home favored (e.g. -6.5)
# mean_margin = -spread_home_line (home expected margin)
df["mean_margin"] = -df["spread_home_line"]

# Win prob: P(margin > 0)
df["home_model"] = 1 - norm.cdf((0 - df["mean_margin"]) / df["sd_margin"])

# Edge vs de-vig ML market
df["edge"] = df["home_model"] - df["home_mkt"]

df[["home_team","away_team","spread_home_line","home_model","home_mkt","edge"]].sort_values("edge", ascending=False).head(15)

,home_team,away_team,spread_home_line,home_model,home_mkt,edge
1,UNLV Rebels,Nevada Wolf Pack,-0.5,0.518127,0.514571,0.003556
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1.5,0.445767,0.489879,-0.044112
2,UC Irvine Anteaters,UC Santa Barbara Gauchos,-8.5,0.780158,0.834569,-0.054411


In [24]:
# =========================
# EV + HALF KELLY
# =========================

def ml_to_decimal(ml):
    ml = float(ml)
    return 1 + (ml/100) if ml > 0 else 1 + (100/abs(ml))

def kelly_f(p, dec):
    # f* = (p*(dec) - 1) / (dec - 1)
    b = dec - 1
    return (p*dec - 1) / b if b > 0 else np.nan

UNIT_MIN, UNIT_CAP = 0.25, 1.0

df["dec_home"] = df["home_ml"].apply(ml_to_decimal)
df["ev"] = (df["home_model"] * (df["dec_home"] - 1) - (1 - df["home_model"]))  # per 1u risked

df["kelly_full"] = df.apply(lambda r: kelly_f(r["home_model"], r["dec_home"]), axis=1)
df["kelly_half"] = 0.5 * df["kelly_full"]

# map kelly fraction -> units (pragmatic scaling)
df["units"] = (df["kelly_half"] * 4).clip(lower=0, upper=UNIT_CAP)

# enforce 0.25u minimum for any positive play
df.loc[(df["units"] > 0) & (df["units"] < UNIT_MIN), "units"] = UNIT_MIN
df["units"] = df["units"].round(2)

# filter to positive EV and reasonable juice
df = df[(df["ev"] > 0) & (pd.to_numeric(df["home_ml"], errors="coerce") >= -200)].copy()

top = df.sort_values(["ev","edge"], ascending=False).reset_index(drop=True)

print("✅ +EV ML plays:", len(top))
top[["home_team","away_team","home_ml","spread_home_line","home_model","home_mkt","edge","ev","units"]].head(20)

✅ +EV ML plays: 0


,home_team,away_team,home_ml,spread_home_line,home_model,home_mkt,edge,ev,units


In [25]:
def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

top20 = top.head(20).copy()
top20["matchup"] = top20["away_team"] + " at " + top20["home_team"]

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["home_team"] + " ML (" + top20["home_ml"].apply(fmt_odds) + ") — " + top20["units"].astype(str) + "u\n" +
    "Model: " + top20["home_model"].apply(pct) + " | Market: " + top20["home_mkt"].apply(pct) + "\n" +
    "Edge: " + (top20["edge"]*100).round(1).astype(str) + "% | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))

out_path = "ncaab_stable_2026-03-01_top20.csv"
top20.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

Empty DataFrame
Columns: [discord_text]
Index: []
✅ Exported: ncaab_stable_2026-03-01_top20.csv


In [26]:
print(df[["home_ml","away_ml","spread_home_line"]].isna().mean())

home_ml            NaN
away_ml            NaN
spread_home_line   NaN
dtype: float64


In [27]:
print("games_df rows:", len(games_df))
print("ML non-null:", games_df["home_ml"].notna().sum(), "/", len(games_df))
print("Spread non-null:", games_df["spread_home_line"].notna().sum(), "/", len(games_df))
print("Total non-null:", games_df["total_line"].notna().sum(), "/", len(games_df))

games_df rows: 26
ML non-null: 3 / 26
Spread non-null: 26 / 26
Total non-null: 26 / 26


In [28]:
import numpy as np
import pandas as pd
from scipy.stats import norm

df = games_df.copy()

# numeric safety
for c in ["spread_home_line","spread_home_odds","spread_away_odds","total_line","over_odds","under_odds"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Margin SD proxy (college hoops); keep constant so it behaves like past quick runs
SD_MARGIN = 11.0

df["mean_margin"] = -df["spread_home_line"]   # home expected margin
df["sd_margin"] = SD_MARGIN

# HOME spread cover probability at current line:
# home bet wins if margin > -spread_home_line
df["p_home_cover"] = 1 - norm.cdf(((-df["spread_home_line"]) - df["mean_margin"]) / df["sd_margin"])
# That simplifies to ~0.50 under this baseline, but EV comes from odds/juice & filtering.
# So next we must compute EV correctly vs the price and allow either side.

# Total SD proxy (reuse sd_margin as scoring SD)
df["p_over"]  = 1 - norm.cdf((df["total_line"] - df["total_line"]) / df["sd_margin"])  # baseline 0.50
df["p_under"] = 1 - df["p_over"]

print("✅ Spread/Total prob columns added")
df[["home_team","away_team","spread_home_line","spread_home_odds","total_line","over_odds","under_odds"]].head()

✅ Spread/Total prob columns added


,home_team,away_team,spread_home_line,spread_home_odds,total_line,over_odds,under_odds
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1.5,-154,174.5,-105,-125
1,UC San Diego Tritons,Cal Poly Mustangs,-16.5,-106,145.5,-112,-118
2,UNLV Rebels,Nevada Wolf Pack,-0.5,-122,153.5,-114,-114
3,Utah State Aggies,Grand Canyon Antelopes,-8.5,-102,146.5,-108,-122
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,-8.5,-110,127.5,-122,-108


In [29]:
# Find any existing team metrics DF already loaded in your notebook
candidates = []
for k,v in globals().items():
    if isinstance(v, pd.DataFrame) and v.shape[0] > 50:
        cols = set(map(str.lower, v.columns))
        if any(x in cols for x in ["team","school","team_name"]) and any(x in cols for x in ["adj_o","adj_off","off_eff","adj_off_eff","tempo","adj_d","adj_def","def_eff"]):
            candidates.append((k, v.shape, list(v.columns)[:12]))

candidates[:10]

RuntimeError: dictionary changed size during iteration

In [30]:
games_df = pull_market_data(
    sport="ncaab",
    date="2026-03-01",
    outside=True,
    preferred_book="draftkings"
)

print("games_df rows:", len(games_df))
print("ML non-null:", games_df["home_ml"].notna().sum(), "/", len(games_df))
print("Spread non-null:", games_df["spread_home_line"].notna().sum(), "/", len(games_df))
print("Total non-null:", games_df["total_line"].notna().sum(), "/", len(games_df))

games_df.head()

✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
games_df rows: 26
ML non-null: 15 / 26
Spread non-null: 26 / 26
Total non-null: 24 / 26


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,1300.0,-3200.0,5.5,-125,-5.5,-105,NaN,NaN,NaN,0.071429,0.969697
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-13.5,-125,13.5,-105,NaN,NaN,NaN,NaN,NaN
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-188.0,145.0,-2.5,-120,2.5,-110,154.5,-110.0,-120.0,0.652778,0.408163
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,-298.0,220.0,-3.5,-125,3.5,-105,141.5,-120.0,-110.0,0.748744,0.312500
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-375.0,270.0,-5.5,-110,5.5,-120,125.5,-120.0,-110.0,0.789474,0.270270


In [31]:
import numpy as np
import pandas as pd
from scipy.stats import norm

df = games_df.copy()
for c in ["home_ml","away_ml","spread_home_line"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["home_ml","away_ml","spread_home_line"]).reset_index(drop=True)

def ml_to_prob(ml):
    ml = float(ml)
    return (-ml)/(-ml+100) if ml < 0 else 100/(ml+100)

# De-vig ML market prob
df["home_imp"] = df["home_ml"].apply(ml_to_prob)
df["away_imp"] = df["away_ml"].apply(ml_to_prob)
tot = (df["home_imp"] + df["away_imp"]).replace(0, np.nan)
df["home_mkt"] = df["home_imp"] / tot

# Spread -> model win prob (independent signal)
SD_MARGIN = 11.0
df["mean_margin"] = -df["spread_home_line"]
df["home_model"] = 1 - norm.cdf((0 - df["mean_margin"]) / SD_MARGIN)

# EV on home ML
def ml_to_decimal(ml):
    ml = float(ml)
    return 1 + (ml/100) if ml > 0 else 1 + (100/abs(ml))

df["dec_home"] = df["home_ml"].apply(ml_to_decimal)
df["ev"] = df["home_model"]*(df["dec_home"]-1) - (1-df["home_model"])

# Half-Kelly -> units (0.25–1.0 cap)
def kelly_f(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b>0 else np.nan

UNIT_MIN, UNIT_CAP = 0.25, 1.0
df["kelly_half"] = 0.5 * df.apply(lambda r: kelly_f(r["home_model"], r["dec_home"]), axis=1)
df["units"] = (df["kelly_half"]*4).clip(lower=0, upper=UNIT_CAP)
df.loc[(df["units"]>0) & (df["units"]<UNIT_MIN), "units"] = UNIT_MIN
df["units"] = df["units"].round(2)

card = df[(df["ev"]>0) & (df["home_ml"]>=-200) & (df["units"]>0)].copy()
card = card.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Card plays:", len(card))
card[["home_team","away_team","home_ml","spread_home_line","home_model","home_mkt","ev","units"]].head(20)

✅ Card plays: 3


,home_team,away_team,home_ml,spread_home_line,home_model,home_mkt,ev,units
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1300.0,5.5,0.308538,0.068607,3.319526,0.51
1,Bradley Braves,Murray St Racers,-192.0,-4.5,0.658764,0.630940,0.001870,0.25
2,Marquette Golden Eagles,DePaul Blue Demons,-192.0,-4.5,0.658764,0.630940,0.001870,0.25


In [32]:
books = ["draftkings","fanduel","betmgm","caesars","betrivers","pointsbetus"]
best = None

for bk in books:
    try:
        tmp = pull_market_data("ncaab","2026-03-01",True, preferred_book=bk)
        ml_ct = tmp["home_ml"].notna().sum()
        print(f"{bk:10s}  ML:{ml_ct:2d}/{len(tmp):2d}  Spread:{tmp['spread_home_line'].notna().sum():2d}/{len(tmp):2d}")
        if best is None or ml_ct > best[1]:
            best = (bk, ml_ct)
    except Exception as e:
        print(f"{bk:10s}  error: {str(e)[:90]}")

print("\n✅ Best book:", best)

✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
draftkings  ML:15/26  Spread:26/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
fanduel     ML:23/26  Spread:26/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
betmgm      ML:24/26  Spread:24/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
caesars     ML: 3/26  Spread:26/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
betrivers   ML:25/26  Spread:25/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
pointsbetus  ML: 3/26  Spread:26/26

✅ Best book: ('betrivers', np.int64(25))


In [33]:
BEST_BOOK = "BEST_BOOK_HERE"

games_df = pull_market_data(
    sport="ncaab",
    date="2026-03-01",
    outside=True,
    preferred_book=BEST_BOOK
)

print("games_df rows:", len(games_df))
print("ML non-null:", games_df["home_ml"].notna().sum(), "/", len(games_df))
games_df.head()

✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
games_df rows: 26
ML non-null: 3 / 26


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,1300.0,-3200.0,5.5,-125,-5.5,-105,NaN,NaN,NaN,0.071429,0.969697
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-13.5,104,13.5,-138,151.5,-102.0,-130.0,NaN,NaN
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-174.0,136.0,-2.5,-102,2.5,-130,155.5,-114.0,-114.0,0.635036,0.423729
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,NaN,NaN,-3.5,-130,3.5,-102,143.5,-112.0,-118.0,NaN,NaN
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-400.0,285.0,-5.5,-108,5.5,-122,124.5,-125.0,-106.0,0.800000,0.259740


In [34]:
import numpy as np
import pandas as pd
from scipy.stats import norm

df = games_df.copy()
for c in ["home_ml","away_ml","spread_home_line"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["home_ml","away_ml","spread_home_line"]).reset_index(drop=True)

def ml_to_prob(ml):
    ml = float(ml)
    return (-ml)/(-ml+100) if ml < 0 else 100/(ml+100)

# De-vig ML market prob
df["home_imp"] = df["home_ml"].apply(ml_to_prob)
df["away_imp"] = df["away_ml"].apply(ml_to_prob)
tot = (df["home_imp"] + df["away_imp"]).replace(0, np.nan)
df["home_mkt"] = df["home_imp"] / tot

# Spread -> model win prob
SD_MARGIN = 11.0
df["mean_margin"] = -df["spread_home_line"]
df["home_model"] = 1 - norm.cdf((0 - df["mean_margin"]) / SD_MARGIN)

# Mismatch flag: if ML implies huge dog but spread is near pick (or vice versa)
df["ml_spread_gap"] = (df["home_model"] - df["home_mkt"]).abs()
df["bad_mapping_flag"] = df["ml_spread_gap"] > 0.25   # 25% absolute prob gap is usually “wrong side / bad book data”

df[["home_team","away_team","home_ml","spread_home_line","home_model","home_mkt","ml_spread_gap","bad_mapping_flag"]].sort_values("ml_spread_gap", ascending=False).head(10)

,home_team,away_team,home_ml,spread_home_line,home_model,home_mkt,ml_spread_gap,bad_mapping_flag
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1300.0,5.5,0.308538,0.068607,0.239930,False
2,UC Irvine Anteaters,UC Santa Barbara Gauchos,-400.0,-5.5,0.691462,0.754902,0.063439,False
1,UNLV Rebels,Nevada Wolf Pack,-174.0,-2.5,0.589894,0.599790,0.009896,False


In [35]:
def ml_to_decimal(ml):
    ml = float(ml)
    return 1 + (ml/100) if ml > 0 else 1 + (100/abs(ml))

def kelly_f(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b>0 else np.nan

UNIT_MIN, UNIT_CAP = 0.25, 1.0

df["dec_home"] = df["home_ml"].apply(ml_to_decimal)
df["ev"] = df["home_model"]*(df["dec_home"]-1) - (1-df["home_model"])

df["kelly_half"] = 0.5 * df.apply(lambda r: kelly_f(r["home_model"], r["dec_home"]), axis=1)
df["units"] = (df["kelly_half"]*4).clip(lower=0, upper=UNIT_CAP)
df.loc[(df["units"]>0) & (df["units"]<UNIT_MIN), "units"] = UNIT_MIN
df["units"] = df["units"].round(2)

card = df[
    (df["ev"]>0) &
    (df["home_ml"]>=-200) &
    (df["units"]>0) &
    (~df["bad_mapping_flag"])
].copy()

card = card.sort_values("ev", ascending=False).reset_index(drop=True)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

card["matchup"] = card["away_team"] + " at " + card["home_team"]
card["discord_text"] = (
    card["matchup"] + "\n" +
    card["home_team"] + " ML (" + card["home_ml"].apply(fmt_odds) + ") — " + card["units"].astype(str) + "u\n" +
    "Model: " + card["home_model"].apply(pct) + " | Market: " + card["home_mkt"].apply(pct) + "\n" +
    "EV: " + (card["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ Final card plays:", len(card))
print(card[["discord_text"]].head(10).to_string(index=False))

out_path = "ncaab_stable_2026-03-01_card.csv"
card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

✅ Final card plays: 1
                                                                                                                                 discord_text
Long Beach St 49ers at CSU Bakersfield Roadrunners\nCSU Bakersfield Roadrunners ML (+1300) — 0.51u\nModel: 30.9% | Market: 6.9%\nEV: 332.0%\n
✅ Exported: ncaab_stable_2026-03-01_card.csv


In [36]:
mask = (games_df["home_team"].str.contains("Bakersfield", na=False)) | (games_df["away_team"].str.contains("Bakersfield", na=False))
games_df.loc[mask].T

,0
home_team,CSU Bakersfield Roadrunners
away_team,Long Beach St 49ers
commence_time,2026-03-01T02:35:00Z
home_ml,1300.0
away_ml,-3200.0
spread_home_line,5.5
spread_home_odds,-125
spread_away_line,-5.5
spread_away_odds,-105
total_line,NaN


In [37]:
import numpy as np
from scipy.stats import norm

def ml_to_prob(ml):
    ml = float(ml)
    return (-ml)/(-ml+100) if ml < 0 else 100/(ml+100)

tmp = games_df.copy()
for c in ["home_ml","away_ml","spread_home_line"]:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

tmp = tmp.dropna(subset=["home_ml","away_ml","spread_home_line"]).reset_index(drop=True)

# de-vig ML market prob
tmp["home_imp"] = tmp["home_ml"].apply(ml_to_prob)
tmp["away_imp"] = tmp["away_ml"].apply(ml_to_prob)
tot = (tmp["home_imp"] + tmp["away_imp"]).replace(0, np.nan)
tmp["home_mkt"] = tmp["home_imp"] / tot

# spread-based win prob
SD = 11.0
tmp["home_model"] = 1 - norm.cdf((0 - (-tmp["spread_home_line"])) / SD)

# MUCH stricter mismatch threshold (10% not 25%)
tmp["abs_gap"] = (tmp["home_model"] - tmp["home_mkt"]).abs()
tmp["mapping_ok"] = tmp["abs_gap"] <= 0.10

tmp.sort_values("abs_gap", ascending=False)[
    ["home_team","away_team","home_ml","away_ml","spread_home_line","home_model","home_mkt","abs_gap","mapping_ok"]
].head(15)

,home_team,away_team,home_ml,away_ml,spread_home_line,home_model,home_mkt,abs_gap,mapping_ok
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,1300.0,-3200.0,5.5,0.308538,0.068607,0.239930,False
2,UC Irvine Anteaters,UC Santa Barbara Gauchos,-400.0,285.0,-5.5,0.691462,0.754902,0.063439,True
1,UNLV Rebels,Nevada Wolf Pack,-174.0,136.0,-2.5,0.589894,0.599790,0.009896,True


In [38]:
books = ["draftkings","fanduel","betmgm","caesars","betrivers","pointsbetus"]
for bk in books:
    try:
        t = pull_market_data("ncaab","2026-03-01",True, preferred_book=bk)
        print(f"{bk:10s} ML:{t['home_ml'].notna().sum():2d}/{len(t):2d}")
    except Exception as e:
        print(f"{bk:10s} error: {str(e)[:80]}")

✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
draftkings ML:15/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
fanduel    ML:23/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
betmgm     ML:24/26
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
caesars    ML: 3/26
betrivers  error: Odds API Error 429: {"message":"Requests are too frequent","error_code":"EXCEEDE
✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
pointsbetus ML: 3/26


In [39]:
BEST_BOOK = "fanduel"  # replace with what printed best

games_df = pull_market_data("ncaab","2026-03-01",True, preferred_book=BEST_BOOK)
print("ML non-null:", games_df["home_ml"].notna().sum(), "/", len(games_df))

✅ pull_market_data: pulled 26 games for basketball_ncaab on 2026-03-01 (date_filter=True)
ML non-null: 23 / 26


In [40]:
import pandas as pd

items = list(globals().items())
candidates = []

for k, v in items:
    if isinstance(v, pd.DataFrame) and v.shape[0] > 100:
        cols = set(map(str.lower, v.columns))
        if any(x in cols for x in ["team","school","team_name"]) and \
           any(x in cols for x in ["adj_o","adj_off","off_eff","adj_off_eff"]) and \
           any(x in cols for x in ["adj_d","adj_def","def_eff"]) and \
           any(x in cols for x in ["tempo","pace"]):
            candidates.append((k, v.shape, list(v.columns)))

candidates

[]

In [41]:
# ============================================
# NCAAB STABLE — INTERNAL RATING + MONTE CARLO
# ============================================

import numpy as np
import pandas as pd

SIMS = 50000
SD_MARGIN = 11.0
UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200
MAPPING_TOLERANCE = 0.12

rng = np.random.default_rng(42)

df = games_df.copy()

# ---------- CLEAN ----------
for c in ["home_ml","away_ml","spread_home_line","total_line"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["spread_home_line"]).reset_index(drop=True)

# ============================================
# STEP 1 — BUILD INTERNAL POWER RATINGS
# ============================================

# Spread gives implied margin.
# We convert each game into rating contributions.

ratings = {}

for _, row in df.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    spread = row["spread_home_line"]

    # implied margin for home
    margin = -spread

    ratings.setdefault(home, []).append(margin)
    ratings.setdefault(away, []).append(-margin)

# Average margin contribution
power = {team: np.mean(vals) for team, vals in ratings.items()}

# Normalize to league mean 0
league_mean = np.mean(list(power.values()))
power = {team: val - league_mean for team, val in power.items()}

# ============================================
# STEP 2 — PROJECT MARGINS FROM RATINGS
# ============================================

proj_margins = []
proj_totals = []

for _, row in df.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    home_rating = power.get(home, 0)
    away_rating = power.get(away, 0)

    mean_margin = home_rating - away_rating
    mean_total = row.get("total_line", 140)  # fallback if total missing

    proj_margins.append(mean_margin)
    proj_totals.append(mean_total)

df["mean_margin"] = proj_margins
df["mean_total"] = proj_totals

# ============================================
# STEP 3 — MONTE CARLO SIMULATION
# ============================================

margins = rng.normal(
    loc=df["mean_margin"].to_numpy()[:, None],
    scale=SD_MARGIN,
    size=(len(df), SIMS)
)

df["p_home_win"] = (margins > 0).mean(axis=1)

# ============================================
# STEP 4 — ML MARKET + EV
# ============================================

def ml_to_prob(ml):
    ml = float(ml)
    return (-ml)/(-ml+100) if ml < 0 else 100/(ml+100)

def ml_to_decimal(ml):
    ml = float(ml)
    return 1 + (ml/100) if ml > 0 else 1 + (100/abs(ml))

df = df.dropna(subset=["home_ml","away_ml"]).reset_index(drop=True)

df["home_imp"] = df["home_ml"].apply(ml_to_prob)
df["away_imp"] = df["away_ml"].apply(ml_to_prob)
tot = (df["home_imp"] + df["away_imp"]).replace(0, np.nan)
df["home_mkt"] = df["home_imp"] / tot

df["abs_gap"] = (df["p_home_win"] - df["home_mkt"]).abs()
df["mapping_ok"] = df["abs_gap"] <= MAPPING_TOLERANCE

df["dec_home"] = df["home_ml"].apply(ml_to_decimal)
df["ev_ml"] = df["p_home_win"]*(df["dec_home"]-1) - (1-df["p_home_win"])

def kelly(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b>0 else 0

df["kelly_half"] = 0.5 * df.apply(lambda r: kelly(r["p_home_win"], r["dec_home"]), axis=1)
df["units_ml"] = (df["kelly_half"]*4).clip(lower=0, upper=UNIT_CAP)
df.loc[(df["units_ml"]>0)&(df["units_ml"]<UNIT_MIN),"units_ml"]=UNIT_MIN
df["units_ml"] = df["units_ml"].round(2)

card = df[
    (df["mapping_ok"]) &
    (df["ev_ml"]>0) &
    (df["home_ml"]>=MAX_JUICE) &
    (df["units_ml"]>0)
].copy()

card = card.sort_values("ev_ml", ascending=False).reset_index(drop=True)

print("✅ Sims per game:", SIMS)
print("✅ Final ML plays:", len(card))

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

if len(card) > 0:
    card["matchup"] = card["away_team"] + " at " + card["home_team"]
    card["discord_text"] = (
        card["matchup"] + "\n" +
        card["home_team"] + " ML (" + card["home_ml"].apply(fmt_odds) + ") — " +
        card["units_ml"].astype(str) + "u\n" +
        "Sim Win%: " + card["p_home_win"].apply(pct) +
        " | Market: " + card["home_mkt"].apply(pct) + "\n" +
        "EV: " + (card["ev_ml"]*100).round(1).astype(str) + "%\n"
    )

    print(card[["discord_text"]].head(20).to_string(index=False))
    card.head(20).to_csv("ncaab_stable_internal_sim_card.csv", index=False)
    print("✅ Exported: ncaab_stable_internal_sim_card.csv")
else:
    print("No ML plays passed filters.")

✅ Sims per game: 50000
✅ Final ML plays: 2
                                                                                                               discord_text
Belmont Bruins at Illinois St Redbirds\nIllinois St Redbirds ML (-120) — 0.3u\nSim Win%: 61.3% | Market: 52.2%\nEV: 12.5%\n
                Nevada Wolf Pack at UNLV Rebels\nUNLV Rebels ML (-148) — 0.25u\nSim Win%: 60.5% | Market: 56.3%\nEV: 1.4%\n
✅ Exported: ncaab_stable_internal_sim_card.csv


In [42]:
# ============================================
# INITIALIZE PERSISTENT TEAM RATINGS
# ============================================

import os
import pandas as pd

RATINGS_FILE = "ncaab_team_ratings.csv"

if os.path.exists(RATINGS_FILE):
    ratings_df = pd.read_csv(RATINGS_FILE)
    print("Loaded existing ratings:", len(ratings_df))
else:
    ratings_df = pd.DataFrame(columns=["team","rating"])
    ratings_df.to_csv(RATINGS_FILE, index=False)
    print("Initialized new ratings file.")

Initialized new ratings file.


In [43]:
# ============================================
# NCAAB STABLE — ROLLING RATING MONTE CARLO
# ============================================

import numpy as np
import pandas as pd
import os

SIMS = 50000
SD_MARGIN = 11.0
K_FACTOR = 0.15
HOME_EDGE = 1.5   # home court in rating points
UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200

RATINGS_FILE = "ncaab_team_ratings.csv"
rng = np.random.default_rng(7)

# ---------- LOAD RATINGS ----------
if os.path.exists(RATINGS_FILE):
    ratings_df = pd.read_csv(RATINGS_FILE)
else:
    ratings_df = pd.DataFrame(columns=["team","rating"])

ratings = dict(zip(ratings_df.team, ratings_df.rating))

# ---------- PREP SLATE ----------
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","total_line"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["spread_home_line"]).reset_index(drop=True)

# Ensure all teams have ratings
for team in pd.concat([df.home_team, df.away_team]).unique():
    if team not in ratings:
        ratings[team] = 0.0

# ---------- PROJECT MARGINS ----------
proj_margins = []

for _, row in df.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    home_rating = ratings[home]
    away_rating = ratings[away]

    mean_margin = (home_rating + HOME_EDGE) - away_rating
    proj_margins.append(mean_margin)

df["mean_margin"] = proj_margins

# ---------- MONTE CARLO ----------
margins = rng.normal(
    loc=df["mean_margin"].to_numpy()[:, None],
    scale=SD_MARGIN,
    size=(len(df), SIMS)
)

df["p_home_win"] = (margins > 0).mean(axis=1)

# ---------- ML EV ----------
def ml_to_prob(ml):
    ml = float(ml)
    return (-ml)/(-ml+100) if ml < 0 else 100/(ml+100)

def ml_to_decimal(ml):
    ml = float(ml)
    return 1 + (ml/100) if ml > 0 else 1 + (100/abs(ml))

df = df.dropna(subset=["home_ml","away_ml"]).reset_index(drop=True)

df["home_imp"] = df["home_ml"].apply(ml_to_prob)
df["away_imp"] = df["away_ml"].apply(ml_to_prob)
tot = (df["home_imp"] + df["away_imp"]).replace(0, np.nan)
df["home_mkt"] = df["home_imp"] / tot

df["dec_home"] = df["home_ml"].apply(ml_to_decimal)
df["ev_ml"] = df["p_home_win"]*(df["dec_home"]-1) - (1-df["p_home_win"])

def kelly(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b>0 else 0

df["kelly_half"] = 0.5 * df.apply(lambda r: kelly(r["p_home_win"], r["dec_home"]), axis=1)
df["units_ml"] = (df["kelly_half"]*4).clip(lower=0, upper=UNIT_CAP)
df.loc[(df["units_ml"]>0)&(df["units_ml"]<UNIT_MIN),"units_ml"]=UNIT_MIN
df["units_ml"] = df["units_ml"].round(2)

card = df[
    (df["ev_ml"]>0) &
    (df["home_ml"]>=MAX_JUICE) &
    (df["units_ml"]>0)
].copy()

card = card.sort_values("ev_ml", ascending=False).reset_index(drop=True)

print("✅ Sims:", SIMS)
print("✅ ML Plays:", len(card))

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

if len(card)>0:
    card["matchup"] = card["away_team"] + " at " + card["home_team"]
    card["discord_text"] = (
        card["matchup"] + "\n" +
        card["home_team"] + " ML (" + card["home_ml"].apply(fmt_odds) + ") — " +
        card["units_ml"].astype(str) + "u\n" +
        "Sim Win%: " + card["p_home_win"].apply(pct) +
        " | Market: " + card["home_mkt"].apply(pct) + "\n" +
        "EV: " + (card["ev_ml"]*100).round(1).astype(str) + "%\n"
    )
    print(card[["discord_text"]].head(20).to_string(index=False))
    card.head(20).to_csv("ncaab_stable_card.csv", index=False)
    print("✅ Exported: ncaab_stable_card.csv")
else:
    print("No ML plays passed filters.")

# ---------- SAVE RATINGS ----------
pd.DataFrame(list(ratings.items()), columns=["team","rating"]).to_csv(RATINGS_FILE, index=False)

✅ Sims: 50000
✅ ML Plays: 9
                                                                                                                             discord_text
      Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles ML (+360) — 0.86u\nSim Win%: 55.5% | Market: 20.8%\nEV: 155.4%\n
Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins ML (+290) — 0.81u\nSim Win%: 55.6% | Market: 24.5%\nEV: 116.8%\n
          Purdue Boilermakers at Ohio State Buckeyes\nOhio State Buckeyes ML (+255) — 0.76u\nSim Win%: 55.6% | Market: 27.0%\nEV: 97.4%\n
            Memphis Tigers at East Carolina Pirates\nEast Carolina Pirates ML (+180) — 0.6u\nSim Win%: 55.0% | Market: 34.0%\nEV: 53.9%\n
                 UIC Flames at Indiana St Sycamores\nIndiana St Sycamores ML (+158) — 0.55u\nSim Win%: 55.6% | Market: 37.1%\nEV: 43.5%\n
                 Northern Iowa Panthers at Drake Bulldogs\nDrake Bulldogs ML (+158) — 0.54u\nSim Win%: 55.3% | Market: 37.1%\nEV: 42.8%\n
      

In [45]:
# ==========================================
# PULL 60 DAYS HISTORICAL RESULTS (NCAAB)
# ==========================================

import requests
import pandas as pd

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"  # <-- replace if needed

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/scores"

params = {
    "apiKey": API_KEY,
    "daysFrom": 60
}

response = requests.get(url, params=params)

if response.status_code != 200:
    raise Exception(response.text)

data = response.json()

print("✅ Historical games pulled:", len(data))

Exception: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}


In [46]:
# ==========================================
# PULL 30 DAYS HISTORICAL RESULTS (SAFE LOOP)
# ==========================================

import requests
import pandas as pd
from datetime import datetime, timedelta

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

all_games = []

for days_back in range(1, 31):  # pull last 30 days
    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/scores"

    params = {
        "apiKey": API_KEY,
        "daysFrom": days_back
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        continue

    data = response.json()
    all_games.extend(data)

# Remove duplicates
seen = set()
unique_games = []

for g in all_games:
    key = (g.get("id"), g.get("commence_time"))
    if key not in seen:
        seen.add(key)
        unique_games.append(g)

print("✅ Unique historical games pulled:", len(unique_games))

data = unique_games

✅ Unique historical games pulled: 243


In [47]:
# ==========================================
# BUILD HISTORICAL DF WITH DATES
# ==========================================

import pandas as pd
from datetime import datetime

historical_rows = []

for game in data:
    if not game.get("completed"):
        continue

    home = game["home_team"]
    away = game["away_team"]
    commence = game.get("commence_time")

    if not commence:
        continue

    game_date = datetime.fromisoformat(commence.replace("Z",""))

    scores = game.get("scores", [])
    home_score = None
    away_score = None

    for s in scores:
        if s["name"] == home:
            home_score = int(s["score"])
        if s["name"] == away:
            away_score = int(s["score"])

    if home_score is None or away_score is None:
        continue

    historical_rows.append({
        "home_team": home,
        "away_team": away,
        "home_score": home_score,
        "away_score": away_score,
        "date": game_date
    })

historical_df = pd.DataFrame(historical_rows)

print("✅ Completed games stored:", len(historical_df))
historical_df.head()

✅ Completed games stored: 217


,home_team,away_team,home_score,away_score,date
0,American Eagles,Boston Univ. Terriers,65,68,2026-02-28 17:00:00
1,Holy Cross Crusaders,Loyola (MD) Greyhounds,62,76,2026-02-28 17:00:00
2,Penn State Nittany Lions,Iowa Hawkeyes,71,69,2026-02-28 17:00:17
3,Notre Dame Fighting Irish,NC State Wolfpack,96,90,2026-02-28 17:00:20
4,Georgia Tech Yellow Jackets,Florida St Seminoles,71,80,2026-02-28 17:00:48


In [48]:
# ==========================================
# ADD RECENCY WEIGHTS
# ==========================================

from datetime import datetime

today = historical_df["date"].max()
historical_df["days_ago"] = (today - historical_df["date"]).dt.days

def weight_func(days):
    if days <= 7:
        return 1.5
    elif days <= 21:
        return 1.0
    else:
        return 0.7

historical_df["weight"] = historical_df["days_ago"].apply(weight_func)

print("✅ Recency weights applied")
historical_df[["home_team","days_ago","weight"]].head()

✅ Recency weights applied


,home_team,days_ago,weight
0,American Eagles,0,1.5
1,Holy Cross Crusaders,0,1.5
2,Penn State Nittany Lions,0,1.5
3,Notre Dame Fighting Irish,0,1.5
4,Georgia Tech Yellow Jackets,0,1.5


In [49]:
# ==========================================
# BUILD WEIGHTED HOME / AWAY SPLITS
# ==========================================

# Home perspective
home_stats = historical_df[[
    "home_team","home_score","away_score","weight"
]].copy()

home_stats.columns = ["team","points_scored","points_allowed","weight"]
home_stats["location"] = "home"

# Away perspective
away_stats = historical_df[[
    "away_team","away_score","home_score","weight"
]].copy()

away_stats.columns = ["team","points_scored","points_allowed","weight"]
away_stats["location"] = "away"

all_stats = pd.concat([home_stats, away_stats], ignore_index=True)

# Weighted averages
def weighted_avg(x, value_col):
    return (x[value_col] * x["weight"]).sum() / x["weight"].sum()

team_splits = (
    all_stats
    .groupby(["team","location"])
    .apply(lambda x: pd.Series({
        "off_rating": weighted_avg(x, "points_scored"),
        "def_rating": weighted_avg(x, "points_allowed")
    }))
    .reset_index()
)

print("✅ Team splits built:", len(team_splits))
team_splits.head()

✅ Team splits built: 356


/tmp/ipython-input-926/2783797761.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,team,location,off_rating,def_rating
0,Abilene Christian Wildcats,away,74.0,79.5
1,Air Force Falcons,away,62.0,66.0
2,Akron Zips,away,92.0,70.0
3,Alabama A&M Bulldogs,home,88.0,89.0
4,Alabama Crimson Tide,away,71.0,69.0


In [50]:
# ==========================================
# PIVOT SPLITS INTO PROJECTION FORMAT
# ==========================================

home_split = team_splits[team_splits["location"]=="home"].copy()
home_split = home_split.rename(columns={
    "off_rating":"home_off",
    "def_rating":"home_def"
}).drop(columns=["location"])

away_split = team_splits[team_splits["location"]=="away"].copy()
away_split = away_split.rename(columns={
    "off_rating":"away_off",
    "def_rating":"away_def"
}).drop(columns=["location"])

team_projection_table = home_split.merge(
    away_split,
    on="team",
    how="outer"
)

print("✅ Projection table ready:", len(team_projection_table))
team_projection_table.head()

✅ Projection table ready: 329


,team,home_off,home_def,away_off,away_def
0,Abilene Christian Wildcats,NaN,NaN,74.0,79.5
1,Air Force Falcons,NaN,NaN,62.0,66.0
2,Akron Zips,NaN,NaN,92.0,70.0
3,Alabama A&M Bulldogs,88.0,89.0,NaN,NaN
4,Alabama Crimson Tide,NaN,NaN,71.0,69.0


In [51]:
# ==========================================
# FILL MISSING SPLITS WITH LEAGUE AVERAGES
# ==========================================

league_home_off = team_projection_table["home_off"].mean()
league_home_def = team_projection_table["home_def"].mean()
league_away_off = team_projection_table["away_off"].mean()
league_away_def = team_projection_table["away_def"].mean()

team_projection_table["home_off"] = team_projection_table["home_off"].fillna(league_home_off)
team_projection_table["home_def"] = team_projection_table["home_def"].fillna(league_home_def)
team_projection_table["away_off"] = team_projection_table["away_off"].fillna(league_away_off)
team_projection_table["away_def"] = team_projection_table["away_def"].fillna(league_away_def)

print("✅ Missing splits filled")
print(team_projection_table.isna().sum())

✅ Missing splits filled
team        0
home_off    0
home_def    0
away_off    0
away_def    0
dtype: int64


In [52]:
# ==========================================
# MERGE PROJECTION SPLITS INTO SLATE
# ==========================================

games_df = games_df.merge(
    team_projection_table,
    left_on="home_team",
    right_on="team",
    how="left"
).rename(columns={
    "home_off":"home_off_home",
    "home_def":"home_def_home",
    "away_off":"home_off_away_context",
    "away_def":"home_def_away_context"
})

games_df = games_df.merge(
    team_projection_table,
    left_on="away_team",
    right_on="team",
    how="left"
).rename(columns={
    "home_off":"away_off_home_context",
    "home_def":"away_def_home_context",
    "away_off":"away_off_away",
    "away_def":"away_def_away"
})

print("✅ Projection metrics merged into slate")
games_df.head()

✅ Projection metrics merged into slate


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied,team_x,home_off_home,home_def_home,home_off_away_context,home_def_away_context,team_y,away_off_home_context,away_def_home_context,away_off_away,away_def_away
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,2026-03-01T02:35:00Z,NaN,NaN,4.5,-112,-4.5,-118,NaN,NaN,NaN,NaN,NaN,CSU Bakersfield Roadrunners,87.000000,88.000000,72.000000,84.000000,Long Beach St 49ers,76.561798,72.196629,90.000000,94.500000
1,UC San Diego Tritons,Cal Poly Mustangs,2026-03-01T03:03:21Z,NaN,NaN,-14.5,-136,14.5,102,NaN,NaN,NaN,NaN,NaN,UC San Diego Tritons,82.000000,68.000000,72.036517,76.373596,Cal Poly Mustangs,102.000000,92.000000,64.000000,80.000000
2,UNLV Rebels,Nevada Wolf Pack,2026-03-01T03:14:00Z,-148.0,116.0,-1.5,-118,1.5,-112,152.5,-118.0,-112.0,0.596774,0.462963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Utah State Aggies,Grand Canyon Antelopes,2026-03-01T03:15:00Z,NaN,NaN,-2.5,-104,2.5,-128,142.5,-120.0,-110.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,2026-03-01T03:40:00Z,-400.0,285.0,-5.5,-108,5.5,-122,124.5,-125.0,-106.0,0.800000,0.259740,UC Irvine Anteaters,76.561798,72.196629,68.000000,67.000000,UC Santa Barbara Gauchos,70.000000,59.000000,72.036517,76.373596


In [53]:
# ==========================================
# BUILD INDEPENDENT POINT PROJECTIONS
# ==========================================

HOME_EDGE = 1.5

games_df["home_points_proj"] = (
    (games_df["home_off_home"] + games_df["away_def_away"]) / 2
)

games_df["away_points_proj"] = (
    (games_df["away_off_away"] + games_df["home_def_home"]) / 2
)

games_df["mean_margin"] = (
    games_df["home_points_proj"] -
    games_df["away_points_proj"] +
    HOME_EDGE
)

games_df["mean_total"] = (
    games_df["home_points_proj"] +
    games_df["away_points_proj"]
)

print("✅ Independent projections built")
games_df[["home_team","away_team","mean_margin","mean_total"]].head()

✅ Independent projections built


,home_team,away_team,mean_margin,mean_total
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,3.250000,179.75000
1,UC San Diego Tritons,Cal Poly Mustangs,16.500000,147.00000
2,UNLV Rebels,Nevada Wolf Pack,NaN,NaN
3,Utah State Aggies,Grand Canyon Antelopes,NaN,NaN
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,5.851124,148.58427


In [54]:
# ==========================================
# NORMALIZE TEAM NAMES
# ==========================================

def normalize_name(name):
    if pd.isna(name):
        return name
    name = name.lower()
    name = name.replace("state", "st")
    name = name.replace("university", "")
    name = name.replace("college", "")
    name = name.replace(".", "")
    name = name.replace("  ", " ")
    return name.strip()

team_projection_table["team_norm"] = team_projection_table["team"].apply(normalize_name)

games_df["home_team_norm"] = games_df["home_team"].apply(normalize_name)
games_df["away_team_norm"] = games_df["away_team"].apply(normalize_name)

print("✅ Normalization complete")

✅ Normalization complete


In [55]:
# ==========================================
# RE-MERGE USING NORMALIZED NAMES
# ==========================================

games_df = games_df.drop(columns=[
    col for col in games_df.columns
    if "home_off" in col or "home_def" in col or "away_off" in col or "away_def" in col
], errors="ignore")

games_df = games_df.merge(
    team_projection_table,
    left_on="home_team_norm",
    right_on="team_norm",
    how="left"
).rename(columns={
    "home_off":"home_off_home",
    "home_def":"home_def_home",
    "away_off":"home_off_away_context",
    "away_def":"home_def_away_context"
})

games_df = games_df.merge(
    team_projection_table,
    left_on="away_team_norm",
    right_on="team_norm",
    how="left"
).rename(columns={
    "home_off":"away_off_home_context",
    "home_def":"away_def_home_context",
    "away_off":"away_off_away",
    "away_def":"away_def_away"
})

print("✅ Re-merge complete")

MergeError: Passing 'suffixes' which cause duplicate columns {'team_x'} is not allowed.

In [56]:
import pandas as pd

# 1) Remove previous merge junk columns that cause collisions
junk_cols = [
    "team", "team_x", "team_y",
    "team_norm", "team_norm_x", "team_norm_y",
    "location"
]
games_df = games_df.drop(columns=[c for c in junk_cols if c in games_df.columns], errors="ignore")

# Also remove any old projection columns so we don't reuse stale ones
proj_cols = [c for c in games_df.columns if any(k in c for k in [
    "home_off", "home_def", "away_off", "away_def",
    "home_points_proj", "away_points_proj",
    "mean_margin", "mean_total"
])]
games_df = games_df.drop(columns=proj_cols, errors="ignore")

# 2) Ensure normalization columns exist
def normalize_name(name):
    if pd.isna(name):
        return name
    s = str(name).lower()
    s = s.replace("state", "st")
    s = s.replace("university", "")
    s = s.replace("college", "")
    s = s.replace(".", "")
    s = " ".join(s.split())
    return s.strip()

if "team_norm" not in team_projection_table.columns:
    team_projection_table["team_norm"] = team_projection_table["team"].apply(normalize_name)

games_df["home_team_norm"] = games_df["home_team"].apply(normalize_name)
games_df["away_team_norm"] = games_df["away_team"].apply(normalize_name)

# 3) Merge HOME metrics (suffix avoids collisions)
games_df = games_df.merge(
    team_projection_table[["team_norm","home_off","home_def","away_off","away_def"]],
    left_on="home_team_norm",
    right_on="team_norm",
    how="left",
    suffixes=("", "_homejoin")
)

# Rename home-joined columns into clear names
games_df = games_df.rename(columns={
    "home_off": "home_off_home",
    "home_def": "home_def_home",
    "away_off": "home_off_away_context",
    "away_def": "home_def_away_context",
})

# Drop join key from right
games_df = games_df.drop(columns=["team_norm"], errors="ignore")

# 4) Merge AWAY metrics (use suffixes and then rename)
games_df = games_df.merge(
    team_projection_table[["team_norm","home_off","home_def","away_off","away_def"]],
    left_on="away_team_norm",
    right_on="team_norm",
    how="left",
    suffixes=("", "_awayjoin")
)

games_df = games_df.rename(columns={
    "home_off": "away_off_home_context",
    "home_def": "away_def_home_context",
    "away_off": "away_off_away",
    "away_def": "away_def_away",
})

games_df = games_df.drop(columns=["team_norm"], errors="ignore")

print("✅ Re-merge complete. Null check:")
print(games_df[["home_off_home","home_def_home","away_off_away","away_def_away"]].isna().sum())

# 5) Rebuild projections
HOME_EDGE = 1.5

games_df["home_points_proj"] = (games_df["home_off_home"] + games_df["away_def_away"]) / 2
games_df["away_points_proj"] = (games_df["away_off_away"] + games_df["home_def_home"]) / 2

games_df["mean_margin"] = (games_df["home_points_proj"] - games_df["away_points_proj"] + HOME_EDGE)
games_df["mean_total"] = (games_df["home_points_proj"] + games_df["away_points_proj"])

print("✅ Projections rebuilt. NaNs in mean_margin:", games_df["mean_margin"].isna().sum())
games_df[["home_team","away_team","mean_margin","mean_total"]].head()

✅ Re-merge complete. Null check:
home_off_home    15
home_def_home    15
away_off_away    13
away_def_away    13
dtype: int64
✅ Projections rebuilt. NaNs in mean_margin: 17


,home_team,away_team,mean_margin,mean_total
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,3.250000,179.75000
1,UC San Diego Tritons,Cal Poly Mustangs,16.500000,147.00000
2,UNLV Rebels,Nevada Wolf Pack,NaN,NaN
3,Utah State Aggies,Grand Canyon Antelopes,NaN,NaN
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,5.851124,148.58427


In [57]:
import pandas as pd
import difflib

# Build lookup list of canonical team_norm values we DO have metrics for
canon_norms = team_projection_table["team_norm"].dropna().unique().tolist()

def fuzzy_best_match(name_norm, choices, cutoff=0.86):
    """
    Returns best match string if similarity >= cutoff, else None.
    cutoff in [0,1]; 0.86 is conservative.
    """
    if pd.isna(name_norm) or not isinstance(name_norm, str) or len(name_norm) == 0:
        return None
    matches = difflib.get_close_matches(name_norm, choices, n=1, cutoff=cutoff)
    return matches[0] if matches else None

# Identify which teams are missing metrics after normalization merge
missing_home = games_df.loc[games_df["home_off_home"].isna(), "home_team_norm"].dropna().unique().tolist()
missing_away = games_df.loc[games_df["away_off_away"].isna(), "away_team_norm"].dropna().unique().tolist()

missing_all = sorted(set(missing_home + missing_away))
print("Missing team_norm count:", len(missing_all))

# Create fuzzy mapping
fuzzy_map = {}
for tnorm in missing_all:
    m = fuzzy_best_match(tnorm, canon_norms, cutoff=0.86)
    if m:
        fuzzy_map[tnorm] = m

print("✅ Fuzzy matches found:", len(fuzzy_map))

# Show the mappings so you can verify
preview = pd.DataFrame(
    [{"missing_norm": k, "matched_norm": v} for k, v in fuzzy_map.items()]
).sort_values("missing_norm")
display(preview.head(50))

# Apply fuzzy mapping to normalized team columns (only when missing)
games_df["home_team_norm_fuzzy"] = games_df["home_team_norm"].map(lambda x: fuzzy_map.get(x, x))
games_df["away_team_norm_fuzzy"] = games_df["away_team_norm"].map(lambda x: fuzzy_map.get(x, x))

# Drop the metric columns and re-merge one last time using the fuzzy norms
drop_cols = [c for c in games_df.columns if c in [
    "home_off_home","home_def_home","home_off_away_context","home_def_away_context",
    "away_off_home_context","away_def_home_context","away_off_away","away_def_away",
    "home_points_proj","away_points_proj","mean_margin","mean_total"
]]
games_df = games_df.drop(columns=drop_cols, errors="ignore")

# HOME join
games_df = games_df.merge(
    team_projection_table[["team_norm","home_off","home_def","away_off","away_def"]],
    left_on="home_team_norm_fuzzy",
    right_on="team_norm",
    how="left",
    suffixes=("", "_hj")
).rename(columns={
    "home_off": "home_off_home",
    "home_def": "home_def_home",
    "away_off": "home_off_away_context",
    "away_def": "home_def_away_context",
}).drop(columns=["team_norm"], errors="ignore")

# AWAY join
games_df = games_df.merge(
    team_projection_table[["team_norm","home_off","home_def","away_off","away_def"]],
    left_on="away_team_norm_fuzzy",
    right_on="team_norm",
    how="left",
    suffixes=("", "_aj")
).rename(columns={
    "home_off": "away_off_home_context",
    "home_def": "away_def_home_context",
    "away_off": "away_off_away",
    "away_def": "away_def_away",
}).drop(columns=["team_norm"], errors="ignore")

print("✅ Post-fuzzy merge null check:")
print(games_df[["home_off_home","home_def_home","away_off_away","away_def_away"]].isna().sum())

# Rebuild projections
HOME_EDGE = 1.5
games_df["home_points_proj"] = (games_df["home_off_home"] + games_df["away_def_away"]) / 2
games_df["away_points_proj"] = (games_df["away_off_away"] + games_df["home_def_home"]) / 2
games_df["mean_margin"] = (games_df["home_points_proj"] - games_df["away_points_proj"] + HOME_EDGE)
games_df["mean_total"]  = (games_df["home_points_proj"] + games_df["away_points_proj"])

print("✅ NaNs in mean_margin:", games_df["mean_margin"].isna().sum())
games_df[["home_team","away_team","mean_margin","mean_total"]].head(10)

Missing team_norm count: 28
✅ Fuzzy matches found: 0


KeyError: 'missing_norm'

In [58]:
import pandas as pd
import re

# 1) Build a stronger "school only" normalizer
STOPWORDS = {
    "st","state","university","college","the","at","of","and",
    "fighting","golden","blue","red","green","wildcats","bulldogs","eagles","bears","tigers","wolves",
    "wolf","pack","rebels","aggies","antelopes","49ers","roadunners","anteaters","gauchos",
    "sparti","spartans","hoosiers","boilermakers","buckeyes","pirates","bruins","redbirds",
    "demons","jaspers","gaels","sycamores","panthers","bulldogs","braves","racers",
    "tritons","mustangs"
}

def school_key(name):
    if pd.isna(name):
        return None
    s = str(name).lower()
    s = s.replace("&", " and ")
    s = s.replace(".", " ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)      # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()

    # convert "state" -> "st" but keep original token too via stopword set
    s = s.replace("state", " st ")

    tokens = [t for t in s.split() if t not in STOPWORDS and not t.isdigit()]
    # If stripping leaves nothing, fall back to first 2 tokens of original cleaned string
    if not tokens:
        tokens = s.split()[:2]
    return " ".join(tokens).strip()

# 2) Add keys to both tables
team_projection_table["school_key"] = team_projection_table["team"].apply(school_key)
games_df["home_key"] = games_df["home_team"].apply(school_key)
games_df["away_key"] = games_df["away_team"].apply(school_key)

# 3) Drop prior merge artifacts + projection cols to avoid collisions
junk_cols = [c for c in games_df.columns if c in ["team","team_x","team_y","team_norm","team_norm_x","team_norm_y","location"]]
games_df = games_df.drop(columns=junk_cols, errors="ignore")

proj_cols = [c for c in games_df.columns if any(k in c for k in [
    "home_off","home_def","away_off","away_def",
    "home_points_proj","away_points_proj",
    "mean_margin","mean_total"
])]
games_df = games_df.drop(columns=proj_cols, errors="ignore")

# 4) Merge HOME via school_key
metrics = team_projection_table[["school_key","home_off","home_def","away_off","away_def"]].copy()

games_df = games_df.merge(
    metrics,
    left_on="home_key",
    right_on="school_key",
    how="left",
    suffixes=("", "_home")
).rename(columns={
    "home_off": "home_off_home",
    "home_def": "home_def_home",
    "away_off": "home_off_away_context",
    "away_def": "home_def_away_context",
}).drop(columns=["school_key"], errors="ignore")

# 5) Merge AWAY via school_key
games_df = games_df.merge(
    metrics,
    left_on="away_key",
    right_on="school_key",
    how="left",
    suffixes=("", "_away")
).rename(columns={
    "home_off": "away_off_home_context",
    "home_def": "away_def_home_context",
    "away_off": "away_off_away",
    "away_def": "away_def_away",
}).drop(columns=["school_key"], errors="ignore")

print("✅ Post school-key merge null check:")
print(games_df[["home_off_home","home_def_home","away_off_away","away_def_away"]].isna().sum())

# 6) Rebuild projections
HOME_EDGE = 1.5
games_df["home_points_proj"] = (games_df["home_off_home"] + games_df["away_def_away"]) / 2
games_df["away_points_proj"] = (games_df["away_off_away"] + games_df["home_def_home"]) / 2
games_df["mean_margin"] = (games_df["home_points_proj"] - games_df["away_points_proj"] + HOME_EDGE)
games_df["mean_total"]  = (games_df["home_points_proj"] + games_df["away_points_proj"])

print("✅ NaNs in mean_margin:", games_df["mean_margin"].isna().sum())
games_df[["home_team","away_team","mean_margin","mean_total"]].head(10)

✅ Post school-key merge null check:
home_off_home    15
home_def_home    15
away_off_away    13
away_def_away    13
dtype: int64
✅ NaNs in mean_margin: 17


,home_team,away_team,mean_margin,mean_total
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,3.250000,179.75000
1,UC San Diego Tritons,Cal Poly Mustangs,16.500000,147.00000
2,UNLV Rebels,Nevada Wolf Pack,NaN,NaN
3,Utah State Aggies,Grand Canyon Antelopes,NaN,NaN
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,5.851124,148.58427
5,Davidson Wildcats,La Salle Explorers,NaN,NaN
6,Maryland Terrapins,Rutgers Scarlet Knights,NaN,NaN
7,UAB Blazers,North Texas Mean Green,NaN,NaN
8,South Florida Bulls,Tulane Green Wave,NaN,NaN
9,Canisius Golden Griffins,Quinnipiac Bobcats,5.000000,141.50000


In [59]:
import numpy as np

# League averages from the projection table (already weighted/split)
league_home_off = team_projection_table["home_off"].mean()
league_home_def = team_projection_table["home_def"].mean()
league_away_off = team_projection_table["away_off"].mean()
league_away_def = team_projection_table["away_def"].mean()

# Fill missing HOME team metrics
games_df["home_off_home"] = games_df["home_off_home"].fillna(league_home_off)
games_df["home_def_home"] = games_df["home_def_home"].fillna(league_home_def)

# Fill missing AWAY team metrics
games_df["away_off_away"] = games_df["away_off_away"].fillna(league_away_off)
games_df["away_def_away"] = games_df["away_def_away"].fillna(league_away_def)

# (Optional) also fill the context columns if you use them later
for c, v in [
    ("home_off_away_context", league_away_off),
    ("home_def_away_context", league_away_def),
    ("away_off_home_context", league_home_off),
    ("away_def_home_context", league_home_def),
]:
    if c in games_df.columns:
        games_df[c] = games_df[c].fillna(v)

print("✅ After fill null check:")
print(games_df[["home_off_home","home_def_home","away_off_away","away_def_away"]].isna().sum())

# Rebuild projections
HOME_EDGE = 1.5
games_df["home_points_proj"] = (games_df["home_off_home"] + games_df["away_def_away"]) / 2
games_df["away_points_proj"] = (games_df["away_off_away"] + games_df["home_def_home"]) / 2
games_df["mean_margin"] = (games_df["home_points_proj"] - games_df["away_points_proj"] + HOME_EDGE)
games_df["mean_total"]  = (games_df["home_points_proj"] + games_df["away_points_proj"])

print("✅ NaNs in mean_margin:", games_df["mean_margin"].isna().sum())
games_df[["home_team","away_team","mean_margin","mean_total"]].head(10)

✅ After fill null check:
home_off_home    0
home_def_home    0
away_off_away    0
away_def_away    0
dtype: int64
✅ NaNs in mean_margin: 0


,home_team,away_team,mean_margin,mean_total
0,CSU Bakersfield Roadrunners,Long Beach St 49ers,3.250000,179.75000
1,UC San Diego Tritons,Cal Poly Mustangs,16.500000,147.00000
2,UNLV Rebels,Nevada Wolf Pack,5.851124,148.58427
3,Utah State Aggies,Grand Canyon Antelopes,5.851124,148.58427
4,UC Irvine Anteaters,UC Santa Barbara Gauchos,5.851124,148.58427
5,Davidson Wildcats,La Salle Explorers,5.851124,148.58427
6,Maryland Terrapins,Rutgers Scarlet Knights,5.851124,148.58427
7,UAB Blazers,North Texas Mean Green,5.851124,148.58427
8,South Florida Bulls,Tulane Green Wave,5.851124,148.58427
9,Canisius Golden Griffins,Quinnipiac Bobcats,5.000000,141.50000


In [60]:
import numpy as np
import pandas as pd

# =========================
# SETTINGS
# =========================
SIMS = 50000
SD_MARGIN = 11.0
SD_TOTAL  = 13.0   # totals tend to be a bit wider; keep stable
HOME_EDGE = 1.5

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(7)

df = games_df.copy()

# Numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Require spreads/totals for spread/total cards; ML card also needs ML
df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# SIMULATE
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]    = (margins > 0).mean(axis=1)
df["p_home_cover"]  = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"]  = 1 - df["p_home_cover"]
df["p_over"]        = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]       = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)          # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))         # scale to "units"
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# ML CARD (HOME ONLY; we can add away leg later if you want)
# =========================
ml = df.dropna(subset=["home_ml","away_ml"]).copy()

ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev_ml"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev_ml"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# =========================
# SPREAD CARD (choose best side per game)
# =========================
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()

sp["ev_home_spread"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away_spread"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

# pick side with higher EV
sp["side"] = np.where(sp["ev_home_spread"] >= sp["ev_away_spread"], "home", "away")
sp["ev_spread"] = np.where(sp["side"]=="home", sp["ev_home_spread"], sp["ev_away_spread"])
sp["p_spread"]  = np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds_spread"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line_spread"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team_spread"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])

sp["units"] = sp.apply(lambda r: to_units(r["p_spread"], r["odds_spread"]), axis=1)
sp_card = sp[(sp["ev_spread"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team_spread"] + " " + sp_card["line_spread"].astype(str) + " (" + sp_card["odds_spread"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_spread"]

# =========================
# TOTAL CARD (choose best side per game)
# =========================
to = df.dropna(subset=["over_odds","under_odds"]).copy()

to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev_total"] = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_total"]  = np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds_total"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])

to["units"] = to.apply(lambda r: to_units(r["p_total"], r["odds_total"]), axis=1)
to_card = to[(to["ev_total"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds_total"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_total"]

# =========================
# COMBINE + TOP 20
# =========================
def base_cols(x):
    return x.assign(
        matchup = x["away_team"] + " at " + x["home_team"],
        ev = x.get("ev_ml", x.get("ev_spread", x.get("ev_total")))
    )[["matchup","market","bet","units","p_win","ev","commence_time"]]

out = pd.concat([
    base_cols(ml_card),
    base_cols(sp_card),
    base_cols(to_card)
], ignore_index=True)

out = out.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Plays found:", len(out))
top20 = out.head(20).copy()

# Discord text
top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " — " + top20["units"].astype(str) + "u\n" +
    "Sim Win%: " + top20["p_win"].apply(pct) + " | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))

top20.to_csv("ncaab_stable_sim_top20.csv", index=False)
print("✅ Exported: ncaab_stable_sim_top20.csv")

✅ Plays found: 56
                                                                                                             discord_text
      Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles ML (+360) — 1.0u\nSim Win%: 67.5% | EV: 210.7%\n
Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins ML (+290) — 1.0u\nSim Win%: 67.5% | EV: 163.4%\n
         Purdue Boilermakers at Ohio State Buckeyes\nOhio State Buckeyes ML (+255) — 1.0u\nSim Win%: 70.5% | EV: 150.2%\n
           Memphis Tigers at East Carolina Pirates\nEast Carolina Pirates ML (+180) — 1.0u\nSim Win%: 70.2% | EV: 96.5%\n
                 Northern Iowa Panthers at Drake Bulldogs\nDrake Bulldogs ML (+158) — 1.0u\nSim Win%: 70.3% | EV: 81.5%\n
                 UIC Flames at Indiana St Sycamores\nIndiana St Sycamores ML (+158) — 1.0u\nSim Win%: 70.2% | EV: 81.2%\n
                 UC Santa Barbara Gauchos at UC Irvine Anteaters\nOVER 124.5 (-125) — 1.0u\nSim Win%: 96.9% | EV: 74.4%\n
      

In [61]:
import numpy as np
import pandas as pd

# =========================
# PPP / POSSESSION CALIBRATION (FREE, PERMANENT)
# =========================
PPP_LEAGUE = 1.02   # stable NCAAB prior; keeps poss estimates sane
HOME_EDGE_PTS = 1.5

# Build per-team, per-location possession proxy + PPP using historical_df splits
# We reuse all_stats from earlier if it exists; otherwise rebuild quickly
try:
    all_stats
except NameError:
    home_stats = historical_df[["home_team","home_score","away_score","weight"]].copy()
    home_stats.columns = ["team","points_scored","points_allowed","weight"]
    home_stats["location"] = "home"

    away_stats = historical_df[["away_team","away_score","home_score","weight"]].copy()
    away_stats.columns = ["team","points_scored","points_allowed","weight"]
    away_stats["location"] = "away"

    all_stats = pd.concat([home_stats, away_stats], ignore_index=True)

# Possession proxy per game for that team:
# poss ≈ avg of both teams’ possessions; with only scores, approximate from total points / PPP_LEAGUE
all_stats["poss_proxy"] = ((all_stats["points_scored"] + all_stats["points_allowed"]) / 2) / PPP_LEAGUE

def wavg(df, col):
    return (df[col] * df["weight"]).sum() / df["weight"].sum()

team_loc = (
    all_stats.groupby(["team","location"])
    .apply(lambda x: pd.Series({
        "poss": wavg(x, "poss_proxy"),
        "ppp_for": wavg(x, "points_scored") / max(wavg(x, "poss_proxy"), 1e-6),
        "ppp_against": wavg(x, "points_allowed") / max(wavg(x, "poss_proxy"), 1e-6),
    }))
    .reset_index()
)

# Pivot to home/away contexts
home_loc = team_loc[team_loc["location"]=="home"].rename(columns={
    "poss":"home_poss","ppp_for":"home_ppp_for","ppp_against":"home_ppp_against"
}).drop(columns=["location"])

away_loc = team_loc[team_loc["location"]=="away"].rename(columns={
    "poss":"away_poss","ppp_for":"away_ppp_for","ppp_against":"away_ppp_against"
}).drop(columns=["location"])

team_ppp = home_loc.merge(away_loc, on="team", how="outer")

# Fill missing with league averages
for col in ["home_poss","home_ppp_for","home_ppp_against","away_poss","away_ppp_for","away_ppp_against"]:
    team_ppp[col] = team_ppp[col].fillna(team_ppp[col].mean())

# Merge into games_df by exact team name (works because these came from Odds API scores)
g = games_df.copy()
g = g.drop(columns=[c for c in g.columns if c in ["home_points_proj","away_points_proj","mean_margin","mean_total"]], errors="ignore")

g = g.merge(team_ppp, left_on="home_team", right_on="team", how="left", suffixes=("", "_home"))
g = g.rename(columns={
    "home_poss":"h_poss","home_ppp_for":"h_ppp_for","home_ppp_against":"h_ppp_against",
    "away_poss":"h_poss_awayctx","away_ppp_for":"h_ppp_for_awayctx","away_ppp_against":"h_ppp_against_awayctx",
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_ppp, left_on="away_team", right_on="team", how="left", suffixes=("", "_away"))
g = g.rename(columns={
    "home_poss":"a_poss_homectx","home_ppp_for":"a_ppp_for_homectx","home_ppp_against":"a_ppp_against_homectx",
    "away_poss":"a_poss","away_ppp_for":"a_ppp_for","away_ppp_against":"a_ppp_against",
}).drop(columns=["team"], errors="ignore")

# Expected possessions = average of home home-possession and away away-possession
g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2

# Blend PPP: offense vs opponent defense
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

# Points projections
g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"] + (HOME_EDGE_PTS/2)
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"] - (HOME_EDGE_PTS/2)

g["mean_margin"] = g["home_points_proj"] - g["away_points_proj"]
g["mean_total"]  = g["home_points_proj"] + g["away_points_proj"]

print("✅ PPP-based projections rebuilt")
print("mean_margin range:", float(g["mean_margin"].min()), "to", float(g["mean_margin"].max()))
print("mean_total  range:", float(g["mean_total"].min()),  "to", float(g["mean_total"].max()))

games_df = g  # overwrite with calibrated projections

✅ PPP-based projections rebuilt
mean_margin range: 3.178501742160279 to 16.526666666666685
mean_total  range: 137.99999999999997 to 179.75


/tmp/ipython-input-926/1839556400.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


In [62]:
import pandas as pd
import numpy as np

print("Games:", len(games_df))
print("Mean margin < 0:", (games_df["mean_margin"] < 0).sum())
print("Mean margin == 0:", (games_df["mean_margin"] == 0).sum())
print("Mean margin > 0:", (games_df["mean_margin"] > 0).sum())

# Show the 10 biggest away-favored games (most negative margins)
display(
    games_df.sort_values("mean_margin").head(10)[
        ["away_team","home_team","mean_margin","mean_total","spread_home_line","total_line","home_ml","away_ml"]
    ]
)

# Show the 10 biggest home-favored games
display(
    games_df.sort_values("mean_margin", ascending=False).head(10)[
        ["away_team","home_team","mean_margin","mean_total","spread_home_line","total_line","home_ml","away_ml"]
    ]
)

Games: 26
Mean margin < 0: 0
Mean margin == 0: 0
Mean margin > 0: 9


,away_team,home_team,mean_margin,mean_total,spread_home_line,total_line,home_ml,away_ml
0,Long Beach St 49ers,CSU Bakersfield Roadrunners,3.178502,179.750000,4.5,NaN,NaN,NaN
9,Quinnipiac Bobcats,Canisius Golden Griffins,5.161079,141.500000,7.5,137.5,290.0,-375.0
18,Merrimack Warriors,Niagara Purple Eagles,5.161079,141.500000,8.5,125.5,360.0,-480.0
4,UC Santa Barbara Gauchos,UC Irvine Anteaters,5.841254,148.584270,-5.5,124.5,-400.0,285.0
16,Iona Gaels,Manhattan Jaspers,5.841254,148.584270,2.5,150.5,118.0,-142.0
25,Charleston Cougars,UNC Wilmington Seahawks,6.180053,154.500000,-4.5,144.5,-225.0,184.0
15,Mt. St. Mary's Mountaineers,Fairfield Stags,12.711591,138.000000,-4.5,143.5,-230.0,188.0
20,Rider Broncs,Siena Saints,15.022359,143.379213,-15.5,137.5,-2000.0,980.0
1,Cal Poly Mustangs,UC San Diego Tritons,16.526667,147.000000,-14.5,NaN,NaN,NaN
2,Nevada Wolf Pack,UNLV Rebels,NaN,NaN,-1.5,152.5,-148.0,116.0


,away_team,home_team,mean_margin,mean_total,spread_home_line,total_line,home_ml,away_ml
1,Cal Poly Mustangs,UC San Diego Tritons,16.526667,147.000000,-14.5,NaN,NaN,NaN
20,Rider Broncs,Siena Saints,15.022359,143.379213,-15.5,137.5,-2000.0,980.0
15,Mt. St. Mary's Mountaineers,Fairfield Stags,12.711591,138.000000,-4.5,143.5,-230.0,188.0
25,Charleston Cougars,UNC Wilmington Seahawks,6.180053,154.500000,-4.5,144.5,-225.0,184.0
16,Iona Gaels,Manhattan Jaspers,5.841254,148.584270,2.5,150.5,118.0,-142.0
4,UC Santa Barbara Gauchos,UC Irvine Anteaters,5.841254,148.584270,-5.5,124.5,-400.0,285.0
9,Quinnipiac Bobcats,Canisius Golden Griffins,5.161079,141.500000,7.5,137.5,290.0,-375.0
18,Merrimack Warriors,Niagara Purple Eagles,5.161079,141.500000,8.5,125.5,360.0,-480.0
0,Long Beach St 49ers,CSU Bakersfield Roadrunners,3.178502,179.750000,4.5,NaN,NaN,NaN
2,Nevada Wolf Pack,UNLV Rebels,NaN,NaN,-1.5,152.5,-148.0,116.0


In [63]:
import numpy as np

HOME_EDGE_PTS = 1.5

# Sanity: ensure required columns exist
required = ["h_poss","a_poss","h_ppp_for","a_ppp_against","a_ppp_for","h_ppp_against"]
missing = [c for c in required if c not in games_df.columns]
if missing:
    raise ValueError(f"Missing required columns in games_df: {missing}")

# Expected possessions
games_df["poss_proj"] = (games_df["h_poss"] + games_df["a_poss"]) / 2

# PPP projections
games_df["home_ppp_proj"] = (games_df["h_ppp_for"] + games_df["a_ppp_against"]) / 2
games_df["away_ppp_proj"] = (games_df["a_ppp_for"] + games_df["h_ppp_against"]) / 2

# Raw points (no home edge baked into points)
games_df["home_points_proj"] = games_df["home_ppp_proj"] * games_df["poss_proj"]
games_df["away_points_proj"] = games_df["away_ppp_proj"] * games_df["poss_proj"]

# Apply home edge ONLY to margin
games_df["mean_margin"] = (games_df["home_points_proj"] - games_df["away_points_proj"] + HOME_EDGE_PTS)
games_df["mean_total"]  = (games_df["home_points_proj"] + games_df["away_points_proj"])

print("✅ Projection override applied")
print("Mean margin < 0:", int((games_df['mean_margin'] < 0).sum()))
print("Mean margin > 0:", int((games_df['mean_margin'] > 0).sum()))
print("Margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))

✅ Projection override applied
Mean margin < 0: 0
Mean margin > 0: 9
Margin range: 3.178501742160279 to 16.526666666666685


In [64]:
import pandas as pd
import numpy as np

# 1) How many teams are using filled league averages (proxy test: exact equality to column mean)
cols = ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]
means = {c: games_df[c].mean() for c in cols}

def near_mean(series, m, tol=1e-9):
    return (series - m).abs() < tol

print("=== Near-league-mean counts (exact/near) ===")
for c in cols:
    print(c, int(near_mean(games_df[c], means[c], 1e-9).sum()), "/", len(games_df))

# 2) Show where home_ppp_proj - away_ppp_proj is smallest (should include negatives if model is sane)
games_df["ppp_diff"] = games_df["home_ppp_proj"] - games_df["away_ppp_proj"]

display(
    games_df.sort_values("ppp_diff").head(12)[
        ["away_team","home_team","home_ppp_proj","away_ppp_proj","ppp_diff",
         "h_ppp_for","a_ppp_against","a_ppp_for","h_ppp_against",
         "h_poss","a_poss","poss_proj",
         "mean_margin","spread_home_line","home_ml","away_ml"]
    ]
)

print("ppp_diff min/max:", float(games_df["ppp_diff"].min()), float(games_df["ppp_diff"].max()))

=== Near-league-mean counts (exact/near) ===
h_poss 0 / 26
a_poss 0 / 26
h_ppp_for 0 / 26
h_ppp_against 0 / 26
a_ppp_for 0 / 26
a_ppp_against 0 / 26


,away_team,home_team,home_ppp_proj,away_ppp_proj,ppp_diff,h_ppp_for,a_ppp_against,a_ppp_for,h_ppp_against,h_poss,a_poss,poss_proj,mean_margin,spread_home_line,home_ml,away_ml
0,Long Beach St 49ers,CSU Bakersfield Roadrunners,1.029525,1.010475,0.019049,1.014171,1.044878,0.995122,1.025829,85.784314,90.441176,88.112745,3.178502,4.5,NaN,NaN
9,Quinnipiac Bobcats,Canisius Golden Griffins,1.046391,0.993609,0.052782,1.059535,1.033247,1.006753,0.980465,63.235294,75.490196,69.362745,5.161079,7.5,290.0,-375.0
18,Merrimack Warriors,Niagara Purple Eagles,1.046391,0.993609,0.052782,1.033247,1.059535,0.980465,1.006753,75.490196,63.235294,69.362745,5.161079,8.5,360.0,-480.0
4,UC Santa Barbara Gauchos,UC Irvine Anteaters,1.049802,0.990198,0.059604,1.049787,1.049816,0.990184,0.990213,72.920798,72.750055,72.835426,5.841254,-5.5,-400.0,285.0
16,Iona Gaels,Manhattan Jaspers,1.049802,0.990198,0.059604,1.049787,1.049816,0.990184,0.990213,72.920798,72.750055,72.835426,5.841254,2.5,118.0,-142.0
25,Charleston Cougars,UNC Wilmington Seahawks,1.050897,0.989103,0.061795,1.173333,0.928462,1.111538,0.866667,75.000000,76.470588,75.735294,6.180053,-4.5,-225.0,184.0
15,Mt. St. Mary's Mountaineers,Fairfield Stags,1.102868,0.937132,0.165737,1.129846,1.075890,0.964110,0.910154,63.725490,71.568627,67.647059,12.711591,-4.5,-230.0,188.0
20,Rider Broncs,Siena Saints,1.116198,0.923802,0.192396,1.049787,1.182609,0.857391,0.990213,72.920798,67.647059,70.283928,15.022359,-15.5,-2000.0,980.0
1,Cal Poly Mustangs,UC San Diego Tritons,1.124267,0.915733,0.208533,1.115200,1.133333,0.906667,0.924800,73.529412,70.588235,72.058824,16.526667,-14.5,NaN,NaN
2,Nevada Wolf Pack,UNLV Rebels,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.5,-148.0,116.0


ppp_diff min/max: 0.01904947735191631 0.20853333333333346


In [65]:
import numpy as np

HOME_EDGE_PTS = 1.5

# =====================================
# Build overall PPP (not split-based)
# =====================================

team_overall = (
    all_stats.groupby("team")
    .apply(lambda x: pd.Series({
        "poss": (x["poss_proxy"] * x["weight"]).sum() / x["weight"].sum(),
        "ppp_for": (x["points_scored"] * x["weight"]).sum() /
                   max((x["poss_proxy"] * x["weight"]).sum(), 1e-6),
        "ppp_against": (x["points_allowed"] * x["weight"]).sum() /
                       max((x["poss_proxy"] * x["weight"]).sum(), 1e-6),
    }))
    .reset_index()
)

# Fill missing with league means
for col in ["poss","ppp_for","ppp_against"]:
    team_overall[col] = team_overall[col].fillna(team_overall[col].mean())

# Merge
g = games_df.drop(columns=[
    "home_points_proj","away_points_proj",
    "mean_margin","mean_total"
], errors="ignore")

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left")
g = g.rename(columns={
    "poss":"h_poss_overall",
    "ppp_for":"h_ppp_for_overall",
    "ppp_against":"h_ppp_against_overall"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left")
g = g.rename(columns={
    "poss":"a_poss_overall",
    "ppp_for":"a_ppp_for_overall",
    "ppp_against":"a_ppp_against_overall"
}).drop(columns=["team"], errors="ignore")

# Expected possessions
g["poss_proj"] = (g["h_poss_overall"] + g["a_poss_overall"]) / 2

# Blend offense vs defense (NO SPLIT BIAS)
g["home_ppp_proj"] = (g["h_ppp_for_overall"] + g["a_ppp_against_overall"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for_overall"] + g["h_ppp_against_overall"]) / 2

# Points
g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

# Margin
g["mean_margin"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total"]  = g["home_points_proj"] + g["away_points_proj"]

games_df = g

print("✅ Overall PPP projections applied")
print("Mean margin < 0:", int((games_df['mean_margin'] < 0).sum()))
print("Margin range:",
      float(games_df["mean_margin"].min()),
      "to",
      float(games_df["mean_margin"].max()))

✅ Overall PPP projections applied
Mean margin < 0: 2
Margin range: -14.543995859213247 to 12.711591148577433


/tmp/ipython-input-926/443311746.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


In [66]:
import numpy as np
import pandas as pd

# =========================
# SETTINGS
# =========================
SIMS = 50000
SD_MARGIN = 11.5
SD_TOTAL  = 14.0

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# Numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds",
          "spread_away_odds","total_line","over_odds","under_odds"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# MONEYLINE CARD
# =========================
ml = df.dropna(subset=["home_ml","away_ml"]).copy()

ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev_ml"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev_ml"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# =========================
# SPREAD CARD
# =========================
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()

sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"] = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"] = np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])

sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# =========================
# TOTAL CARD
# =========================
to = df.dropna(subset=["over_odds","under_odds"]).copy()

to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"] = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"] = np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])

to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

# =========================
# COMBINE + TOP 20
# =========================
def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    return x[["matchup","market","bet","units","p_win","ev","commence_time"]]

out = pd.concat([
    finalize(ml_card),
    finalize(sp_card),
    finalize(to_card)
], ignore_index=True)

out = out.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Plays found:", len(out))
top20 = out.head(20).copy()

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " — " + top20["units"].astype(str) + "u\n" +
    "Sim Win%: " + top20["p_win"].apply(pct) +
    " | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))
top20.to_csv("ncaab_stable_final_top20.csv", index=False)
print("✅ Exported: ncaab_stable_final_top20.csv")

KeyError: "['ev'] not in index"

In [67]:
import pandas as pd
import numpy as np

# Ensure each card has a unified EV column named 'ev'
if "ev" not in ml_card.columns:
    ml_card = ml_card.copy()
    ml_card["ev"] = ml_card["ev_ml"]

if "ev" not in sp_card.columns:
    sp_card = sp_card.copy()
    # spread card already uses 'ev' in our build, but just in case:
    if "ev" not in sp_card.columns and "ev_spread" in sp_card.columns:
        sp_card["ev"] = sp_card["ev_spread"]

if "ev" not in to_card.columns:
    to_card = to_card.copy()
    # total card already uses 'ev' in our build, but just in case:
    if "ev" not in to_card.columns and "ev_total" in to_card.columns:
        to_card["ev"] = to_card["ev_total"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    # commence_time might not exist in some pulls
    cols = ["matchup","market","bet","units","p_win","ev"]
    if "commence_time" in x.columns:
        cols.append("commence_time")
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Plays found:", len(out))
top20 = out.head(20).copy()

def pct(x):
    return f"{round(float(x)*100,1)}%"

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " — " + top20["units"].astype(str) + "u\n" +
    "Sim Win%: " + top20["p_win"].apply(pct) +
    " | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))
top20.to_csv("ncaab_stable_final_top20.csv", index=False)
print("✅ Exported: ncaab_stable_final_top20.csv")

✅ Plays found: 18
                                                                                                                     discord_text
              Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles ML (+360) — 1.0u\nSim Win%: 67.4% | EV: 210.0%\n
        Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins ML (+290) — 1.0u\nSim Win%: 67.6% | EV: 163.7%\n
                                    Iona Gaels at Manhattan Jaspers\nIona Gaels -2.5 (-102) — 1.0u\nSim Win%: 85.4% | EV: 69.1%\n
              Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles 8.5 (-110) — 1.0u\nSim Win%: 88.2% | EV: 68.4%\n
        Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins 7.5 (-110) — 1.0u\nSim Win%: 86.5% | EV: 65.2%\n
                             Merrimack Warriors at Niagara Purple Eagles\nOVER 125.5 (-115) — 1.0u\nSim Win%: 87.5% | EV: 63.5%\n
                                     Rider Broncs at Siena Saints\nRider

In [68]:
import numpy as np
import pandas as pd

# =========================
# CALIBRATED NCAAB SETTINGS
# =========================
SIMS = 50000
SD_MARGIN = 15.5   # widened for NCAAB
SD_TOTAL  = 17.0   # widened for NCAAB

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# Numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds",
          "spread_away_odds","total_line","over_odds","under_odds"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

# Markets need lines
if "spread_home_line" in df.columns and df["spread_home_line"].notna().any():
    df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
    df["p_away_cover"] = 1 - df["p_home_cover"]
else:
    df["p_home_cover"] = np.nan
    df["p_away_cover"] = np.nan

if "total_line" in df.columns and df["total_line"].notna().any():
    df["p_over"]  = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
    df["p_under"] = 1 - df["p_over"]
else:
    df["p_over"] = np.nan
    df["p_under"] = np.nan

df["p_home_win"] = (margins > 0).mean(axis=1)

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# MONEYLINE CARD (home side)
# =========================
ml = df.dropna(subset=["home_ml","away_ml"]).copy()

ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# =========================
# SPREAD CARD
# =========================
sp = df.dropna(subset=["spread_home_line","spread_home_odds","spread_away_odds"]).copy()
if len(sp) > 0:
    sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
    sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

    sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
    sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
    sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
    sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
    sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
    sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])

    sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
    sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
    sp_card["market"] = "Spread"
    sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
else:
    sp_card = pd.DataFrame(columns=df.columns.tolist() + ["market","bet","p_win","ev","units"])

# =========================
# TOTAL CARD
# =========================
to = df.dropna(subset=["total_line","over_odds","under_odds"]).copy()
if len(to) > 0:
    to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
    to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

    to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
    to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
    to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
    to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])

    to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
    to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
    to_card["market"] = "Total"
    to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
else:
    to_card = pd.DataFrame(columns=df.columns.tolist() + ["market","bet","p_win","ev","units"])

# =========================
# COMBINE + TOP 20
# =========================
def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev"]
    if "commence_time" in x.columns:
        cols.append("commence_time")
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Plays found:", len(out))
top20 = out.head(20).copy()

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " — " + top20["units"].astype(str) + "u\n" +
    "Sim Win%: " + top20["p_win"].apply(pct) +
    " | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))

top20.to_csv("ncaab_stable_calibrated_top20.csv", index=False)
print("✅ Exported: ncaab_stable_calibrated_top20.csv")

✅ Plays found: 18
                                                                                                                     discord_text
              Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles ML (+360) — 1.0u\nSim Win%: 63.0% | EV: 190.0%\n
        Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins ML (+290) — 1.0u\nSim Win%: 63.3% | EV: 146.9%\n
              Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles 8.5 (-110) — 1.0u\nSim Win%: 81.2% | EV: 55.0%\n
                                    Iona Gaels at Manhattan Jaspers\nIona Gaels -2.5 (-102) — 1.0u\nSim Win%: 78.2% | EV: 54.9%\n
                             Merrimack Warriors at Niagara Purple Eagles\nOVER 125.5 (-115) — 1.0u\nSim Win%: 82.8% | EV: 54.9%\n
        Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins 7.5 (-110) — 1.0u\nSim Win%: 79.7% | EV: 52.1%\n
                                       Iona Gaels at Manhattan Jaspers\n

In [69]:
import numpy as np

# ======================================
# MARKET BLEND CALIBRATION
# ======================================

BLEND_MODEL = 0.6
BLEND_MARKET = 0.4

if "spread_home_line" not in games_df.columns:
    raise ValueError("spread_home_line missing")

# Market implied margin (home perspective)
games_df["market_margin"] = -games_df["spread_home_line"]

# Blend model and market
games_df["mean_margin"] = (
    BLEND_MODEL * games_df["mean_margin"] +
    BLEND_MARKET * games_df["market_margin"]
)

print("✅ Market blend applied")
print("New margin range:",
      float(games_df["mean_margin"].min()),
      "to",
      float(games_df["mean_margin"].max()))

✅ Market blend applied
New margin range: -9.726397515527948 to 12.01540828402366


In [70]:
import numpy as np
import pandas as pd

SIMS = 50000
SD_MARGIN = 15.5
SD_TOTAL  = 17.0
UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_home_odds",
          "spread_away_odds","total_line","over_odds","under_odds"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)
ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread
sp = df.dropna(subset=["spread_home_line","spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])
sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# Total
to = df.dropna(subset=["total_line","over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])
to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev"]
    if "commence_time" in x.columns:
        cols.append("commence_time")
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Plays found:", len(out))
top20 = out.head(20).copy()

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " — " + top20["units"].astype(str) + "u\n" +
    "Sim Win%: " + top20["p_win"].apply(pct) +
    " | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))
top20.to_csv("ncaab_stable_blended_top20.csv", index=False)
print("✅ Exported: ncaab_stable_blended_top20.csv")

✅ Plays found: 17
                                                                                                                    discord_text
             Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles ML (+360) — 0.7u\nSim Win%: 49.2% | EV: 126.4%\n
       Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins ML (+290) — 0.67u\nSim Win%: 50.7% | EV: 97.5%\n
                            Merrimack Warriors at Niagara Purple Eagles\nOVER 125.5 (-115) — 1.0u\nSim Win%: 82.8% | EV: 54.9%\n
                                      Iona Gaels at Manhattan Jaspers\nUNDER 150.5 (-105) — 0.96u\nSim Win%: 74.7% | EV: 45.9%\n
                         Charleston Cougars at UNC Wilmington Seahawks\nOVER 144.5 (-110) — 0.85u\nSim Win%: 72.5% | EV: 38.4%\n
                                  Iona Gaels at Manhattan Jaspers\nIona Gaels -2.5 (-102) — 0.71u\nSim Win%: 68.0% | EV: 34.7%\n
            Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles 

In [71]:
# ======================================
# STRONGER MARKET BLEND (50/50)
# ======================================

BLEND_MODEL = 0.5
BLEND_MARKET = 0.5

games_df["market_margin"] = -games_df["spread_home_line"]

games_df["mean_margin"] = (
    BLEND_MODEL * games_df["mean_margin"] +
    BLEND_MARKET * games_df["market_margin"]
)

print("✅ 50/50 blend applied")
print("New margin range:",
      float(games_df["mean_margin"].min()),
      "to",
      float(games_df["mean_margin"].max()))

✅ 50/50 blend applied
New margin range: -6.113198757763974 to 13.25770414201183


In [74]:
import numpy as np
import pandas as pd

SIMS = 50000
SD_MARGIN = 15.5
SD_TOTAL  = 17.0
UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_home_odds",
          "spread_away_odds","total_line","over_odds","under_odds"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)
ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread
sp = df.dropna(subset=["spread_home_line","spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])
sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total
to = df.dropna(subset=["total_line","over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])
to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)
to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev"]
    if "commence_time" in x.columns:
        cols.append("commence_time")
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Plays found:", len(out))
top20 = out.head(20).copy()

top20["discord_text"] = (
    top20["matchup"] + "\n" +
    top20["bet"] + " — " + top20["units"].astype(str) + "u\n" +
    "Sim Win%: " + top20["p_win"].apply(pct) +
    " | EV: " + (top20["ev"]*100).round(1).astype(str) + "%\n"
)

print(top20[["discord_text"]].to_string(index=False))
top20.to_csv("ncaab_stable_blended50_top20.csv", index=False)
print("✅ Exported: ncaab_stable_blended50_top20.csv")

✅ Plays found: 15
                                                                                                              discord_text
       Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles ML (+360) — 0.44u\nSim Win%: 39.0% | EV: 79.4%\n
 Quinnipiac Bobcats at Canisius Golden Griffins\nCanisius Golden Griffins ML (+290) — 0.39u\nSim Win%: 40.1% | EV: 56.5%\n
                     Merrimack Warriors at Niagara Purple Eagles\nOVER 125.5 (-115) — 0.61u\nSim Win%: 67.8% | EV: 26.7%\n
                                Iona Gaels at Manhattan Jaspers\nUNDER 150.5 (-105) — 0.49u\nSim Win%: 63.2% | EV: 23.3%\n
                    Charleston Cougars at UNC Wilmington Seahawks\nOVER 144.5 (-110) — 0.4u\nSim Win%: 61.9% | EV: 18.1%\n
                            Iona Gaels at Manhattan Jaspers\nIona Gaels -2.5 (-102) — 0.33u\nSim Win%: 58.7% | EV: 16.3%\n
      Merrimack Warriors at Niagara Purple Eagles\nNiagara Purple Eagles 8.5 (-110) — 0.35u\nSim Win%: 60.7% | EV: 15.8%\

In [73]:
# ======================================
# 50/50 TOTAL BLEND
# ======================================

if "total_line" not in games_df.columns:
    raise ValueError("total_line missing")

games_df["mean_total"] = (
    0.5 * games_df["mean_total"] +
    0.5 * games_df["total_line"]
)

print("✅ Total blend applied")
print("New total range:",
      float(games_df["mean_total"].min()),
      "to",
      float(games_df["mean_total"].max()))

✅ Total blend applied
New total range: 128.25 to 149.5
